# DIMER Table Intelligence Workshop
## Table Detection → Structure Recognition → Table Reconstruction → TAPAS QA

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Workflow scope:** `COMPOSED-PIPELINE`  
**Standalone:** yes  
**Canonical workflow:** frozen inference only

This notebook treats Table Intelligence as a system: three frozen models joined by notebook-local composition rules.

### Who this notebook is for

Learners who can run cells in Colab or Jupyter and read short Python functions, and who are new to composed document-intelligence pipelines. No prior experience with Table Transformer or TAPAS is assumed. A Tesla T4-class GPU is recommended; the notebook also runs on CPU, more slowly (TAPAS Large dominates the run time). It runs on **Linux x86_64 only** (Google Colab, Kaggle or a Linux Jupyter server), because it builds its own isolated Python environment there; Windows and macOS kernels are refused with a message.

### How to use this notebook

1. Select a T4 GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU*).
2. Choose *Runtime → Run all*. The defaults in the Configuration form (§3) are the reference settings; change a setting only in the activity marked **Predict → Change one thing** (after §18). The first run spends several minutes building an isolated environment (§4); nothing is installed into the notebook's own Python, and `Run all` needs no restart.
3. While the notebook runs, read the concept cells and write down your prediction whenever a cell asks for one, before its output appears.
4. Each section is labelled by role:
   - **Core concept** — what the models do and how the stages fit together;
   - **Evaluation practice** — how each stage is measured and how to read the numbers;
   - **Infrastructure** — runtime setup, pinned downloads, integrity checks, data plumbing and exports. You may run these cells without studying their implementation; their code is collapsed where your notebook viewer supports it.
5. Checkpoint questions have a collapsed **Sample answer**. Answer first, then open it.

### Task at a glance

```text
Input:  a document page containing one scientific table + a natural-language question
  → Table Transformer Detection       where is the table on the page?
  → crop
  → Table Transformer Structure       where are its rows and columns?
  → grid reconstruction + cell text   which string belongs in which cell?
  → TAPAS Large WTQ                   which cells answer the question, with which aggregation?
Output: an answer (cell strings or a number) + per-stage metrics showing where errors entered
```

Models:

1. **Table Transformer Detection**
2. **Table Transformer Structure Recognition v1.1-all**
3. **TAPAS Large WTQ**

The canonical corpus is the public-domain **SciTSR-PD** scientific-table dataset. Its logical cell text is used as a controlled text provider so this notebook can isolate table geometry and QA without mixing in OCR quality.

### Roadmap

1. Distinguish the three table-intelligence tasks and predict where errors will enter (§1–2).
2. Set up the runtime, models and SciTSR-PD inputs (§3–10, mostly Infrastructure).
3. Run table detection and measure it (§11–13).
4. Recognize structure on gold and detected crops, reconstruct grids and measure them (§14–18); then change the structure threshold on validation tables (activity).
5. Probe tables with spanning cells (§19).
6. Ask 50 questions through three paths with TAPAS and score them (§20–22).
7. Read the error waterfall and trace failures to their stage (§23–24).
8. Inspect resources, panels and exports (§25–28).
9. Write an evidence-based conclusion (§29); troubleshooting and a glossary follow.

### Learning objectives

By the end you should be able to:

- **distinguish** table detection, structure recognition and table QA, naming each one's input and output;
- **reconstruct** a rectangular cell grid from predicted row and column boxes;
- **trace** a box through the page → crop → table coordinate frames;
- **explain** why TAPAS needs cell strings rather than boxes, and where OCR would supply them in production;
- **compare** gold-table, structure-only and end-to-end QA accuracy on the same questions;
- **diagnose** a wrong answer by locating the stage in the error waterfall where it first went wrong;
- **predict, then test,** how one structure-threshold change moves predicted row and column counts.

> **Important boundary:** Table Transformer models recover geometry, not text. The canonical workflow uses SciTSR annotation text, not OCR inference. OCR / Document Extraction is the next notebook in the Document Intelligence track.

**AI Use Disclosure:** Generative AI assisted with this notebook’s code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

## 1. Three different tasks

> **Core concept**

### Table detection
**Where is the table on the page?**

### Structure recognition
**Where are rows, columns and spanning cells inside the table?**

### Table QA
**What does the reconstructed table say?**

A correct final answer depends on all upstream interfaces.

## 2. Error waterfall

> **Core concept**

The same TAPAS questions are evaluated through three paths:

```text
Path 1:
gold logical table → TAPAS

Path 2:
gold table crop → predicted structure → reconstructed table → TAPAS

Path 3:
page → predicted table crop → predicted structure → reconstructed table → TAPAS
```

This separates QA-model error from structure error and table-detection error.

### Before you run: make predictions

Write down short answers now; §24 returns to them with the measured waterfall and sample answers.

1. Which stage is most likely to reduce end-to-end accuracy?
2. Can perfect table detection guarantee correct structure?
3. Can perfect structure guarantee correct QA?
4. Why does TAPAS need strings rather than row/column boxes?
5. Where would OCR enter a production version of this workflow?
6. What happens if one predicted row is missing?
7. What happens if the first reconstructed row is mistaken?

## 3. Configuration

> **Configuration** — the form below holds the reference settings. Keep them for `Run all`; the activity after §18 is the place to experiment.

The defaults define the canonical `Run all` path.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}

DETECTION_THRESHOLD = 0.90  # @param {type:"number"}
STRUCTURE_THRESHOLD = 0.50  # @param {type:"number"}
CROP_PADDING = 10  # @param {type:"integer"}
GRID_NMS_IOU = 0.50  # @param {type:"number"}

# The QA set is fixed: ten test tables, five questions each (50 questions). These two are not form fields.
END_TO_END_TABLES = 10
QUESTIONS_PER_TABLE = 5

RUN_COMPLEX_STRUCTURE_PROBE = True  # @param {type:"boolean"}
COMPLEX_STRUCTURE_TABLES = 3  # @param {type:"integer"}

OUTPUT_DIR = "outputs/table_intelligence"
WORK_DIR = "work"  # isolated environment, carried stage file, stage logs and state

if not 0 <= DETECTION_THRESHOLD <= 1:
    raise ValueError("DETECTION_THRESHOLD must be in [0,1]")
if not 0 <= STRUCTURE_THRESHOLD <= 1:
    raise ValueError("STRUCTURE_THRESHOLD must be in [0,1]")
if not (isinstance(CROP_PADDING,int) and 0 <= CROP_PADDING <= 100):
    raise ValueError(f"CROP_PADDING must be a whole number of pixels in [0,100], got {CROP_PADDING!r}")
if not 0 < GRID_NMS_IOU <= 1:
    raise ValueError(f"GRID_NMS_IOU must be in (0,1], got {GRID_NMS_IOU!r}")
if not (isinstance(COMPLEX_STRUCTURE_TABLES,int) and 0 <= COMPLEX_STRUCTURE_TABLES <= 10):
    raise ValueError(f"COMPLEX_STRUCTURE_TABLES must be a whole number in [0,10], got {COMPLEX_STRUCTURE_TABLES!r}")
if END_TO_END_TABLES != 10:
    raise ValueError("END_TO_END_TABLES is fixed at 10: the canonical QA set is ten test tables and 50 questions")
print({
    "detection_threshold":DETECTION_THRESHOLD,
    "structure_threshold":STRUCTURE_THRESHOLD,
    "crop_padding":CROP_PADDING,
    "tables":END_TO_END_TABLES,
})

{'detection_threshold': 0.9, 'structure_threshold': 0.5, 'crop_padding': 10, 'tables': 10}


## 4. Runtime

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

The model code does not run in this notebook's own Python. The next two cells:

1. write two carried files under `work/runner/` and check their SHA-256: `table_intelligence_workshop.py`, which holds every function and stage of this workshop, and `requirements.lock.txt`, a hash lock of every package (the same versions this notebook pinned before: torch 2.14.0, transformers 4.57.6, timm 1.0.30, numpy 2.5.3, datasets 4.1.1, pandas 2.2.3 and the rest);
2. download a pinned `uv` (0.12.15, checked by size and SHA-256), create a CPython 3.12.12 virtual environment in `work/env` and install the lock with `uv pip install --require-hashes --only-binary :all:` — only wheels whose hashes are in the lock.

Each later section runs one stage of the carried file as a separate process with that environment's Python (`run_stage`), and shows the functions it runs (`show_source`). Objects that pass from one section to the next (tables, crops, predictions) are saved under `work/state/` between stages. Nothing is installed into this kernel, so `Run all` needs no restart, and a second `Run all` reuses the environment.

This works on **Linux x86_64 only** (Google Colab, Kaggle, a Linux Jupyter server). `datasets` and `pyarrow` are used only to read the small pinned SciTSR-PD parquet files.

A Tesla T4 is recommended because TAPAS Large is approximately 1.35 GB.

In [ ]:
# Carried stage file and hash lock (infrastructure)
# Generated by tools/build_table_intelligence_workshop.py; do not edit by hand.
from pathlib import Path
CARRIED_FILES = {
 'table_intelligence_workshop.py': (
  '"""Stages of DIMER_Table_Intelligence_Workshop.ipynb, run in its isolated environment.\n\nThe notebook carries this file byte-for-byte, writes it next to the hash-locked requirements, and runs\nevery stage as ``python table_intelligence_workshop.py --config <json> --stage <name>`` with the\ninterpreter of a uv-built CPython 3.12 virtual environment. Nothing is installed into the notebook\nkernel: it shows the functions of this file and streams what each stage prints.\n\nFunction bodies are the notebook\'s former cell code (revision 0.2.0-candidate), moved unchanged\nexcept where noted:\n\n* defaults that read a control (``CROP_PADDING``, ``GRID_NMS_IOU``, ``DETECTION_THRESHOLD``,\n  ``STRUCTURE_THRESHOLD``) are resolved when the function is called, because the controls arrive\n  after import;\n* objects that lived in the kernel between cells (tables, pages, crops, predictions, timings) are\n  pickled to ``WORK_ROOT/state/state.pkl`` after each stage and restored by the next one;\n'
  '* each stage process unloads the model it loaded; the former section 21 statements that loaded\n  TAPAS are ``load_tapas()``, used by the QA and BYOD stages;\n* the SciTSR-PD source-table count is kept as ``SOURCE_TABLES`` (the ``datasets`` object is not\n  carried between processes).\n"""\n# ruff: noqa: E401,E402,E501,E701,E702,E731,F841,I001,UP017,B007,B905,SIM108\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport pickle\nimport random\nimport re\nimport shutil\nimport sys\nimport time\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nSTAGE_FORMAT = "dimer.table-intelligence-workshop.stages.v1"\n\n# ---- Controls (the notebook\'s section 3 defaults; overwritten by configure()) -----------------------\nUSE_BYOD = False\nBYOD_PATH = ""\nDETECTION_THRESHOLD = 0.90\nSTRUCTURE_THRESHOLD = 0.50\n'
  'CROP_PADDING = 10\nGRID_NMS_IOU = 0.50\nEND_TO_END_TABLES = 10\nQUESTIONS_PER_TABLE = 5\nRUN_COMPLEX_STRUCTURE_PROBE = True\nCOMPLEX_STRUCTURE_TABLES = 3\nOUTPUT_DIR = "outputs/table_intelligence"\nWORK_ROOT = Path("work")\n\n\ndef configure(config):\n    """Apply the notebook controls (a JSON object written by the notebook\'s run_stage)."""\n    global USE_BYOD, BYOD_PATH, DETECTION_THRESHOLD, STRUCTURE_THRESHOLD, CROP_PADDING, GRID_NMS_IOU\n    global END_TO_END_TABLES, QUESTIONS_PER_TABLE, RUN_COMPLEX_STRUCTURE_PROBE, COMPLEX_STRUCTURE_TABLES\n    global OUTPUT_DIR, WORK_ROOT\n    USE_BYOD = bool(config["use_byod"])\n    BYOD_PATH = str(config["byod_path"])\n    DETECTION_THRESHOLD = float(config["detection_threshold"])\n    STRUCTURE_THRESHOLD = float(config["structure_threshold"])\n    CROP_PADDING = config["crop_padding"]\n    GRID_NMS_IOU = float(config["grid_nms_iou"])\n    END_TO_END_TABLES = int(config["end_to_end_tables"])\n    QUESTIONS_PER_TABLE = int(config["questions_per_table"])\n'
  '    RUN_COMPLEX_STRUCTURE_PROBE = bool(config["run_complex_structure_probe"])\n    COMPLEX_STRUCTURE_TABLES = config["complex_structure_tables"]\n    OUTPUT_DIR = str(config["output_dir"])\n    WORK_ROOT = Path(config["work_dir"])\n\n\n# ---- Libraries that exist only in the isolated environment (loaded by the stages that use them) -----\ntorch = None\npd = None\nhf_hub_download = None\nAutoImageProcessor = TableTransformerForObjectDetection = None\nTapasTokenizer = TapasForQuestionAnswering = None\nDEVICE = "cpu"\n\n\ndef load_torch():\n    global torch, DEVICE\n    import torch as _torch\n    torch = _torch\n    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"\n\n\ndef load_transformers():\n    global AutoImageProcessor, TableTransformerForObjectDetection, TapasTokenizer, TapasForQuestionAnswering\n    load_torch()\n    from transformers import AutoImageProcessor as _AIP, TableTransformerForObjectDetection as _TTD\n    from transformers import TapasTokenizer as _TT, TapasForQuestionAnswering as _TQA\n'
  '    AutoImageProcessor, TableTransformerForObjectDetection = _AIP, _TTD\n    TapasTokenizer, TapasForQuestionAnswering = _TT, _TQA\n\n\ndef load_pandas():\n    global pd\n    import pandas as _pd\n    pd = _pd\n\n\ndef load_hub():\n    global hf_hub_download\n    from huggingface_hub import hf_hub_download as _download\n    hf_hub_download = _download\n\n\n# ---- State carried between stage processes ---------------------------------------------------------\n# Names a later stage reads. They are pickled together, so tables shared between `derived`, `splits` and\n# `canonical_tables` stay one object, as they were in the kernel. The file is written and read only by\n# this file, under WORK_ROOT.\nSTATE_NAMES = (\n    "RUNTIME", "scitsr_paths", "SOURCE_TABLES", "derived", "failed", "splits", "canonical_tables",\n    "det_load_seconds", "det_params", "det_times", "det_peak", "det_rows", "det_metrics",\n    "struct_load_seconds", "struct_params", "struct_times", "struct_peak", "structure_rows",\n'
  '    "activity_rows", "complex_tables", "qa_records", "tapas_load_seconds", "tapas_params",\n    "gold_qa", "structure_qa", "end_to_end_qa", "gold_tapas_times", "structure_tapas_times",\n    "end_tapas_times", "tapas_peak", "waterfall", "resource_rows", "byod_result",\n)\n\n\ndef state_file():\n    return WORK_ROOT/"state"/"state.pkl"\n\n\ndef restore_state():\n    path=state_file()\n    if path.is_file():\n        with open(path,"rb") as f:\n            globals().update(pickle.load(f))\n\n\ndef persist_state():\n    path=state_file()\n    path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(".tmp")\n    with open(tmp,"wb") as f:\n        pickle.dump({name:globals()[name] for name in STATE_NAMES if name in globals()},f,protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp,path)\n\n\n# ---- former cell 8 (constants)\n'
  'DETECTION_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-detection", "modelId": "microsoft/table-transformer-detection", "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab", "files": [{"path": "README.md", "bytes": 1174, "sha256": "c91e7f8199313c4d24b09e73a2f6df3141268666969346cb7b94ccb59900f5dd"}, {"path": "config.json", "bytes": 1228, "sha256": "ed5b93df2c3a59d473ddea853553a6d545d52bd4e9f8f72bf40b8a974aba4c1d"}, {"path": "model.safetensors", "bytes": 115317516, "sha256": "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"}, {"path": "preprocessor_config.json", "bytes": 273, "sha256": "86a8837ae440456b0a9aef788b064921df29c20f0b67040954ce5c2fbd352c4f"}], "totalBytes": 115320191}""")\n'
  'STRUCTURE_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-structure-v1.1-all", "modelId": "microsoft/table-transformer-structure-recognition-v1.1-all", "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c", "files": [{"path": "README.md", "bytes": 1056, "sha256": "01eed360e0f54ad297ab347cfef208d68d8ac2c108b808147bb7af9f8d9132c8"}, {"path": "config.json", "bytes": 76761, "sha256": "17a8a6edfb9e394263fa6ba9b82176ebccdfcc5d6cd29121ec91572c7d6be22c"}, {"path": "model.safetensors", "bytes": 115437156, "sha256": "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"}, {"path": "preprocessor_config.json", "bytes": 374, "sha256": "eead409bb80e36ae85b8377642c54550f0504f65688ba3a4967950cafe461df2"}], "totalBytes": 115515347}""")\n'
  'TAPAS_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "tapas-large-wtq", "modelId": "google/tapas-large-finetuned-wtq", "revision": "f58317ab2577d17647d9acafa790c744a0388b30", "files": [{"path": "README.md", "bytes": 7224, "sha256": "2397bdae46316684903465e6f425a9fe10c85d491a9755ec7a31a74353c034c8"}, {"path": "config.json", "bytes": 1659, "sha256": "834f44a325a349d2d209ecb18e4fa2ca3cdb16cef5b169e5c0a8f5e16c444d13"}, {"path": "model.safetensors", "bytes": 1346985282, "sha256": "149247e13732c222ba621e0c4e7b90ba260869b36adcbe115dc872c2c98bccf0"}, {"path": "special_tokens_map.json", "bytes": 154, "sha256": "e3ec7abc6bcd45aba696cb95e9945186dad920aea78733f8b45c83d696ed0dea"}, {"path": "tokenizer_config.json", "bytes": 490, "sha256": "ab033df4f3c902cbc66bae2736bc99f4a59c43e7d9e53e40046576826b2cc5e9"}, {"path": "vocab.txt", "bytes": 262028, "sha256": "4d96f9308bcf9019684fcc109aa8c042b9b745edabba0162fbe66c75ebee2db4"}], "totalBytes": 1347256837}""")\n'
  'DETECTION_DIR=Path("weights/table-transformer-detection")\nSTRUCTURE_DIR=Path("weights/table-transformer-structure-v1.1-all")\nTAPAS_DIR=Path("weights/tapas-large-wtq")\n\n\n# ---- former cell 8\ndef sha256_file(path):\n    h=hashlib.sha256()\n    with open(path,"rb") as f:\n        for chunk in iter(lambda:f.read(1<<20),b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef stage_snapshot(root,manifest):\n    root.mkdir(parents=True,exist_ok=True)\n    (root/"dimer-base-manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")\n    for entry in manifest["files"]:\n        path=root/entry["path"]\n        if not path.is_file():\n            hf_hub_download(\n                repo_id=manifest["modelId"],\n                filename=entry["path"],\n                revision=manifest["revision"],\n                local_dir=str(root),\n            )\n    for entry in manifest["files"]:\n        path=root/entry["path"]\n        if path.stat().st_size!=entry["bytes"]:\n'
  '            raise RuntimeError(f"{manifest[\'modelId\']} {entry[\'path\']} size mismatch")\n        if sha256_file(path)!=entry["sha256"]:\n            raise RuntimeError(f"{manifest[\'modelId\']} {entry[\'path\']} SHA-256 mismatch")\n    return {"model_id":manifest["modelId"],"revision":manifest["revision"],"bytes":manifest["totalBytes"]}\n\n\n# ---- former cell 10 (constants)\nSCITSR_REPO="bevaya/SciTSR-pd"\nSCITSR_REVISION="dae336efa7af07d69194a510ff5568980e8ef253"\nSCITSR_FILES=json.loads(r"""{"train": {"path": "data/train-00000-of-00001.parquet", "bytes": 5185141, "sha256": "d295b918777425a26fa13a5c3ba076517429682ede8c71b671a8b6115968e465", "rows": 89}, "test": {"path": "data/test-00000-of-00001.parquet", "bytes": 1066042, "sha256": "73af6317219349a08acdf1c8fee7cff21b463cdded64111b643123f7df4f8bed", "rows": 19}}""")\nSCITSR_DIR=Path("weights/scitsr-pd")\n\n\n# ---- former cell 12 (pinned carrier geometry)\n'
  '# Infrastructure: pinned carrier geometry for the 94 SciTSR-PD tables of the DIMER structure carrier.\n# Per table: paper, chunk offset (dx, dy in pixels), grid size, pixel digest, and structure boxes\n# [label, x_min, y_min, x_max, y_max] with t=table, r=table row, c=table column, s=table spanning cell.\nCARRIER_GEOMETRY_SHA256="d6d524c367a151ed8a41bda7eadf7679977256953888862c9d85186eeda2c66a"\nCARRIER_GEOMETRY_JSON=r"""{\n"1003.3684v1.1":{"paper_id":"1003.3684","dx":20,"dy":7,"n_rows":3,"n_cols":4,"pixel_sha256":"f81a6fd18dac7606a157db5a95e666000dc72017d41f82dd65d2ade9a8d47491","objects":[["t",3.0,3.0,563.0,84.0],["r",3.0,3.0,563.0,30.9],["r",3.0,30.9,563.0,58.3],["r",3.0,58.3,563.0,84.0],["c",3.0,3.0,113.1,84.0],["c",113.1,3.0,255.3,84.0],["c",255.3,3.0,391.0,84.0],["c",391.0,3.0,563.0,84.0]]},\n'
  '"1003.3684v1.2":{"paper_id":"1003.3684","dx":20,"dy":15,"n_rows":5,"n_cols":3,"pixel_sha256":"e9980448fa314c2941757c1c59bd86edc8ede136bdf01c0485a922899797af92","objects":[["t",3.0,3.0,577.0,142.0],["r",3.0,3.0,577.0,35.1],["r",3.0,35.1,577.0,62.5],["r",3.0,62.5,577.0,87.4],["r",3.0,87.4,577.0,112.3],["r",3.0,112.3,577.0,142.0],["c",3.0,3.0,161.2,142.0],["c",161.2,3.0,352.2,142.0],["c",352.2,3.0,577.0,142.0]]},\n"1004.5186v1.1":{"paper_id":"1004.5186","dx":16,"dy":11,"n_rows":6,"n_cols":3,"pixel_sha256":"c3766c7010adb1c83c11546bdf2fc677937e4d10d058e2141aa09e8c868bbc02","objects":[["t",3.0,3.0,426.0,155.0],["r",3.0,3.0,426.0,29.1],["r",3.0,29.1,426.0,54.4],["r",3.0,54.4,426.0,79.3],["r",3.0,79.3,426.0,104.2],["r",3.0,104.2,426.0,129.1],["r",3.0,129.1,426.0,155.0],["c",3.0,3.0,184.2,155.0],["c",184.2,3.0,283.6,155.0],["c",283.6,3.0,426.0,155.0]]},\n'
  '"1007.0920v1.1":{"paper_id":"1007.0920","dx":16,"dy":11,"n_rows":8,"n_cols":4,"pixel_sha256":"40a7a095db0e2cebb6ab5ca5745f168fb4924962f03598ebba7890ed817f0cf7","objects":[["t",3.0,3.0,543.0,213.0],["r",3.0,3.0,543.0,31.1],["r",3.0,31.1,543.0,58.5],["r",3.0,58.5,543.0,83.4],["r",3.0,83.4,543.0,108.3],["r",3.0,108.3,543.0,133.3],["r",3.0,133.3,543.0,158.2],["r",3.0,158.2,543.0,183.1],["r",3.0,183.1,543.0,213.0],["c",3.0,3.0,115.4,213.0],["c",115.4,3.0,263.6,213.0],["c",263.6,3.0,396.7,213.0],["c",396.7,3.0,543.0,213.0]]},\n'
  '"1007.0920v1.2":{"paper_id":"1007.0920","dx":16,"dy":12,"n_rows":13,"n_cols":4,"pixel_sha256":"175b2b08e0b134f8ca6a6b1adfa2f3555c15ec637d60ec937415d9f2aa0ca8a3","objects":[["t",3.0,3.0,549.0,338.0],["r",3.0,3.0,549.0,32.1],["r",3.0,32.1,549.0,59.5],["r",3.0,59.5,549.0,84.4],["r",3.0,84.4,549.0,109.3],["r",3.0,109.3,549.0,134.3],["r",3.0,134.3,549.0,159.2],["r",3.0,159.2,549.0,184.1],["r",3.0,184.1,549.0,209.0],["r",3.0,209.0,549.0,233.9],["r",3.0,233.9,549.0,258.8],["r",3.0,258.8,549.0,283.7],["r",3.0,283.7,549.0,308.6],["r",3.0,308.6,549.0,338.0],["c",3.0,3.0,118.4,338.0],["c",118.4,3.0,266.5,338.0],["c",266.5,3.0,402.7,338.0],["c",402.7,3.0,549.0,338.0]]},\n'
  '"1108.4723v1.1":{"paper_id":"1108.4723","dx":16,"dy":12,"n_rows":5,"n_cols":7,"pixel_sha256":"085f89838b3a7b2dd6765406ac806e07217ccfce3dcfb221b207efe93c6df0cd","objects":[["t",3.0,3.0,641.0,137.0],["r",3.0,3.0,641.0,32.1],["r",3.0,32.1,641.0,59.9],["r",3.0,59.9,641.0,87.2],["r",3.0,87.2,641.0,113.0],["r",3.0,113.0,641.0,137.0],["c",3.0,3.0,138.7,137.0],["c",138.7,3.0,224.1,137.0],["c",224.1,3.0,307.4,137.0],["c",307.4,3.0,390.7,137.0],["c",390.7,3.0,474.0,137.0],["c",474.0,3.0,557.3,137.0],["c",557.3,3.0,641.0,137.0],["s",474.0,3.0,641.0,32.1],["s",138.7,3.0,307.4,32.1],["s",307.4,3.0,474.0,32.1]]},\n'
  '"1108.4723v1.2":{"paper_id":"1108.4723","dx":16,"dy":11,"n_rows":6,"n_cols":10,"pixel_sha256":"6decf00a18f5937e7f01a59de2a6504b5ef515bcf870f1a1d5ea79294aadadb8","objects":[["t",3.0,3.0,706.0,162.0],["r",3.0,3.0,706.0,29.1],["r",3.0,29.1,706.0,56.9],["r",3.0,56.9,706.0,84.7],["r",3.0,84.7,706.0,110.4],["r",3.0,110.4,706.0,136.2],["r",3.0,136.2,706.0,162.0],["c",3.0,3.0,138.7,162.0],["c",138.7,3.0,195.4,162.0],["c",195.4,3.0,262.1,162.0],["c",262.1,3.0,328.2,162.0],["c",328.2,3.0,384.2,162.0],["c",384.2,3.0,450.9,162.0],["c",450.9,3.0,517.1,162.0],["c",517.1,3.0,573.1,162.0],["c",573.1,3.0,639.8,162.0],["c",639.8,3.0,706.0,162.0],["s",328.2,3.0,517.1,29.1],["s",138.7,3.0,328.2,29.1],["s",517.1,3.0,706.0,29.1]]},\n'
  '"1108.4723v1.3":{"paper_id":"1108.4723","dx":16,"dy":12,"n_rows":4,"n_cols":4,"pixel_sha256":"668cb76ec6d117e60025839f730cb21c8e1648909d357792571066d98e5884f4","objects":[["t",3.0,3.0,395.0,111.0],["r",3.0,3.0,395.0,32.1],["r",3.0,32.1,395.0,59.9],["r",3.0,59.9,395.0,85.7],["r",3.0,85.7,395.0,111.0],["c",3.0,3.0,113.6,111.0],["c",113.6,3.0,208.6,111.0],["c",208.6,3.0,301.6,111.0],["c",301.6,3.0,395.0,111.0]]},\n"1108.4723v1.4":{"paper_id":"1108.4723","dx":16,"dy":11,"n_rows":3,"n_cols":4,"pixel_sha256":"38801d9a4ce81bcf62b0cf301a6a6e79182fa8b019e1605757bf14cee0ebd2fa","objects":[["t",3.0,3.0,374.0,86.0],["r",3.0,3.0,374.0,31.1],["r",3.0,31.1,374.0,58.9],["r",3.0,58.9,374.0,86.0],["c",3.0,3.0,92.8,86.0],["c",92.8,3.0,187.8,86.0],["c",187.8,3.0,280.7,86.0],["c",280.7,3.0,374.0,86.0]]},\n'
  '"1109.4653v2.11":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"d4ce0897e5e92fc558df075300c7ec8b1f944eaf541fdd1df7967282f36fba4f","objects":[["t",3.0,3.0,401.0,184.0],["r",3.0,3.0,401.0,30.8],["r",3.0,30.8,401.0,56.6],["r",3.0,56.6,401.0,81.5],["r",3.0,81.5,401.0,106.4],["r",3.0,106.4,401.0,131.4],["r",3.0,131.4,401.0,156.3],["r",3.0,156.3,401.0,184.0],["c",3.0,3.0,232.7,184.0],["c",232.7,3.0,317.4,184.0],["c",317.4,3.0,401.0,184.0]]},\n"1109.4653v2.12":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"172f29e8a1e10efb4d5e27498a8e920f399b3f3665458f482d08e35dcbdca1cb","objects":[["t",3.0,3.0,432.0,159.0],["r",3.0,3.0,432.0,30.8],["r",3.0,30.8,432.0,56.6],["r",3.0,56.6,432.0,81.5],["r",3.0,81.5,432.0,106.4],["r",3.0,106.4,432.0,131.4],["r",3.0,131.4,432.0,159.0],["c",3.0,3.0,346.5,159.0],["c",346.5,3.0,432.0,159.0]]},\n'
  '"1109.4653v2.2":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":8,"n_cols":2,"pixel_sha256":"74ce1641c8ef8a4593a612a484893e9657af34133390f9d4edb76d863e4236d2","objects":[["t",3.0,3.0,371.0,205.0],["r",3.0,3.0,371.0,29.1],["r",3.0,29.1,371.0,54.4],["r",3.0,54.4,371.0,79.3],["r",3.0,79.3,371.0,102.3],["r",3.0,102.3,371.0,129.1],["r",3.0,129.1,371.0,154.0],["r",3.0,154.0,371.0,178.9],["r",3.0,178.9,371.0,205.0],["c",3.0,3.0,143.1,205.0],["c",143.1,3.0,371.0,205.0]]},\n'
  '"1109.4653v2.6":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":17,"n_cols":3,"pixel_sha256":"9a6c61d297851c0f0545b71b76ce6ed8cd3d7d50a8f93240c34691b409e2ddb3","objects":[["t",3.0,3.0,318.0,435.0],["r",3.0,3.0,318.0,30.6],["r",3.0,30.6,318.0,54.4],["r",3.0,54.4,318.0,79.3],["r",3.0,79.3,318.0,104.2],["r",3.0,104.2,318.0,129.1],["r",3.0,129.1,318.0,154.0],["r",3.0,154.0,318.0,178.9],["r",3.0,178.9,318.0,206.3],["r",3.0,206.3,318.0,235.7],["r",3.0,235.7,318.0,261.0],["r",3.0,261.0,318.0,285.9],["r",3.0,285.9,318.0,310.8],["r",3.0,310.8,318.0,335.7],["r",3.0,335.7,318.0,360.6],["r",3.0,360.6,318.0,385.5],["r",3.0,385.5,318.0,410.4],["r",3.0,410.4,318.0,435.0],["c",3.0,3.0,143.1,435.0],["c",143.1,3.0,234.6,435.0],["c",234.6,3.0,318.0,435.0]]},\n'
  '"1109.4653v2.8":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"78a47c83e462d7e89825dd107ba341471cbc91fda9ab9ce878f033f51f4eec26","objects":[["t",3.0,3.0,441.0,184.0],["r",3.0,3.0,441.0,30.8],["r",3.0,30.8,441.0,56.6],["r",3.0,56.6,441.0,81.5],["r",3.0,81.5,441.0,106.4],["r",3.0,106.4,441.0,131.4],["r",3.0,131.4,441.0,156.3],["r",3.0,156.3,441.0,184.0],["c",3.0,3.0,273.5,184.0],["c",273.5,3.0,358.2,184.0],["c",358.2,3.0,441.0,184.0]]},\n"1109.4653v2.9":{"paper_id":"1109.4653","dx":17,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"e375479a3eeaff2e3b4a66006fdda11fc781b2d489d69832e0d5c1b7f8f35686","objects":[["t",3.0,3.0,464.0,159.0],["r",3.0,3.0,464.0,30.8],["r",3.0,30.8,464.0,56.6],["r",3.0,56.6,464.0,81.5],["r",3.0,81.5,464.0,106.4],["r",3.0,106.4,464.0,131.4],["r",3.0,131.4,464.0,159.0],["c",3.0,3.0,379.4,159.0],["c",379.4,3.0,464.0,159.0]]},\n'
  '"1301.0302v2.1":{"paper_id":"1301.0302","dx":17,"dy":11,"n_rows":8,"n_cols":6,"pixel_sha256":"9b1bb9f3d19c50a7a850dcd9f78bca5fcedf10e2aa46bb39091f158ce2cc0cf1","objects":[["t",3.0,3.0,958.0,214.0],["r",3.0,3.0,958.0,31.1],["r",3.0,31.1,958.0,58.9],["r",3.0,58.9,958.0,84.7],["r",3.0,84.7,958.0,110.4],["r",3.0,110.4,958.0,136.2],["r",3.0,136.2,958.0,161.9],["r",3.0,161.9,958.0,187.6],["r",3.0,187.6,958.0,214.0],["c",3.0,3.0,415.2,214.0],["c",415.2,3.0,549.4,214.0],["c",549.4,3.0,654.2,214.0],["c",654.2,3.0,748.9,214.0],["c",748.9,3.0,843.5,214.0],["c",843.5,3.0,958.0,214.0]]},\n'
  '"1305.1199v4.1":{"paper_id":"1305.1199","dx":23,"dy":16,"n_rows":7,"n_cols":2,"pixel_sha256":"9a3b144c34d0a82d6bb55918ce1244bd6b3f5526ff9c09e2b3e6df32536f4da5","objects":[["t",3.0,3.0,962.0,228.0],["r",3.0,3.0,962.0,37.2],["r",3.0,37.2,962.0,69.4],["r",3.0,69.4,962.0,101.5],["r",3.0,101.5,962.0,133.6],["r",3.0,133.6,962.0,165.7],["r",3.0,165.7,962.0,197.8],["r",3.0,197.8,962.0,228.0],["c",3.0,3.0,553.1,228.0],["c",553.1,3.0,962.0,228.0]]},\n"1401.1475v1.1":{"paper_id":"1401.1475","dx":16,"dy":9,"n_rows":7,"n_cols":2,"pixel_sha256":"2a545f13c0112038304e5f3436ecea06257ea9cb41e4a04512429eca2e65986b","objects":[["t",3.0,3.0,699.0,149.0],["r",3.0,3.0,699.0,25.5],["r",3.0,25.5,699.0,47.7],["r",3.0,47.7,699.0,67.8],["r",3.0,67.8,699.0,88.0],["r",3.0,88.0,699.0,108.1],["r",3.0,108.1,699.0,128.2],["r",3.0,128.2,699.0,149.0],["c",3.0,3.0,352.9,149.0],["c",352.9,3.0,699.0,149.0]]},\n'
  '"1404.5002v1.1":{"paper_id":"1404.5002","dx":16,"dy":10,"n_rows":13,"n_cols":5,"pixel_sha256":"72c4ddb559b2468d4888c4922bfed0b0eef18929bb2ce17553de07d22c59fc49","objects":[["t",3.0,3.0,488.0,263.0],["r",3.0,3.0,488.0,24.0],["r",3.0,24.0,488.0,44.1],["r",3.0,44.1,488.0,64.3],["r",3.0,64.3,488.0,84.0],["r",3.0,84.0,488.0,103.7],["r",3.0,103.7,488.0,123.4],["r",3.0,123.4,488.0,143.1],["r",3.0,143.1,488.0,162.9],["r",3.0,162.9,488.0,182.6],["r",3.0,182.6,488.0,202.3],["r",3.0,202.3,488.0,222.4],["r",3.0,222.4,488.0,242.6],["r",3.0,242.6,488.0,263.0],["c",3.0,3.0,133.9,263.0],["c",133.9,3.0,220.7,263.0],["c",220.7,3.0,305.7,263.0],["c",305.7,3.0,366.8,263.0],["c",366.8,3.0,488.0,263.0]]},\n'
  '"1404.5002v1.2":{"paper_id":"1404.5002","dx":16,"dy":9,"n_rows":12,"n_cols":5,"pixel_sha256":"2b78851e895d47604289323e6ca1c439ca8b9438b132a14ef9ca755b7727fa11","objects":[["t",3.0,3.0,453.0,243.0],["r",3.0,3.0,453.0,23.4],["r",3.0,23.4,453.0,43.6],["r",3.0,43.6,453.0,63.3],["r",3.0,63.3,453.0,83.0],["r",3.0,83.0,453.0,102.7],["r",3.0,102.7,453.0,122.4],["r",3.0,122.4,453.0,142.1],["r",3.0,142.1,453.0,161.9],["r",3.0,161.9,453.0,181.6],["r",3.0,181.6,453.0,201.7],["r",3.0,201.7,453.0,221.8],["r",3.0,221.8,453.0,243.0],["c",3.0,3.0,133.9,243.0],["c",133.9,3.0,251.1,243.0],["c",251.1,3.0,319.9,243.0],["c",319.9,3.0,385.5,243.0],["c",385.5,3.0,453.0,243.0]]},\n'
  '"1407.3745v1.1":{"paper_id":"1407.3745","dx":16,"dy":15,"n_rows":4,"n_cols":4,"pixel_sha256":"2fafa59648427ee3b5cf512202acb49df2d0d29892abd6ef324fd7fad7af2c0d","objects":[["t",3.0,3.0,717.0,111.0],["r",3.0,3.0,717.0,33.1],["r",3.0,33.1,717.0,58.8],["r",3.0,58.8,717.0,84.5],["r",3.0,84.5,717.0,111.0],["c",3.0,3.0,326.6,111.0],["c",326.6,3.0,487.7,111.0],["c",487.7,3.0,596.8,111.0],["c",596.8,3.0,717.0,111.0]]},\n"1409.5313v2.2":{"paper_id":"1409.5313","dx":16,"dy":10,"n_rows":7,"n_cols":5,"pixel_sha256":"cb07864095b3591ea250d54ec109a836aee2937118eb87a094f166c8f1060877","objects":[["t",3.0,3.0,587.0,148.0],["r",3.0,3.0,587.0,24.4],["r",3.0,24.4,587.0,45.0],["r",3.0,45.0,587.0,65.5],["r",3.0,65.5,587.0,86.1],["r",3.0,86.1,587.0,106.6],["r",3.0,106.6,587.0,127.2],["r",3.0,127.2,587.0,148.0],["c",3.0,3.0,129.3,148.0],["c",129.3,3.0,248.0,148.0],["c",248.0,3.0,354.6,148.0],["c",354.6,3.0,457.2,148.0],["c",457.2,3.0,587.0,148.0]]},\n'
  '"1410.1237v2.1":{"paper_id":"1410.1237","dx":16,"dy":11,"n_rows":13,"n_cols":6,"pixel_sha256":"9b0b863353c18d2813c92258ad395841c746e6f30aff0216ea3ded5dfaff738f","objects":[["t",3.0,3.0,743.0,346.0],["r",3.0,3.0,743.0,28.6],["r",3.0,28.6,743.0,56.0],["r",3.0,56.0,743.0,83.9],["r",3.0,83.9,743.0,109.6],["r",3.0,109.6,743.0,135.3],["r",3.0,135.3,743.0,161.1],["r",3.0,161.1,743.0,186.8],["r",3.0,186.8,743.0,212.5],["r",3.0,212.5,743.0,238.3],["r",3.0,238.3,743.0,264.0],["r",3.0,264.0,743.0,289.7],["r",3.0,289.7,743.0,315.5],["r",3.0,315.5,743.0,346.0],["c",3.0,3.0,184.2,346.0],["c",184.2,3.0,311.9,346.0],["c",311.9,3.0,457.9,346.0],["c",457.9,3.0,566.9,346.0],["c",566.9,3.0,659.9,346.0],["c",659.9,3.0,743.0,346.0],["s",457.9,3.0,743.0,28.6]]},\n'
  '"1510.01234v1.1":{"paper_id":"1510.01234","dx":16,"dy":11,"n_rows":13,"n_cols":2,"pixel_sha256":"e27955a634e3bf33e8aab42ba14d0f79464b31e1edd3b408f2d5e74dc5f60570","objects":[["t",3.0,3.0,365.0,329.0],["r",3.0,3.0,365.0,29.1],["r",3.0,29.1,365.0,54.4],["r",3.0,54.4,365.0,79.3],["r",3.0,79.3,365.0,104.2],["r",3.0,104.2,365.0,129.1],["r",3.0,129.1,365.0,154.0],["r",3.0,154.0,365.0,178.9],["r",3.0,178.9,365.0,203.8],["r",3.0,203.8,365.0,228.7],["r",3.0,228.7,365.0,253.6],["r",3.0,253.6,365.0,278.5],["r",3.0,278.5,365.0,303.4],["r",3.0,303.4,365.0,329.0],["c",3.0,3.0,261.4,329.0],["c",261.4,3.0,365.0,329.0]]},\n'
  '"1510.01234v1.2":{"paper_id":"1510.01234","dx":2,"dy":11,"n_rows":17,"n_cols":2,"pixel_sha256":"7f39f46a90ab914af315df25d34720c1a961b415a7fb90ac6c887e8d8a547c86","objects":[["t",3.0,3.0,390.0,443.0],["r",3.0,3.0,390.0,29.1],["r",3.0,29.1,390.0,54.4],["r",3.0,54.4,390.0,79.3],["r",3.0,79.3,390.0,104.2],["r",3.0,104.2,390.0,129.1],["r",3.0,129.1,390.0,154.0],["r",3.0,154.0,390.0,178.9],["r",3.0,178.9,390.0,203.8],["r",3.0,203.8,390.0,228.7],["r",3.0,228.7,390.0,253.6],["r",3.0,253.6,390.0,278.5],["r",3.0,278.5,390.0,303.4],["r",3.0,303.4,390.0,328.4],["r",3.0,328.4,390.0,353.3],["r",3.0,353.3,390.0,378.2],["r",3.0,378.2,390.0,403.1],["r",3.0,403.1,390.0,443.0],["c",3.0,3.0,261.4,443.0],["c",261.4,3.0,390.0,443.0]]},\n'
  '"1510.04780v1.6":{"paper_id":"1510.04780","dx":16,"dy":12,"n_rows":31,"n_cols":2,"pixel_sha256":"abd6b413d0a56a3513e09a46db71ec5583a1aa541fb9a99ccaa6272b42b82ecc","objects":[["t",3.0,3.0,703.0,778.0],["r",3.0,3.0,703.0,30.1],["r",3.0,30.1,703.0,55.4],["r",3.0,55.4,703.0,80.3],["r",3.0,80.3,703.0,105.2],["r",3.0,105.2,703.0,130.1],["r",3.0,130.1,703.0,155.0],["r",3.0,155.0,703.0,179.9],["r",3.0,179.9,703.0,204.8],["r",3.0,204.8,703.0,229.7],["r",3.0,229.7,703.0,254.6],["r",3.0,254.6,703.0,279.5],["r",3.0,279.5,703.0,304.4],["r",3.0,304.4,703.0,329.4],["r",3.0,329.4,703.0,354.3],["r",3.0,354.3,703.0,379.2],["r",3.0,379.2,703.0,404.1],["r",3.0,404.1,703.0,429.0],["r",3.0,429.0,703.0,453.9],["r",3.0,453.9,703.0,478.8],["r",3.0,478.8,703.0,503.7],["r",3.0,503.7,703.0,528.6],["r",3.0,528.6,703.0,553.5],["r",3.0,553.5,703.0,578.4],["r",3.0,578.4,703.0,603.3],["r",3.0,603.3,703.0,628.2],["r",3.0,628.2,703.0,653.1],["r",3.0,653.1,703.0,678.0],["r",3.0,678.0,703.0,703.0],["r",3.0,703.0,703.0,727.9],["r",3.0,727.9,703.0,752.8],["r",3.0,752.8,703.0,778.0],["c",3.0,3.0,75.7,778.0],["c",75.7,3.0,703.0,778.0]]},\n'
  '"1510.04780v1.7":{"paper_id":"1510.04780","dx":16,"dy":11,"n_rows":14,"n_cols":2,"pixel_sha256":"8973451847fdfba22d0674bd75dbf117ef16c90f88c8425eb7c6d2733c3a4b4b","objects":[["t",3.0,3.0,814.0,354.0],["r",3.0,3.0,814.0,29.1],["r",3.0,29.1,814.0,54.4],["r",3.0,54.4,814.0,79.3],["r",3.0,79.3,814.0,104.2],["r",3.0,104.2,814.0,129.1],["r",3.0,129.1,814.0,154.0],["r",3.0,154.0,814.0,178.9],["r",3.0,178.9,814.0,203.8],["r",3.0,203.8,814.0,228.7],["r",3.0,228.7,814.0,253.6],["r",3.0,253.6,814.0,278.5],["r",3.0,278.5,814.0,303.4],["r",3.0,303.4,814.0,328.4],["r",3.0,328.4,814.0,354.0],["c",3.0,3.0,65.4,354.0],["c",65.4,3.0,814.0,354.0]]},\n'
  '"1511.06278v1.1":{"paper_id":"1511.06278","dx":4,"dy":11,"n_rows":6,"n_cols":12,"pixel_sha256":"6a32a72562a3d07b494415feb93a7aa19cb2d0aff6ae2e97f9259d03d1aa88c2","objects":[["t",3.0,3.0,585.0,162.0],["r",3.0,3.0,585.0,31.1],["r",3.0,31.1,585.0,58.9],["r",3.0,58.9,585.0,84.7],["r",3.0,84.7,585.0,110.4],["r",3.0,110.4,585.0,136.2],["r",3.0,136.2,585.0,162.0],["c",3.0,3.0,66.2,162.0],["c",66.2,3.0,119.3,162.0],["c",119.3,3.0,166.7,162.0],["c",166.7,3.0,212.4,162.0],["c",212.4,3.0,258.0,162.0],["c",258.0,3.0,303.7,162.0],["c",303.7,3.0,349.4,162.0],["c",349.4,3.0,395.0,162.0],["c",395.0,3.0,440.7,162.0],["c",440.7,3.0,486.3,162.0],["c",486.3,3.0,532.0,162.0],["c",532.0,3.0,585.0,162.0]]},\n'
  '"1602.02332v1.3":{"paper_id":"1602.02332","dx":16,"dy":11,"n_rows":12,"n_cols":3,"pixel_sha256":"5beed8244b2508e66133342d2604d074a201da7df3558207126a65723752fd1d","objects":[["t",3.0,3.0,584.0,305.0],["r",3.0,3.0,584.0,29.1],["r",3.0,29.1,584.0,54.4],["r",3.0,54.4,584.0,79.3],["r",3.0,79.3,584.0,104.2],["r",3.0,104.2,584.0,129.1],["r",3.0,129.1,584.0,154.0],["r",3.0,154.0,584.0,178.9],["r",3.0,178.9,584.0,203.8],["r",3.0,203.8,584.0,228.7],["r",3.0,228.7,584.0,253.6],["r",3.0,253.6,584.0,278.5],["r",3.0,278.5,584.0,305.0],["c",3.0,3.0,210.7,305.0],["c",210.7,3.0,454.9,305.0],["c",454.9,3.0,584.0,305.0]]},\n'
  '"1603.01595v1.1":{"paper_id":"1603.01595","dx":17,"dy":12,"n_rows":8,"n_cols":4,"pixel_sha256":"6101172539a340efbd8097c085dfc091d859b04cb2f6cb21ca6abcc1ca2ec47d","objects":[["t",3.0,3.0,508.0,205.0],["r",3.0,3.0,508.0,30.1],["r",3.0,30.1,508.0,55.4],["r",3.0,55.4,508.0,80.3],["r",3.0,80.3,508.0,105.2],["r",3.0,105.2,508.0,130.1],["r",3.0,130.1,508.0,155.0],["r",3.0,155.0,508.0,179.9],["r",3.0,179.9,508.0,205.0],["c",3.0,3.0,185.7,205.0],["c",185.7,3.0,278.3,205.0],["c",278.3,3.0,397.9,205.0],["c",397.9,3.0,508.0,205.0]]},\n"1603.01595v1.2":{"paper_id":"1603.01595","dx":16,"dy":12,"n_rows":6,"n_cols":4,"pixel_sha256":"71b224d18b7b4fee54e66f3cd7a6a167cd487c387d2e23949756e03a6d1fa81f","objects":[["t",3.0,3.0,596.0,155.0],["r",3.0,3.0,596.0,30.1],["r",3.0,30.1,596.0,55.4],["r",3.0,55.4,596.0,80.3],["r",3.0,80.3,596.0,105.2],["r",3.0,105.2,596.0,130.1],["r",3.0,130.1,596.0,155.0],["c",3.0,3.0,301.0,155.0],["c",301.0,3.0,383.5,155.0],["c",383.5,3.0,490.8,155.0],["c",490.8,3.0,596.0,155.0]]},\n'
  '"1603.01595v1.3":{"paper_id":"1603.01595","dx":16,"dy":11,"n_rows":12,"n_cols":2,"pixel_sha256":"7220c236a75da7df29eaf041892f949055f685423e91ed4a280fa30fbab1f055","objects":[["t",3.0,3.0,317.0,304.0],["r",3.0,3.0,317.0,29.1],["r",3.0,29.1,317.0,54.4],["r",3.0,54.4,317.0,79.3],["r",3.0,79.3,317.0,104.2],["r",3.0,104.2,317.0,129.1],["r",3.0,129.1,317.0,154.0],["r",3.0,154.0,317.0,178.9],["r",3.0,178.9,317.0,203.8],["r",3.0,203.8,317.0,228.7],["r",3.0,228.7,317.0,253.6],["r",3.0,253.6,317.0,278.5],["r",3.0,278.5,317.0,304.0],["c",3.0,3.0,151.7,304.0],["c",151.7,3.0,317.0,304.0]]},\n"1603.02514v3.1":{"paper_id":"1603.02514","dx":16,"dy":11,"n_rows":3,"n_cols":5,"pixel_sha256":"3e7f4a09541fb2069e99b9ff1e58ffc56ab32aea8853d935e34408207cc22313","objects":[["t",3.0,3.0,583.0,74.0],["r",3.0,3.0,583.0,27.5],["r",3.0,27.5,583.0,50.7],["r",3.0,50.7,583.0,74.0],["c",3.0,3.0,121.1,74.0],["c",121.1,3.0,232.9,74.0],["c",232.9,3.0,369.2,74.0],["c",369.2,3.0,476.1,74.0],["c",476.1,3.0,583.0,74.0]]},\n'
  '"1604.06285v1.2":{"paper_id":"1604.06285","dx":16,"dy":11,"n_rows":14,"n_cols":2,"pixel_sha256":"4821c2aa4c1da202523777ee33a0c8f64a704738a096b8434aae91aa7a3cec9a","objects":[["t",3.0,3.0,498.0,359.0],["r",3.0,3.0,498.0,29.1],["r",3.0,29.1,498.0,54.8],["r",3.0,54.8,498.0,80.1],["r",3.0,80.1,498.0,105.0],["r",3.0,105.0,498.0,129.9],["r",3.0,129.9,498.0,155.3],["r",3.0,155.3,498.0,181.0],["r",3.0,181.0,498.0,206.3],["r",3.0,206.3,498.0,231.2],["r",3.0,231.2,498.0,256.1],["r",3.0,256.1,498.0,281.4],["r",3.0,281.4,498.0,307.2],["r",3.0,307.2,498.0,330.6],["r",3.0,330.6,498.0,359.0],["c",3.0,3.0,62.4,359.0],["c",62.4,3.0,498.0,359.0],["s",3.0,29.1,498.0,54.8],["s",3.0,155.3,498.0,181.0],["s",3.0,281.4,498.0,307.2]]},\n'
  '"1604.06285v1.4":{"paper_id":"1604.06285","dx":16,"dy":11,"n_rows":5,"n_cols":5,"pixel_sha256":"49a9915485b4745da5f107d9807bb5d5e537fccb6636dbbc3732de912ea4452e","objects":[["t",3.0,3.0,409.0,131.0],["r",3.0,3.0,409.0,29.1],["r",3.0,29.1,409.0,54.4],["r",3.0,54.4,409.0,79.7],["r",3.0,79.7,409.0,105.0],["r",3.0,105.0,409.0,131.0],["c",3.0,3.0,159.1,131.0],["c",159.1,3.0,222.8,131.0],["c",222.8,3.0,284.6,131.0],["c",284.6,3.0,346.4,131.0],["c",346.4,3.0,409.0,131.0],["s",3.0,79.7,159.1,131.0],["s",3.0,29.1,159.1,79.7]]},\n"1604.06979v1.1":{"paper_id":"1604.06979","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"9ea947d633a7fde0fef6cc4a891d4c44b086ea63b272d8e23f72844d098d246f","objects":[["t",3.0,3.0,580.0,188.0],["r",3.0,3.0,580.0,31.1],["r",3.0,31.1,580.0,58.9],["r",3.0,58.9,580.0,84.7],["r",3.0,84.7,580.0,110.4],["r",3.0,110.4,580.0,136.2],["r",3.0,136.2,580.0,161.9],["r",3.0,161.9,580.0,188.0],["c",3.0,3.0,287.5,188.0],["c",287.5,3.0,444.8,188.0],["c",444.8,3.0,580.0,188.0]]},\n'
  '"1604.06979v1.2":{"paper_id":"1604.06979","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"cdaedeba73097ba9446c521800d9c546796adc5f70ef1fa24343581d46287760","objects":[["t",3.0,3.0,476.0,188.0],["r",3.0,3.0,476.0,31.1],["r",3.0,31.1,476.0,58.9],["r",3.0,58.9,476.0,84.7],["r",3.0,84.7,476.0,110.4],["r",3.0,110.4,476.0,136.2],["r",3.0,136.2,476.0,161.9],["r",3.0,161.9,476.0,188.0],["c",3.0,3.0,190.9,188.0],["c",190.9,3.0,341.2,188.0],["c",341.2,3.0,476.0,188.0]]},\n"1605.04635v2.1":{"paper_id":"1605.04635","dx":16,"dy":12,"n_rows":4,"n_cols":5,"pixel_sha256":"be421a209c4bd21f5bfdc26f83bace51de13925e35fd80e1ea38ecb4675bcc6a","objects":[["t",3.0,3.0,470.0,107.0],["r",3.0,3.0,470.0,29.0],["r",3.0,29.0,470.0,54.8],["r",3.0,54.8,470.0,80.5],["r",3.0,80.5,470.0,107.0],["c",3.0,3.0,106.6,107.0],["c",106.6,3.0,197.6,107.0],["c",197.6,3.0,280.4,107.0],["c",280.4,3.0,401.1,107.0],["c",401.1,3.0,470.0,107.0]]},\n'
  '"1605.06770v1.1":{"paper_id":"1605.06770","dx":16,"dy":12,"n_rows":8,"n_cols":2,"pixel_sha256":"fef7a93d63404699c3a70726df6a5eee1c913f837cffc7cd3a5745619c9b2f87","objects":[["t",3.0,3.0,513.0,209.0],["r",3.0,3.0,513.0,32.1],["r",3.0,32.1,513.0,59.5],["r",3.0,59.5,513.0,84.4],["r",3.0,84.4,513.0,109.3],["r",3.0,109.3,513.0,134.3],["r",3.0,134.3,513.0,159.2],["r",3.0,159.2,513.0,184.1],["r",3.0,184.1,513.0,209.0],["c",3.0,3.0,419.6,209.0],["c",419.6,3.0,513.0,209.0]]},\n"1605.06770v1.2":{"paper_id":"1605.06770","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"7c7e6332e382b62ce80b89ca72f56c0b413f0c9252c402904c75fe5094018b1d","objects":[["t",3.0,3.0,473.0,140.0],["r",3.0,3.0,473.0,31.1],["r",3.0,31.1,473.0,58.5],["r",3.0,58.5,473.0,87.5],["r",3.0,87.5,473.0,113.3],["r",3.0,113.3,473.0,140.0],["c",3.0,3.0,101.9,140.0],["c",101.9,3.0,226.3,140.0],["c",226.3,3.0,324.3,140.0],["c",324.3,3.0,473.0,140.0],["s",3.0,87.5,101.9,140.0],["s",3.0,31.1,101.9,87.5]]},\n'
  '"1606.09370v1.1":{"paper_id":"1606.09370","dx":16,"dy":12,"n_rows":7,"n_cols":2,"pixel_sha256":"860111df9549d7448d973fd315aeadb65ff1c2d9d6de5c140a3501a4e30677d1","objects":[["t",3.0,3.0,351.0,184.0],["r",3.0,3.0,351.0,30.1],["r",3.0,30.1,351.0,55.8],["r",3.0,55.8,351.0,81.5],["r",3.0,81.5,351.0,107.3],["r",3.0,107.3,351.0,133.0],["r",3.0,133.0,351.0,158.7],["r",3.0,158.7,351.0,184.0],["c",3.0,3.0,137.7,184.0],["c",137.7,3.0,351.0,184.0]]},\n"1606.09370v1.4":{"paper_id":"1606.09370","dx":16,"dy":11,"n_rows":7,"n_cols":4,"pixel_sha256":"a7cd36495249046b748241e70dcfd25286acfe2e669aa2aae18b8786efb51f48","objects":[["t",3.0,3.0,417.0,193.0],["r",3.0,3.0,417.0,31.1],["r",3.0,31.1,417.0,58.9],["r",3.0,58.9,417.0,86.2],["r",3.0,86.2,417.0,112.5],["r",3.0,112.5,417.0,140.3],["r",3.0,140.3,417.0,167.6],["r",3.0,167.6,417.0,193.0],["c",3.0,3.0,177.0,193.0],["c",177.0,3.0,258.1,193.0],["c",258.1,3.0,337.4,193.0],["c",337.4,3.0,417.0,193.0]]},\n'
  '"1606.09370v1.5":{"paper_id":"1606.09370","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"cff6f8adb3315f0e03b5388957145e0f8020048d314382cd90df11624e217f07","objects":[["t",3.0,3.0,475.0,137.0],["r",3.0,3.0,475.0,31.1],["r",3.0,31.1,475.0,58.9],["r",3.0,58.9,475.0,84.7],["r",3.0,84.7,475.0,110.4],["r",3.0,110.4,475.0,137.0],["c",3.0,3.0,236.4,137.0],["c",236.4,3.0,315.6,137.0],["c",315.6,3.0,394.9,137.0],["c",394.9,3.0,475.0,137.0]]},\n"1606.09371v1.2":{"paper_id":"1606.09371","dx":17,"dy":12,"n_rows":7,"n_cols":4,"pixel_sha256":"c1ecab46480e0ddbc9786879c4b972839ef5cb53f0003246a8420583123525b9","objects":[["t",3.0,3.0,475.0,184.0],["r",3.0,3.0,475.0,30.1],["r",3.0,30.1,475.0,55.8],["r",3.0,55.8,475.0,81.6],["r",3.0,81.6,475.0,107.3],["r",3.0,107.3,475.0,133.0],["r",3.0,133.0,475.0,158.8],["r",3.0,158.8,475.0,184.0],["c",3.0,3.0,140.6,184.0],["c",140.6,3.0,261.3,184.0],["c",261.3,3.0,366.2,184.0],["c",366.2,3.0,475.0,184.0]]},\n'
  '"1606.09371v1.5":{"paper_id":"1606.09371","dx":16,"dy":7,"n_rows":5,"n_cols":4,"pixel_sha256":"c58047166868fbbda58f123c6b47b3844bc66b541d1884804eea6fec62e3cd5a","objects":[["t",3.0,3.0,383.0,133.0],["r",3.0,3.0,383.0,25.1],["r",3.0,25.1,383.0,50.8],["r",3.0,50.8,383.0,76.6],["r",3.0,76.6,383.0,102.3],["r",3.0,102.3,383.0,133.0],["c",3.0,3.0,166.0,133.0],["c",166.0,3.0,238.1,133.0],["c",238.1,3.0,310.3,133.0],["c",310.3,3.0,383.0,133.0]]},\n"1609.01344v1.1":{"paper_id":"1609.01344","dx":15,"dy":8,"n_rows":8,"n_cols":2,"pixel_sha256":"f0a82941854339181ef210517b4278cc20654478714cb4d31b6c74b661ef3df2","objects":[["t",3.0,3.0,567.0,142.0],["r",3.0,3.0,567.0,22.4],["r",3.0,22.4,567.0,41.5],["r",3.0,41.5,567.0,58.1],["r",3.0,58.1,567.0,74.7],["r",3.0,74.7,567.0,91.3],["r",3.0,91.3,567.0,107.9],["r",3.0,107.9,567.0,124.5],["r",3.0,124.5,567.0,142.0],["c",3.0,3.0,147.4,142.0],["c",147.4,3.0,567.0,142.0]]},\n'
  '"1609.01344v1.2":{"paper_id":"1609.01344","dx":16,"dy":11,"n_rows":5,"n_cols":5,"pixel_sha256":"8cb9c077e06a5d2d4ed81c3535a3f14dafd12ee7a463c40488937391729024d2","objects":[["t",3.0,3.0,306.0,134.0],["r",3.0,3.0,306.0,32.7],["r",3.0,32.7,306.0,60.1],["r",3.0,60.1,306.0,85.0],["r",3.0,85.0,306.0,109.9],["r",3.0,109.9,306.0,134.0],["c",3.0,3.0,75.7,134.0],["c",75.7,3.0,132.5,134.0],["c",132.5,3.0,189.9,134.0],["c",189.9,3.0,247.2,134.0],["c",247.2,3.0,306.0,134.0]]},\n"1610.06272v2.1":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":4,"n_cols":5,"pixel_sha256":"1d45e646e13542ab33e8958ca306df8e4eb9c98e6d4b37735ae8bf2986684d2e","objects":[["t",3.0,3.0,360.0,100.0],["r",3.0,3.0,360.0,28.6],["r",3.0,28.6,360.0,53.9],["r",3.0,53.9,360.0,76.7],["r",3.0,76.7,360.0,100.0],["c",3.0,3.0,60.0,100.0],["c",60.0,3.0,130.7,100.0],["c",130.7,3.0,208.9,100.0],["c",208.9,3.0,279.6,100.0],["c",279.6,3.0,360.0,100.0]]},\n'
  '"1610.06272v2.2":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":4,"n_cols":7,"pixel_sha256":"12af47e92fa608b88fbb3124f0059a04a12346450973d5bbd5ef9d96c85c8fb2","objects":[["t",3.0,3.0,452.0,96.0],["r",3.0,3.0,452.0,26.5],["r",3.0,26.5,452.0,49.7],["r",3.0,49.7,452.0,72.6],["r",3.0,72.6,452.0,96.0],["c",3.0,3.0,59.9,96.0],["c",59.9,3.0,125.3,96.0],["c",125.3,3.0,188.6,96.0],["c",188.6,3.0,251.9,96.0],["c",251.9,3.0,315.2,96.0],["c",315.2,3.0,380.6,96.0],["c",380.6,3.0,452.0,96.0]]},\n'
  '"1610.06272v2.3":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":5,"n_cols":5,"pixel_sha256":"4670255eb46ff1169a9c2ba3be859a0dfc2e97c0182a1dbb08f3bfcce9df59c0","objects":[["t",3.0,3.0,355.0,123.0],["r",3.0,3.0,355.0,26.1],["r",3.0,26.1,355.0,51.4],["r",3.0,51.4,355.0,76.7],["r",3.0,76.7,355.0,99.6],["r",3.0,99.6,355.0,123.0],["c",3.0,3.0,59.9,123.0],["c",59.9,3.0,130.7,123.0],["c",130.7,3.0,201.4,123.0],["c",201.4,3.0,276.5,123.0],["c",276.5,3.0,355.0,123.0],["s",201.4,3.0,355.0,26.1],["s",59.9,3.0,201.4,26.1]]},\n"1611.04741v2.1":{"paper_id":"1611.04741","dx":16,"dy":12,"n_rows":8,"n_cols":2,"pixel_sha256":"12d69600e054bb44986fa7e47518b1a87d5a0089452ba0c5a1577521f2b5758f","objects":[["t",3.0,3.0,453.0,214.0],["r",3.0,3.0,453.0,32.1],["r",3.0,32.1,453.0,59.9],["r",3.0,59.9,453.0,85.7],["r",3.0,85.7,453.0,111.4],["r",3.0,111.4,453.0,137.2],["r",3.0,137.2,453.0,162.9],["r",3.0,162.9,453.0,188.6],["r",3.0,188.6,453.0,214.0],["c",3.0,3.0,368.8,214.0],["c",368.8,3.0,453.0,214.0]]},\n'
  '"1611.04741v2.2":{"paper_id":"1611.04741","dx":16,"dy":11,"n_rows":12,"n_cols":4,"pixel_sha256":"5d946428031580b23a962499cf3031428c4de8d3c5687be4b797d663e96451ca","objects":[["t",3.0,3.0,902.0,321.0],["r",3.0,3.0,902.0,31.1],["r",3.0,31.1,902.0,58.9],["r",3.0,58.9,902.0,84.7],["r",3.0,84.7,902.0,110.4],["r",3.0,110.4,902.0,136.2],["r",3.0,136.2,902.0,161.9],["r",3.0,161.9,902.0,187.6],["r",3.0,187.6,902.0,213.4],["r",3.0,213.4,902.0,239.1],["r",3.0,239.1,902.0,266.9],["r",3.0,266.9,902.0,294.7],["r",3.0,294.7,902.0,321.0],["c",3.0,3.0,376.7,321.0],["c",376.7,3.0,562.8,321.0],["c",562.8,3.0,737.4,321.0],["c",737.4,3.0,902.0,321.0]]},\n'
  '"1611.04741v2.3":{"paper_id":"1611.04741","dx":16,"dy":11,"n_rows":6,"n_cols":4,"pixel_sha256":"3f9e474e3ebba0a5ee84d98d6ca687fea43fabd7d7ed4d15f7faa15e17a3ceff","objects":[["t",3.0,3.0,580.0,167.0],["r",3.0,3.0,580.0,31.1],["r",3.0,31.1,580.0,58.9],["r",3.0,58.9,580.0,84.7],["r",3.0,84.7,580.0,112.5],["r",3.0,112.5,580.0,140.3],["r",3.0,140.3,580.0,167.0],["c",3.0,3.0,376.7,167.0],["c",376.7,3.0,444.0,167.0],["c",444.0,3.0,511.4,167.0],["c",511.4,3.0,580.0,167.0]]},\n"1611.09235v1.1":{"paper_id":"1611.09235","dx":16,"dy":10,"n_rows":7,"n_cols":3,"pixel_sha256":"330d090fef8547f312b608296616ae88e913725a0371968e776b40cb0a571f5b","objects":[["t",3.0,3.0,442.0,165.0],["r",3.0,3.0,442.0,26.5],["r",3.0,26.5,442.0,49.7],["r",3.0,49.7,442.0,72.6],["r",3.0,72.6,442.0,95.4],["r",3.0,95.4,442.0,118.2],["r",3.0,118.2,442.0,141.1],["r",3.0,141.1,442.0,165.0],["c",3.0,3.0,149.3,165.0],["c",149.3,3.0,301.2,165.0],["c",301.2,3.0,442.0,165.0]]},\n'
  '"1611.09238v1.1":{"paper_id":"1611.09238","dx":16,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"b96b6fdd04fd740c86dd88e32f3d7d58c1c41365be954c83326171d787339519","objects":[["t",3.0,3.0,491.0,99.0],["r",3.0,3.0,491.0,27.5],["r",3.0,27.5,491.0,51.2],["r",3.0,51.2,491.0,74.8],["r",3.0,74.8,491.0,99.0],["c",3.0,3.0,97.0,99.0],["c",97.0,3.0,205.2,99.0],["c",205.2,3.0,293.3,99.0],["c",293.3,3.0,376.6,99.0],["c",376.6,3.0,491.0,99.0]]},\n'
  '"1702.02925v1.1":{"paper_id":"1702.02925","dx":16,"dy":12,"n_rows":13,"n_cols":3,"pixel_sha256":"11f88fcf82a68acfa1bcd29d39354d92a7d02ad79b8c461a1f848898cbf3d271","objects":[["t",3.0,3.0,613.0,330.0],["r",3.0,3.0,613.0,30.1],["r",3.0,30.1,613.0,55.4],["r",3.0,55.4,613.0,80.3],["r",3.0,80.3,613.0,105.2],["r",3.0,105.2,613.0,130.1],["r",3.0,130.1,613.0,155.0],["r",3.0,155.0,613.0,179.9],["r",3.0,179.9,613.0,204.8],["r",3.0,204.8,613.0,229.7],["r",3.0,229.7,613.0,254.6],["r",3.0,254.6,613.0,279.5],["r",3.0,279.5,613.0,304.4],["r",3.0,304.4,613.0,330.0],["c",3.0,3.0,114.9,330.0],["c",114.9,3.0,335.2,330.0],["c",335.2,3.0,613.0,330.0]]},\n'
  '"1702.02925v1.2":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":4,"n_cols":13,"pixel_sha256":"c3bd82c3ada30ee69508b52269815805c6e6b07e5a9af2486b212b1e58da6aaa","objects":[["t",3.0,3.0,948.0,106.0],["r",3.0,3.0,948.0,29.1],["r",3.0,29.1,948.0,54.4],["r",3.0,54.4,948.0,79.3],["r",3.0,79.3,948.0,106.0],["c",3.0,3.0,205.6,106.0],["c",205.6,3.0,267.4,106.0],["c",267.4,3.0,329.2,106.0],["c",329.2,3.0,391.0,106.0],["c",391.0,3.0,452.8,106.0],["c",452.8,3.0,514.7,106.0],["c",514.7,3.0,576.5,106.0],["c",576.5,3.0,638.3,106.0],["c",638.3,3.0,700.1,106.0],["c",700.1,3.0,761.9,106.0],["c",761.9,3.0,823.7,106.0],["c",823.7,3.0,885.5,106.0],["c",885.5,3.0,948.0,106.0]]},\n'
  '"1702.02925v1.3":{"paper_id":"1702.02925","dx":15,"dy":11,"n_rows":14,"n_cols":7,"pixel_sha256":"fc407ab9dba9a5c6124ce4bb541a34cb4a751c74bc845cb59e011eba302ba37a","objects":[["t",3.0,3.0,553.0,354.0],["r",3.0,3.0,553.0,29.1],["r",3.0,29.1,553.0,54.4],["r",3.0,54.4,553.0,79.3],["r",3.0,79.3,553.0,104.2],["r",3.0,104.2,553.0,129.1],["r",3.0,129.1,553.0,154.0],["r",3.0,154.0,553.0,178.9],["r",3.0,178.9,553.0,203.8],["r",3.0,203.8,553.0,228.7],["r",3.0,228.7,553.0,253.6],["r",3.0,253.6,553.0,278.5],["r",3.0,278.5,553.0,303.4],["r",3.0,303.4,553.0,328.4],["r",3.0,328.4,553.0,354.0],["c",3.0,3.0,64.4,354.0],["c",64.4,3.0,148.4,354.0],["c",148.4,3.0,230.0,354.0],["c",230.0,3.0,318.1,354.0],["c",318.1,3.0,404.1,354.0],["c",404.1,3.0,482.9,354.0],["c",482.9,3.0,553.0,354.0]]},\n'
  '"1702.02925v1.6":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":10,"n_cols":7,"pixel_sha256":"7b93018a32162cf93026690dff697b9d0a205adec3f7ce2d1160d0146e03efc9","objects":[["t",3.0,3.0,522.0,254.0],["r",3.0,3.0,522.0,29.1],["r",3.0,29.1,522.0,54.4],["r",3.0,54.4,522.0,79.3],["r",3.0,79.3,522.0,104.2],["r",3.0,104.2,522.0,129.1],["r",3.0,129.1,522.0,154.0],["r",3.0,154.0,522.0,178.9],["r",3.0,178.9,522.0,203.8],["r",3.0,203.8,522.0,228.7],["r",3.0,228.7,522.0,254.0],["c",3.0,3.0,65.4,254.0],["c",65.4,3.0,149.4,254.0],["c",149.4,3.0,216.9,254.0],["c",216.9,3.0,305.0,254.0],["c",305.0,3.0,391.0,254.0],["c",391.0,3.0,452.8,254.0],["c",452.8,3.0,522.0,254.0]]},\n'
  '"1702.02925v1.7":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":10,"n_cols":7,"pixel_sha256":"19e70e7985c56361f38f906b2b43a8cdc0c102ac03314d2dff1a49f193d1b570","objects":[["t",3.0,3.0,522.0,254.0],["r",3.0,3.0,522.0,29.1],["r",3.0,29.1,522.0,54.4],["r",3.0,54.4,522.0,79.3],["r",3.0,79.3,522.0,104.2],["r",3.0,104.2,522.0,129.1],["r",3.0,129.1,522.0,154.0],["r",3.0,154.0,522.0,178.9],["r",3.0,178.9,522.0,203.8],["r",3.0,203.8,522.0,228.7],["r",3.0,228.7,522.0,254.0],["c",3.0,3.0,65.4,254.0],["c",65.4,3.0,149.4,254.0],["c",149.4,3.0,216.9,254.0],["c",216.9,3.0,305.0,254.0],["c",305.0,3.0,391.0,254.0],["c",391.0,3.0,452.8,254.0],["c",452.8,3.0,522.0,254.0]]},\n'
  '"1705.02407v2.1":{"paper_id":"1705.02407","dx":16,"dy":11,"n_rows":8,"n_cols":9,"pixel_sha256":"1e1622f46c133090f5a49069be9879e02a63cce36ffaac035c190aaa8793e176","objects":[["t",3.0,3.0,729.0,194.0],["r",3.0,3.0,729.0,29.6],["r",3.0,29.6,729.0,54.9],["r",3.0,54.9,729.0,77.7],["r",3.0,77.7,729.0,101.0],["r",3.0,101.0,729.0,124.2],["r",3.0,124.2,729.0,147.1],["r",3.0,147.1,729.0,169.9],["r",3.0,169.9,729.0,194.0],["c",3.0,3.0,208.3,194.0],["c",208.3,3.0,277.5,194.0],["c",277.5,3.0,339.8,194.0],["c",339.8,3.0,400.2,194.0],["c",400.2,3.0,462.4,194.0],["c",462.4,3.0,522.6,194.0],["c",522.6,3.0,591.2,194.0],["c",591.2,3.0,657.7,194.0],["c",657.7,3.0,729.0,194.0]]},\n'
  '"1705.04915v1.2":{"paper_id":"1705.04915","dx":3,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"3421f5de988ea16e5262d3cc0c9f6905f7d4d765acab19781dd70bb504ba5705","objects":[["t",3.0,3.0,421.0,105.0],["r",3.0,3.0,421.0,28.6],["r",3.0,28.6,421.0,53.5],["r",3.0,53.5,421.0,78.5],["r",3.0,78.5,421.0,105.0],["c",3.0,3.0,127.4,105.0],["c",127.4,3.0,216.8,105.0],["c",216.8,3.0,283.4,105.0],["c",283.4,3.0,358.0,105.0],["c",358.0,3.0,421.0,105.0]]},\n'
  '"1706.08653v2.10":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":4,"n_cols":7,"pixel_sha256":"dd69fc59998a9872fbd23dc10670224cd196aa99d1d401f4548cedf9a1f595a5","objects":[["t",3.0,3.0,508.0,105.0],["r",3.0,3.0,508.0,28.6],["r",3.0,28.6,508.0,55.5],["r",3.0,55.5,508.0,79.3],["r",3.0,79.3,508.0,105.0],["c",3.0,3.0,114.6,105.0],["c",114.6,3.0,180.6,105.0],["c",180.6,3.0,248.4,105.0],["c",248.4,3.0,316.9,105.0],["c",316.9,3.0,379.9,105.0],["c",379.9,3.0,442.3,105.0],["c",442.3,3.0,508.0,105.0],["s",114.6,3.0,180.6,55.5],["s",180.6,3.0,248.4,55.5],["s",316.9,3.0,508.0,28.6],["s",248.4,3.0,316.9,55.5]]},\n'
  '"1706.08653v2.11":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":8,"n_cols":7,"pixel_sha256":"8a3b16600f91f1bbbb561ab1d5bfe7cd5b05774b5fad49d7b739bef7afe7b324","objects":[["t",3.0,3.0,518.0,207.0],["r",3.0,3.0,518.0,28.6],["r",3.0,28.6,518.0,55.5],["r",3.0,55.5,518.0,79.3],["r",3.0,79.3,518.0,104.6],["r",3.0,104.6,518.0,129.9],["r",3.0,129.9,518.0,155.3],["r",3.0,155.3,518.0,180.6],["r",3.0,180.6,518.0,207.0],["c",3.0,3.0,124.4,207.0],["c",124.4,3.0,190.4,207.0],["c",190.4,3.0,258.2,207.0],["c",258.2,3.0,326.7,207.0],["c",326.7,3.0,389.7,207.0],["c",389.7,3.0,452.1,207.0],["c",452.1,3.0,518.0,207.0],["s",326.7,3.0,518.0,28.6],["s",258.2,3.0,326.7,55.5],["s",190.4,3.0,258.2,55.5],["s",124.4,3.0,190.4,55.5]]},\n'
  '"1706.08653v2.12":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":8,"n_cols":7,"pixel_sha256":"c78e6546d49fcb9de99cd67fcc7bf3e6412bbe6c8adfa46b93a39f2c5b7a1a79","objects":[["t",3.0,3.0,518.0,207.0],["r",3.0,3.0,518.0,28.6],["r",3.0,28.6,518.0,55.5],["r",3.0,55.5,518.0,79.3],["r",3.0,79.3,518.0,104.6],["r",3.0,104.6,518.0,129.9],["r",3.0,129.9,518.0,155.3],["r",3.0,155.3,518.0,180.6],["r",3.0,180.6,518.0,207.0],["c",3.0,3.0,124.4,207.0],["c",124.4,3.0,190.4,207.0],["c",190.4,3.0,258.2,207.0],["c",258.2,3.0,326.7,207.0],["c",326.7,3.0,389.7,207.0],["c",389.7,3.0,452.1,207.0],["c",452.1,3.0,518.0,207.0],["s",124.4,3.0,190.4,55.5],["s",258.2,3.0,326.7,55.5],["s",190.4,3.0,258.2,55.5],["s",326.7,3.0,518.0,28.6]]},\n'
  '"1706.08653v2.4":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":7,"n_cols":5,"pixel_sha256":"6a4ebced83a8d94321fcfa5662aefc9dc781271f2c54a43854ceaae61ec757d8","objects":[["t",3.0,3.0,470.0,185.0],["r",3.0,3.0,470.0,31.1],["r",3.0,31.1,470.0,58.5],["r",3.0,58.5,470.0,83.4],["r",3.0,83.4,470.0,108.3],["r",3.0,108.3,470.0,133.2],["r",3.0,133.2,470.0,158.6],["r",3.0,158.6,470.0,185.0],["c",3.0,3.0,152.1,185.0],["c",152.1,3.0,231.4,185.0],["c",231.4,3.0,310.7,185.0],["c",310.7,3.0,389.9,185.0],["c",389.9,3.0,470.0,185.0]]},\n'
  '"1706.08653v2.5":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":15,"n_cols":5,"pixel_sha256":"283ff8e5bc177a911e1a2c3e003a5b3d0b64fc75b2eaf65ccce083c715ed86ed","objects":[["t",3.0,3.0,512.0,385.0],["r",3.0,3.0,512.0,28.6],["r",3.0,28.6,512.0,54.0],["r",3.0,54.0,512.0,79.3],["r",3.0,79.3,512.0,104.2],["r",3.0,104.2,512.0,129.1],["r",3.0,129.1,512.0,154.0],["r",3.0,154.0,512.0,178.9],["r",3.0,178.9,512.0,204.2],["r",3.0,204.2,512.0,231.4],["r",3.0,231.4,512.0,258.2],["r",3.0,258.2,512.0,283.5],["r",3.0,283.5,512.0,308.8],["r",3.0,308.8,512.0,333.7],["r",3.0,333.7,512.0,359.1],["r",3.0,359.1,512.0,385.0],["c",3.0,3.0,197.4,385.0],["c",197.4,3.0,273.8,385.0],["c",273.8,3.0,353.0,385.0],["c",353.0,3.0,432.3,385.0],["c",432.3,3.0,512.0,385.0],["s",353.0,231.4,512.0,258.2],["s",353.0,3.0,512.0,28.6],["s",197.4,231.4,353.0,258.2],["s",3.0,3.0,197.4,54.0],["s",197.4,3.0,353.0,28.6],["s",3.0,231.4,197.4,283.5]]},\n'
  '"1706.08653v2.6":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":14,"n_cols":13,"pixel_sha256":"e8a04469e787bc3295883f4a7952273c6a43587091b1fb9dc4efe6344f3b9ef4","objects":[["t",3.0,3.0,933.0,355.0],["r",3.0,3.0,933.0,28.6],["r",3.0,28.6,933.0,55.5],["r",3.0,55.5,933.0,79.3],["r",3.0,79.3,933.0,104.2],["r",3.0,104.2,933.0,129.1],["r",3.0,129.1,933.0,154.0],["r",3.0,154.0,933.0,178.9],["r",3.0,178.9,933.0,203.8],["r",3.0,203.8,933.0,228.7],["r",3.0,228.7,933.0,253.6],["r",3.0,253.6,933.0,278.5],["r",3.0,278.5,933.0,303.9],["r",3.0,303.9,933.0,329.2],["r",3.0,329.2,933.0,355.0],["c",3.0,3.0,124.6,355.0],["c",124.6,3.0,192.0,355.0],["c",192.0,3.0,259.3,355.0],["c",259.3,3.0,326.7,355.0],["c",326.7,3.0,394.0,355.0],["c",394.0,3.0,461.3,355.0],["c",461.3,3.0,528.7,355.0],["c",528.7,3.0,596.0,355.0],["c",596.0,3.0,663.4,355.0],["c",663.4,3.0,730.7,355.0],["c",730.7,3.0,798.0,355.0],["c",798.0,3.0,865.4,355.0],["c",865.4,3.0,933.0,355.0],["s",730.7,3.0,933.0,28.6],["s",124.6,329.2,326.7,355.0],["s",326.7,329.2,528.7,355.0],["s",528.7,329.2,730.7,355.0],["s",730.7,329.2,933.0,355.0],["s",326.7,3.0,528.7,28.6],["s",124.6,3.0,326.7,28.6],["s",528.7,3.0,730.7,28.6]]},\n'
  '"1706.08653v2.7":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":9,"n_cols":7,"pixel_sha256":"23ae472d525832ba1046b545e4a3d4480a910018606fb18092e4fc07269536ac","objects":[["t",3.0,3.0,538.0,231.0],["r",3.0,3.0,538.0,28.6],["r",3.0,28.6,538.0,55.5],["r",3.0,55.5,538.0,79.3],["r",3.0,79.3,538.0,104.2],["r",3.0,104.2,538.0,129.1],["r",3.0,129.1,538.0,154.0],["r",3.0,154.0,538.0,179.3],["r",3.0,179.3,538.0,205.1],["r",3.0,205.1,538.0,231.0],["c",3.0,3.0,124.6,231.0],["c",124.6,3.0,196.8,231.0],["c",196.8,3.0,264.2,231.0],["c",264.2,3.0,333.6,231.0],["c",333.6,3.0,403.0,231.0],["c",403.0,3.0,470.3,231.0],["c",470.3,3.0,538.0,231.0],["s",124.6,3.0,333.6,28.6],["s",333.6,3.0,538.0,28.6],["s",124.6,205.1,333.6,231.0],["s",333.6,205.1,538.0,231.0],["s",3.0,3.0,124.6,55.5]]},\n'
  '"1706.08653v2.8":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"cc1f4cd24da50a1e75669b42b35710b5ddb6a3a411b1fd3be9bab5493c7fba69","objects":[["t",3.0,3.0,442.0,105.0],["r",3.0,3.0,442.0,28.6],["r",3.0,28.6,442.0,54.0],["r",3.0,54.0,442.0,79.3],["r",3.0,79.3,442.0,105.0],["c",3.0,3.0,127.3,105.0],["c",127.3,3.0,203.9,105.0],["c",203.9,3.0,283.2,105.0],["c",283.2,3.0,362.5,105.0],["c",362.5,3.0,442.0,105.0],["s",3.0,3.0,127.3,54.0],["s",127.3,3.0,283.2,28.6],["s",283.2,3.0,442.0,28.6]]},\n"1706.10239v2.1":{"paper_id":"1706.10239","dx":15,"dy":11,"n_rows":3,"n_cols":4,"pixel_sha256":"d3a1571d64a8acaaebfc2f980d2956ebd5f68404dc946fc65fb652c0b1de4ff1","objects":[["t",3.0,3.0,584.0,80.0],["r",3.0,3.0,584.0,26.8],["r",3.0,26.8,584.0,52.5],["r",3.0,52.5,584.0,80.0],["c",3.0,3.0,179.1,80.0],["c",179.1,3.0,313.5,80.0],["c",313.5,3.0,448.0,80.0],["c",448.0,3.0,584.0,80.0]]},\n'
  '"1708.06828v1.5":{"paper_id":"1708.06828","dx":16,"dy":10,"n_rows":7,"n_cols":5,"pixel_sha256":"71bf8f6ca8aca6cc2bfb0b88947ecff1426e3b19b9bd942eee15c03f6df549ff","objects":[["t",3.0,3.0,608.0,183.0],["r",3.0,3.0,608.0,27.6],["r",3.0,27.6,608.0,55.0],["r",3.0,55.0,608.0,82.4],["r",3.0,82.4,608.0,107.3],["r",3.0,107.3,608.0,132.3],["r",3.0,132.3,608.0,157.2],["r",3.0,157.2,608.0,183.0],["c",3.0,3.0,90.6,183.0],["c",90.6,3.0,291.0,183.0],["c",291.0,3.0,461.9,183.0],["c",461.9,3.0,532.9,183.0],["c",532.9,3.0,608.0,183.0],["s",3.0,3.0,90.6,55.0],["s",90.6,3.0,291.0,55.0],["s",291.0,3.0,608.0,27.6]]},\n'
  '"1709.04959v2.2":{"paper_id":"1709.04959","dx":16,"dy":11,"n_rows":11,"n_cols":4,"pixel_sha256":"18ddbfd4c2d3400a05dbe7b02ae990dd88ce358a8fdefadc84b03e1bb02e9ad9","objects":[["t",3.0,3.0,606.0,257.0],["r",3.0,3.0,606.0,27.5],["r",3.0,27.5,606.0,50.7],["r",3.0,50.7,606.0,73.6],["r",3.0,73.6,606.0,96.4],["r",3.0,96.4,606.0,119.2],["r",3.0,119.2,606.0,142.5],["r",3.0,142.5,606.0,165.7],["r",3.0,165.7,606.0,188.6],["r",3.0,188.6,606.0,211.4],["r",3.0,211.4,606.0,234.2],["r",3.0,234.2,606.0,257.0],["c",3.0,3.0,86.1,257.0],["c",86.1,3.0,328.9,257.0],["c",328.9,3.0,427.2,257.0],["c",427.2,3.0,606.0,257.0]]},\n'
  '"1709.08718v1.4":{"paper_id":"1709.08718","dx":16,"dy":8,"n_rows":7,"n_cols":2,"pixel_sha256":"824edda886f5b57bb6c87877e277f07c90963cbde52856af33d17ed60c1fe3b0","objects":[["t",3.0,3.0,344.0,170.0],["r",3.0,3.0,344.0,26.1],["r",3.0,26.1,344.0,47.8],["r",3.0,47.8,344.0,76.3],["r",3.0,76.3,344.0,101.2],["r",3.0,101.2,344.0,126.1],["r",3.0,126.1,344.0,151.0],["r",3.0,151.0,344.0,170.0],["c",3.0,3.0,214.6,170.0],["c",214.6,3.0,344.0,170.0]]},\n"1709.08718v1.5":{"paper_id":"1709.08718","dx":16,"dy":6,"n_rows":6,"n_cols":4,"pixel_sha256":"0065c3d2ff916ad80e573836f1351f8f8dbfd15917d1596b1e851a17b6e5f7cd","objects":[["t",3.0,2.0,463.0,143.0],["r",3.0,2.0,463.0,24.1],["r",3.0,24.1,463.0,49.4],["r",3.0,49.4,463.0,74.3],["r",3.0,74.3,463.0,99.2],["r",3.0,99.2,463.0,124.1],["r",3.0,124.1,463.0,143.0],["c",3.0,2.0,167.3,143.0],["c",167.3,2.0,246.3,143.0],["c",246.3,2.0,325.0,143.0],["c",325.0,2.0,463.0,143.0]]},\n'
  '"1711.04434v1.4":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"cae9d3bd4a5a57c30f5dadb130fde3a2e20130436c6f8fda4cfdb8ee2e03071e","objects":[["t",3.0,3.0,367.0,105.0],["r",3.0,3.0,367.0,29.1],["r",3.0,29.1,367.0,54.4],["r",3.0,54.4,367.0,79.3],["r",3.0,79.3,367.0,105.0],["c",3.0,3.0,159.1,105.0],["c",159.1,3.0,233.1,105.0],["c",233.1,3.0,300.1,105.0],["c",300.1,3.0,367.0,105.0]]},\n"1711.04434v1.5":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"8ec06f5209b98116642aa688630c5b5fb2b25f8957adb72404acbc03e0f8119f","objects":[["t",3.0,3.0,264.0,155.0],["r",3.0,3.0,264.0,25.6],["r",3.0,25.6,264.0,51.1],["r",3.0,51.1,264.0,79.7],["r",3.0,79.7,264.0,104.6],["r",3.0,104.6,264.0,131.0],["r",3.0,131.0,264.0,155.0],["c",3.0,3.0,145.0,155.0],["c",145.0,3.0,264.0,155.0]]},\n'
  '"1711.04434v1.7":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"b47cd58dd3955aa6b724fcf7fd36237ef97bc0c5c367c0b01cdf07f98f625f5e","objects":[["t",3.0,3.0,320.0,181.0],["r",3.0,3.0,320.0,29.1],["r",3.0,29.1,320.0,54.4],["r",3.0,54.4,320.0,79.3],["r",3.0,79.3,320.0,104.6],["r",3.0,104.6,320.0,129.9],["r",3.0,129.9,320.0,154.8],["r",3.0,154.8,320.0,181.0],["c",3.0,3.0,106.2,181.0],["c",106.2,3.0,238.9,181.0],["c",238.9,3.0,320.0,181.0],["s",3.0,104.6,106.2,181.0],["s",3.0,29.1,106.2,104.6]]},\n"1801.00005v1.1":{"paper_id":"1801.00005","dx":33,"dy":11,"n_rows":6,"n_cols":4,"pixel_sha256":"8aa61bdd7bd4ff8e9527af5ed13d44dede1c8a2883e1377371b9541d2ea43704","objects":[["t",3.0,3.0,496.0,163.0],["r",3.0,3.0,496.0,32.7],["r",3.0,32.7,496.0,58.9],["r",3.0,58.9,496.0,84.7],["r",3.0,84.7,496.0,110.4],["r",3.0,110.4,496.0,136.2],["r",3.0,136.2,496.0,163.0],["c",3.0,3.0,104.1,163.0],["c",104.1,3.0,224.1,163.0],["c",224.1,3.0,356.1,163.0],["c",356.1,3.0,496.0,163.0]]},\n'
  '"1803.02632v2.1":{"paper_id":"1803.02632","dx":16,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"a1b7ef58d372302c65a3e854679b4a31dd23f9eb949adc2cbd2e34bd3b03568b","objects":[["t",3.0,3.0,811.0,105.0],["r",3.0,3.0,811.0,29.1],["r",3.0,29.1,811.0,54.4],["r",3.0,54.4,811.0,79.3],["r",3.0,79.3,811.0,105.0],["c",3.0,3.0,569.5,105.0],["c",569.5,3.0,651.1,105.0],["c",651.1,3.0,723.2,105.0],["c",723.2,3.0,811.0,105.0]]},\n"1803.03670v1.1":{"paper_id":"1803.03670","dx":15,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"f2b8a3c9184f76e246881f7dbe28a5c697fdfe4a778961b62db23690ab61310f","objects":[["t",3.0,3.0,519.0,97.0],["r",3.0,3.0,519.0,27.1],["r",3.0,27.1,519.0,50.3],["r",3.0,50.3,519.0,73.6],["r",3.0,73.6,519.0,97.0],["c",3.0,3.0,76.8,97.0],["c",76.8,3.0,186.5,97.0],["c",186.5,3.0,292.5,97.0],["c",292.5,3.0,407.1,97.0],["c",407.1,3.0,519.0,97.0]]},\n'
  '"1803.03670v1.3":{"paper_id":"1803.03670","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"475383bd4841e7f55226a3cd920a6377f9074d77ccc299c72ae177bf64f0b775","objects":[["t",3.0,3.0,190.0,155.0],["r",3.0,3.0,190.0,28.6],["r",3.0,28.6,190.0,53.5],["r",3.0,53.5,190.0,78.5],["r",3.0,78.5,190.0,103.4],["r",3.0,103.4,190.0,128.7],["r",3.0,128.7,190.0,155.0],["c",3.0,3.0,117.4,155.0],["c",117.4,3.0,190.0,155.0]]},\n"1803.07835v1.2":{"paper_id":"1803.07835","dx":16,"dy":12,"n_rows":2,"n_cols":6,"pixel_sha256":"505e4bd3a121dba1c98c7f8f665f790fcba408f933c43ef964032112c78d8853","objects":[["t",3.0,3.0,644.0,56.0],["r",3.0,3.0,644.0,30.1],["r",3.0,30.1,644.0,56.0],["c",3.0,3.0,97.4,56.0],["c",97.4,3.0,174.1,56.0],["c",174.1,3.0,274.6,56.0],["c",274.6,3.0,367.8,56.0],["c",367.8,3.0,511.9,56.0],["c",511.9,3.0,644.0,56.0]]},\n'
  '"1806.04450v1.1":{"paper_id":"1806.04450","dx":16,"dy":11,"n_rows":7,"n_cols":2,"pixel_sha256":"23b9363e6cb21d1c2ecb0a1f2c59555e3f10b9359711bf3674434834d94fc7c8","objects":[["t",3.0,3.0,332.0,180.0],["r",3.0,3.0,332.0,29.1],["r",3.0,29.1,332.0,54.4],["r",3.0,54.4,332.0,79.3],["r",3.0,79.3,332.0,104.2],["r",3.0,104.2,332.0,129.1],["r",3.0,129.1,332.0,154.0],["r",3.0,154.0,332.0,180.0],["c",3.0,3.0,228.3,180.0],["c",228.3,3.0,332.0,180.0]]},\n'
  '"1806.04450v1.3":{"paper_id":"1806.04450","dx":4,"dy":9,"n_rows":10,"n_cols":5,"pixel_sha256":"b0c09047bb81ce7d2a587b9f4a4edfe01ad3dc37ee7aadbbb6e27069e2eca185","objects":[["t",3.0,63.0,737.0,317.0],["r",3.0,63.0,737.0,88.9],["r",3.0,88.9,737.0,114.2],["r",3.0,114.2,737.0,139.1],["r",3.0,139.1,737.0,164.0],["r",3.0,164.0,737.0,189.4],["r",3.0,189.4,737.0,215.1],["r",3.0,215.1,737.0,240.4],["r",3.0,240.4,737.0,265.3],["r",3.0,265.3,737.0,290.7],["r",3.0,290.7,737.0,317.0],["c",3.0,63.0,288.6,317.0],["c",288.6,63.0,410.7,317.0],["c",410.7,63.0,531.2,317.0],["c",531.2,63.0,620.4,317.0],["c",620.4,63.0,737.0,317.0]]},\n'
  '"1807.06535v1.2":{"paper_id":"1807.06535","dx":16,"dy":12,"n_rows":3,"n_cols":4,"pixel_sha256":"0ac7a89e8ff3bb3b5c9a717e1130c0f8916dd24ee681e0dc0c6cb41599542e9d","objects":[["t",3.0,3.0,629.0,105.0],["r",3.0,3.0,629.0,36.3],["r",3.0,36.3,629.0,70.4],["r",3.0,70.4,629.0,105.0],["c",3.0,3.0,87.4,105.0],["c",87.4,3.0,245.3,105.0],["c",245.3,3.0,483.9,105.0],["c",483.9,3.0,629.0,105.0]]},\n"1808.00179v1.1":{"paper_id":"1808.00179","dx":15,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"6abb00b361bdec53333dcf014737924df228aad26bb2878719153484f7a32ba8","objects":[["t",3.0,3.0,321.0,107.0],["r",3.0,3.0,321.0,27.2],["r",3.0,27.2,321.0,52.9],["r",3.0,52.9,321.0,78.6],["r",3.0,78.6,321.0,107.0],["c",3.0,3.0,103.6,107.0],["c",103.6,3.0,157.9,107.0],["c",157.9,3.0,238.7,107.0],["c",238.7,3.0,321.0,107.0]]},\n'
  '"1808.00179v1.11":{"paper_id":"1808.00179","dx":15,"dy":10,"n_rows":3,"n_cols":4,"pixel_sha256":"7e3e9be248d06db1ca3a671784b8d7a974c37b27ff941e7b3ab9f910d58f55ec","objects":[["t",3.0,3.0,321.0,79.0],["r",3.0,3.0,321.0,29.2],["r",3.0,29.2,321.0,53.0],["r",3.0,53.0,321.0,79.0],["c",3.0,3.0,64.1,79.0],["c",64.1,3.0,151.2,79.0],["c",151.2,3.0,254.5,79.0],["c",254.5,3.0,321.0,79.0],["s",3.0,3.0,64.1,53.0],["s",64.1,3.0,321.0,29.2]]},\n"1808.00179v1.14":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"37487f768ea7a20f4f0be872e69ee8ed5238d8c39bd56496ca016b59228ba523","objects":[["t",3.0,3.0,434.0,132.0],["r",3.0,3.0,434.0,30.2],["r",3.0,30.2,434.0,54.0],["r",3.0,54.0,434.0,79.7],["r",3.0,79.7,434.0,105.4],["r",3.0,105.4,434.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,190.5,132.0],["c",190.5,3.0,312.2,132.0],["c",312.2,3.0,434.0,132.0],["s",68.8,3.0,434.0,30.2],["s",3.0,3.0,68.8,54.0]]},\n'
  '"1808.00179v1.3":{"paper_id":"1808.00179","dx":15,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"61e86f0ddaf0796786f136a935d7696c52b105e8e5161c14bc0fb13ab9ed633e","objects":[["t",3.0,3.0,343.0,132.0],["r",3.0,3.0,343.0,30.2],["r",3.0,30.2,343.0,54.0],["r",3.0,54.0,343.0,79.7],["r",3.0,79.7,343.0,105.4],["r",3.0,105.4,343.0,132.0],["c",3.0,3.0,67.8,132.0],["c",67.8,3.0,190.8,132.0],["c",190.8,3.0,266.1,132.0],["c",266.1,3.0,343.0,132.0],["s",3.0,3.0,67.8,54.0],["s",67.8,3.0,343.0,30.2]]},\n"1808.00179v1.6":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"9462fe171476d288a6524a59ca03f3304bb1f81040cbbf40d82256ce502c8def","objects":[["t",3.0,3.0,434.0,132.0],["r",3.0,3.0,434.0,30.2],["r",3.0,30.2,434.0,54.0],["r",3.0,54.0,434.0,79.7],["r",3.0,79.7,434.0,105.4],["r",3.0,105.4,434.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,190.5,132.0],["c",190.5,3.0,312.2,132.0],["c",312.2,3.0,434.0,132.0],["s",68.8,3.0,434.0,30.2],["s",3.0,3.0,68.8,54.0]]},\n'
  '"1808.00179v1.8":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"d8e356c260b34373267d0f71f78e5a422ec9b5bed16a04f2d332f5771bc7ad86","objects":[["t",3.0,3.0,482.0,132.0],["r",3.0,3.0,482.0,30.2],["r",3.0,30.2,482.0,54.0],["r",3.0,54.0,482.0,79.7],["r",3.0,79.7,482.0,105.4],["r",3.0,105.4,482.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,206.4,132.0],["c",206.4,3.0,344.1,132.0],["c",344.1,3.0,482.0,132.0],["s",3.0,3.0,68.8,54.0],["s",68.8,3.0,482.0,30.2]]},\n"1808.00179v1.9":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"5bc5a6bfe5b08bd7fa3d403dc0f233f4bb8e1f939497aa691ce2f5395ab464f4","objects":[["t",3.0,3.0,482.0,132.0],["r",3.0,3.0,482.0,30.2],["r",3.0,30.2,482.0,54.0],["r",3.0,54.0,482.0,79.7],["r",3.0,79.7,482.0,105.4],["r",3.0,105.4,482.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,206.4,132.0],["c",206.4,3.0,344.1,132.0],["c",344.1,3.0,482.0,132.0],["s",3.0,3.0,68.8,54.0],["s",68.8,3.0,482.0,30.2]]},\n'
  '"1808.03399v1.1":{"paper_id":"1808.03399","dx":16,"dy":8,"n_rows":7,"n_cols":7,"pixel_sha256":"7e52c59a38a088887301eb00f27cc801e85d0c0b8926a1e2e9c2bc7602ea0c22","objects":[["t",3.0,3.0,596.0,123.0],["r",3.0,3.0,596.0,19.9],["r",3.0,19.9,596.0,36.5],["r",3.0,36.5,596.0,53.6],["r",3.0,53.6,596.0,70.6],["r",3.0,70.6,596.0,87.2],["r",3.0,87.2,596.0,104.2],["r",3.0,104.2,596.0,123.0],["c",3.0,3.0,171.4,123.0],["c",171.4,3.0,247.2,123.0],["c",247.2,3.0,321.5,123.0],["c",321.5,3.0,384.2,123.0],["c",384.2,3.0,460.0,123.0],["c",460.0,3.0,534.3,123.0],["c",534.3,3.0,596.0,123.0],["s",384.2,3.0,596.0,19.9],["s",171.4,3.0,384.2,19.9],["s",384.2,19.9,596.0,36.5],["s",171.4,19.9,384.2,36.5]]}\n}"""\nif hashlib.sha256(CARRIER_GEOMETRY_JSON.encode()).hexdigest()!=CARRIER_GEOMETRY_SHA256:\n    raise RuntimeError("Embedded carrier geometry does not match its SHA-256")\nCARRIER_GEOMETRY=json.loads(CARRIER_GEOMETRY_JSON)\n'
  'CARRIER_LABELS={"t":"table","r":"table row","c":"table column","s":"table spanning cell"}\n\n\n# ---- former cell 13\ndef norm_text(s):\n    # Loose normalization for eligibility checks (headers, row identifiers).\n    return re.sub(r"\\s+"," ",re.sub(r"[^0-9A-Za-z]+"," ",str(s))).strip().lower()\n\n\ndef image_digest(image):\n    rgb=image.convert("RGB")\n    return hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode()+rgb.tobytes()).hexdigest()\n\n\ndef place_chunks(chunks,dx,dy,scale=150.0/72.0):\n    """Chunk boxes (PDF points, bottom-left origin) in image pixels: the chunks\' extent is rendered at\n    150 DPI and shifted by the carrier\'s whole-pixel offset."""\n    x_min=min(float(c["x1"]) for c in chunks)\n    y_top=max(float(c["y2"]) for c in chunks)\n    return [\n        [(float(c["x1"])-x_min)*scale+dx,(y_top-float(c["y2"]))*scale+dy,\n         (float(c["x2"])-x_min)*scale+dx,(y_top-float(c["y1"]))*scale+dy]\n        for c in chunks\n    ]\n\n\ndef match_key(s):\n'
  '    # Alphanumeric characters only; a punctuation-only cell such as "..." keeps its punctuation.\n    s=str(s).lower()\n    return re.sub(r"[^0-9a-z]+","",s) or re.sub(r"\\s+","",s)\n\n\ndef cell_text(cell):\n    # Plain cell text; SciTSR\'s `tex` field holds LaTeX markup that never matches the PDF chunks.\n    content=[str(x) for x in (cell.get("content") or []) if str(x).strip()]\n    return " ".join(content) if content else str(cell.get("tex") or "")\n\n\ndef match_cells_to_chunks(cells,chunks,chunk_boxes):\n    texts=[str(c.get("text","")) for c in chunks]\n    # SciTSR\'s conversion doubles the final character of each table\'s last chunk ("2.53" -> "2.533").\n    if texts and len(texts[-1])>1 and texts[-1][-1]==texts[-1][-2]:\n        texts[-1]=texts[-1][:-1]\n    keys=[match_key(x) for x in texts]\n    used=set()\n    out=[]\n    for cell in sorted(cells,key=lambda c:(int(c["start_row"]),int(c["start_col"]),int(c.get("id",0)))):\n        text=cell_text(cell)\n        target=match_key(text)\n'
  '        if not target:\n            continue\n        exact=[i for i,k in enumerate(keys) if i not in used and k==target]\n        matches=[]\n        if exact:\n            matches=[exact[0]]\n        else:\n            # A cell may span several chunks: join unused chunks in reading order until they spell the cell.\n            for i in range(len(chunks)):\n                if i in used or not keys[i] or not target.startswith(keys[i]): continue\n                joined=""\n                idxs=[]\n                for j in range(i,len(chunks)):\n                    if j in used or not keys[j]: continue\n                    if not target.startswith(joined+keys[j]): break\n                    joined+=keys[j]; idxs.append(j)\n                    if joined==target:\n                        matches=idxs\n                        break\n                if matches: break\n        if not matches:\n            return None\n        used.update(matches)\n        b=[chunk_boxes[i] for i in matches]\n'
  '        box=[min(x[0] for x in b),min(x[1] for x in b),max(x[2] for x in b),max(x[3] for x in b)]\n        out.append({\n            "text":text,\n            "row_start":int(cell["start_row"]),"row_end":int(cell["end_row"]),\n            "col_start":int(cell["start_col"]),"col_end":int(cell["end_col"]),\n            "box":box,\n        })\n    return out\n\n\ndef decode_source_image(value):\n    if isinstance(value, Image.Image):\n        return value.convert("RGB")\n    if isinstance(value, dict):\n        if value.get("bytes") is not None:\n            image=Image.open(io.BytesIO(value["bytes"]))\n            image.load()\n            return image.convert("RGB")\n        if value.get("path") and Path(value["path"]).is_file():\n            image=Image.open(value["path"])\n            image.load()\n            return image.convert("RGB")\n    if isinstance(value,(bytes,bytearray)):\n        image=Image.open(io.BytesIO(value))\n        image.load()\n        return image.convert("RGB")\n'
  '    raise TypeError(f"Unsupported SciTSR image representation: {type(value).__name__}")\n\n\ndef derive_table(row,geometry):\n    table_id=str(row["table_id"])\n    image=decode_source_image(row["image"])\n    if image_digest(image)!=geometry["pixel_sha256"]:\n        raise RuntimeError(f"{table_id}: source pixels differ from the carrier\'s")\n    chunk_boxes=place_chunks(row["chunks"],geometry["dx"],geometry["dy"])\n    cell_records=match_cells_to_chunks(row["cells"],row["chunks"],chunk_boxes)\n    if not cell_records:\n        return None\n    n_rows=1+max(c["row_end"] for c in cell_records)\n    n_cols=1+max(c["col_end"] for c in cell_records)\n    if (n_rows,n_cols)!=(geometry["n_rows"],geometry["n_cols"]):\n        return None\n\n    objects=[{"label":CARRIER_LABELS[o[0]],"box":[float(v) for v in o[1:]]} for o in geometry["objects"]]\n\n    grid=[["" for _ in range(n_cols)] for __ in range(n_rows)]\n    for c in cell_records:\n        if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]:\n'
  '            grid[c["row_start"]][c["col_start"]]=c["text"]\n\n    return {\n        "source_id":table_id,\n        "paper_id":str(row["paper_id"]),\n        "paper_title":str(row["paper_title"]),\n        "paper_license":str(row["paper_license"]),\n        "image":image,\n        "pixel_sha256":geometry["pixel_sha256"],\n        "cells":cell_records,\n        "objects":objects,\n        "n_rows":n_rows,\n        "n_cols":n_cols,\n        "grid":grid,\n        "has_spans":any(c["row_start"]!=c["row_end"] or c["col_start"]!=c["col_end"] for c in cell_records),\n        "alignment":{"dx":geometry["dx"],"dy":geometry["dy"]},\n    }\n\n\n# ---- former cell 17 (constants)\nNUMBER_RE=re.compile(r"^[-+]?(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?%?$")\n\n\n# ---- former cell 17\ndef parse_number(text):\n    s=str(text).strip().replace(",","")\n    if s.endswith("%"): s=s[:-1]\n    if not NUMBER_RE.match(str(text).strip()):\n        try:\n            # permit comma-stripped representation\n            float(s)\n'
  '        except Exception:\n            return None\n    try:return float(s)\n    except Exception:return None\n\n\ndef eligible(item):\n    if item["has_spans"]: return False\n    if not (3<=item["n_rows"]<=20 and 2<=item["n_cols"]<=12): return False\n    grid=item["grid"]\n    if any(not str(x).strip() or len(str(x))>200 for row in grid for x in row): return False\n    headers=[norm_text(x) for x in grid[0]]\n    if any(not x for x in headers) or len(set(headers))!=len(headers): return False\n    row_ids=[norm_text(row[0]) for row in grid[1:]]\n    if any(not x for x in row_ids) or len(set(row_ids))!=len(row_ids): return False\n    numeric_cols=[]\n    for c in range(1,item["n_cols"]):\n        vals=[parse_number(row[c]) for row in grid[1:]]\n        if len(vals)>=2 and all(v is not None for v in vals):\n            numeric_cols.append(c)\n    return bool(numeric_cols)\n\n\n# ---- former cell 19 (constants)\nFONT=ImageFont.load_default(size=22)\n\n\n# ---- former cell 19\ndef make_page(item):\n'
  '    table=item["image"]\n    left,top,right,bottom=120,160,120,120\n    page=Image.new("RGB",(table.width+left+right,table.height+top+bottom),"white")\n    draw=ImageDraw.Draw(page)\n    draw.text((left,48),f"Table {item[\'source_id\']}",font=FONT,fill="black")\n    page.paste(table,(left,top))\n    draw.text((left,top+table.height+40),"Scientific table excerpt",font=FONT,fill="gray")\n    gt_box=[left,top,left+table.width,top+table.height]\n    return page,gt_box\n\n\n# ---- former cell 21\ndef box_iou(a,b):\n    x0=max(float(a[0]),float(b[0])); y0=max(float(a[1]),float(b[1]))\n    x1=min(float(a[2]),float(b[2])); y1=min(float(a[3]),float(b[3]))\n    inter=max(0,x1-x0)*max(0,y1-y0)\n    aa=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))\n    bb=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))\n    u=aa+bb-inter\n    return inter/u if u>0 else 0.0\n\n\ndef ap_for_label(preds,refs,label,thr):\n    det=[]\n    n_gt=0\n    for i,(p,r) in enumerate(zip(preds,refs,strict=True)):\n'
  '        gt=[x["box"] for x in r if x["label"]==label]\n        n_gt+=len(gt)\n        det += [(float(x["score"]),i,x["box"]) for x in p if x["label"]==label]\n    det.sort(key=lambda x:-x[0])\n    matched=defaultdict(set); hits=[]\n    for score,i,box in det:\n        gt=[x["box"] for x in refs[i] if x["label"]==label]\n        best,bj=0,None\n        for j,g in enumerate(gt):\n            if j in matched[i]: continue\n            v=box_iou(box,g)\n            if v>best: best,bj=v,j\n        if bj is not None and best>=thr:\n            matched[i].add(bj); hits.append(1)\n        else:hits.append(0)\n    if not hits or n_gt==0:return 0.0\n    tp=np.cumsum(hits); recall=tp/n_gt; precision=tp/np.arange(1,len(hits)+1)\n    mr=np.concatenate([[0],recall,[1]]); mp=np.concatenate([[0],precision,[0]])\n    for i in range(len(mp)-2,-1,-1):mp[i]=max(mp[i],mp[i+1])\n    idx=np.where(mr[1:]!=mr[:-1])[0]\n    return float(np.sum((mr[idx+1]-mr[idx])*mp[idx+1]))\n\n\n# ---- former cell 23\ndef load_detection_model():\n'
  '    """Load the verified detection snapshot (used here and by BYOD in §28)."""\n    processor=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)\n    model=TableTransformerForObjectDetection.from_pretrained(\n        str(DETECTION_DIR),\n        local_files_only=True,\n        trust_remote_code=False,\n        use_pretrained_backbone=False,\n    ).to(DEVICE).eval()\n    for p in model.parameters():p.requires_grad_(False)\n    return processor,model\n\n\ndef detect_table(image,threshold=None):\n    threshold=DETECTION_THRESHOLD if threshold is None else threshold\n    inputs=det_processor(images=image,return_tensors="pt").to(DEVICE)\n    with torch.inference_mode():\n        out=det_model(**inputs)\n    result=det_processor.post_process_object_detection(\n        out,target_sizes=[image.size[::-1]],threshold=float(threshold)\n    )[0]\n    rows=[]\n    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):\n'
  '        name=det_model.config.id2label[int(label)]\n        if name in ("table","table rotated"):\n            rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})\n    rows.sort(key=lambda x:-x["score"])\n    return rows\n\n\n# ---- former cell 28\ndef padded_crop(page,box,pad=None):\n    pad=CROP_PADDING if pad is None else pad\n    x0,y0,x1,y1=[float(v) for v in box]\n    cx0=max(0,int(math.floor(x0-pad))); cy0=max(0,int(math.floor(y0-pad)))\n    cx1=min(page.width,int(math.ceil(x1+pad))); cy1=min(page.height,int(math.ceil(y1+pad)))\n    return page.crop((cx0,cy0,cx1,cy1)),[cx0,cy0,cx1,cy1]\n\n\ndef translate_gold_objects(item,crop_box):\n    page_x0,page_y0=120,160\n    cx0,cy0,cx1,cy1=crop_box\n    out=[]\n    for obj in item["objects"]:\n        x0,y0,x1,y1=obj["box"]\n        b=[x0+page_x0-cx0,y0+page_y0-cy0,x1+page_x0-cx0,y1+page_y0-cy0]\n        b=[max(0,b[0]),max(0,b[1]),min(cx1-cx0,b[2]),min(cy1-cy0,b[3])]\n        if b[2]>b[0] and b[3]>b[1]:\n'
  '            out.append({"label":obj["label"],"box":b})\n    return out\n\n\ndef to_crop_frame(box,crop_box):\n    return [box[0]-crop_box[0],box[1]-crop_box[1],box[2]-crop_box[0],box[3]-crop_box[1]]\n\n\n# ---- former cell 30\ndef load_structure_model():\n    """Load the verified structure snapshot; §15, the activity after §18 and BYOD (§28) all use this loader."""\n    processor=AutoImageProcessor.from_pretrained(\n        str(STRUCTURE_DIR),\n        local_files_only=True,\n        trust_remote_code=False,\n        size={"shortest_edge":800,"longest_edge":800},\n    )\n    model=TableTransformerForObjectDetection.from_pretrained(\n        str(STRUCTURE_DIR),\n        local_files_only=True,\n        trust_remote_code=False,\n        use_pretrained_backbone=False,\n    ).to(DEVICE).eval()\n    for p in model.parameters():p.requires_grad_(False)\n    return processor,model\n\n\ndef recognize_structure(image,threshold=None):\n    threshold=STRUCTURE_THRESHOLD if threshold is None else threshold\n'
  '    inputs=struct_processor(images=image,return_tensors="pt").to(DEVICE)\n    with torch.inference_mode():\n        out=struct_model(**inputs)\n    result=struct_processor.post_process_object_detection(\n        out,target_sizes=[image.size[::-1]],threshold=float(threshold)\n    )[0]\n    rows=[]\n    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):\n        name=struct_model.config.id2label[int(label)]\n        rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})\n    rows.sort(key=lambda x:-x["score"])\n    return rows\n\n\ndef timed_structure(crop):\n    if torch.cuda.is_available():torch.cuda.synchronize()\n    t=time.perf_counter(); raw=recognize_structure(crop)\n    if torch.cuda.is_available():torch.cuda.synchronize()\n    struct_times.append(time.perf_counter()-t)\n    return raw\n\n\n# ---- former cell 32\ndef nms_items(items,iou_threshold=None):\n    iou_threshold=GRID_NMS_IOU if iou_threshold is None else iou_threshold\n'
  '    kept=[]\n    for item in sorted(items,key=lambda x:-x["score"]):\n        if all(box_iou(item["box"],k["box"])<iou_threshold for k in kept):\n            kept.append(item)\n    return kept\n\n\ndef grid_from_structure(raw):\n    rows=nms_items([x for x in raw if x["label"]=="table row"])\n    cols=nms_items([x for x in raw if x["label"]=="table column"])\n    rows.sort(key=lambda x:(x["box"][1]+x["box"][3])/2)\n    cols.sort(key=lambda x:(x["box"][0]+x["box"][2])/2)\n    cells=[]\n    valid=bool(rows and cols)\n    for ri,r in enumerate(rows):\n        row=[]\n        for ci,c in enumerate(cols):\n            b=[\n                max(r["box"][0],c["box"][0]),max(r["box"][1],c["box"][1]),\n                min(r["box"][2],c["box"][2]),min(r["box"][3],c["box"][3]),\n            ]\n            if b[2]<=b[0] or b[3]<=b[1]:valid=False\n            row.append(b)\n        cells.append(row)\n    return {"rows":rows,"columns":cols,"cells":cells,"valid_geometry":valid}\n\n\n# ---- former cell 34\n'
  'def source_cells_in_crop(item,crop_box):\n    cx0,cy0,_,_=crop_box\n    page_left,page_top=120,160\n    out=[]\n    for c in item["cells"]:\n        b=c["box"]\n        out.append({\n            **c,\n            "crop_box":[b[0]+page_left-cx0,b[1]+page_top-cy0,b[2]+page_left-cx0,b[3]+page_top-cy0],\n        })\n    return out\n\n\ndef contains(box,x,y):\n    return box[0]<=x<=box[2] and box[1]<=y<=box[3]\n\n\ndef reconstruct_table(item,grid,crop_box):\n    if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:\n        return {"valid":False,"reason":"grid_invalid","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}\n    strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]\n    mapped=unmapped=ambiguous=correct=0\n    simple=[c for c in source_cells_in_crop(item,crop_box) if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]]\n    for c in simple:\n        b=c["crop_box"]; x=(b[0]+b[2])/2; y=(b[1]+b[3])/2\n'
  '        ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]\n        cis=[i for i,col in enumerate(grid["columns"]) if contains(col["box"],x,y)]\n        if len(ris)==1 and len(cis)==1:\n            ri,ci=ris[0],cis[0]\n            strings[ri][ci].append(c["text"])\n            mapped+=1\n            if ri==c["row_start"] and ci==c["col_start"]:correct+=1\n        elif len(ris)==0 or len(cis)==0:\n            unmapped+=1\n        else:\n            ambiguous+=1\n    table=[[" ".join(cell).strip() for cell in row] for row in strings]\n    headers=table[0] if table else []\n    norm_headers=[norm_text(x) for x in headers]\n    valid_headers=bool(headers) and all(norm_headers) and len(set(norm_headers))==len(norm_headers)\n    valid=grid["valid_geometry"] and valid_headers and len(table)<=64 and len(headers)<=32\n    return {\n        "valid":valid,\n        "reason":"ok" if valid else "header_or_limit_invalid",\n        "table":table,\n'
  '        "mapped":mapped,"unmapped":unmapped,"ambiguous":ambiguous,\n        "assignment_accuracy":correct/max(1,len(simple)),\n        "headers_valid":valid_headers,\n    }\n\n\n# ---- former cell 36\ndef refs_for(item,key,label):\n    return [x for x in item[key] if x["label"]==label]\n\n\ndef preds_for(item,key,label):\n    return [x for x in item[key] if x["label"]==label]\n\n\n# ---- former cell 38\ndef grid_counts(crop,threshold):\n    grid=grid_from_structure(recognize_structure(crop,threshold=threshold))\n    return len(grid["rows"]),len(grid["columns"])\n\n\n# ---- former cell 40 (constants)\nPROBE_COLORS={"table row":"royalblue","table column":"seagreen","table spanning cell":"magenta"}\n\n\n# ---- former cell 40\ndef outline(image,items,colors):\n    out=image.convert("RGB").copy()\n    d=ImageDraw.Draw(out)\n    for x in items:\n        d.rectangle(x["box"],outline=colors.get(x["label"],"orange"),width=2)\n    return out\n\n\n# ---- former cell 42\ndef gold_table_dict(item):\n    headers=item["grid"][0]\n'
  '    body=item["grid"][1:]\n    return {headers[c]:[row[c] for row in body] for c in range(len(headers))}\n\n\ndef numeric_columns(item):\n    out=[]\n    for c in range(1,item["n_cols"]):\n        vals=[parse_number(row[c]) for row in item["grid"][1:]]\n        if len(vals)>=2 and all(v is not None for v in vals):\n            out.append((c,vals))\n    return out\n\n\n# ---- former cell 44 (constants)\nAGGREGATIONS=("NONE","SUM","AVERAGE","COUNT")\n\n\n# ---- former cell 44\ndef compute_numeric(cells,aggregation):\n    vals=[parse_number(x) for x in cells]\n    if aggregation=="COUNT":return float(len(cells))\n    if any(v is None for v in vals):return None\n    if aggregation=="SUM":return float(sum(vals))\n    if aggregation=="AVERAGE":return float(sum(vals)/len(vals)) if vals else None\n    return None\n\n\ndef table_to_df(table):\n    headers=table[0]\n    body=table[1:]\n    return pd.DataFrame(body,columns=headers,dtype=object)\n\n\ndef tapas_answer(table,question):\n    df=table_to_df(table)\n'
  '    inputs=tapas_tokenizer(\n        table=df,\n        queries=[question],\n        padding="max_length",\n        truncation=True,\n        return_tensors="pt",\n    )\n    model_inputs={k:v.to(DEVICE) for k,v in inputs.items()}\n    with torch.inference_mode():\n        out=tapas_model(**model_inputs)\n    coords,agg=tapas_tokenizer.convert_logits_to_predictions(\n        inputs,\n        out.logits.detach().cpu(),\n        out.logits_aggregation.detach().cpu(),\n    )\n    selected=[list(x) for x in coords[0]]\n    aggregation=AGGREGATIONS[int(agg[0])]\n    cells=[str(df.iat[r,c]) for r,c in selected]\n    numeric=compute_numeric(cells,aggregation)\n    answer=(f"{aggregation} > " if aggregation!="NONE" else "")+", ".join(cells)\n    return {\n        "coordinates":selected,"cells":cells,"aggregation":aggregation,\n        "numeric_answer":numeric,"answer":answer,\n    }\n\n\n# ---- former cell 46\ndef norm_cell(s):\n    return " ".join(str(s).lower().split())\n\n\ndef denotation_correct(result,record):\n'
  '    gold=record["gold_denotation"]\n    if isinstance(gold,list):\n        return sorted(norm_cell(x) for x in result["cells"])==sorted(norm_cell(x) for x in gold)\n    value=result["numeric_answer"]\n    return value is not None and abs(float(value)-float(gold))<=1e-6\n\n\ndef score_path(path_name,table_provider):\n    rows=[]\n    times=[]\n    for record in qa_records:\n        item=next(x for x in canonical_tables if x["source_id"]==record["table_id"])\n        table,status=table_provider(item)\n        if table is None:\n            rows.append({\n                "question_id":record["id"],"table_id":record["table_id"],"path":path_name,\n                "question":record["question"],"gold_aggregation":record["gold_aggregation"],\n                "predicted_aggregation":None,"gold_denotation":record["gold_denotation"],\n                "predicted_denotation":None,"denotation_correct":False,\n                "aggregation_correct":False,"selected_cells":[],"coordinates":[],\n'
  '                "pipeline_status":status,"latency_seconds":None,\n            })\n            continue\n        t=time.perf_counter()\n        result=tapas_answer(table,record["question"])\n        times.append(time.perf_counter()-t)\n        rows.append({\n            "question_id":record["id"],"table_id":record["table_id"],"path":path_name,\n            "question":record["question"],"gold_aggregation":record["gold_aggregation"],\n            "predicted_aggregation":result["aggregation"],"gold_denotation":record["gold_denotation"],\n            "predicted_denotation":result["numeric_answer"] if record["gold_aggregation"]!="NONE" else result["cells"],\n            "denotation_correct":denotation_correct(result,record),\n            "aggregation_correct":result["aggregation"]==record["gold_aggregation"],\n            "selected_cells":result["cells"],"coordinates":result["coordinates"],\n            "pipeline_status":"ok","latency_seconds":times[-1],\n        })\n    return rows,times\n\n\n'
  'def gold_provider(item):\n    return item["grid"],"ok"\n\n\ndef struct_provider(item):\n    r=item["gold_recon"]\n    return (r["table"],"ok") if r["valid"] else (None,r["reason"])\n\n\ndef full_provider(item):\n    if item["selected_detection"] is None:return None,"detection_miss"\n    r=item["det_recon"]\n    return (r["table"],"ok") if r["valid"] else (None,r["reason"])\n\n\ndef qa_summary(rows):\n    return {\n        "n":len(rows),\n        "correct":int(sum(r["denotation_correct"] for r in rows)),\n        "denotation_accuracy":float(np.mean([r["denotation_correct"] for r in rows])),\n        "aggregation_accuracy":float(np.mean([r["aggregation_correct"] for r in rows])),\n        "valid_questions":sum(r["pipeline_status"]=="ok" for r in rows),\n    }\n\n\n# ---- former cell 54\ndef median(x):return float(np.median(x)) if x else None\n\n\n# ---- former cell 56 (constants)\nBOX_COLORS={"table":"red","table row":"royalblue","table column":"seagreen","table spanning cell":"magenta"}\n\n\n# ---- former cell 56\n'
  'def draw_boxes(image,items):\n    out=image.convert("RGB").copy()\n    d=ImageDraw.Draw(out)\n    for item in items:\n        # PIL draws no outline unless a colour is given.\n        d.rectangle(item["box"],outline=BOX_COLORS.get(item["label"],"orange"),width=2)\n        if item["label"]=="table":\n            d.text((item["box"][0]+4,item["box"][1]-14),f"table {item.get(\'score\',0):.2f}",fill="red")\n    return out\n\n\n# ---- former cell 58\ndef write_csv(path,rows,fieldnames):\n    with open(path,"w",encoding="utf-8",newline="") as f:\n        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader()\n        for row in rows:\n            out={}\n            for k in fieldnames:\n                v=row.get(k)\n                if isinstance(v,(list,dict)):v=json.dumps(v)\n                out[k]=v\n            w.writerow(out)\n\n\n# ---- former cell 60 (constants)\nBYOD_IMAGE_SUFFIXES={".png",".jpg",".jpeg",".webp",".tif",".tiff",".bmp"}\n\n\n# ---- former cell 60\ndef accepted_answer_matches(result,accepted):\n'
  '    """Numbers compare within 1e-6 when both sides parse as numbers; anything else compares as normalized text."""\n    predicted=result["numeric_answer"] if result["aggregation"]!="NONE" else ", ".join(result["cells"])\n    if predicted is None:\n        return False\n    p,a=parse_number(predicted),parse_number(accepted)\n    if p is not None and a is not None:\n        return abs(p-a)<=1e-6\n    return norm_cell(predicted)==norm_cell(accepted)\n\n\ndef read_byod_csv(path,required):\n    """Rows with their file line numbers (line 1 is the header); names any missing column."""\n    with open(path,encoding="utf-8-sig",newline="") as f:\n        reader=csv.DictReader(f)\n        lacking=sorted(required-set(reader.fieldnames or []))\n        if lacking:\n            raise ValueError(f"{path.name}: missing column(s) {lacking}; required columns are {sorted(required)}")\n        return list(enumerate(reader,start=2))\n\n\n'
  '# ---- Former section 21 load statements (TAPAS), shared by the QA and BYOD stages -------------------\ndef load_tapas():\n    global tapas_tokenizer,tapas_model,tapas_load_seconds,tapas_params\n    tapas_tokenizer=TapasTokenizer.from_pretrained(str(TAPAS_DIR),local_files_only=True)\n    t0=time.perf_counter()\n    tapas_model=TapasForQuestionAnswering.from_pretrained(\n        str(TAPAS_DIR),local_files_only=True,dtype=torch.float32\n    ).to(DEVICE).eval()\n    tapas_load_seconds=time.perf_counter()-t0\n    for p in tapas_model.parameters():p.requires_grad_(False)\n    tapas_params=sum(p.numel() for p in tapas_model.parameters())\n\n\n# ---- Stages ------------------------------------------------------------------------------------------\nSTAGES = {}\n\n\ndef stage(name, fresh=False):\n    """Register a stage: restore the carried state, run, and persist it for the next stage."""\n    def register(function):\n        def run(args):\n            if fresh:  # a new Run all starts from an empty state\n'
  '                if state_file().is_file():\n                    state_file().unlink()\n                for name in STATE_NAMES:\n                    globals().pop(name, None)\n            restore_state()\n            try:\n                function(args)\n            finally:\n                persist_state()  # as in a kernel, what a failing stage changed before its error is kept\n        STAGES[name] = run\n        return function\n    return register\n\n\n@stage("environment", fresh=True)\ndef stage_environment(args):\n    global RUNTIME\n    load_transformers()\n    load_pandas()\n    import importlib.metadata as importlib_metadata\n    import datasets, torchvision, transformers\n    lock=Path(__file__).with_name("requirements.lock.txt")\n    pins=dict(re.findall(r"^([A-Za-z0-9_.-]+)==(\\S+)",lock.read_text(encoding="utf-8"),re.M))\n    drift={name:(importlib_metadata.version(name),pin) for name,pin in pins.items() if importlib_metadata.version(name)!=pin}\n    if drift:\n'
  '        raise RuntimeError(f"Installed versions differ from the hash lock: {drift}")\n    RUNTIME={\n        "python":sys.version.split()[0],\n        "executable":sys.executable,\n        "torch":str(torch.__version__),\n        "torchvision":str(torchvision.__version__),\n        "transformers":transformers.__version__,\n        "timm":importlib_metadata.version("timm"),\n        "datasets":datasets.__version__,\n        "numpy":np.__version__,\n        "numpy_source":"hash lock (isolated environment)",\n        "pandas":pd.__version__,\n        "pillow":importlib_metadata.version("pillow"),\n        "lock_pins_checked":len(pins),\n        "device":DEVICE,\n    }\n    if torch.cuda.is_available():\n        RUNTIME["gpu_name"]=torch.cuda.get_device_name(0)\n    print(RUNTIME)\n\n\n@stage("models")\ndef stage_models(args):\n    load_hub()\n    print(stage_snapshot(DETECTION_DIR,DETECTION_MANIFEST))\n    print(stage_snapshot(STRUCTURE_DIR,STRUCTURE_MANIFEST))\n    print(stage_snapshot(TAPAS_DIR,TAPAS_MANIFEST))\n'
  '\n\n@stage("dataset")\ndef stage_dataset(args):\n    global scitsr_paths\n    load_hub()\n    SCITSR_DIR.mkdir(parents=True,exist_ok=True)\n\n    scitsr_paths={}\n    for split,spec in SCITSR_FILES.items():\n        path=Path(hf_hub_download(\n            repo_id=SCITSR_REPO,\n            repo_type="dataset",\n            filename=spec["path"],\n            revision=SCITSR_REVISION,\n            local_dir=str(SCITSR_DIR),\n        ))\n        if path.stat().st_size!=spec["bytes"]:\n            raise RuntimeError(f"{split} SciTSR-PD shard size mismatch")\n        digest=sha256_file(path)\n        if digest!=spec["sha256"]:\n            raise RuntimeError(f"{split} SciTSR-PD shard SHA-256 mismatch")\n        scitsr_paths[split]=path\n        print(split,path,digest)\n\n\n@stage("geometry")\ndef stage_geometry(args):\n    print({"carrier_tables":len(CARRIER_GEOMETRY),"geometry_sha256":CARRIER_GEOMETRY_SHA256})\n\n\n@stage("companion")\ndef stage_companion(args):\n    global derived,failed,SOURCE_TABLES\n'
  '    from datasets import load_dataset\n    data_files={k:str(v) for k,v in scitsr_paths.items()}\n    ds=load_dataset("parquet",data_files=data_files)\n\n    derived=[]\n    failed=[]\n    for split in ("train","test"):\n        for row in ds[split]:\n            geometry=CARRIER_GEOMETRY.get(str(row["table_id"]))\n            if geometry is None:\n                continue  # one of the 14 tables the carrier dropped\n            item=derive_table(row,geometry)\n            if item is None:\n                failed.append(str(row["table_id"]))\n            else:\n                derived.append(item)\n\n    missing=sorted(set(CARRIER_GEOMETRY)-{x["source_id"] for x in derived})\n    if missing:\n        raise RuntimeError(f"{len(missing)} carrier tables failed to derive: {missing[:10]}")\n\n    SOURCE_TABLES=sum(len(ds[s]) for s in ("train","test"))\n    print({\n        "source_tables":SOURCE_TABLES,\n        "carrier_tables":len(CARRIER_GEOMETRY),\n        "derived_tables":len(derived),\n    })\n\n\n'
  '@stage("split")\ndef stage_split(args):\n    global splits\n    rng=random.Random(42)\n    by_paper=defaultdict(list)\n    for item in derived:\n        by_paper[item["paper_id"]].append(item)\n\n    papers=sorted(by_paper)\n    rng.shuffle(papers)\n    if len(papers)!=46:\n        raise RuntimeError(f"Expected the carrier\'s 46 papers, derived {len(papers)}")\n\n    # Iterate the shuffled paper list, never a set: set order depends on per-process string hashing.\n    test_papers=papers[:10]\n    val_papers=papers[10:17]\n    train_papers=papers[17:]\n\n    splits={\n        "test":[x for p in test_papers for x in by_paper[p]],\n        "validation":[x for p in val_papers for x in by_paper[p]],\n        "train":[x for p in train_papers for x in by_paper[p]],\n    }\n    expected={"test":21,"validation":23,"train":50}\n    if {k:len(v) for k,v in splits.items()}!=expected:\n        raise RuntimeError(f"Paper-disjoint split does not match the carrier: {({k:len(v) for k,v in splits.items()})}")\n'
  '    print({k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()})\n\n\n@stage("select")\ndef stage_select(args):\n    global canonical_tables\n    eligible_tables=[x for x in splits["test"] if eligible(x)]\n    eligible_tables.sort(key=lambda x:hashlib.sha256(f"42:{x[\'source_id\']}".encode()).hexdigest())\n    canonical_tables=eligible_tables[:END_TO_END_TABLES]\n    if len(canonical_tables)!=END_TO_END_TABLES:\n        # Spec §18: canonical tables come from the paper-disjoint test split only; never borrow train/validation tables.\n        raise RuntimeError(f"Only {len(canonical_tables)} QA-eligible test tables were derived")\n\n    print([x["source_id"] for x in canonical_tables])\n\n\n@stage("pages")\ndef stage_pages(args):\n    for item in canonical_tables:\n        item["page"],item["page_gt_box"]=make_page(item)\n        item["page_sha256"]=image_digest(item["page"])\n\n'
  '    print({"pages":len(canonical_tables),"page_sizes":[x["page"].size for x in canonical_tables[:3]]})\n\n\n@stage("detection")\ndef stage_detection(args):\n    global det_processor,det_model,det_load_seconds,det_params,det_times,det_peak\n    load_transformers()\n    t0=time.perf_counter()\n    det_processor,det_model=load_detection_model()\n    det_load_seconds=time.perf_counter()-t0\n    det_params=sum(p.numel() for p in det_model.parameters())\n\n    _ = detect_table(canonical_tables[0]["page"])\n    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()\n    det_times=[]\n    for i,item in enumerate(canonical_tables):\n        if torch.cuda.is_available():torch.cuda.synchronize()\n        t=time.perf_counter(); preds=detect_table(item["page"])\n        if torch.cuda.is_available():torch.cuda.synchronize()\n        det_times.append(time.perf_counter()-t)\n        item["detections"]=preds\n        item["selected_detection"]=preds[0] if preds else None\n'
  '    det_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None\n\n    print({"mean_latency":float(np.mean(det_times)),"detections":[len(x["detections"]) for x in canonical_tables]})\n\n    # The detector is no longer needed: unload it before the structure model (sequential loading, spec §67).\n    del det_model,det_processor\n    gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n\n\n@stage("detection-metrics")\ndef stage_detection_metrics(args):\n    global det_rows,det_metrics\n    det_pred_lists=[]\n    det_refs=[]\n    det_rows=[]\n    for item in canonical_tables:\n        preds=[{"label":"table","score":x["score"],"box":x["box"]} for x in item["detections"]]\n        det_pred_lists.append(preds)\n        det_refs.append([{"label":"table","box":item["page_gt_box"]}])\n        best=max([box_iou(x["box"],item["page_gt_box"]) for x in item["detections"]],default=0.0)\n        det_rows.append({\n'
  '            "table_id":item["source_id"],"predicted":bool(item["detections"]),\n            "best_iou":best,"hit50":best>=0.5,"hit75":best>=0.75,\n        })\n\n    det_metrics={\n        "ap50":ap_for_label(det_pred_lists,det_refs,"table",0.5),\n        "ap75":ap_for_label(det_pred_lists,det_refs,"table",0.75),\n        "hit50":float(np.mean([r["hit50"] for r in det_rows])),\n        "hit75":float(np.mean([r["hit75"] for r in det_rows])),\n        "mean_best_iou":float(np.mean([r["best_iou"] for r in det_rows])),\n        "median_best_iou":float(np.median([r["best_iou"] for r in det_rows])),\n    }\n    print(det_metrics)\n\n\n@stage("crops")\ndef stage_crops(args):\n    for item in canonical_tables:\n        gold_crop,gold_crop_box=padded_crop(item["page"],item["page_gt_box"])\n        item["gold_crop"]=gold_crop; item["gold_crop_box"]=gold_crop_box\n        item["gold_crop_objects"]=translate_gold_objects(item,gold_crop_box)\n\n        if item["selected_detection"] is not None:\n'
  '            det_crop,det_crop_box=padded_crop(item["page"],item["selected_detection"]["box"])\n            item["det_crop"]=det_crop; item["det_crop_box"]=det_crop_box\n            item["det_crop_objects"]=translate_gold_objects(item,det_crop_box)\n        else:\n            item["det_crop"]=None; item["det_crop_box"]=None; item["det_crop_objects"]=[]\n\n    # Trace one reference row box through the frames: table -> page -> gold crop / detected crop.\n    trace_item=next((x for x in canonical_tables if x["selected_detection"] is not None),canonical_tables[0])\n    trace_box=next(o["box"] for o in trace_item["objects"] if o["label"]=="table row")\n    trace_page=[trace_box[0]+120,trace_box[1]+160,trace_box[2]+120,trace_box[3]+160]\n    print({\n        "table":trace_item["source_id"],\n        "row_box_table_frame":[round(v,1) for v in trace_box],\n        "page_frame":[round(v,1) for v in trace_page],\n        "gold_crop_origin":trace_item["gold_crop_box"][:2],\n'
  '        "gold_crop_frame":[round(v,1) for v in to_crop_frame(trace_page,trace_item["gold_crop_box"])],\n        "detected_crop_origin":trace_item["det_crop_box"][:2] if trace_item["det_crop_box"] else None,\n        "detected_crop_frame":[round(v,1) for v in to_crop_frame(trace_page,trace_item["det_crop_box"])] if trace_item["det_crop_box"] else None,\n    })\n\n\n@stage("structure")\ndef stage_structure(args):\n    global struct_processor,struct_model,struct_load_seconds,struct_params,struct_times,struct_peak\n    load_transformers()\n    t0=time.perf_counter()\n    struct_processor,struct_model=load_structure_model()\n    struct_load_seconds=time.perf_counter()-t0\n    struct_params=sum(p.numel() for p in struct_model.parameters())\n\n    _ = recognize_structure(canonical_tables[0]["gold_crop"])\n    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()\n    struct_times=[]  # one entry per crop (gold or detected), synchronised like the detection timings\n\n'
  '    for item in canonical_tables:\n        item["gold_structure_raw"]=timed_structure(item["gold_crop"])\n        item["det_structure_raw"]=timed_structure(item["det_crop"]) if item["det_crop"] is not None else []\n    struct_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None\n    print({"mean_seconds_per_crop":float(np.mean(struct_times)),"crops":len(struct_times)})\n\n    del struct_model,struct_processor\n    gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n\n\n@stage("grid")\ndef stage_grid(args):\n    for item in canonical_tables:\n        item["gold_grid_pred"]=grid_from_structure(item["gold_structure_raw"])\n        item["det_grid_pred"]=grid_from_structure(item["det_structure_raw"]) if item["det_crop"] is not None else {"rows":[],"columns":[],"cells":[],"valid_geometry":False}\n\n\n@stage("reconstruct")\ndef stage_reconstruct(args):\n    for item in canonical_tables:\n'
  '        item["gold_recon"]=reconstruct_table(item,item["gold_grid_pred"],item["gold_crop_box"])\n        if item["det_crop"] is not None:\n            item["det_recon"]=reconstruct_table(item,item["det_grid_pred"],item["det_crop_box"])\n        else:\n            item["det_recon"]={"valid":False,"reason":"detection_miss","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}\n\n\n@stage("structure-metrics")\ndef stage_structure_metrics(args):\n    global structure_rows\n    structure_rows=[]\n    for path_name,pred_key,ref_key in [\n        ("gold_crop","gold_structure_raw","gold_crop_objects"),\n        ("detected_crop","det_structure_raw","det_crop_objects"),\n    ]:\n        for item in canonical_tables:\n            gp=item["gold_grid_pred"] if path_name=="gold_crop" else item["det_grid_pred"]\n            # A page without a detection has no crop to recognize: a failed grid on this path, never an exact one.\n            missed=path_name=="detected_crop" and item["det_crop"] is None\n'
  '            gold_rows=refs_for(item,ref_key,"table row")\n            gold_cols=refs_for(item,ref_key,"table column")\n            pred_rows=preds_for(item,pred_key,"table row")\n            pred_cols=preds_for(item,pred_key,"table column")\n            best_r=[max([box_iou(g["box"],p["box"]) for p in pred_rows],default=0) for g in gold_rows]\n            best_c=[max([box_iou(g["box"],p["box"]) for p in pred_cols],default=0) for g in gold_cols]\n            # Counts are compared with the table\'s own grid, not with the reference boxes left inside a (possibly clipped) crop.\n            rows_exact=not missed and len(gp["rows"])==item["n_rows"]\n            cols_exact=not missed and len(gp["columns"])==item["n_cols"]\n            structure_rows.append({\n                "table_id":item["source_id"],"path":path_name,\n                "status":"detection_miss" if missed else "ok",\n                "gold_rows":item["n_rows"],"predicted_rows":len(gp["rows"]),\n'
  '                "gold_columns":item["n_cols"],"predicted_columns":len(gp["columns"]),\n                "row_count_exact":rows_exact,\n                "column_count_exact":cols_exact,\n                "grid_shape_exact":rows_exact and cols_exact,\n                "mean_row_iou":float(np.mean(best_r)) if best_r else 0.0,\n                "mean_column_iou":float(np.mean(best_c)) if best_c else 0.0,\n            })\n\n    for path_name in ("gold_crop","detected_crop"):\n        subset=[r for r in structure_rows if r["path"]==path_name]\n        print(path_name,{\n            "grid_exact":float(np.mean([r["grid_shape_exact"] for r in subset])),\n            "mean_row_iou":float(np.mean([r["mean_row_iou"] for r in subset])),\n            "mean_column_iou":float(np.mean([r["mean_column_iou"] for r in subset])),\n        })\n\n\n@stage("activity")\ndef stage_activity(args):\n    global struct_processor,struct_model,activity_rows\n    load_transformers()\n'
  '    # Change one thing: the structure threshold, on validation tables only (canonical outputs are unaffected).\n    ACTIVITY_STRUCTURE_THRESHOLD = float(args.threshold)\n    if not 0 <= ACTIVITY_STRUCTURE_THRESHOLD <= 1:\n        raise ValueError("ACTIVITY_STRUCTURE_THRESHOLD must be in [0,1]")\n\n    # Each stage process loads the structure model only if it is absent, and unloads what it loaded.\n    activity_loaded_model="struct_model" not in globals()\n    if activity_loaded_model:\n        struct_processor,struct_model=load_structure_model()\n\n    activity_rows=[]\n    for item in splits["validation"]:\n        page,page_box=make_page(item)\n        crop,_=padded_crop(page,page_box)\n        activity_rows.append({\n            "table":item["source_id"],\n            "gold":(item["n_rows"],item["n_cols"]),\n            "at_reference":grid_counts(crop,STRUCTURE_THRESHOLD),\n            "at_changed":grid_counts(crop,ACTIVITY_STRUCTURE_THRESHOLD),\n        })\n\n'
  '    for name,threshold in (("at_reference",STRUCTURE_THRESHOLD),("at_changed",ACTIVITY_STRUCTURE_THRESHOLD)):\n        exact=sum(r[name]==r["gold"] for r in activity_rows)\n        print(f"threshold {threshold:.2f}: exact grids {exact} / {len(activity_rows)} validation tables")\n    changed=[r for r in activity_rows if r["at_reference"]!=r["at_changed"]]\n    print(f"Tables whose (rows, columns) changed: {len(changed)}")\n    for r in changed:\n        print(" ",r)\n\n    if activity_loaded_model:\n        del struct_model,struct_processor\n        gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n\n\n@stage("complex-probe")\ndef stage_complex_probe(args):\n    global struct_processor,struct_model,complex_tables\n    load_transformers()\n    complex_tables=[x for x in splits["test"] if x["has_spans"]][:COMPLEX_STRUCTURE_TABLES] if RUN_COMPLEX_STRUCTURE_PROBE else []\n    complex_dir=Path(OUTPUT_DIR)/"complex_probe"\n    complex_dir.mkdir(parents=True,exist_ok=True)\n'
  '    for old_panel in complex_dir.glob("*.png"):\n        old_panel.unlink()  # keep only this run\'s panels\n\n    if complex_tables and "struct_model" not in globals():\n        struct_processor,struct_model=load_structure_model()\n    for item in complex_tables:\n        page,page_box=make_page(item)\n        crop,crop_box=padded_crop(page,page_box)\n        gold_spans=[x for x in translate_gold_objects(item,crop_box) if x["label"]=="table spanning cell"]\n        raw=recognize_structure(crop)\n        grid=grid_from_structure(raw)\n        pred_spans=[x for x in raw if x["label"]=="table spanning cell"]\n        left=outline(crop,gold_spans,{"table spanning cell":"orange"})\n        right=outline(crop,grid["rows"]+grid["columns"]+pred_spans,PROBE_COLORS)\n        panel=Image.new("RGB",(left.width+right.width+20,left.height+60),"white")\n        panel.paste(left,(0,40)); panel.paste(right,(left.width+20,40))\n        d=ImageDraw.Draw(panel)\n'
  '        d.text((5,8),"gold spanning cells (orange)",fill="black")\n        d.text((left.width+25,8),"predicted rows / columns / spanning cells",fill="black")\n        panel.save(complex_dir/f"{item[\'source_id\'].replace(\'/\',\'_\')}.png")\n        print({\n            "table":item["source_id"],\n            "gold_grid":[item["n_rows"],item["n_cols"]],"predicted_grid":[len(grid["rows"]),len(grid["columns"])],\n            "gold_spanning_cells":len(gold_spans),"predicted_spanning_cells":len(pred_spans),\n        })\n    if complex_tables:\n        print("These tables are intentionally excluded from the simple rectangular QA reconstruction contract.")\n\n    # The structure recognizer is no longer needed: unload it before TAPAS (sequential loading, spec §67).\n    if "struct_model" in globals():\n        del struct_model,struct_processor\n    gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n\n\n@stage("questions")\ndef stage_questions(args):\n    global qa_records\n    qa_records=[]\n'
  '    for item in canonical_tables:\n        grid=item["grid"]; headers=grid[0]; body=grid[1:]\n        row_ids=[row[0] for row in body]\n        candidates=[]\n        for r in range(len(body)):\n            for c in range(1,len(headers)):\n                candidates.append((r,c))\n        candidates.sort(key=lambda rc:hashlib.sha256(f"{item[\'source_id\']}:{rc[0]}:{rc[1]}".encode()).hexdigest())\n        for j,(r,c) in enumerate(candidates[:2]):\n            qa_records.append({\n                "id":f"{item[\'source_id\']}-lookup-{j}",\n                "table_id":item["source_id"],\n                "question":f"What is the {headers[c]} for {row_ids[r]}?",\n                "category":"NONE",\n                "gold_aggregation":"NONE",\n                "gold_coordinates":[[r,c]],\n                "gold_denotation":[body[r][c]],\n            })\n        c,vals=numeric_columns(item)[0]\n        coords=[[r,c] for r in range(len(body))]\n        qa_records += [\n            {\n'
  '                "id":f"{item[\'source_id\']}-sum","table_id":item["source_id"],\n                "question":f"What is the total {headers[c]}?","category":"SUM",\n                "gold_aggregation":"SUM","gold_coordinates":coords,\n                "gold_denotation":float(sum(vals)),\n            },\n            {\n                "id":f"{item[\'source_id\']}-avg","table_id":item["source_id"],\n                "question":f"What is the average {headers[c]}?","category":"AVERAGE",\n                "gold_aggregation":"AVERAGE","gold_coordinates":coords,\n                "gold_denotation":float(sum(vals)/len(vals)),\n            },\n            {\n                "id":f"{item[\'source_id\']}-count","table_id":item["source_id"],\n                "question":f"How many {headers[c]} entries are listed?","category":"COUNT",\n                "gold_aggregation":"COUNT","gold_coordinates":coords,\n                "gold_denotation":float(len(vals)),\n            },\n        ]\n\n    if len(qa_records)!=50:\n'
  '        raise RuntimeError(f"Expected 50 questions, got {len(qa_records)}")\n    counts=defaultdict(int)\n    for r in qa_records:counts[r["category"]]+=1\n    if dict(counts)!={"NONE":20,"SUM":10,"AVERAGE":10,"COUNT":10}:\n        raise RuntimeError(f"Question mix mismatch: {dict(counts)}")\n    print(dict(counts))\n\n\n@stage("qa")\ndef stage_qa(args):\n    global gold_qa,gold_tapas_times,structure_qa,structure_tapas_times,end_to_end_qa,end_tapas_times,tapas_peak\n    load_transformers()\n    load_pandas()\n    load_tapas()\n    print({"parameters":tapas_params,"load_seconds":tapas_load_seconds})\n    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()  # TAPAS peak only, not the earlier stages\n    gold_qa,gold_tapas_times=score_path("gold_table",gold_provider)\n    structure_qa,structure_tapas_times=score_path("structure_only",struct_provider)\n    end_to_end_qa,end_tapas_times=score_path("end_to_end",full_provider)\n'
  '    tapas_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None\n\n    print("Gold:",qa_summary(gold_qa))\n    print("Structure-only:",qa_summary(structure_qa))\n    print("End-to-end:",qa_summary(end_to_end_qa))\n\n\n@stage("waterfall")\ndef stage_waterfall(args):\n    global waterfall\n    waterfall={\n        "tables":len(canonical_tables),\n        "unit":"detection, structure and reconstruction values count tables out of `tables`; qa gives accuracy and correct-answer counts",\n        "detection":{\n            "hit50":sum(r["hit50"] for r in det_rows),\n            "hit75":sum(r["hit75"] for r in det_rows),\n        },\n        "structure_gold_crop":{\n            "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="gold_crop"),\n        },\n        "structure_detected_crop":{\n            "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="detected_crop"),\n        },\n        "reconstruction":{\n'
  '            "valid_gold_crop":sum(x["gold_recon"]["valid"] for x in canonical_tables),\n            "valid_detected_crop":sum(x["det_recon"]["valid"] for x in canonical_tables),\n        },\n        "qa":{\n            "questions":len(qa_records),\n            "gold_table_correct":qa_summary(gold_qa)["correct"],\n            "structure_only_correct":qa_summary(structure_qa)["correct"],\n            "end_to_end_correct":qa_summary(end_to_end_qa)["correct"],\n            "gold_table_denotation_accuracy":qa_summary(gold_qa)["denotation_accuracy"],\n            "structure_only_denotation_accuracy":qa_summary(structure_qa)["denotation_accuracy"],\n            "end_to_end_denotation_accuracy":qa_summary(end_to_end_qa)["denotation_accuracy"],\n        },\n    }\n    print(json.dumps(waterfall,indent=2))\n\n\n@stage("checkpoint")\ndef stage_checkpoint(args):\n    for item in canonical_tables:\n        print({\n            "table":item["source_id"],\n'
  '            "detection_iou":next(r["best_iou"] for r in det_rows if r["table_id"]==item["source_id"]),\n            "gold_crop_valid":item["gold_recon"]["valid"],\n            "detected_crop_valid":item["det_recon"]["valid"],\n            "gold_shape":[item["n_rows"],item["n_cols"]],\n            "gold_crop_pred_shape":[len(item["gold_grid_pred"]["rows"]),len(item["gold_grid_pred"]["columns"])],\n            "det_crop_pred_shape":[len(item["det_grid_pred"]["rows"]),len(item["det_grid_pred"]["columns"])],\n            "cell_accuracy_gold_crop":item["gold_recon"].get("assignment_accuracy"),\n            "cell_accuracy_detected_crop":item["det_recon"].get("assignment_accuracy"),\n        })\n\n\n@stage("resources")\ndef stage_resources(args):\n    global resource_rows\n    resource_rows=[\n        {\n            "stage":"detection","model":"Table Transformer Detection",\n            "model_id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"],\n'
  '            "parameter_count":det_params,\n            "weight_bytes":next(x["bytes"] for x in DETECTION_MANIFEST["files"] if x["path"]=="model.safetensors"),\n            "load_seconds":det_load_seconds,"latency_unit":"per page","mean_latency_seconds":float(np.mean(det_times)),\n            "median_latency_seconds":median(det_times),"peak_gpu_memory_bytes":det_peak,\n        },\n        {\n            "stage":"structure","model":"Table Transformer Structure",\n            "model_id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"],\n            "parameter_count":struct_params,\n            "weight_bytes":next(x["bytes"] for x in STRUCTURE_MANIFEST["files"] if x["path"]=="model.safetensors"),\n            "load_seconds":struct_load_seconds,"latency_unit":"per crop","mean_latency_seconds":float(np.mean(struct_times)),\n            "median_latency_seconds":median(struct_times),"peak_gpu_memory_bytes":struct_peak,\n        },\n        {\n'
  '            "stage":"qa","model":"TAPAS Large WTQ",\n            "model_id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"],\n            "parameter_count":tapas_params,\n            "weight_bytes":next(x["bytes"] for x in TAPAS_MANIFEST["files"] if x["path"]=="model.safetensors"),\n            "load_seconds":tapas_load_seconds,"latency_unit":"per question",\n            "mean_latency_seconds":float(np.mean(gold_tapas_times+structure_tapas_times+end_tapas_times)),\n            "median_latency_seconds":median(gold_tapas_times+structure_tapas_times+end_tapas_times),\n            "peak_gpu_memory_bytes":tapas_peak,\n        },\n    ]\n    for row in resource_rows:print(row)\n\n\n@stage("panels")\ndef stage_panels(args):\n    examples_dir=Path(OUTPUT_DIR)/"examples"\n    examples_dir.mkdir(parents=True,exist_ok=True)\n\n    for item in canonical_tables:\n        page=draw_boxes(item["page"],item["detections"][:3])\n'
  '        crop=item["det_crop"] if item["det_crop"] is not None else item["gold_crop"]\n        struct=item["det_structure_raw"] if item["det_crop"] is not None else item["gold_structure_raw"]\n        crop_vis=draw_boxes(crop,[x for x in struct if x["label"] in ("table row","table column","table spanning cell")])\n        page.thumbnail((700,700)); crop_vis.thumbnail((700,700))\n        recon=item["det_recon"] if item["det_crop"] is not None else item["gold_recon"]\n        recon_lines=[" | ".join(row) for row in (recon["table"] or [])[:3]] or [f"(no table: {recon[\'reason\']})"]\n        canvas=Image.new("RGB",(max(page.width,crop_vis.width),page.height+crop_vis.height+120+30*len(recon_lines)),"white")\n        canvas.paste(page,(0,0)); canvas.paste(crop_vis,(0,page.height))\n        q=next(r for r in end_to_end_qa if r["table_id"]==item["source_id"])\n        d=ImageDraw.Draw(canvas)\n        y=page.height+crop_vis.height+8\n        for line in recon_lines:\n'
  '            d.text((5,y),f"Reconstructed: {line[:110]}",fill="black"); y+=30\n        d.text((5,y),f"Q: {q[\'question\']}",fill="black")\n        d.text((5,y+30),f"Gold: {q[\'gold_denotation\']}",fill="black")\n        d.text((5,y+60),f"Path status: {q[\'pipeline_status\']}  TAPAS: {q[\'predicted_denotation\']}  aggregation: {q[\'predicted_aggregation\']}  cells: {q[\'selected_cells\'][:4]}",fill="black")\n        path=examples_dir/f"{item[\'source_id\'].replace(\'/\',\'_\')}.png"\n        canvas.save(path)\n        print(path)\n\n\n@stage("exports")\ndef stage_exports(args):\n    out_dir=Path(OUTPUT_DIR)\n    out_dir.mkdir(parents=True,exist_ok=True)\n    tables_dir=out_dir/"tables"\n    tables_dir.mkdir(exist_ok=True)\n\n    write_csv(out_dir/"table_detection.csv",det_rows,["table_id","predicted","best_iou","hit50","hit75"])\n    write_csv(\n        out_dir/"structure_metrics.csv",structure_rows,\n        ["table_id","path","status","gold_rows","predicted_rows","gold_columns","predicted_columns",\n'
  '         "row_count_exact","column_count_exact","grid_shape_exact","mean_row_iou","mean_column_iou"]\n    )\n\n    recon_rows=[]\n    for item in canonical_tables:\n        for path_name,key in [("gold_crop","gold_recon"),("detected_crop","det_recon")]:\n            r=item[key]\n            recon_rows.append({\n                "table_id":item["source_id"],"path":path_name,\n                "gold_rows":item["n_rows"],"gold_columns":item["n_cols"],\n                "predicted_rows":len(item["gold_grid_pred"]["rows"] if path_name=="gold_crop" else item["det_grid_pred"]["rows"]),\n                "predicted_columns":len(item["gold_grid_pred"]["columns"] if path_name=="gold_crop" else item["det_grid_pred"]["columns"]),\n                "mapped_cells":r.get("mapped",0),"unmapped_cells":r.get("unmapped",0),\n                "ambiguous_cells":r.get("ambiguous",0),"cell_assignment_accuracy":r.get("assignment_accuracy"),\n'
  '                "header_exact":r.get("headers_valid",False),"valid_for_tapas":r.get("valid",False),\n            })\n            if r.get("table") is not None:\n                (tables_dir/f"{item[\'source_id\'].replace(\'/\',\'_\')}_{path_name}.json").write_text(\n                    json.dumps({"table":r["table"],"findings":r["reason"]},indent=2),encoding="utf-8"\n                )\n        (tables_dir/f"{item[\'source_id\'].replace(\'/\',\'_\')}_gold.json").write_text(\n            json.dumps({"table":item["grid"]},indent=2),encoding="utf-8"\n        )\n    write_csv(\n        out_dir/"reconstruction_metrics.csv",recon_rows,\n        ["table_id","path","gold_rows","gold_columns","predicted_rows","predicted_columns","mapped_cells",\n         "unmapped_cells","ambiguous_cells","cell_assignment_accuracy","header_exact","valid_for_tapas"]\n    )\n\n    with open(out_dir/"qa_questions.jsonl","w",encoding="utf-8") as f:\n        for r in qa_records:f.write(json.dumps(r,ensure_ascii=False)+"\\n")\n\n'
  '    qa_all=gold_qa+structure_qa+end_to_end_qa\n    write_csv(\n        out_dir/"qa_predictions.csv",qa_all,\n        ["question_id","table_id","path","question","gold_aggregation","predicted_aggregation",\n         "gold_denotation","predicted_denotation","denotation_correct","aggregation_correct",\n         "selected_cells","coordinates","pipeline_status","latency_seconds"]\n    )\n    write_csv(\n        out_dir/"resource_metrics.csv",resource_rows,\n        ["stage","model","model_id","revision","parameter_count","weight_bytes","load_seconds",\n         "latency_unit","mean_latency_seconds","median_latency_seconds","peak_gpu_memory_bytes"]\n    )\n    (out_dir/"waterfall.json").write_text(json.dumps(waterfall,indent=2),encoding="utf-8")\n\n    input_manifest={\n        "source_tables":SOURCE_TABLES,\n        "derived_tables":len(derived),\n        "paper_split":{k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()},\n'
  '        "canonical_table_ids":[x["source_id"] for x in canonical_tables],\n        "carrier_geometry_sha256":CARRIER_GEOMETRY_SHA256,\n        "pages":[\n            {"table_id":x["source_id"],"source_pixel_sha256":x["pixel_sha256"],\n             "page_sha256":x["page_sha256"],"placement":x["page_gt_box"]}\n            for x in canonical_tables\n        ],\n        "complex_probe_ids":[x["source_id"] for x in complex_tables],\n    }\n    (out_dir/"input_manifest.json").write_text(json.dumps(input_manifest,indent=2),encoding="utf-8")\n\n    provenance={\n        "created_utc":datetime.now(timezone.utc).isoformat(),\n        "notebook_spec":"2.1","profile":"TASK-INFERENCE","pedagogical_mode":"WORKSHOP",\n        "dataset":{\n            "id":SCITSR_REPO,"revision":SCITSR_REVISION,"license":"CC0-1.0",\n            "source_files":SCITSR_FILES,"split_seed":42,\n            "derivation":"pinned DIMER carrier geometry (sha256 recorded in input_manifest.json) plus cell text matched to SciTSR text chunks",\n'
  '        },\n        "models":{\n            "detection":{"id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"]},\n            "structure":{"id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"]},\n            "tapas":{"id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"]},\n        },\n        "thresholds":{\n            "detection":DETECTION_THRESHOLD,"structure":STRUCTURE_THRESHOLD,"grid_nms_iou":GRID_NMS_IOU\n        },\n        "runtime":RUNTIME,\n    }\n    (out_dir/"provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")\n\n    print("Outputs:")\n    for p in sorted(out_dir.iterdir()):print(" ",p)\n\n\n@stage("byod")\ndef stage_byod(args):\n    global det_processor,det_model,struct_processor,struct_model,byod_result\n    load_transformers()\n    load_pandas()\n    byod_result=None\n    if USE_BYOD:\n        if not str(BYOD_PATH).strip():\n'
  '            raise ValueError("USE_BYOD is True but BYOD_PATH is empty: set BYOD_PATH in §3 to the folder that holds pages/ and ocr.csv")\n        root=Path(BYOD_PATH)\n        pages_dir=root/"pages"\n        ocr_path=root/"ocr.csv"\n        questions_path=root/"questions.csv"\n        absent=[name for name,ok in (("pages/",pages_dir.is_dir()),("ocr.csv",ocr_path.is_file())) if not ok]\n        if absent:\n            raise FileNotFoundError(f"BYOD_PATH {str(root)!r} is missing {absent}; use the layout shown in §28")\n\n        # Validate every row before any model runs.\n        ocr_by_page=defaultdict(list)\n        for line,row in read_byod_csv(ocr_path,{"page_id","text","x0","y0","x1","y1","order"}):\n            page_id=str(row["page_id"] or "").strip()\n            if not page_id:\n                raise ValueError(f"ocr.csv line {line}: page_id is empty")\n            try:\n                box=[float(row[k]) for k in ("x0","y0","x1","y1")]\n            except (TypeError,ValueError):\n'
  '                raise ValueError(\n                    f"ocr.csv line {line}: x0,y0,x1,y1 must be numbers in page-image pixels, got "\n                    f"{[row[k] for k in (\'x0\',\'y0\',\'x1\',\'y1\')]}"\n                ) from None\n            if not all(math.isfinite(v) for v in box) or box[0]>box[2] or box[1]>box[3]:\n                raise ValueError(f"ocr.csv line {line}: box {box} needs finite values with x0<=x1 and y0<=y1 (top-left origin)")\n            try:\n                order=int(str(row["order"]).strip())\n            except ValueError:\n                raise ValueError(f"ocr.csv line {line}: order must be a whole number, got {row[\'order\']!r}") from None\n            ocr_by_page[page_id].append({"text":str(row["text"] or ""),"box":box,"order":order,"line":line})\n        if not ocr_by_page:\n            raise ValueError("ocr.csv has a header but no rows")\n        for page_id in ocr_by_page:\n            ocr_by_page[page_id].sort(key=lambda x:x["order"])\n\n'
  '        q_by_page=defaultdict(list)\n        if questions_path.is_file():\n            for line,row in read_byod_csv(questions_path,{"id","page_id","question"}):\n                if not str(row["question"] or "").strip() or not str(row["page_id"] or "").strip():\n                    raise ValueError(f"questions.csv line {line}: page_id and question must be non-empty")\n                q_by_page[str(row["page_id"]).strip()].append(row)\n\n        byod_skipped=[]\n        byod_inputs=[]\n        for image_path in sorted(pages_dir.iterdir()):\n            if not image_path.is_file():\n                continue\n            if image_path.suffix.lower() not in BYOD_IMAGE_SUFFIXES:\n                byod_skipped.append({"file":image_path.name,"reason":"unsupported_extension"})\n            elif image_path.stem not in ocr_by_page:\n                byod_skipped.append({"file":image_path.name,"reason":"no_ocr_rows"})\n            else:\n                byod_inputs.append(image_path)\n'
  '        ocr_without_image=sorted(set(ocr_by_page)-{p.stem for p in byod_inputs})\n        if not byod_inputs:\n            raise ValueError(\n                f"No page in {pages_dir} has OCR rows. Skipped: {byod_skipped}. "\n                f"ocr.csv page_ids: {sorted(ocr_by_page)[:10]}. A page_id must equal its image file name without the extension."\n            )\n\n        byod_images={}\n        for image_path in byod_inputs:\n            image=Image.open(image_path); image.load(); image=image.convert("RGB")\n            if min(image.size)<16 or max(image.size)>4096:\n                raise ValueError(f"{image_path.name}: page is {image.width}x{image.height}; each side must be 16..4096 px")\n            outside=[w["line"] for w in ocr_by_page[image_path.stem]\n                     if w["box"][0]<-1 or w["box"][1]<-1 or w["box"][2]>image.width+1 or w["box"][3]>image.height+1]\n            if outside:\n                raise ValueError(\n'
  '                    f"ocr.csv lines {outside[:5]}: boxes for {image_path.stem} extend beyond its {image.width}x{image.height} image. "\n                    "Boxes must be page-image pixels with a top-left origin; convert PDF points or a bottom-left origin first"\n                )\n            byod_images[image_path.stem]=image\n\n        # Detection, then structure: loaded one at a time with the same loaders as the sample path.\n        det_processor,det_model=load_detection_model()\n        byod_pages=[]\n        for page_id,image in byod_images.items():\n            detections=detect_table(image)\n            byod_pages.append({"page_id":page_id,"image":image,"detection":detections[0] if detections else None})\n        del det_model,det_processor\n        gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n\n        struct_processor,struct_model=load_structure_model()\n        if any(q_by_page.values()) and "tapas_model" not in globals():\n            load_tapas()\n'
  '        byod_outputs=[]\n        for page in byod_pages:\n            record={"page_id":page["page_id"],"detection":page["detection"],"table":None,"answers":[],\n                    "words":len(ocr_by_page[page["page_id"]]),"words_assigned":0}\n            if page["detection"] is None:\n                byod_outputs.append({**record,"status":"detection_miss"})\n                continue\n            crop,crop_box=padded_crop(page["image"],page["detection"]["box"])\n            grid=grid_from_structure(recognize_structure(crop))\n            if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:\n                byod_outputs.append({**record,"status":"grid_invalid"})\n                continue\n\n            # Assign OCR word centers into predicted row/column cells (crop frame = page frame - crop origin).\n            strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]\n            cx0,cy0,_,_=crop_box\n            for word in ocr_by_page[page["page_id"]]:\n'
  '                x=(word["box"][0]+word["box"][2])/2-cx0\n                y=(word["box"][1]+word["box"][3])/2-cy0\n                ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]\n                cis=[i for i,c in enumerate(grid["columns"]) if contains(c["box"],x,y)]\n                if len(ris)==1 and len(cis)==1:\n                    strings[ris[0]][cis[0]].append((word["order"],word["text"]))\n                    record["words_assigned"]+=1\n            table=[[" ".join(t for _o,t in sorted(cell)).strip() for cell in row] for row in strings]\n            record["table"]=table\n            headers=[norm_text(x) for x in table[0]] if table else []\n            valid=bool(table and all(headers) and len(set(headers))==len(headers) and len(table)<=64 and len(table[0])<=32)\n            if not valid:\n                byod_outputs.append({**record,"status":"header_or_limit_invalid"})\n                continue\n            for q in q_by_page.get(page["page_id"],[]):\n'
  '                result=tapas_answer(table,q["question"])\n                row={"id":q["id"],"question":q["question"],"answer":result["answer"],"aggregation":result["aggregation"],\n                     "cells":result["cells"],"numeric_answer":result["numeric_answer"]}\n                accepted=(q.get("accepted_answer") or "").strip()\n                if accepted:\n                    row["accepted_answer"]=accepted\n                    row["accepted_answer_match"]=accepted_answer_matches(result,accepted)\n                record["answers"].append(row)\n            byod_outputs.append({**record,"status":"ok"})\n        del struct_model,struct_processor\n        gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n\n        # Show and export this run\'s BYOD results; earlier BYOD outputs are removed first.\n        byod_dir=Path(OUTPUT_DIR)/"byod"\n        if byod_dir.exists():\n            shutil.rmtree(byod_dir)\n        (byod_dir/"tables").mkdir(parents=True)\n'
  '        for out in byod_outputs:\n            print(f"\\n[{out[\'page_id\']}] status={out[\'status\']}  OCR words assigned to cells: {out[\'words_assigned\']}/{out[\'words\']}")\n            if out["table"] is not None:\n                with open(byod_dir/"tables"/f"{out[\'page_id\']}.csv","w",encoding="utf-8",newline="") as f:\n                    csv.writer(f).writerows(out["table"])\n                header=out["table"][0]\n                columns=header if len(set(header))==len(header) else [f"{i}:{h}" for i,h in enumerate(header)]\n                print(pd.DataFrame(out["table"][1:],columns=columns).to_string(index=False))\n            for a in out["answers"]:\n                verdict=f"  (accepted {a[\'accepted_answer\']!r}: {\'match\' if a[\'accepted_answer_match\'] else \'no match\'})" if "accepted_answer" in a else ""\n                print(f"  Q {a[\'id\']}: {a[\'question\']}\\n    -> {a[\'answer\']}{verdict}")\n        byod_result={\n'
  '            "byod_path":str(root),"pages":len(byod_outputs),"skipped":byod_skipped,\n            "ocr_page_ids_without_image":ocr_without_image,\n            "thresholds":{"detection":DETECTION_THRESHOLD,"structure":STRUCTURE_THRESHOLD,"grid_nms_iou":GRID_NMS_IOU},\n            "outputs":byod_outputs,\n        }\n        (byod_dir/"byod_results.json").write_text(json.dumps(byod_result,indent=2,ensure_ascii=False,default=str),encoding="utf-8")\n        print({"byod_pages":len(byod_outputs),"status_counts":dict(Counter(x["status"] for x in byod_outputs)),\n               "skipped":byod_skipped,"outputs":str(byod_dir)})\n    else:\n        print("BYOD disabled on canonical Run all path.")\n\n\n@stage("summary")\ndef stage_summary(args):\n    out_dir=Path(OUTPUT_DIR)\n    required=[\n        out_dir/"table_detection.csv",\n        out_dir/"structure_metrics.csv",\n        out_dir/"reconstruction_metrics.csv",\n        out_dir/"qa_questions.jsonl",\n        out_dir/"qa_predictions.csv",\n'
  '        out_dir/"waterfall.json",\n        out_dir/"resource_metrics.csv",\n        out_dir/"input_manifest.json",\n        out_dir/"provenance.json",\n    ]\n    missing=[str(p) for p in required if not p.is_file()]\n    if missing:raise RuntimeError(f"Required outputs missing: {missing}")\n\n    print("DIMER Table Intelligence Workshop")\n    print("-"*34)\n    print(f"Canonical tables: {len(canonical_tables)}")\n    print(f"Questions: {len(qa_records)}")\n    print()\n    print("Stage 1 — Detection")\n    print(f"  AP50: {det_metrics[\'ap50\']:.3f}")\n    print(f"  hit@0.50: {det_metrics[\'hit50\']:.3f}")\n    print(f"  mean IoU: {det_metrics[\'mean_best_iou\']:.3f}")\n    print()\n    print("Stage 2 — Structure")\n    print(f"  gold-crop exact grids: {waterfall[\'structure_gold_crop\'][\'grid_exact\']} / {len(canonical_tables)}")\n    print(f"  detected-crop exact grids (a detection miss is not exact): {waterfall[\'structure_detected_crop\'][\'grid_exact\']} / {len(canonical_tables)}")\n    print()\n'
  '    print("Stage 3 — Reconstruction")\n    print(f"  valid gold-crop tables: {waterfall[\'reconstruction\'][\'valid_gold_crop\']} / {len(canonical_tables)}")\n    print(f"  valid detected tables: {waterfall[\'reconstruction\'][\'valid_detected_crop\']} / {len(canonical_tables)}")\n    print()\n    print("Stage 4 — TAPAS")\n    print(f"  gold-table denotation: {waterfall[\'qa\'][\'gold_table_denotation_accuracy\']:.3f}")\n    print(f"  structure-only denotation: {waterfall[\'qa\'][\'structure_only_denotation_accuracy\']:.3f}")\n    print(f"  end-to-end denotation: {waterfall[\'qa\'][\'end_to_end_denotation_accuracy\']:.3f}")\n    print(f"  correct answers of {len(qa_records)}: gold {waterfall[\'qa\'][\'gold_table_correct\']}, "\n          f"structure-only {waterfall[\'qa\'][\'structure_only_correct\']}, end-to-end {waterfall[\'qa\'][\'end_to_end_correct\']}")\n    if globals().get("byod_result") is not None:\n        print()\n'
  '        print(f"BYOD: {byod_result[\'pages\']} page(s) processed, {len(byod_result[\'skipped\'])} skipped; see {out_dir}/byod/")\n    print()\n    print(f"Outputs: {out_dir}/")\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument("--config",required=True)\n    parser.add_argument("--stage",required=True,choices=sorted(STAGES))\n    parser.add_argument("--threshold",default=None,type=float)\n    args=parser.parse_args(argv)\n    configure(json.loads(Path(args.config).read_text(encoding="utf-8")))\n    STAGES[args.stage](args)\n    return 0\n\n\nif __name__=="__main__":\n    sys.exit(main())\n'
 ),
 'requirements.lock.txt': (
  '# Hash lock of the isolated environment of tutorials/DIMER_Table_Intelligence_Workshop.ipynb.\n# Target: CPython 3.12.12 on Linux x86_64 (manylinux_2_28), wheels only, installed with\n#   uv pip install --require-hashes --only-binary :all: -r requirements.lock.txt\n# Direct pins (table-intelligence-workshop-requirements.in) are the versions the notebook pinned for its former\n# in-kernel install: torch 2.14.0, torchvision 0.29.0, torchaudio 2.11.0, transformers 4.57.6, timm 1.0.30,\n# safetensors 0.8.0, numpy 2.5.3, pillow 11.3.0, huggingface-hub 0.36.2, datasets 4.1.1, pyarrow 25.0.1,\n# pandas 2.2.3. No version changed; every package resolves to a cp312 manylinux or pure-Python wheel.\n# Regenerate from tools/ with the uv command below, then run build_table_intelligence_workshop.py.\n# This file was autogenerated by uv via the following command:\n'
  '#    uv pip compile table-intelligence-workshop-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: --output-file table-intelligence-workshop-requirements.lock\naiohappyeyeballs==2.7.1 \\\n    --hash=sha256:065665c041c42a5938ed220bdcd7230f22527fbec085e1853d2402c8a3615d9d \\\n    --hash=sha256:9243213661e29250eb41368e5daa826fc017156c3b8a11440826b2e3ed376472\n    # via aiohttp\naiohttp==3.14.3 \\\n    --hash=sha256:03cd2bde3d7f085b64e549c985f4bb928cad7e8ecf5323bfca320db548d81b39 \\\n    --hash=sha256:041badb8f84396357c4d3ad26de6afd7a32b112f43d3c63045c0c8278cfd2043 \\\n    --hash=sha256:0a5ff2dfbb9ce645fa5b8ef3e02c6c0b9cc3f6030ff863d0c51fffc50cb5541b \\\n    --hash=sha256:0fdea2281997af69da84c77ffa6f5938a0285f21fb3887c249d67419ca865b3d \\\n    --hash=sha256:11fb37ef075669eee52ab1928fbf6e1741fada40409fa309ebde9607a962aebf \\\n    --hash=sha256:134ac5ddcf61c6fad984b9a5727d83492ada43d63471db20fb73042c13fca62f \\\n'
  '    --hash=sha256:152516815ef926786a0b6ae2b8f1fd2e0c71582dee0b435636865316fd4891b7 \\\n    --hash=sha256:1576145bdceeb92382d899751e12743a3a5b8e460a841e3e50543859e54864dc \\\n    --hash=sha256:16100ad3ab8d649fdfbee87602d9d2dcdca9df0b9eda8a1b5fdc0d41f96da559 \\\n    --hash=sha256:16ea7e24c309fb7c0bbd505d149abe4fe4dccfb8db911db7dbec0921bc889a6f \\\n    --hash=sha256:18c441d0a8fca6de8d1f546849b9f0ab20d435993e2c5b59562b2fae6be2f929 \\\n    --hash=sha256:18cb43369747b2ae007bd2655fb8e63a099c2ff1d207962943636dac989b3147 \\\n    --hash=sha256:1b59533861b70a2185c8f4f350f791f39d64358ef6944ce71c5240c9ec0982c9 \\\n    --hash=sha256:1c5281acc88b92396f88c7e1e2748f8466689df22b80170e4f51efa712fb47a8 \\\n    --hash=sha256:1c5ec8fb1bcc31a8466f74aaf26c345d5c386fa4bd08a3f0eb9c7a4a3fe8b5bf \\\n    --hash=sha256:1caa7b0d05f3e3a36f87788c59e970a7ee1cefcfcbb924a9f138c4a6551c9cb7 \\\n    --hash=sha256:21c016079415ed3fd676963e9793700a566d85dbbd6bfc564b9b2d209147dcc8 \\\n'
  '    --hash=sha256:2498f0fe69ead802f9675beca44a7c21c62fdaa4ec5145ea1c3ad6edbee29f85 \\\n    --hash=sha256:25bd2708db6bdf6a6630dd37bdcdfcb47c4434d22ac69c64665b802910140b30 \\\n    --hash=sha256:270d3dace9ca2f10f0da5d8ebe519b7a310fc6112ed916e32df5866df0888553 \\\n    --hash=sha256:2e1161602f45a54de2ce0905243a95f58cb42dcd378402f3697f5e0b21e9d2e7 \\\n    --hash=sha256:2e9878ae68e4a5f1c0abe4dd497dbc3d51946f5837b56759e2a02e78fa90ef86 \\\n    --hash=sha256:30402d03a7c0ff52bce290b57e564e9079fd9d0cb545c8aba73f86a103162d2e \\\n    --hash=sha256:33a2d7c28d33797a2e99923dffa63f83d908a19b6bf26cfe80fa790aa5e1a75a \\\n    --hash=sha256:362a3fd481769cac1a824514bcd86fda51c65e8fe6e051099e008fddde6db17c \\\n    --hash=sha256:38901a84da3ce22249f6e860bf8f90d141bcab7da090cc398f8bb58c0e44b7da \\\n    --hash=sha256:39aded8c7f3b935b54aab1d8d73c70ec0ee2d3ec3b943e0e86611bc150ba47f5 \\\n    --hash=sha256:3a26434dafe408229ff3403458ca58de24fb51936504decac49ce6755f77e59d \\\n'
  '    --hash=sha256:3ae5b3a59436d089b5395d910121a390feed4d00578eb95a0fd1a329fe963100 \\\n    --hash=sha256:3d4f72af88ac2474bb5bca640030320e3d38a0163a1d7533500e87be458eef71 \\\n    --hash=sha256:3f42e9b78301f11c8f861746175d8b9c1ccef713fcad9eab396e2f6db8ed4a22 \\\n    --hash=sha256:42a67efc36300d052fb4508a53e8b6901b9284b599ae63945c377569c5fcc1e1 \\\n    --hash=sha256:48d67b87db6279c044760787eb01f6413032c2e6f3ba1cafaa492b1c8e578479 \\\n    --hash=sha256:498c6c623134f8e09a3c4e60bcd607a0b4590dd7dbf08dd40851b27cbb520ccb \\\n    --hash=sha256:49f7325beb0f85ef4aef5f48f490269575f83e6e2acad00a1d80b807eb027062 \\\n    --hash=sha256:4e3ac92d90e92773b2362d506068e9a948192bd553e743c5b2429e28527c8661 \\\n    --hash=sha256:530125ee1163c4219af35dc3aa1206e541e7b31b6efc1a3f93b70a136f65d427 \\\n    --hash=sha256:5373dc80ad1aa2fb9ad95c83f24eef418bbda3a61375f128e5b0192e4f3f9b32 \\\n    --hash=sha256:53e5179d8abb5710f8e83ba207c41c8d1261fcffd4616500e15ca2b7a33be10a \\\n'
  '    --hash=sha256:53e7b4ce82b54a8bcc71b3b67a5cbd177ca1d7f592cbc92cd38b7349f73482db \\\n    --hash=sha256:543906c127fb1d929b95076db19b83fa2d46751006ff1e23b093aa5ac4d8db42 \\\n    --hash=sha256:54cfcdee2770dac994417cbb0ee1f3eb0e7cb6b30c79bf44f2c02ff79ec5124a \\\n    --hash=sha256:55bdcc472aafe2de4a253045cc128007a64f1e0264fb675791e132ea5edaa3bd \\\n    --hash=sha256:56f355e79f71aef2a85c80305cc915f894b170dba76de5fe84f6351939b83c06 \\\n    --hash=sha256:5895ef58c4620afe02fa16044f023dc4dafec08158f9d08874a46a7dbc0341b8 \\\n    --hash=sha256:5bcb6ff3fdab1258a192679ff1a05d44f59626430aa05cd1a9d2447423599228 \\\n    --hash=sha256:5f08ec777f35ee70720233b8b9811d3bb5d728137f30ac91b7457709c3261ac0 \\\n    --hash=sha256:614c61d478b83953e261d02bb2df750f17227cd33ef8002945bf5aebbde21919 \\\n    --hash=sha256:617105e2c3018ee38d0c8ce5ee3c84f621a6d8b9f723202aacaff28449ca91ee \\\n    --hash=sha256:6debfa7312ff9d4c124dc71d72e9a0a4b9e0879e48ba6fcb42bef5c3300289e2 \\\n'
  '    --hash=sha256:7041d52c3a7fa20c9e8c182b534704abb19502c8bdcbde7ab23bfda6f642394f \\\n    --hash=sha256:70c987b27534f9ae1a723f47ae921571d616da21d3208282bf4c52af5164ac43 \\\n    --hash=sha256:74ab5b6a9fb13e873e5a90946588baecaf488745e1db1a4a5c433f971f035098 \\\n    --hash=sha256:78253b573e6ffab5028924fc98bc281aae05445969982a10864bc360dea2016c \\\n    --hash=sha256:7a75aa63cbf9b21cfaf60dc2657e19df2c2867d91707d653fee171ffeedd1371 \\\n    --hash=sha256:8800c996b01c2772a783e3e46f3e1abd5823029adca0df54231960de9bfefa5b \\\n    --hash=sha256:89176250f686cb9853c0fb7ead90e639e915b84a6f43eedc2a4e7ec21f1037f0 \\\n    --hash=sha256:8a5fd34f7f7410d1730d5c2ba873cacb2eed3fede366feb268a70ba22581ed8f \\\n    --hash=sha256:8b3b60de05f3dcb6f6a00f818bb2ec781cee4de0645f59ccaf99b1d1823b6100 \\\n    --hash=sha256:8f2f1c4c032c7cedd7d8da6f54c97b70266c6570c3108d3fdffee7188bb70529 \\\n    --hash=sha256:9491196535a88924a60afd5b5f434b5b203b6cc616250878dbdb223a8f7844bc \\\n'
  '    --hash=sha256:9aa6e61fdf20105c4144e755bd586008ff450791d67b1c8146fdc15959c4d51c \\\n    --hash=sha256:9d9edccfe496b476db5f398d97b865e9a6752bcf8aec4eef8390ce20fb64bb41 \\\n    --hash=sha256:9fc7b5bfec6573f3ae844f457fdde5adeb713f8b8e4a81ad64fc207b49383716 \\\n    --hash=sha256:a0dc483c00da8b673abbb367eb6f8d8f4bcec30eb58529ea13cb42e7fd2dfa33 \\\n    --hash=sha256:a3a8296e7ab5c295f53f1041487cb088e1480775aafbf7fe545d93b770a0f96f \\\n    --hash=sha256:a3e22975f905b89a55a488c2a08f2fdb2186175349e917d48985cc468a3d4c6e \\\n    --hash=sha256:a4af35c443e0b1a1bd6a8af3f3485d7fda15c142751a00f3ff8090f0b93346fa \\\n    --hash=sha256:a94dbaae5ae27bd849c93570669bff91e0510f33a80805738e3de72a7be0447b \\\n    --hash=sha256:ac74facc01463f138b0da5580329cfcc82818dea5656e83ddcd11268fc12ff80 \\\n    --hash=sha256:ad4c8b7488d745d2ca4838ebd8ae5ba9b56341d30b1da43640e4ce87f9f49646 \\\n    --hash=sha256:b014a6ed7cf912e787149fdc529166d3ceabac23f26efeea3158c9aba2354e7e \\\n'
  '    --hash=sha256:b20032766aedf6261c7a566585a40867d092ac03a0d81592d5370ef9b054f99b \\\n    --hash=sha256:b2466434105a4e03113c36ec775cc2ebe6676b62eae326fa670bb607ef788c1c \\\n    --hash=sha256:b304db572b4368edd8dda8a2274f73156fe15558fca4a917cb8a09fc47af5963 \\\n    --hash=sha256:ba59d59aba08ac02fc03b0c8983ccd5ee39a199d0552ce9e6d2b4845b34d59ae \\\n    --hash=sha256:bd52f811e65f6fb634b1047159657c98f52b407f8efec907bcfc09da9a4c0a25 \\\n    --hash=sha256:bdd0e2834dce1a26c1bbe26464861e16bbe217042cbff619247c11594472518c \\\n    --hash=sha256:c23ec8ee9d5ab2f5421f9c7fffce208435607af27fd46d4a44e031954352838f \\\n    --hash=sha256:c39846c3aad97a8530c89d7a3869a8f8e9e3762c6ac0504481e5c80948f7e807 \\\n    --hash=sha256:c3c200cf9757edd785051dc699c7ecbec22110dbfcb3fefc7a9f9695eda8ea7a \\\n    --hash=sha256:c7d3a97c678d34fc5b59da671ee9cd630096ddc643e7b5a30d54a2a6f3574d3f \\\n    --hash=sha256:c8653fd547c93a61aadc612007790f5555cdd18946fa48cf45e26d8ea4ea473d \\\n'
  '    --hash=sha256:cc7cb243a68167172f48c1fd43cee91ec4b1d40cefd190edd43369d1a6bc9c82 \\\n    --hash=sha256:ccd4893707b3e2a13e39c90d43cf80edf2e4d0457935bcc103bf2346214c3f15 \\\n    --hash=sha256:cd817772b2fcf2b8c0905795318485f9ec16eae60b29feb7f4c77085311637f0 \\\n    --hash=sha256:cda5fd5c95ad7a125a2e8464acc78b98b94c475a3780d6aa0aa157c93f470f4d \\\n    --hash=sha256:cef89a58e628c4efcac3275c2d68083f82426dcdc89c1492a6f654f9f7ea6ab9 \\\n    --hash=sha256:d1558173930a5a8d3069cee5c92fc91c87c4dbcb099debbb3622053717145a19 \\\n    --hash=sha256:d6088ec9894113802bddb3c09e974929aed2c7b3a8c456219b8aab4481f1a239 \\\n    --hash=sha256:d6218d92e450824e9b4881f44e8c09f1853b490f9a64130801024a4793b1b3b0 \\\n    --hash=sha256:d77640cc618c1d99fc4f8589c0f24a730adfa54eb1e57ef7bf0c8dfb78da898c \\\n    --hash=sha256:d7d2deec16eeedf55f2c7cf75b521ea3856a5177e123844f8fd0f114ce252cb5 \\\n    --hash=sha256:db332af25642007330fca8be5c4d194caf2bea7a7fc84415aff3497af5dfee6b \\\n'
  '    --hash=sha256:dd54d0e8717de95939766febac482ac0474d8ac3b048115f9f2b1d23a16e7db4 \\\n    --hash=sha256:ddcac3c6b382e81f1dd0499199d4136b877beb4cb5ef770bbbfba56c4b8f55d2 \\\n    --hash=sha256:df82f3787c940c94986b34222d59c9e38843fba85139f36e85255a82ad5355a9 \\\n    --hash=sha256:dfa68deb2a443bdaa3ea5297b0699c1464f08aef3812b486d1348eee61b07dc0 \\\n    --hash=sha256:dff9461ec275f22135650d5ba4b4931a11f3958df7dfbb8db630000d4dee0883 \\\n    --hash=sha256:e1e74298bab6ee0d6e749ed4fd1901c7e604bdda32c03d787a2cc71c46d0433d \\\n    --hash=sha256:e2667f0bbe7eb6c74eae5e9691441ad186e5845ca3cff63230fc09c4e7514f5d \\\n    --hash=sha256:e3be98a7c30b8c25d573dafba7171d66dfb05ee6a9070fc46535464ff97700a6 \\\n    --hash=sha256:e568e14940c09955aa51f4e645b6daa18a581c5dcfcd73744dcc86a856e3ced3 \\\n    --hash=sha256:e72ee89e28d907a18f46959b4eb0bb06701cc7f8cf4366e00029e2ccfaaf5924 \\\n    --hash=sha256:e92eb8acc45eb6a9f4935071a77edf5b85cc6f8dfad5cd99e97653c26593cdde \\\n'
  '    --hash=sha256:ea05e1f97ceea523942d9b2a7d7c0359d781d683d6b043f5943a602b14da4787 \\\n    --hash=sha256:eac645b09bcfdf73df7536331f0678c1086ea250981118ddb5199e17ccef72bb \\\n    --hash=sha256:eb0495d778817619273c108784292be161a924b9f5ae5cbbc70a2caa6838250b \\\n    --hash=sha256:ebe8e504f058fe91223351cecd2d9d6946c9d241bb0250d898ffbdf584cc72b0 \\\n    --hash=sha256:ed099d105449c4f9e84f24af203cd131349d4761d8813fa7e02c32e7128cd910 \\\n    --hash=sha256:f0f177d1b195b9e06376cfd7d308d8a1b920909a609d03ac82a8c73bbb16d3b9 \\\n    --hash=sha256:f3d2669fe7dec7fc359ecdb5984b29b50d85d5d00f8c1cb61de4f4a24ee42627 \\\n    --hash=sha256:f4e05329faa0ea1a404b37de4f034fd2c2defcca06a68dc6745e4e56c88e8a48 \\\n    --hash=sha256:f53bcd52f585e1ac3e590d61434eb61f9a88c38df041b4ea126d97144344a77b \\\n    --hash=sha256:f55119f7bf25f49ed210f6096090715da24f2943c62102448915fde3c62877ce \\\n    --hash=sha256:f631fe87a6f30df5fbe6d79640b25e4cffb38c31c7fb6f10871517b84b0f8c1a \\\n'
  '    --hash=sha256:f8fb78a83c9e5f741ca3a68cfb455c1f5bb83b4e7249a3848b3cd78d0a8563b0 \\\n    --hash=sha256:fa9467a8113aa69d3d7c55a70ef0b7c636010a40993f3df9d9d0d73b3eb7ef24 \\\n    --hash=sha256:fd51ebf9d3a00c074df4ede271023f4d2dba289bcc740b88191872716014e3c5\n    # via fsspec\naiosignal==1.4.0 \\\n    --hash=sha256:053243f8b92b990551949e63930a839ff0cf0b0ebbe0597b0f3fb19e1a0fe82e \\\n    --hash=sha256:f47eecd9468083c2029cc99945502cb7708b082c232f9aca65da147157b251c7\n    # via aiohttp\nattrs==26.1.0 \\\n    --hash=sha256:c647aa4a12dfbad9333ca4e71fe62ddc36f4e63b2d260a37a8b83d2f043ac309 \\\n    --hash=sha256:d03ceb89cb322a8fd706d4fb91940737b6642aa36998fe130a9bc96c985eff32\n    # via aiohttp\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n'
  '    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \\\n    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \\\n'
  '    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \\\n    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \\\n    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \\\n    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \\\n    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \\\n    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \\\n    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n'
  '    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \\\n    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \\\n    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \\\n    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \\\n    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \\\n    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n'
  '    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \\\n    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \\\n    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \\\n    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n'
  '    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \\\n    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \\\n    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \\\n    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \\\n    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \\\n    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \\\n'
  '    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \\\n    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \\\n    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \\\n    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \\\n    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \\\n    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \\\n    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \\\n    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \\\n    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \\\n'
  '    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \\\n    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \\\n    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \\\n    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \\\n    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \\\n    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n'
  '    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \\\n    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \\\n    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \\\n    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \\\n    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \\\n    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \\\n'
  '    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \\\n    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \\\n    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \\\n    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \\\n    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \\\n    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \\\n'
  '    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \\\n    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \\\n    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \\\n    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \\\n    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \\\n    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \\\n    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \\\n    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n'
  '    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n'
  '    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \\\n    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \\\n    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \\\n    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \\\n    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \\\n    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \\\n    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \\\n    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \\\n    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \\\n'
  '    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \\\n    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \\\n    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n'
  '    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \\\n    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \\\n    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \\\n    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \\\n    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \\\n    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \\\n    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \\\n    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \\\n'
  '    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \\\n    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \\\n    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \\\n    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \\\n    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \\\n    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \\\n    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \\\n'
  '    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \\\n    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \\\n    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \\\n    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \\\n    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \\\n    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f\n    # via requests\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n'
  '    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n'
  '    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.3 \\\n    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f\n    # via cuda-bindings\ncuda-toolkit==13.0.3.0 \\\n    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n    # via torch\ndatasets==4.1.1 \\\n    --hash=sha256:62e4f6899a36be9ec74a7e759a6951253cc85b3fcfa0a759b0efa8353b149dac \\\n'
  '    --hash=sha256:7d8d5ba8b12861d2c44bfff9c83484ebfafff1ff553371e5901a8d3aab5450e2\n    # via -r table-intelligence-workshop-requirements.in\ndill==0.4.0 \\\n    --hash=sha256:0633f1d2df477324f53a895b02c901fb961bdbf65a17122586ea7019292cbcf0 \\\n    --hash=sha256:44f54bf6412c2c8464c14e8243eb163690a9800dbe2c367330883b19c7561049\n    # via\n    #   datasets\n    #   multiprocess\nfilelock==4.0.9 \\\n    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \\\n    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfrozenlist==1.8.0 \\\n    --hash=sha256:0325024fe97f94c41c08872db482cf8ac4800d80e79222c6b0b7b162d5b13686 \\\n    --hash=sha256:032efa2674356903cd0261c4317a561a6850f3ac864a63fc1583147fb05a79b0 \\\n    --hash=sha256:03ae967b4e297f58f8c774c7eabcce57fe3c2434817d4385c50661845a058121 \\\n'
  '    --hash=sha256:06be8f67f39c8b1dc671f5d83aaefd3358ae5cdcf8314552c57e7ed3e6475bdd \\\n    --hash=sha256:073f8bf8becba60aa931eb3bc420b217bb7d5b8f4750e6f8b3be7f3da85d38b7 \\\n    --hash=sha256:07cdca25a91a4386d2e76ad992916a85038a9b97561bf7a3fd12d5d9ce31870c \\\n    --hash=sha256:09474e9831bc2b2199fad6da3c14c7b0fbdd377cce9d3d77131be28906cb7d84 \\\n    --hash=sha256:0c18a16eab41e82c295618a77502e17b195883241c563b00f0aa5106fc4eaa0d \\\n    --hash=sha256:0f96534f8bfebc1a394209427d0f8a63d343c9779cda6fc25e8e121b5fd8555b \\\n    --hash=sha256:102e6314ca4da683dca92e3b1355490fed5f313b768500084fbe6371fddfdb79 \\\n    --hash=sha256:11847b53d722050808926e785df837353bd4d75f1d494377e59b23594d834967 \\\n    --hash=sha256:119fb2a1bd47307e899c2fac7f28e85b9a543864df47aa7ec9d3c1b4545f096f \\\n    --hash=sha256:13d23a45c4cebade99340c4165bd90eeb4a56c6d8a9d8aa49568cac19a6d0dc4 \\\n    --hash=sha256:154e55ec0655291b5dd1b8731c637ecdb50975a2ae70c606d100750a540082f7 \\\n'
  '    --hash=sha256:168c0969a329b416119507ba30b9ea13688fafffac1b7822802537569a1cb0ef \\\n    --hash=sha256:17c883ab0ab67200b5f964d2b9ed6b00971917d5d8a92df149dc2c9779208ee9 \\\n    --hash=sha256:1a7607e17ad33361677adcd1443edf6f5da0ce5e5377b798fba20fae194825f3 \\\n    --hash=sha256:1a7fa382a4a223773ed64242dbe1c9c326ec09457e6b8428efb4118c685c3dfd \\\n    --hash=sha256:1aa77cb5697069af47472e39612976ed05343ff2e84a3dcf15437b232cbfd087 \\\n    --hash=sha256:1b9290cf81e95e93fdf90548ce9d3c1211cf574b8e3f4b3b7cb0537cf2227068 \\\n    --hash=sha256:20e63c9493d33ee48536600d1a5c95eefc870cd71e7ab037763d1fbb89cc51e7 \\\n    --hash=sha256:21900c48ae04d13d416f0e1e0c4d81f7931f73a9dfa0b7a8746fb2fe7dd970ed \\\n    --hash=sha256:229bf37d2e4acdaf808fd3f06e854a4a7a3661e871b10dc1f8f1896a3b05f18b \\\n    --hash=sha256:2552f44204b744fba866e573be4c1f9048d6a324dfe14475103fd51613eb1d1f \\\n    --hash=sha256:27c6e8077956cf73eadd514be8fb04d77fc946a7fe9f7fe167648b0b9085cc25 \\\n'
  '    --hash=sha256:28bd570e8e189d7f7b001966435f9dac6718324b5be2990ac496cf1ea9ddb7fe \\\n    --hash=sha256:294e487f9ec720bd8ffcebc99d575f7eff3568a08a253d1ee1a0378754b74143 \\\n    --hash=sha256:29548f9b5b5e3460ce7378144c3010363d8035cea44bc0bf02d57f5a685e084e \\\n    --hash=sha256:2c5dcbbc55383e5883246d11fd179782a9d07a986c40f49abe89ddf865913930 \\\n    --hash=sha256:2dc43a022e555de94c3b68a4ef0b11c4f747d12c024a520c7101709a2144fb37 \\\n    --hash=sha256:2f05983daecab868a31e1da44462873306d3cbfd76d1f0b5b69c473d21dbb128 \\\n    --hash=sha256:33139dc858c580ea50e7e60a1b0ea003efa1fd42e6ec7fdbad78fff65fad2fd2 \\\n    --hash=sha256:332db6b2563333c5671fecacd085141b5800cb866be16d5e3eb15a2086476675 \\\n    --hash=sha256:33f48f51a446114bc5d251fb2954ab0164d5be02ad3382abcbfe07e2531d650f \\\n    --hash=sha256:34187385b08f866104f0c0617404c8eb08165ab1272e884abc89c112e9c00746 \\\n    --hash=sha256:342c97bf697ac5480c0a7ec73cd700ecfa5a8a40ac923bd035484616efecc2df \\\n'
  '    --hash=sha256:3462dd9475af2025c31cc61be6652dfa25cbfb56cbbf52f4ccfe029f38decaf8 \\\n    --hash=sha256:39ecbc32f1390387d2aa4f5a995e465e9e2f79ba3adcac92d68e3e0afae6657c \\\n    --hash=sha256:3e0761f4d1a44f1d1a47996511752cf3dcec5bbdd9cc2b4fe595caf97754b7a0 \\\n    --hash=sha256:3ede829ed8d842f6cd48fc7081d7a41001a56f1f38603f9d49bf3020d59a31ad \\\n    --hash=sha256:3ef2d026f16a2b1866e1d86fc4e1291e1ed8a387b2c333809419a2f8b3a77b82 \\\n    --hash=sha256:405e8fe955c2280ce66428b3ca55e12b3c4e9c336fb2103a4937e891c69a4a29 \\\n    --hash=sha256:42145cd2748ca39f32801dad54aeea10039da6f86e303659db90db1c4b614c8c \\\n    --hash=sha256:4314debad13beb564b708b4a496020e5306c7333fa9a3ab90374169a20ffab30 \\\n    --hash=sha256:433403ae80709741ce34038da08511d4a77062aa924baf411ef73d1146e74faf \\\n    --hash=sha256:44389d135b3ff43ba8cc89ff7f51f5a0bb6b63d829c8300f79a2fe4fe61bcc62 \\\n    --hash=sha256:48e6d3f4ec5c7273dfe83ff27c91083c6c9065af655dc2684d2c200c94308bb5 \\\n'
  '    --hash=sha256:494a5952b1c597ba44e0e78113a7266e656b9794eec897b19ead706bd7074383 \\\n    --hash=sha256:4970ece02dbc8c3a92fcc5228e36a3e933a01a999f7094ff7c23fbd2beeaa67c \\\n    --hash=sha256:4e0c11f2cc6717e0a741f84a527c52616140741cd812a50422f83dc31749fb52 \\\n    --hash=sha256:50066c3997d0091c411a66e710f4e11752251e6d2d73d70d8d5d4c76442a199d \\\n    --hash=sha256:517279f58009d0b1f2e7c1b130b377a349405da3f7621ed6bfae50b10adf20c1 \\\n    --hash=sha256:54b2077180eb7f83dd52c40b2750d0a9f175e06a42e3213ce047219de902717a \\\n    --hash=sha256:5500ef82073f599ac84d888e3a8c1f77ac831183244bfd7f11eaa0289fb30714 \\\n    --hash=sha256:581ef5194c48035a7de2aefc72ac6539823bb71508189e5de01d60c9dcd5fa65 \\\n    --hash=sha256:59a6a5876ca59d1b63af8cd5e7ffffb024c3dc1e9cf9301b21a2e76286505c95 \\\n    --hash=sha256:5a3a935c3a4e89c733303a2d5a7c257ea44af3a56c8202df486b7f5de40f37e1 \\\n    --hash=sha256:5c1c8e78426e59b3f8005e9b19f6ff46e5845895adbde20ece9218319eca6506 \\\n'
  '    --hash=sha256:5d63a068f978fc69421fb0e6eb91a9603187527c86b7cd3f534a5b77a592b888 \\\n    --hash=sha256:667c3777ca571e5dbeb76f331562ff98b957431df140b54c85fd4d52eea8d8f6 \\\n    --hash=sha256:6da155091429aeba16851ecb10a9104a108bcd32f6c1642867eadaee401c1c41 \\\n    --hash=sha256:6dc4126390929823e2d2d9dc79ab4046ed74680360fc5f38b585c12c66cdf459 \\\n    --hash=sha256:7398c222d1d405e796970320036b1b563892b65809d9e5261487bb2c7f7b5c6a \\\n    --hash=sha256:74c51543498289c0c43656701be6b077f4b265868fa7f8a8859c197006efb608 \\\n    --hash=sha256:776f352e8329135506a1d6bf16ac3f87bc25b28e765949282dcc627af36123aa \\\n    --hash=sha256:778a11b15673f6f1df23d9586f83c4846c471a8af693a22e066508b77d201ec8 \\\n    --hash=sha256:78f7b9e5d6f2fdb88cdde9440dc147259b62b9d3b019924def9f6478be254ac1 \\\n    --hash=sha256:799345ab092bee59f01a915620b5d014698547afd011e691a208637312db9186 \\\n    --hash=sha256:7bf6cdf8e07c8151fba6fe85735441240ec7f619f935a5205953d58009aef8c6 \\\n'
  '    --hash=sha256:8009897cdef112072f93a0efdce29cd819e717fd2f649ee3016efd3cd885a7ed \\\n    --hash=sha256:80f85f0a7cc86e7a54c46d99c9e1318ff01f4687c172ede30fd52d19d1da1c8e \\\n    --hash=sha256:8585e3bb2cdea02fc88ffa245069c36555557ad3609e83be0ec71f54fd4abb52 \\\n    --hash=sha256:878be833caa6a3821caf85eb39c5ba92d28e85df26d57afb06b35b2efd937231 \\\n    --hash=sha256:8a76ea0f0b9dfa06f254ee06053d93a600865b3274358ca48a352ce4f0798450 \\\n    --hash=sha256:8b7b94a067d1c504ee0b16def57ad5738701e4ba10cec90529f13fa03c833496 \\\n    --hash=sha256:8d92f1a84bb12d9e56f818b3a746f3efba93c1b63c8387a73dde655e1e42282a \\\n    --hash=sha256:908bd3f6439f2fef9e85031b59fd4f1297af54415fb60e4254a95f75b3cab3f3 \\\n    --hash=sha256:92db2bf818d5cc8d9c1f1fc56b897662e24ea5adb36ad1f1d82875bd64e03c24 \\\n    --hash=sha256:940d4a017dbfed9daf46a3b086e1d2167e7012ee297fef9e1c545c4d022f5178 \\\n    --hash=sha256:957e7c38f250991e48a9a73e6423db1bb9dd14e722a10f6b8bb8e16a0f55f695 \\\n'
  '    --hash=sha256:96153e77a591c8adc2ee805756c61f59fef4cf4073a9275ee86fe8cba41241f7 \\\n    --hash=sha256:96f423a119f4777a4a056b66ce11527366a8bb92f54e541ade21f2374433f6d4 \\\n    --hash=sha256:97260ff46b207a82a7567b581ab4190bd4dfa09f4db8a8b49d1a958f6aa4940e \\\n    --hash=sha256:974b28cf63cc99dfb2188d8d222bc6843656188164848c4f679e63dae4b0708e \\\n    --hash=sha256:9ff15928d62a0b80bb875655c39bf517938c7d589554cbd2669be42d97c2cb61 \\\n    --hash=sha256:a6483e309ca809f1efd154b4d37dc6d9f61037d6c6a81c2dc7a15cb22c8c5dca \\\n    --hash=sha256:a88f062f072d1589b7b46e951698950e7da00442fc1cacbe17e19e025dc327ad \\\n    --hash=sha256:ac913f8403b36a2c8610bbfd25b8013488533e71e62b4b4adce9c86c8cea905b \\\n    --hash=sha256:adbeebaebae3526afc3c96fad434367cafbfd1b25d72369a9e5858453b1bb71a \\\n    --hash=sha256:b2a095d45c5d46e5e79ba1e5b9cb787f541a8dee0433836cea4b96a2c439dcd8 \\\n    --hash=sha256:b3210649ee28062ea6099cfda39e147fa1bc039583c8ee4481cb7811e2448c51 \\\n'
  '    --hash=sha256:b37f6d31b3dcea7deb5e9696e529a6aa4a898adc33db82da12e4c60a7c4d2011 \\\n    --hash=sha256:b4dec9482a65c54a5044486847b8a66bf10c9cb4926d42927ec4e8fd5db7fed8 \\\n    --hash=sha256:b4f3b365f31c6cd4af24545ca0a244a53688cad8834e32f56831c4923b50a103 \\\n    --hash=sha256:b6db2185db9be0a04fecf2f241c70b63b1a242e2805be291855078f2b404dd6b \\\n    --hash=sha256:b9be22a69a014bc47e78072d0ecae716f5eb56c15238acca0f43d6eb8e4a5bda \\\n    --hash=sha256:bac9c42ba2ac65ddc115d930c78d24ab8d4f465fd3fc473cdedfccadb9429806 \\\n    --hash=sha256:bf0a7e10b077bf5fb9380ad3ae8ce20ef919a6ad93b4552896419ac7e1d8e042 \\\n    --hash=sha256:c23c3ff005322a6e16f71bf8692fcf4d5a304aaafe1e262c98c6d4adc7be863e \\\n    --hash=sha256:c4c800524c9cd9bac5166cd6f55285957fcfc907db323e193f2afcd4d9abd69b \\\n    --hash=sha256:c7366fe1418a6133d5aa824ee53d406550110984de7637d65a178010f759c6ef \\\n    --hash=sha256:c8d1634419f39ea6f5c427ea2f90ca85126b54b50837f31497f3bf38266e853d \\\n'
  '    --hash=sha256:c9a63152fe95756b85f31186bddf42e4c02c6321207fd6601a1c89ebac4fe567 \\\n    --hash=sha256:cb89a7f2de3602cfed448095bab3f178399646ab7c61454315089787df07733a \\\n    --hash=sha256:cba69cb73723c3f329622e34bdbf5ce1f80c21c290ff04256cff1cd3c2036ed2 \\\n    --hash=sha256:cee686f1f4cadeb2136007ddedd0aaf928ab95216e7691c63e50a8ec066336d0 \\\n    --hash=sha256:cf253e0e1c3ceb4aaff6df637ce033ff6535fb8c70a764a8f46aafd3d6ab798e \\\n    --hash=sha256:d1eaff1d00c7751b7c6662e9c5ba6eb2c17a2306ba5e2a37f24ddf3cc953402b \\\n    --hash=sha256:d3bb933317c52d7ea5004a1c442eef86f426886fba134ef8cf4226ea6ee1821d \\\n    --hash=sha256:d4d3214a0f8394edfa3e303136d0575eece0745ff2b47bd2cb2e66dd92d4351a \\\n    --hash=sha256:d6a5df73acd3399d893dafc71663ad22534b5aa4f94e8a2fabfe856c3c1b6a52 \\\n    --hash=sha256:d8b7138e5cd0647e4523d6685b0eac5d4be9a184ae9634492f25c6eb38c12a47 \\\n    --hash=sha256:db1e72ede2d0d7ccb213f218df6a078a9c09a7de257c2fe8fcef16d5925230b1 \\\n'
  '    --hash=sha256:e25ac20a2ef37e91c1b39938b591457666a0fa835c7783c3a8f33ea42870db94 \\\n    --hash=sha256:e2de870d16a7a53901e41b64ffdf26f2fbb8917b3e6ebf398098d72c5b20bd7f \\\n    --hash=sha256:e4a3408834f65da56c83528fb52ce7911484f0d1eaf7b761fc66001db1646eff \\\n    --hash=sha256:eaa352d7047a31d87dafcacbabe89df0aa506abb5b1b85a2fb91bc3faa02d822 \\\n    --hash=sha256:eab8145831a0d56ec9c4139b6c3e594c7a83c2c8be25d5bcf2d86136a532287a \\\n    --hash=sha256:ec3cc8c5d4084591b4237c0a272cc4f50a5b03396a47d9caaf76f5d7b38a4f11 \\\n    --hash=sha256:edee74874ce20a373d62dc28b0b18b93f645633c2943fd90ee9d898550770581 \\\n    --hash=sha256:eefdba20de0d938cec6a89bd4d70f346a03108a19b9df4248d3cf0d88f1b0f51 \\\n    --hash=sha256:ef2b7b394f208233e471abc541cc6991f907ffd47dc72584acee3147899d6565 \\\n    --hash=sha256:f21f00a91358803399890ab167098c131ec2ddd5f8f5fd5fe9c9f2c6fcd91e40 \\\n    --hash=sha256:f4be2e3d8bc8aabd566f8d5b8ba7ecc09249d74ba3c9ed52e54dc23a293f0b92 \\\n'
  '    --hash=sha256:f57fb59d9f385710aa7060e89410aeb5058b99e62f4d16b08b91986b9a2140c2 \\\n    --hash=sha256:f6292f1de555ffcc675941d65fffffb0a5bcd992905015f85d0592201793e0e5 \\\n    --hash=sha256:f833670942247a14eafbb675458b4e61c82e002a148f49e68257b79296e865c4 \\\n    --hash=sha256:fa47e444b8ba08fffd1c18e8cdb9a75db1b6a27f17507522834ad13ed5922b93 \\\n    --hash=sha256:fb30f9626572a76dfe4293c7194a09fb1fe93ba94c7d4f720dfae3b646b45027 \\\n    --hash=sha256:fe3c58d2f5db5fbd18c2987cba06d51b0529f52bc3a6cdc33d3f4eab725104bd\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2025.9.0 \\\n    --hash=sha256:19fd429483d25d28b65ec68f9f4adc16c17ea2c7c7bf54ec61360d478fb19c19 \\\n    --hash=sha256:530dc2a2af60a414a832059574df4a6e10cce927f6f4a78209390fe38955cfb7\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n'
  '    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n'
  '    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   datasets\n    #   timm\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via\n    #   requests\n    #   yarl\njinja2==3.1.6 \\\n'
  '    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nmarkupsafe==3.0.4 \\\n    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \\\n    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \\\n    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \\\n    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \\\n    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \\\n    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \\\n    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \\\n    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \\\n    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \\\n'
  '    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \\\n    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \\\n    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \\\n    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \\\n    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \\\n    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \\\n    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \\\n    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \\\n    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \\\n    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \\\n    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \\\n'
  '    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \\\n    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \\\n    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \\\n    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \\\n    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \\\n    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \\\n    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \\\n    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \\\n    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \\\n    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \\\n    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \\\n'
  '    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \\\n    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \\\n    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \\\n    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \\\n    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \\\n    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \\\n    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \\\n    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \\\n    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \\\n    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \\\n    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \\\n'
  '    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \\\n    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \\\n    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \\\n    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \\\n    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \\\n    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \\\n    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \\\n    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \\\n    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \\\n    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \\\n    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \\\n'
  '    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \\\n    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \\\n    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \\\n    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \\\n    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \\\n    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \\\n    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \\\n    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \\\n    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \\\n    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \\\n    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \\\n'
  '    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \\\n    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \\\n    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \\\n    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \\\n    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \\\n    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \\\n    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \\\n    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \\\n    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \\\n    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \\\n    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \\\n'
  '    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \\\n    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \\\n    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \\\n    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \\\n    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \\\n    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \\\n    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \\\n    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \\\n    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \\\n    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \\\n    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \\\n'
  '    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \\\n    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \\\n    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \\\n    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \\\n    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \\\n    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \\\n    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \\\n    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \\\n    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \\\n    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \\\n    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \\\n'
  '    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \\\n    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \\\n    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \\\n    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \\\n    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \\\n    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \\\n    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \\\n    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \\\n    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \\\n    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \\\n    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \\\n'
  '    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \\\n    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \\\n    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \\\n    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \\\n    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \\\n    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \\\n    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \\\n    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \\\n    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \\\n    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \\\n    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \\\n'
  '    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \\\n    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \\\n    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \\\n    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \\\n    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \\\n    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \\\n    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \\\n    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \\\n    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \\\n    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \\\n    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \\\n'
  '    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \\\n    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \\\n    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \\\n    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \\\n    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \\\n    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \\\n    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \\\n    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \\\n    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \\\n    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \\\n    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \\\n'
  '    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \\\n    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \\\n    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \\\n    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \\\n    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \\\n    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \\\n    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \\\n    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \\\n    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \\\n    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3\n    # via jinja2\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n'
  '    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nmultidict==6.9.1 \\\n    --hash=sha256:006c4478de0a1876f4834e14255776286f09b9846b505fe63f67f9d173a9487c \\\n    --hash=sha256:024123f0ab402ab33828e24eb80fa8f25167d0d3783ba5f287e39ed741e6abf9 \\\n    --hash=sha256:02f6d0c4b70f783305e73f9944d8efe6be1022550f0974ba0ae9d8893c0350fa \\\n    --hash=sha256:02fe09dc197b8ae7e355371e51e5dce2f39060cd5a8badf94904527e23a3f188 \\\n    --hash=sha256:03731f6fc036180700c9dc2308205a48e5ca6f3ff03087739ab746f294022201 \\\n    --hash=sha256:03d47df72f084f757c1cb771188d5f4e3a805e4abc4d67e32509272343ae9382 \\\n    --hash=sha256:03fac50ddfd8302175b77863a015eccfae76767cda5506eef86df559ba861e1f \\\n    --hash=sha256:042fb0196047e786936a730bd302de83143950da45f2c16078da8f35e1cf7919 \\\n    --hash=sha256:083735b7f395894e43adb278d5dae901448883a835ff8f1977e285fefdb10418 \\\n    --hash=sha256:095d900c242e00fbe5f321ee072e7278b4153e78c5ce9c1efde167d62c1e4771 \\\n'
  '    --hash=sha256:0a5769559e3312dd96731fbe15b4abb6033368ac1cad5a98dadd21946a4c7d6c \\\n    --hash=sha256:0b2fb8c349d1103863750b5d8cb5ace766917f4b35f3d883c8f778853eaa9f76 \\\n    --hash=sha256:0dd655518f136febd96c05131a76a863e32fc2a1d7acd4e3c959e3ceb77d8345 \\\n    --hash=sha256:0f06e60fa190aa7abd0914c2a766736fdc8e9f34878c4346338534b73d1b20e2 \\\n    --hash=sha256:0f2ce963299d42fa3f22a90adc0fdf174792ffef5ff4c7ffb68260548fb05580 \\\n    --hash=sha256:0f3bd290711c6e9486173a6ee7cd4e7f00c3971c7908c1ec1b6e5437c5e4c6f9 \\\n    --hash=sha256:10083a8a0f4e1b26b599889e90b9802504ce5d3f7722f925bbb7ca47dd22a7c1 \\\n    --hash=sha256:13849a1d4f54c3809ae721e9e83ab28f5ea602f33660cb84eb6ef261eac706c1 \\\n    --hash=sha256:14c56f73e78faa1f68bbb826197cd5871994e70e841b8590829c35912ece5c64 \\\n    --hash=sha256:15db6a102cbaf1949cf028ecf080aac76d20bcd29ad4e092574db6c6b7af78a5 \\\n    --hash=sha256:19e31815d41cefc365489e591d105d2baceb2f65aa75d29471fbdbda8651e006 \\\n'
  '    --hash=sha256:1a53de2772cfb74559df2eb4456ec4eeb908435ec55a84b69370d9d745d62aa8 \\\n    --hash=sha256:1a8adfcaf96f587ab138476eaddef95f29b8a2a8a9afbfea8d2fd62180995d02 \\\n    --hash=sha256:1a938761c77e0e6edb0c93d02f4e988d44a69e5195e5a3b893e5553311347132 \\\n    --hash=sha256:1c9f3c25df6c9d3bbae4f6fd3f514b1c5c740a2110f56ccf36069c85417e289c \\\n    --hash=sha256:1ca5ebe6d454f1e5496cf052386a559f1080bf2de75bf327ebca0a6003b79f19 \\\n    --hash=sha256:1d804e4caf5d5da37d6dac1325da5629ebef1e27a294c2b568b295814aa36c7b \\\n    --hash=sha256:1fee9a16d88a1c4865610de31ef5c666671020d7a81d1f510eaf3c97d00ebeba \\\n    --hash=sha256:23136f5a564654eb61061ec6d5620a4c1ea32c8f552b65e9982a12b72bff601b \\\n    --hash=sha256:2784090c30a586d5b45197bd9c32f87fb927216cde302f6cfd76d76577e90f08 \\\n    --hash=sha256:29138fef49828542e859828107e42e50d0e587c513b7eb4b2d92bade2b0860fe \\\n    --hash=sha256:2c1aeb92eea59d824f004341b26d5e4b47a8a441cf9726769b9a90abf9d0e08f \\\n'
  '    --hash=sha256:2c5d675da8f1cb5650271c8ad5e95c0a3e5a183c105e72d953b12877b1c8d0fd \\\n    --hash=sha256:3100d169ceb7bc8f05f89a6db11d1b21f119975fc26f29dc45a472e3569f0879 \\\n    --hash=sha256:31199204b3ced121ff5407a2c342326a5d27e3870abbf94bd80dbd2451b7bc8f \\\n    --hash=sha256:32217133dddc58c927805cf6c0731d8144584176b768042ee886d51e71860bc9 \\\n    --hash=sha256:33fa55b990f81c2927e01399ace0d18926c69d69baa8cdaa819424132fb97987 \\\n    --hash=sha256:35ba0263bae5dd3ad5aad767cc9afc01a8598c7dae30f1b3b2de98b1b32c28bd \\\n    --hash=sha256:35fc236507fb1b3138f0af5ecd5f94ed752d4d6d826248eae425f86204013eea \\\n    --hash=sha256:361f7206cf341ba94fb015688f5c8b480f8e63bd58a4c14a48aeca7851a241cc \\\n    --hash=sha256:369b5aa01b241cd3fea6890bdbb11a1425d87bf1515831500d518f4223e9d72c \\\n    --hash=sha256:37245ca4105386194dd1d292a6f2aae09bfe1bd7ac6f9ec25093e3cf8e9b143b \\\n    --hash=sha256:372c063f37480f62c1ae32dc3a1a0a5942b180c883f99789075a8a8ec3c4e709 \\\n'
  '    --hash=sha256:37a9ebe00c698279213d56e6c64e1962ab1e092918270649b397cac3dc196ca4 \\\n    --hash=sha256:38c9986f9ce50c459b10de216a05f4bd7ed5ac63887d56e500a52bb464b861ce \\\n    --hash=sha256:3adf06c66041aa21eeb8a71e82379b74773298c8e6d3d839b151aae441a99b94 \\\n    --hash=sha256:3c95601ed98fad3f6e2f8fe809c3b526b0fab31ef525e00a155e227f3d17f58a \\\n    --hash=sha256:40aec5299e1ed71fbb988389059da381c3c1a60e0c649acceb2a35d9b128848e \\\n    --hash=sha256:43124fe172ada86d03ac3c8dc8179091341f6724d5e5d5b160e1587e4cd3761b \\\n    --hash=sha256:4736d350371337825cac1793c9f7c40701c32a03912548c2a7608e51679cbb96 \\\n    --hash=sha256:475d04d5192eba487a3e2f935976340baa24529046e9c1c9c7a3b7bf80445ae1 \\\n    --hash=sha256:4a409ccc42aefec904038695d5d7fd6d8f3af2721b7b6401d55135ec7d6d298e \\\n    --hash=sha256:4d718fa1b5f0d0dd75e86fbbc5b0c93ea3a5d65216c85c61cd5d7cdddfe08455 \\\n    --hash=sha256:501ed8b02a5990c67a91c732843609d43a6be1f7576fcdfc867331239f37fbd3 \\\n'
  '    --hash=sha256:539c2cd5fed0947c135cd7eabaaac55f48300dfa1de0f3ca4edb5efa6606f471 \\\n    --hash=sha256:557a4e1708df428ebe6c3081c83a273d275dad2446ce0d81fc648ac71afdb18d \\\n    --hash=sha256:56d834b74c993a7d7cb2b8ab33a0d55c3e0d4a3d2f2da2808a4ad3d79189711b \\\n    --hash=sha256:5800368526647146978389dfaa46da3356291e9f0fff9a4ef12e8c2bef964a0d \\\n    --hash=sha256:59c123d0e948d760a5f930f316cfefa07e8d632ab84327c0693ee6a88171154f \\\n    --hash=sha256:5b30ddf7234e611ca877575b62840e6af5977f92f1f9d532eedbb05a44ff8004 \\\n    --hash=sha256:5ecace251ccfa705bf3d7d35c5032cf750f5c629809740405697bce5c118c4a4 \\\n    --hash=sha256:5f89dad732280e7a10b74d40b91364f88e13c3f2c08c2ef83a8cd42f7a61af2e \\\n    --hash=sha256:637f4ae36264bd7b8d9a60193acddc1d735ad52e8ed53a19931ea6d921fea8e5 \\\n    --hash=sha256:63ada7ee2e9345695f9e9bc4c65d72222253f07b1ac94fd0e37555cc6f3c7f60 \\\n    --hash=sha256:6441cc837aea58be7d9baef1b2383eb8311ab9303f500f99ac90b584cd78bb14 \\\n'
  '    --hash=sha256:658f90f49cf5af2441cad0a2b801c3ef520471989a1ec55bcb25b255b2ca8d2f \\\n    --hash=sha256:66987aa68b0f7c2a1cc5f388ca962b8ed92b10f79de38d6d0d8c716154f519d9 \\\n    --hash=sha256:696477ad71385c4795e3b8e4cf10b0d2c28c2a1ca6a955e031cb1e62993e9ee3 \\\n    --hash=sha256:6b7e54fd883671d1a8810704851c044b7173287c552b9f5c3d9e0eb9f00ae194 \\\n    --hash=sha256:6bc94fe17c3c56e5418f79515b786b101845f70609b0d19d0c1ba13448e5633a \\\n    --hash=sha256:6e6b7e3a1520c39a2772f414cd9ddf5995f77e4e823dfa1522af383addd465a3 \\\n    --hash=sha256:6ed30be8918e18c8bed0a2e8b70639ecf02feb61ed00ca2e41cfcb2a50fa3f42 \\\n    --hash=sha256:73533644e1f69ea1164d56cc6505f564c6c44072b646b66d67df2d31fed0348c \\\n    --hash=sha256:73bcafa21a78d0776b3ee7cd2a63c66f968eb7db8e8d594e32d7329950f6e828 \\\n    --hash=sha256:7814bbee202acd3bd240204c17d8b87a4c48c81064fd8674dbe527d94d5a4290 \\\n    --hash=sha256:783ba7d845d79ce976afd9c1e91a4e5714671defa198ee789e8b23316083a485 \\\n'
  '    --hash=sha256:79da2491348b30810728050b4a8ec0416f85884125c2fd44655b3d01150d9c3e \\\n    --hash=sha256:7a0c98f6a636adf0d7edd60c61589eaf52d149239af754d0bfeb0effedba53a8 \\\n    --hash=sha256:7ab379f95caee071a37cbd8be86d4c65accc651d391f9f97748fef006f38769c \\\n    --hash=sha256:7bf6478188f4e47bf5686e8a33da4ae28bf43b1b2528d9ee144d28492bfac60b \\\n    --hash=sha256:7c6eecfab7ce4cd9487ff8ba936fe38cdfd68c04faf3d5c710363c6a8e695659 \\\n    --hash=sha256:7c708566da8014b120a64b1eb6d200c6c0c8cb36296383723cdb6fc82038270b \\\n    --hash=sha256:7c8d5882ba25ac0282258be435d8a05aa0cbcacfce15799154a338f847f159b9 \\\n    --hash=sha256:7fc59e9ba821b220944ccfe0f89c9dc4745f6d097569992356eb03869f21e953 \\\n    --hash=sha256:7fd79c521f6290c69125fa2b85fa65d9e657e6a8ffaf722dc881b926bef4aa5c \\\n    --hash=sha256:803f8b575a71b1b299d677c28db0653459c79b5308874efec813f17b7457c7f1 \\\n    --hash=sha256:8050e75af7e4c6e2d5260b84eeedb618f3e452e66473432e085b5d1b81429299 \\\n'
  '    --hash=sha256:81cdc537e0a42e3c0170752fcadbe450246d5c3b4b7231d6eb9672456605ac94 \\\n    --hash=sha256:827c92145b3b976b39430129c89d213b250dacbe5fce678e9a03940e6e848983 \\\n    --hash=sha256:854fd2f1bc6e8a56b89910b5cd7261a8b40f13ebb31572da985ce59c7da0886d \\\n    --hash=sha256:877ca17fdcfdf5c397493a71e5ff97a87bb181417fe717fdadc77c08c09301ac \\\n    --hash=sha256:87cc632c88ee5dc80e12681047839304d98ee5c9a708d686505767001c9b8b9a \\\n    --hash=sha256:8879510a76940670517ea1cb589978da44b86e286ec3e50d664ef330817afce7 \\\n    --hash=sha256:8885e3808aedbd6725b921fb67dacaae0678933561ddd47c2b01315198c70e2e \\\n    --hash=sha256:8b193bf7a443c97d81c47052f60c486071a4bdef4a573fa2514f920089414d45 \\\n    --hash=sha256:8ed78ccad4c7b421804f5d524b7740946a7ef75d89dc0dac52e9d7c24c472410 \\\n    --hash=sha256:8f2973bbd2bebd9d2e0cd6394c1292a1a19ccd56bdcbe1e174059f1a39be5b40 \\\n    --hash=sha256:910d4260512660484c0dc1588a316fbb35a40c081c36fc51d1225351af17cfe4 \\\n'
  '    --hash=sha256:95052e8777a86bae87c0bd0b5ab22d809e3d1d02bf69e3e66ddda5ba75a05805 \\\n    --hash=sha256:95f273bea318a194f656527ee2ed19494327bc500b8e87d9358e3222579aa28d \\\n    --hash=sha256:963a8d8f97057082679523d0fd4c53a38f86bc58cabe4556faef682ae53fa2fa \\\n    --hash=sha256:976fd7689d69ec78d67d31d38d396d8adb562f7e8368279f76aed4aa451fa06d \\\n    --hash=sha256:989261c5f1735a165f2e4e87cf6d5f17ab734fa18f9ad0383d5adfcdaefce701 \\\n    --hash=sha256:9ae9614c317c50836689ce2dfde07c05fe0b16378562e2221746c3913ede3c80 \\\n    --hash=sha256:9b0124b9c17e9890f0819b2e7a5f65ec9a2f5aabd6c8e7ad090c1675cf70dee6 \\\n    --hash=sha256:9c4880d017555d70dea367dd49271830842d48e3891c2da97da7ce8c4abcee40 \\\n    --hash=sha256:9c62e71e6289d78c0108d8aeb495f8bd3cad4bc1632fedddc9297ddf287ecc20 \\\n    --hash=sha256:9d0a21cf76153de8f2d96a877991d6bc59b9ab5180b949e50db73cc193b4a694 \\\n    --hash=sha256:9e14d17773b1b3c758ff153659a1824608a0cb562c45f484b5ed8a433428444a \\\n'
  '    --hash=sha256:a16a1dc8529f9e734a41c3b856f3eae7ebacdc061dde3f8a844e0c7889c97203 \\\n    --hash=sha256:a2212a0c842c723d919ea4a22a9296cb6b244b386e4e6ea92adfc7fbf3095519 \\\n    --hash=sha256:a313bad717dde740959d50850c315b75fd4eb0c5e6b4dc8535db0f1c369be125 \\\n    --hash=sha256:a32b78c1e52ebd8e247bb68300b90b233300d8816faa008ed0713bc539fb6af0 \\\n    --hash=sha256:a3ffe881246d28a862f1985824f484cb7361f44d6b99c4c620436ef56462f38d \\\n    --hash=sha256:a49ff5cdb33654cb7d6a3c377aa2a83ddefaa1db31eb10bcf3c180aa84f9af8a \\\n    --hash=sha256:a8b75dd3d3638d9a19f23e84af4ffab3b8940422c0df2da2a77005ef5aa3d7ea \\\n    --hash=sha256:ab64ace1a68682d191d9bedd9d4c939406ad86b1d9f628180410644249ad46c2 \\\n    --hash=sha256:abc7c2e4b47bfe6a9aea434d3fdebb9597ee636e914e92ba352f2068b9142f4c \\\n    --hash=sha256:ac348379cf4de5538a0a213be1532d289aa801ec5d267c5909446b9ea2f8e2c3 \\\n    --hash=sha256:ac51cd64bae51c462ea58ad2492c9b8209667a4ef60c45c4a304518b67598d5d \\\n'
  '    --hash=sha256:acddcac38adc8342ba48aba98896faa7928854bebb62542362138655b5367ee3 \\\n    --hash=sha256:ad4528cdce058b684f75fad1faf4a6a6c992fe2f08376370ca66e4ce5916a84a \\\n    --hash=sha256:aec65b53a07f580606593f877eefbb29a45939bfc0d3fe6e6d9f42b41b749f68 \\\n    --hash=sha256:aef74e9beabbd6c4aafc091dabff86d046ccf013ce1e4396c0fbb01b4cad9de8 \\\n    --hash=sha256:af1b5a92315048c3e36bbebfa7d4760a9c3e910bc4166f11b20d77d20d6bcfca \\\n    --hash=sha256:b22ff30006a2f28f8bff878fb93413cbe3a4d1fd517c28081d848c90e9cfd2c8 \\\n    --hash=sha256:b2483da477932ad1983d1d33c18bc3771c6fb00cfbaaed70a875fd547ef8e840 \\\n    --hash=sha256:b2f0adc22a4eb31e545221d93fc73a0f6a8cc2379d0f4309f71d1d17ba938b82 \\\n    --hash=sha256:b4c9e5d05b126b267ac048a89a0e2d9b48b1b648add62d4872906304a8590610 \\\n    --hash=sha256:b4da208a63434d21a3df64d29758e650fc4aa8cb05848554b76949c296539cca \\\n    --hash=sha256:b5f8771aaaed7f80e84a4e471d2f29ab6721e4595075e54d03ae1ed951b2000a \\\n'
  '    --hash=sha256:b65121091567847a8cb520d364ab22ba90e00d3cc55fa9eb34bb439f0684bcd1 \\\n    --hash=sha256:b66ccc5c2cdd26e74fa5d4c29ffae424cc6148bf93ce574821783fb3b6d452c5 \\\n    --hash=sha256:b69651732c64afb691e50cdc3387cae305e0eeff8804fe3e3ce203876494932a \\\n    --hash=sha256:b828cf64d62dc09ac183f03c1aeedd164ade96a2ce4934109452edf29de1dd13 \\\n    --hash=sha256:b8a65621b98984a62e59403009591b8a5a7736273aefe1cab64cfb85b365cc07 \\\n    --hash=sha256:bb0d664505f4b112f384cffeee82e91e3f6448d8e574989479db4439b68cba05 \\\n    --hash=sha256:bb58ba73a3f96f9a3e46b1fab69929d7edbbddb3133ee74b5c3c54074a53c4f1 \\\n    --hash=sha256:bc94a68ea5e18f8e85dc6b522bcb53093f692c8eb62b4837ac047da73956cbe4 \\\n    --hash=sha256:bd82c4977196681a499bb6ca9e462afbc5c91c1c15b6a991dfbd72733fea4dd2 \\\n    --hash=sha256:c0c88085affd35c33e124e36930c5ad96aff9294ce195eaa0fd9cec962b64a82 \\\n    --hash=sha256:c148e8b596000dd3e4bfe206e70f3e666be18d72032e0012555f2373c52e35d6 \\\n'
  '    --hash=sha256:c57541034d12b215ab0a2bfa371d1a8a198da18176d0426c27105b9161a6862d \\\n    --hash=sha256:c9648ed33dc8179e4ec04bbc73bd7f0038e1e81217a69467f61f02a78bf07e88 \\\n    --hash=sha256:ca65cced0d67a9039e93bcd98a369920e499bf98296844ff56ead01c9085321f \\\n    --hash=sha256:cf22b43f35b7dbb9f71e8ee2041b5c00029cdfc28ede3a2f31cf8906f9a6c126 \\\n    --hash=sha256:d35a4f1c63f07fbb8c8f9946dea98b21eddf6c57421585f71d91864be3ba2a24 \\\n    --hash=sha256:d3b6e6840421c83ccb60398e333b44f910b0907bb409597685ab2eedd1e22eab \\\n    --hash=sha256:d73fed4e37158ff00cd271871170b79e40138db51e625fd352fa17c6acb34f67 \\\n    --hash=sha256:d7bf9e43282d69561618e8a0ea33368d532ebef42f15c096f427090521dd74f3 \\\n    --hash=sha256:d9d6544790ba50438a9c1a903c3c4afb1ec8a7832db5518549275b40ef0dd4b1 \\\n    --hash=sha256:db4d697b18b6ef5528b1f36bfa25072cd2a421869f5963bc0e92c8a34b9e2800 \\\n    --hash=sha256:dd9a137a4a9becda3094f3831cd026380f75f6855e051eefe4c73ade524f1cc3 \\\n'
  '    --hash=sha256:de7738b8c0bb74c4cc16bbd7fb49fc2bcf6430dba11b3432cd52768ae40933e8 \\\n    --hash=sha256:e3b1aa25f01238886a6baed9e13b9a9240ed344346c79ae280ae65e8603b21d5 \\\n    --hash=sha256:e58952f04772f59f11c6e007471449809a30165188669bca8fdb19dde40a8f24 \\\n    --hash=sha256:e5ccad4b7bac125722f48d6f862bed3b514d8526deea06316bb72f152cd30a7c \\\n    --hash=sha256:e7386aa18d98d6b8af44b92173654ec469237fda35f8e8523e43b581a86f476a \\\n    --hash=sha256:e81ae656b9935ac4528a71f96bb7a14d949778ed1897c573d3e7ebb9187f8841 \\\n    --hash=sha256:e8f1e362c9352b50ed120f001046fdbb80810c9d56580f4c3fc13bbe30823387 \\\n    --hash=sha256:e96ca64383efa107262ee3949f047af5ee4f1845ba09463466c04d35a83bd3ec \\\n    --hash=sha256:ea999ae6e80e66ad5eea287860951b033d0104ca34d6d87c7b5125ebe0e12721 \\\n    --hash=sha256:ecc89dbd4155b2f8a47f4bbd89242a35ed15e8e0ec581cab5ac65fa38407329d \\\n    --hash=sha256:ef01d29fca550ab871fd99154f82c6472aacf8e7def272dfb07b460123850390 \\\n'
  '    --hash=sha256:f04551dce5a7db8c9659f2e4245494c182d0663b83661803e08d46bfcae5eda1 \\\n    --hash=sha256:f0700527dd5bfa8b7204b08330542f4f388899d3c14d885d8a368992e0eb562d \\\n    --hash=sha256:f2524ec55b3e65cbe235a8b3c36e2af3b635be02ed05e20c94e70c5c943c009e \\\n    --hash=sha256:f5844e7befc707367807586f550fa97e23dcfef0728f02b98c7bc498a961a5df \\\n    --hash=sha256:f9dad513626a33670f17cddc6078e30e311f444c957e8dbfc5b2b4603c8b4edb \\\n    --hash=sha256:fc0dcb22fa9aeabfe3fa4e0430099acff985ec5d77a851382f76cc6146e780e5 \\\n    --hash=sha256:fd882aa29bf402b62bf1fd7c19fd5df4b6528cf468a908864b39368572b662a9\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.16 \\\n    --hash=sha256:0dfd078c306e08d46d7a8d06fb120313d87aa43af60d66da43ffff40b44d2f41 \\\n    --hash=sha256:161af703d4652a0e1410be6abccecde4a7ddffd19341be0a7011b94aeb171ac1 \\\n    --hash=sha256:37b55f71c07e2d741374998c043b9520b626a8dddc8b3129222ca4f1a06ef67a \\\n    --hash=sha256:476887be10e2f59ff183c006af746cb6f1fd0eadcfd4ef49e605cbe2659920ee \\\n'
  '    --hash=sha256:a0bafd3ae1b732eac64be2e72038231c1ba97724b60b09400d68f229fcc2fbf3 \\\n    --hash=sha256:a71d82033454891091a226dfc319d0cfa8019a4e888ef9ca910372a446de4435 \\\n    --hash=sha256:af4cabb0dac72abfb1e794fa7855c325fd2b55a10a44628a3c1ad3311c04127a \\\n    --hash=sha256:ba8c31889abf4511c7308a8c52bb4a30b9d590e7f58523302ba00237702ca054 \\\n    --hash=sha256:c4a9944c67bd49f823687463660a2d6daae94c289adff97e0f9d696ba6371d02 \\\n    --hash=sha256:d951bed82c8f73929ac82c61f01a7b5ce8f3e5ef40f5b52553b4f547ce2b08ec \\\n    --hash=sha256:e7b9d0f307cd9bd50851afaac0dba2cb6c44449efff697df7c7645f7d3f2be3a \\\n    --hash=sha256:fc0544c531920dde3b00c29863377f87e1632601092ea2daca74e4beb40faa2e\n    # via datasets\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.3 \\\n    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n'
  '    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n'
  '    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n'
  '    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n'
  '    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n'
  '    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n'
  '    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   datasets\n    #   pandas\n    #   torchvision\n    #   transformers\n'
  'nvidia-cublas==13.1.1.3 \\\n    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via\n    #   cuda-toolkit\n    #   nvidia-cublas\n'
  'nvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.24.0.43 \\\n    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n'
  '    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n'
  '    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.1 \\\n    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n    # via torch\nnvidia-nccl-cu13==2.30.7 \\\n    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n    # via torch\nnvidia-nvjitlink==13.4.92 \\\n    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n'
  '    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   transformers\npandas==2.2.3 \\\n'
  '    --hash=sha256:062309c1b9ea12a50e8ce661145c6aab431b1e99530d3cd60640e255778bd43a \\\n    --hash=sha256:15c0e1e02e93116177d29ff83e8b1619c93ddc9c49083f237d4312337a61165d \\\n    --hash=sha256:1948ddde24197a0f7add2bdc4ca83bf2b1ef84a1bc8ccffd95eda17fd836ecb5 \\\n    --hash=sha256:1db71525a1538b30142094edb9adc10be3f3e176748cd7acc2240c2f2e5aa3a4 \\\n    --hash=sha256:22a9d949bfc9a502d320aa04e5d02feab689d61da4e7764b62c30b991c42c5f0 \\\n    --hash=sha256:29401dbfa9ad77319367d36940cd8a0b3a11aba16063e39632d98b0e931ddf32 \\\n    --hash=sha256:31d0ced62d4ea3e231a9f228366919a5ea0b07440d9d4dac345376fd8e1477ea \\\n    --hash=sha256:3508d914817e153ad359d7e069d752cdd736a247c322d932eb89e6bc84217f28 \\\n    --hash=sha256:37e0aced3e8f539eccf2e099f65cdb9c8aa85109b0be6e93e2baff94264bdc6f \\\n    --hash=sha256:381175499d3802cde0eabbaf6324cce0c4f5d52ca6f8c377c29ad442f50f6348 \\\n    --hash=sha256:38cf8125c40dae9d5acc10fa66af8ea6fdf760b2714ee482ca691fc66e6fcb18 \\\n'
  '    --hash=sha256:3b71f27954685ee685317063bf13c7709a7ba74fc996b84fc6821c59b0f06468 \\\n    --hash=sha256:3fc6873a41186404dad67245896a6e440baacc92f5b716ccd1bc9ed2995ab2c5 \\\n    --hash=sha256:4850ba03528b6dd51d6c5d273c46f183f39a9baf3f0143e566b89450965b105e \\\n    --hash=sha256:4f18ba62b61d7e192368b84517265a99b4d7ee8912f8708660fb4a366cc82667 \\\n    --hash=sha256:56534ce0746a58afaf7942ba4863e0ef81c9c50d3f0ae93e9497d6a41a057645 \\\n    --hash=sha256:59ef3764d0fe818125a5097d2ae867ca3fa64df032331b7e0917cf5d7bf66b13 \\\n    --hash=sha256:5dbca4c1acd72e8eeef4753eeca07de9b1db4f398669d5994086f788a5d7cc30 \\\n    --hash=sha256:5de54125a92bb4d1c051c0659e6fcb75256bf799a732a87184e5ea503965bce3 \\\n    --hash=sha256:61c5ad4043f791b61dd4752191d9f07f0ae412515d59ba8f005832a532f8736d \\\n    --hash=sha256:6374c452ff3ec675a8f46fd9ab25c4ad0ba590b71cf0656f8b6daa5202bca3fb \\\n    --hash=sha256:63cc132e40a2e084cf01adf0775b15ac515ba905d7dcca47e9a251819c575ef3 \\\n'
  '    --hash=sha256:66108071e1b935240e74525006034333f98bcdb87ea116de573a6a0dccb6c039 \\\n    --hash=sha256:6dfcb5ee8d4d50c06a51c2fffa6cff6272098ad6540aed1a76d15fb9318194d8 \\\n    --hash=sha256:7c2875855b0ff77b2a64a0365e24455d9990730d6431b9e0ee18ad8acee13dbd \\\n    --hash=sha256:7eee9e7cea6adf3e3d24e304ac6b8300646e2a5d1cd3a3c2abed9101b0846761 \\\n    --hash=sha256:800250ecdadb6d9c78eae4990da62743b857b470883fa27f652db8bdde7f6659 \\\n    --hash=sha256:86976a1c5b25ae3f8ccae3a5306e443569ee3c3faf444dfd0f41cda24667ad57 \\\n    --hash=sha256:8cd6d7cc958a3910f934ea8dbdf17b2364827bb4dafc38ce6eef6bb3d65ff09c \\\n    --hash=sha256:99df71520d25fade9db7c1076ac94eb994f4d2673ef2aa2e86ee039b6746d20c \\\n    --hash=sha256:a5a1595fe639f5988ba6a8e5bc9649af3baf26df3998a0abe56c02609392e0a4 \\\n    --hash=sha256:ad5b65698ab28ed8d7f18790a0dc58005c7629f227be9ecc1072aa74c0c1d43a \\\n    --hash=sha256:b1d432e8d08679a40e2a6d8b2f9770a5c21793a6f9f47fdd52c5ce1948a5a8a9 \\\n'
  '    --hash=sha256:b8661b0238a69d7aafe156b7fa86c44b881387509653fdf857bebc5e4008ad42 \\\n    --hash=sha256:ba96630bc17c875161df3818780af30e43be9b166ce51c9a18c1feae342906c2 \\\n    --hash=sha256:bc6b93f9b966093cb0fd62ff1a7e4c09e6d546ad7c1de191767baffc57628f39 \\\n    --hash=sha256:c124333816c3a9b03fbeef3a9f230ba9a737e9e5bb4060aa2107a86cc0a497fc \\\n    --hash=sha256:cd8d0c3be0515c12fed0bdbae072551c8b54b7192c7b1fda0ba56059a0179698 \\\n    --hash=sha256:d9c45366def9a3dd85a6454c0e7908f2b3b8e9c138f5dc38fed7ce720d8453ed \\\n    --hash=sha256:f00d1345d84d8c86a63e476bb4955e46458b304b9575dcf71102b5c705320015 \\\n    --hash=sha256:f3a255b2c19987fbbe62a9dfd6cff7ff2aa9ccab3fc75218fd4b7530f01efa24 \\\n    --hash=sha256:fffb8ae78d8af97f849404f21411c95062db1496aeb3e56f146f0355c9989319\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   datasets\npillow==11.3.0 \\\n    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \\\n'
  '    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \\\n    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \\\n    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \\\n    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \\\n    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \\\n    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \\\n    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \\\n    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \\\n    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \\\n    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \\\n    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \\\n'
  '    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \\\n    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \\\n    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \\\n    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \\\n    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \\\n    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \\\n    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \\\n    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \\\n    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \\\n    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \\\n    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \\\n'
  '    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \\\n    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \\\n    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \\\n    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \\\n    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \\\n    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \\\n    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \\\n    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \\\n    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \\\n    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \\\n    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \\\n'
  '    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \\\n    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \\\n    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \\\n    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \\\n    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \\\n    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \\\n    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \\\n    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \\\n    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \\\n    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \\\n    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \\\n'
  '    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \\\n    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \\\n    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \\\n    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \\\n    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \\\n    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \\\n    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \\\n    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \\\n    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \\\n    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \\\n    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \\\n'
  '    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \\\n    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \\\n    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \\\n    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \\\n    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \\\n    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \\\n    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \\\n    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \\\n    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \\\n    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \\\n    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \\\n'
  '    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \\\n    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \\\n    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \\\n    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \\\n    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \\\n    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \\\n    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \\\n    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \\\n    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \\\n    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \\\n    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \\\n'
  '    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \\\n    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \\\n    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \\\n    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \\\n    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \\\n    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \\\n    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \\\n    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \\\n    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \\\n    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \\\n    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \\\n'
  '    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \\\n    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \\\n    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \\\n    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \\\n    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \\\n    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \\\n    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \\\n    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \\\n    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \\\n    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \\\n    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \\\n'
  '    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \\\n    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \\\n    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \\\n    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \\\n    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \\\n    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   torchvision\npropcache==0.5.4 \\\n    --hash=sha256:004e685b315646c410771836e72a44f143bbe624f29653a42687815069a303d5 \\\n    --hash=sha256:02c0a34f16889cf800f10f0247a564d8ce6eeab6ffcd7c87198f769067eb8432 \\\n    --hash=sha256:03969626faf0783a592dfa17e28eac06018bd0b44dafae6943d53b92421a7f72 \\\n    --hash=sha256:03b229037d25b801e7af53fd52b9fc49d9439b036fca1e087e02780631adfa97 \\\n'
  '    --hash=sha256:0951315a6b3142ee2167404d707743f0157c110091342b1aa0accac5cf0e4acf \\\n    --hash=sha256:0a095db8e15a6020db149ecbed6461939fe74f6acaa3ae8b702a1fe8c38cd983 \\\n    --hash=sha256:0c889f6fa84957bc7e8b4eab71fd16a0455068d5045e3aa40c733071d2b2fd77 \\\n    --hash=sha256:0d21d0d2c82bbfeb1677a9711f38df968f9837576102bb4add1bd449d28d88f1 \\\n    --hash=sha256:10ef33a68a61ce317e095fd2e202a592ea92392b90944a78c993f0d9a73ab06c \\\n    --hash=sha256:12682126712ddc19b70ff819debbd279e58adf1f0c8f8f8138c18ade2044b284 \\\n    --hash=sha256:135036c5cfc93864affb0f9af9a27e5d7a71cb7bd745e7b6dbfc2d56cc30e827 \\\n    --hash=sha256:13e52b6e0bde97dee98ab66552dbff2931649c96f1ac432eac299fe689ec373b \\\n    --hash=sha256:141fdbd73748db0cf7636035030aaac383d2efde8f34e7bc24594cc776d225b8 \\\n    --hash=sha256:146f48a9e4812611a7581003b1a39de56c34967046310c4171a68ef908c9a745 \\\n    --hash=sha256:174507f82d3594622acb1dd2dafecf2d899d6d506335494e7107767bf05f3aae \\\n'
  '    --hash=sha256:1783582065a1f07f9d9ee1e992e13f15d7dc8fb1eb3a7476d43eb3f2e69d26bb \\\n    --hash=sha256:17a7400cec0256f0a71ae71f9da398f9894c956ff6668a1c9d317b3367316320 \\\n    --hash=sha256:1b2f3bec4261a94019575481c726c29850f72e27907773c75b1de421e20e9f9d \\\n    --hash=sha256:1d759d05634f1b038fb625a66662a8c85e5a8fec912da381b5149ddac107482b \\\n    --hash=sha256:1df8d8561b21465c5dd56110a01caf897e026d065b4b84e98a488209094272ec \\\n    --hash=sha256:213bb68d9ced5cf2bf717b1071bf2b09b4b04c426256f9fe6d054c60318424c4 \\\n    --hash=sha256:23278f808cd81d5ada7184a76606b925fb3389c60e1077b2cd7da7b1fcf0553c \\\n    --hash=sha256:251c63dd46a0659bb875cb254dc4c1e79ee91a847c737cd62373295afc2235dc \\\n    --hash=sha256:279655a16973f1ee2bd2fe79973137681642fd9ae0d89215bba263726eb0dc3a \\\n    --hash=sha256:2814ecd8e818f487bee4b0f921bc4d1c176cc5fc71ac0f072d0fa67eda4ac14b \\\n    --hash=sha256:286867fb156488c251a3721766e380ac4495e4fd6b51aaa1403d89ce7f4359d9 \\\n'
  '    --hash=sha256:2dba2f02d2d5c09ef8a0e6c1a42aeaa451f4be9898cb00b04fe98717da2eb23b \\\n    --hash=sha256:30cc1cebaf9aef49db06357a50398323ae04d70460c0491837d026ab7d6452ea \\\n    --hash=sha256:31eb43ba2edc704ab2ec27815315dd8a19def0fb16215be4cfe8d32fe78ffd51 \\\n    --hash=sha256:350b272b2279f4135a64fc0c304a5d08e28a137c9573442c606152446638a831 \\\n    --hash=sha256:36c0d9db44b523ef93d03341b1c42d69ff01d673c053d1b1c6c3a363bcaa39ba \\\n    --hash=sha256:3af0c8642b2da4815d86e631232ac8286e17644fad907c19508aa8e7cb4ba8ad \\\n    --hash=sha256:3cd3a7edb6b95b9b33998135ebfa18d709da82290fb8f27c858970b5a12c8b56 \\\n    --hash=sha256:3d605bb239b796e82a81c6709548b2bd460ab73b4590cb0c83de8a2dd9694d0f \\\n    --hash=sha256:3e413d7a4a9b4866b7a761d6060d434b64d23cd35122eda3b026a0bbe8196b25 \\\n    --hash=sha256:3eb2e820e8e2101407da93f17c57cbb7d225461955fc60105daaba14cd421ee2 \\\n    --hash=sha256:3fa15757fea1dfcd5b7745cad9f4638929605531bd4018ab2adff7955f1a403d \\\n'
  '    --hash=sha256:3fc24f209c1b7f7f688b66b98293954f5504279760999b58920ee12dd8471c1d \\\n    --hash=sha256:4054acf80d40456a0537f2913b349718649d8d6458a14ab7f48d0ce28c30869d \\\n    --hash=sha256:40e94adb1e7d39ff28a8bd8d8b8fbd1df6b9f40976dbe379134f1ce058e532dd \\\n    --hash=sha256:420162a77f94eb1cf5ef7893f500016dabd548e73de956785a1dd899cc73006a \\\n    --hash=sha256:425f8cc86ab5018b4b8d4a23bc8e74d964bd3d757c3702e301aa79be76c53f6c \\\n    --hash=sha256:44149f46500a0a41b95b4d99c2e586a77319539730607b9892974a092788b111 \\\n    --hash=sha256:445ee3bfb46e85838387fb3c536a73cc0b994dc192b004e40e170adc54aa2a7e \\\n    --hash=sha256:45488d1a5f9ab5bd90aaa1ca20f50fe1922b8ffad71a2009d2adf41355897aac \\\n    --hash=sha256:45bebbe252550fec975ba3b62bc6f931643cfd3b5464ef47619cf3fef154e01c \\\n    --hash=sha256:45bf2e730ab8905d0527fe05a86500f406e64305c34cc81ebe64b4617cab9760 \\\n    --hash=sha256:48cb48c5346a97de792254af77715aa2529c2a1ebc5f586aa0aae44a02f1fe57 \\\n'
  '    --hash=sha256:4a1f4f5ffa55dce6307631f3cb2948e117e665966ea512e0d502b16c24f567e7 \\\n    --hash=sha256:4cfe0a92ae30151869e67a4b5f5e105e4e03ad30b3f38e5211b5bf77d0881993 \\\n    --hash=sha256:4d86476a935c88963d9b8e1a9a0d38188790e9622169bfbafa173046846709d3 \\\n    --hash=sha256:4e985382be6d15da8d0c2710a6fa7b9070fc9ecdeefb7f580e88373984ec8be3 \\\n    --hash=sha256:4f2d880ff60f45898f4acfa152aac8d04e3ee627d90ff4003491bf92239d5757 \\\n    --hash=sha256:4fbc1a15dc8cd1689508758d626b372b1f09d28d9577667feaf9e6bfcd8efcbc \\\n    --hash=sha256:50e337653721d20ead710da33bf44487fbe8a0db8782714b60306481e9f95b51 \\\n    --hash=sha256:53eaa697c4d0422ff4cb714d00231b43352064d97b944033b30c1d57cc506ec0 \\\n    --hash=sha256:56fc3f7599528db40b1efa0889a620116e2704144495273d66066e8164e45838 \\\n    --hash=sha256:58134228927cee6c047d626c08e60a81be604a20578a12ce752cc5c9a84d4826 \\\n    --hash=sha256:594eb4c6ec35e7179b058481f4e9f02521b56de16fa577c4b85c76fb1bf8a9f8 \\\n'
  '    --hash=sha256:5cacf3c9efd09df409dc33654dd077e1c245ba8fb747b0f0236ef41b7c49b589 \\\n    --hash=sha256:60a64cbccaa11b7760ce705a14ada17ba459e7ca9f23ba587eb013821032d7ef \\\n    --hash=sha256:62530ca89187827e4a4fe733f971abe81a7542eeea48ff61995f19b64d7199c8 \\\n    --hash=sha256:62c60aec739ed00124573cce1178138fd690c7676352d67a37328c1cf51d7468 \\\n    --hash=sha256:69fc35c0779522da366c563e5faf203ffc1f8ff0021d5b1337fa4efa5be73177 \\\n    --hash=sha256:6af4693716bfb03f1752ef1b30faa593db2c01d5272e9b8564a1549452a979ab \\\n    --hash=sha256:6c7599df2b57ebeea8de011b5f2f7b85de95e76037d43d34b95e328430275487 \\\n    --hash=sha256:6e9368e87a3efc285e559131092c5db643eb8e56de4ee42064d5baec22ef2bb5 \\\n    --hash=sha256:6f0093ac3e9daada202c2082439d414a625c57184727a46e112a3fb2a81cb788 \\\n    --hash=sha256:7177c43eddf10a0893c4fec52ebb408fdcd7f7d63962caace9180d8f81b14ece \\\n    --hash=sha256:720cf832eb2d0b0dfee129cb3335a26f6ce3cc45ee1187e8f0731758caa16792 \\\n'
  '    --hash=sha256:770e8209d018175fc0063936fa9583b6d27e88c5ad31543f3383d66080efdd62 \\\n    --hash=sha256:7a8d5ff04eb1f85698a78d20c62a14676e7b960dcafde09a388d60ad377d355d \\\n    --hash=sha256:7b9100a93b372418d8688f3f2a3e5b45c64d70ca4d6176e121aca1e3bfc1e32f \\\n    --hash=sha256:7cc528e760a8af06f2b13e9b9f362cd90c7c718ea61228a96dbd31ba16ed7f47 \\\n    --hash=sha256:7ffafcbfc7b549ab940047e505c831eabac5e67de53e1bc174adbc5285c55944 \\\n    --hash=sha256:87a3caecf8095e48dc72f84bfa42e23a848cf410cc9cc13031fba4869b706a21 \\\n    --hash=sha256:886b59c4d28ca97dd23b025fdfc50a0356be934efbbbca89ad26230067f86fe5 \\\n    --hash=sha256:8876b39961e33d912afe3c1bee18ee564fdad0206f873cc15d522756b7f50737 \\\n    --hash=sha256:897d1ddf6716e8f47200f7aad9a0efa6cc7586df66c6defa572f9eab379c078e \\\n    --hash=sha256:8a1fc236528c457cd739c88abe823da851b7ab645d72792f88658114cc340c12 \\\n    --hash=sha256:8a235f73d6e020855dc29dff012d920c02ee0feab8d73a24185a7569f4be1161 \\\n'
  '    --hash=sha256:8f911c395cef73c510bac566da9507bb6a43e7763d0c79138dc60ee53f11207e \\\n    --hash=sha256:96f7c5c15656040ddcbc51e56dc59b58aa25999d743c126abd425b9766ab43e9 \\\n    --hash=sha256:978f28401afbc76cdc3df9e1717b4229a06b626a1dcc75db4e1f2beb3884c3e9 \\\n    --hash=sha256:98914de2c4d7f0f9f4a8c6ea4bf05841f4175796941e3ef7d47eb718f22311fb \\\n    --hash=sha256:9a2a8a50a93dee0268a860a07fa3b4bd968f8ce4dbd794957da772f395368526 \\\n    --hash=sha256:9cbfff4423eef4cc6cafc021469641a2b835f610b2647a6c5281903e21b8670d \\\n    --hash=sha256:9e9ab13760aa8b6d0881ae7cb04fd891d8d490cd2554ea8e79bb278399169bcc \\\n    --hash=sha256:9f3551b8a35c1df3e7ea4d2d86edee15f0dde1bddd434a71744048683544d0ef \\\n    --hash=sha256:9f86f7259efe2c951f43e57d471c9b41daa5bfc7db9f67189059cf1ae6d77fd9 \\\n    --hash=sha256:9fb0a5be8d9aa213150e8d8148a42aca4984b285bcad1e69587dc4298edd929b \\\n    --hash=sha256:a219f0ac59817a9114dd2aa57c13180f993e819ba658c7ddab4b66ed1ee0d370 \\\n'
  '    --hash=sha256:a419ee85e654927baabda3929c03c0cc1112bf472ff0dfd6142f4e3a81ca4162 \\\n    --hash=sha256:a4d7a54719b67338a305dca2ce6aafe366817df94ddfd4b5514374356f5ca546 \\\n    --hash=sha256:a5793c7698a53f56f4a1889a4737c7eeb1b7ad0842fa6b1abca22913ff79c8c1 \\\n    --hash=sha256:a5e8ef588c109725dc713ba69aadcac00a1ef90c2ce9c0a8c7075128f569f47f \\\n    --hash=sha256:a74bfa37147cc08fb29df10bd9c16f40fa7f860cd3a6d2fff853323a94f6e17f \\\n    --hash=sha256:ada748108a43d29b7c328ba7db3755327cd94f028bcc1a7ee3f0addcfacd9c38 \\\n    --hash=sha256:ae58f361bd5dae942717c65d3413b478c70aea9c462599e7b9adad3731db3894 \\\n    --hash=sha256:b28f41fa3b8c6900457f858ec5b03998f3a6d535fbc1bb2edec5961ea05ec429 \\\n    --hash=sha256:b3083bfe87f95c756e610bd8025f26cbd1cd4aaa03a422f2d65efb7a97cd53d8 \\\n    --hash=sha256:b61805357d966680acf68b3b6d49772631ed9df44ebece10ff1460e117a7da8a \\\n    --hash=sha256:b77c313314524ca9c38fbd70f73515d04597ac58c40c939bc0e71eeb4abff680 \\\n'
  '    --hash=sha256:bee7d3aed13d56f54e681df38c3a23031bc9e3863f687d9d598825c9146acd7d \\\n    --hash=sha256:c02c0e570c5c7e077b0181a9f3cdb7d4c3617d1cda6b5c95bd5d34022923d82c \\\n    --hash=sha256:c174bfd1c48a1b51a3078e95586dde718374bac79719ab3541ec9e74aec40574 \\\n    --hash=sha256:c2ba30a89035b57b73e00475de948521602f543d79ce01db10b04b36c4c76fc8 \\\n    --hash=sha256:c3e98c55bde2bcf7db3c70d1aed7ae9aa8aebbf19a250c66645cde44cdb8b867 \\\n    --hash=sha256:c3ef2818d63bc86071e9d2989ae75a1bc32b8f7059cfd9f5abbbee70c32e2ed6 \\\n    --hash=sha256:c83acbce9f2b5e3f5f5eda9e53d2001fed22fcdfef81274a9e02d8fd53b70a30 \\\n    --hash=sha256:c9281e922c072158c91974d4589f1dbe0fee6d467f284c28e463f9f5a4d933f4 \\\n    --hash=sha256:cc07876cfb079b6f6f36d21ce75784ad6c2c6b563eeac0ed26c2fa2669b85df9 \\\n    --hash=sha256:ccf4f7a79e26bb7efb06ecd50c177833b71df05cbc748701372325e6bcc17f6f \\\n    --hash=sha256:cdee8205a44d0be91bbac4c41b95d86641b72dfc7aef1279400e4fda3f26a937 \\\n'
  '    --hash=sha256:ceb3e879afac028f93d272c957814695dc5569e4904262dbee92f6c41bd5e4a3 \\\n    --hash=sha256:d1f5a500bfcbb2c0ab85e98a0dcd70f5899d34efe365a0187700369a79603031 \\\n    --hash=sha256:d42a9a856a4a6e2f6c10f1318c07e7daa498d6593abe745c71dae4521a26ca39 \\\n    --hash=sha256:d83b12902eb8bce151259c86c03ba746600b2d994543de46e370cecf96c452f2 \\\n    --hash=sha256:d8e017eeb7482bed34cdb0d61cf2bcfc88d104bbab296a17cd16a6af8aabc70e \\\n    --hash=sha256:db3ae52ccc150dbc84704e9d642743897f3e1c54742ff34cacb661e52e3818a9 \\\n    --hash=sha256:dbab5f5ff6897c81f355d079010cdae85b02e5a0b518b5251523b8ad8ae9ac3c \\\n    --hash=sha256:dc4242ca653c9b30ab51c5f8193323e7bc0928f897ee9103201e59a43abcb72e \\\n    --hash=sha256:dcbf346a318a5e30063f547630b02bb787ce2f45b6368d5da143660b6a3835d8 \\\n    --hash=sha256:dd2ac8f5b643454c2cc6b6118b13da16e88f4a6434fc3ba61aca384029f04f36 \\\n    --hash=sha256:e1d52a05dc417279f7e5c7618c5dfbbc29923aaf9bc0a5c1802ddcebf54c61a0 \\\n'
  '    --hash=sha256:e6720ba44ad7e72174314d0e1fb0172494cff5c73a3a8a2159c3d2402ff15565 \\\n    --hash=sha256:e738ab81179510ce79b2eac9a6ecf47feffd9e76d1c72e403005dddb6e36c06c \\\n    --hash=sha256:e904d4d01f36bd6e197590be1533c44e06058771e0746dd073a8ebb3ef880858 \\\n    --hash=sha256:e9f165403b81fea7e89c932d89046a1e3d9a3a60e8d7ef2f249dccdcb0982bf5 \\\n    --hash=sha256:ec6a85f424afa8d23e0d9a094e5dbb6eda01da91c92b9183cd433768247ffc97 \\\n    --hash=sha256:ee19113bce2f3acd46432050688b70f61acd6857d75abb9ec96341b7e9ced123 \\\n    --hash=sha256:ef3b928d9c984322b5c44e6964d8dbc653da87d2d8ee1647fa6da43072e650a9 \\\n    --hash=sha256:f273dcf7149a50527c4fd1f55cfe9eac0f60753f5af544b4c9352578e20c0874 \\\n    --hash=sha256:f5470694918830da62fac9e69133b53d23b736d7070e587b27a4a2be37e08e68 \\\n    --hash=sha256:f574e460d1c8a08384a016fdb09ccf3543433263ed6b2f97104f979e64ea57c2 \\\n    --hash=sha256:f85915e00dcb1cd9f2f890ead064ed40a27df06f0db65be427b29482ae357572 \\\n'
  '    --hash=sha256:fc2461ecc45f17893f8207e73b46ea8ba93e33630e51cf4af3fbc21d47462b1a \\\n    --hash=sha256:ff6b113f50bc066a698db5d944d2c6dc7507168dd3341e255a8892fd0715a558\n    # via\n    #   aiohttp\n    #   yarl\npyarrow==25.0.1 \\\n    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \\\n    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \\\n    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \\\n    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \\\n    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \\\n    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \\\n    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \\\n    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \\\n    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \\\n'
  '    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \\\n    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \\\n    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \\\n    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \\\n    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \\\n    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \\\n    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \\\n    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \\\n    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \\\n    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \\\n    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \\\n'
  '    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \\\n    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \\\n    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \\\n    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \\\n    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \\\n    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \\\n    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \\\n    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \\\n    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \\\n    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \\\n    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \\\n'
  '    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \\\n    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \\\n    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \\\n    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \\\n    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \\\n    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \\\n    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \\\n    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \\\n    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \\\n    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \\\n    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \\\n'
  '    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   datasets\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via pandas\npytz==2026.4 \\\n    --hash=sha256:464303645bafafd72418898368b2429458f709cf1eb6a15372fbcc396b64da63 \\\n    --hash=sha256:9d514388fbc89ca0833203464272ac485b8828568ab73532f5020f17e892a0ff\n    # via pandas\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n'
  '    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n'
  '    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n'
  '    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n'
  '    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n'
  '    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n'
  '    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n'
  '    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   timm\n    #   transformers\nregex==2026.9.29 \\\n    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \\\n    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \\\n    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \\\n    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \\\n    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \\\n    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \\\n    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \\\n'
  '    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \\\n    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \\\n    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \\\n    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \\\n    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \\\n    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \\\n    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \\\n    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \\\n    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \\\n    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \\\n    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \\\n'
  '    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \\\n    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \\\n    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \\\n    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \\\n    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \\\n    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \\\n    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \\\n    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \\\n    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \\\n    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \\\n    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \\\n'
  '    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \\\n    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \\\n    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \\\n    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \\\n    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \\\n    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \\\n    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \\\n    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \\\n    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \\\n    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \\\n    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \\\n'
  '    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \\\n    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \\\n    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \\\n    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \\\n    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \\\n    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \\\n    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \\\n    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \\\n    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \\\n    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \\\n    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \\\n'
  '    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \\\n    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \\\n    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \\\n    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \\\n    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \\\n    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \\\n    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \\\n    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \\\n    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \\\n    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \\\n    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \\\n'
  '    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \\\n    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \\\n    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \\\n    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \\\n    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \\\n    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \\\n    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \\\n    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \\\n    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \\\n    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \\\n    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \\\n'
  '    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \\\n    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \\\n    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \\\n    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \\\n    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \\\n    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \\\n    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \\\n    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \\\n    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \\\n    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \\\n    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \\\n'
  '    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \\\n    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \\\n    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \\\n    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \\\n    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \\\n    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \\\n    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \\\n    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \\\n    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \\\n    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \\\n    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \\\n'
  '    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \\\n    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \\\n    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \\\n    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \\\n    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \\\n    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \\\n    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \\\n    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \\\n    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \\\n    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \\\n    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \\\n'
  '    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \\\n    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \\\n    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \\\n    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \\\n    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \\\n    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \\\n    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \\\n    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \\\n    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \\\n    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \\\n    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \\\n'
  '    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \\\n    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \\\n    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \\\n    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \\\n    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \\\n    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \\\n    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \\\n    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \\\n    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \\\n    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \\\n    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \\\n'
  '    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \\\n    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n'
  '    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n'
  '    #   timm\n    #   transformers\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via torch\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntimm==1.0.30 \\\n    --hash=sha256:1444a3af7e17fdba5440ac2bcf6e4b728da96dd62289ed9e1bcba31a545f5935 \\\n    --hash=sha256:c8e27adf6801a2fdfb570dbbdd171f2f084756eefc673df8cfaab06bd14864f3\n    # via -r table-intelligence-workshop-requirements.in\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n'
  '    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n'
  '    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n'
  '    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.14.0 \\\n    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n'
  '    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n'
  '    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   timm\n    #   torchvision\ntorchaudio==2.11.0 \\\n    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \\\n    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \\\n    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \\\n    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \\\n    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \\\n    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \\\n    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \\\n'
  '    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \\\n    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \\\n    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \\\n    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \\\n    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \\\n    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \\\n    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \\\n    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \\\n    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \\\n    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \\\n    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \\\n'
  '    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \\\n    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \\\n    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \\\n    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \\\n    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \\\n    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \\\n    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \\\n    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \\\n    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \\\n    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84\n    # via -r table-intelligence-workshop-requirements.in\ntorchvision==0.29.0 \\\n'
  '    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n'
  '    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n'
  '    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n    # via\n    #   -r table-intelligence-workshop-requirements.in\n    #   timm\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via -r table-intelligence-workshop-requirements.in\ntriton==3.8.0 \\\n    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n'
  '    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n'
  '    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   huggingface-hub\n    #   torch\ntzdata==2026.4 \\\n    --hash=sha256:c2169a8b0a7a5e9674da5a135ccdfb2b3e671b333ed9fed17b41f73c34476e81 \\\n    --hash=sha256:f1b8bd365d8d210c55353f4d7f8d6d8561c0ba50d704b700d195a9424bba0d79\n    # via pandas\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\nxxhash==4.0.1 \\\n    --hash=sha256:0163b5d259de23ae9e07b7eabf435ce4704f6f205589a2b154e6af4be985ce1b \\\n    --hash=sha256:03600a8987849b2bef7be795a60a6052b635c63fa98b718b08ca5ee823691cfc \\\n    --hash=sha256:04f9a24de11a6647666d5302fd73d6a5224ce50ddc965fb0bb44cee736e6bd7c \\\n    --hash=sha256:06713a5aaf1d0905c5579416c020c02e42b3ceb931e86c7d3b7fb85403dee3f3 \\\n'
  '    --hash=sha256:06d7fbd609503c3be5e65cdb6bb2f040d6a98574404e2e1d5c60815c97fff4aa \\\n    --hash=sha256:0718ad66f4ded2411f8e62bdba549ee71e313a2d26ef5060ca3fdbf29897dd3c \\\n    --hash=sha256:08ed8da18cd4fd0a6a5d6a444852d8fbd0e565388a74a4937085451b5f1a312a \\\n    --hash=sha256:09f9feb118966cc6650e1806205d577eae7ca394aa6acf349a0b62a94bbeb329 \\\n    --hash=sha256:0ab851b45c70d4992be7cdeeee16f97a0b677408c758c4b1efb1cfe8030bfd37 \\\n    --hash=sha256:0b1082fd0f089ce9098ed77aad8b777b5d156f8ac601c69cab73811822b8ef07 \\\n    --hash=sha256:0b20a06454b34f1531fc677c54efe2ecdec691ef9224f7fa919bf2c1363f7ff1 \\\n    --hash=sha256:0b42a5a26607e4b2409fea174773a66f2dff9dfdbf2c1a851bb7b804e2c97535 \\\n    --hash=sha256:101aa300de6ceef3d9c77569706330d8921fc45dd82bceed2084f1e9f2557a24 \\\n    --hash=sha256:1216f7ba5683f17a89eb7dcb4bc50a0b743dfe1902278d7b3d0786f538118433 \\\n    --hash=sha256:1642907941ee4b75aacc3db688af52ea02ca2305ab22af7ee686ed726b332684 \\\n'
  '    --hash=sha256:168dd6b51725a222abc722832e56624d15a63fc2e8249021509c93f1063913f6 \\\n    --hash=sha256:1749f0688020209fe0d357ce1e1cd9ec9c6161ed0405ea949d24581c4c43fa91 \\\n    --hash=sha256:1b3cccf75eeb5b01639b2feadb042a8e07889293b7ca72fa2985e7dcb64763cf \\\n    --hash=sha256:1b50223d92df94d54e1a31469335a2c74b16692e6c1cb726f1e6949514458706 \\\n    --hash=sha256:1bc591533fc975614f7e13594daee76af96b8e1fbcf8de76c8773858fa9e7cea \\\n    --hash=sha256:1c2200b98a805351cb3142ae4e1fdcc9e91b5e20f5d30d4862b0b96f92558f4e \\\n    --hash=sha256:1c7c642a0f79c3e3cf2965475507574d3d1a50ec71060039d60cb87358667cb2 \\\n    --hash=sha256:1ee523f51718e41753f04f7102bb4dc55a18d2ea5cbaceef8ec7ca08571bd428 \\\n    --hash=sha256:1f3346c5c287ac3c7f38b20380f55e8768230e7252af59fabcf3b87ab21e4256 \\\n    --hash=sha256:2194bf96d5f3d4e0cb65deba370ec83dda3edfba42155f9384190ed5e51ea5e2 \\\n    --hash=sha256:237b8f63a2a0fcfb1ffc06e21dad23add44e6d354b2b014364a1d41e419a4dee \\\n'
  '    --hash=sha256:23a4376b4a3183cb50d4d2a3179f887a7773cc695eb2c908e551bec3221b8c60 \\\n    --hash=sha256:247ece770647c0aef080561fa996f9774b4dadce2d0c42eeb98229db7dcf820d \\\n    --hash=sha256:2696bbac613f6880fed60316c298bf3091d4f8eee3ae2e9466f70bb76204fb0c \\\n    --hash=sha256:26fe6238c2d5b11ed5063b9bf4eb290624b004fd074688da6bb079bd564f10d7 \\\n    --hash=sha256:2d52dc7c33c1b83082b707f6b7814dc76d2faaa2ea62bd9c5fab4b36f83c087f \\\n    --hash=sha256:2df3ca8757dc381e75e90a4d7995a6324f58a923c7145220a7b2c0231f66fddc \\\n    --hash=sha256:303121aab4b7f898058582d7962ea79d9e26e2379d7b6d8743f70f2671674481 \\\n    --hash=sha256:3088dadbffa33c29e0518578430a7dff2e901a212e487aefa5faaa0dc06dad34 \\\n    --hash=sha256:31d86f9e81f3e84e00131ac7c54caf5119ae4ddd82c09c31cff597c813ce1ee2 \\\n    --hash=sha256:3358097d333d40657569ec1121e21043dd7d0efa10aead1b50e8b4fa83077d7b \\\n    --hash=sha256:33e270d302c95ec426dfa0f5a4e16bff2ab8d7b8a46faa4746affb05e684ac77 \\\n'
  '    --hash=sha256:33fd538191f47071deef6b1f676535e2aa770f1fd150ae4cc75a34c9e930be3d \\\n    --hash=sha256:348c8f288dc961d6bbd1985c8152a3ed7a85c95df00e82320f0c5215d922a399 \\\n    --hash=sha256:349775ac30372b344d2338b2a168c0a1312a644194da25b8bec476d55761a128 \\\n    --hash=sha256:34ed93e20bfd98d722b902121643791eeb4b1641871e2dc63d0d4c2d93f187df \\\n    --hash=sha256:37f667dee0f867c42894b34e2a6fe26bf195c0ea4683d9d2b713db023f242c3a \\\n    --hash=sha256:3891efe3d7a531ce6da0a4a50a99dd41c75b8fd4ca19d73c86431b4db5c305f0 \\\n    --hash=sha256:38c3d22129a6958846a3098d68bc8e661704461c0be4793ae28836e4690c8478 \\\n    --hash=sha256:3c2445edafc300cc40feb6a25a8356a971c30cd0bf47b5349c2ad74c508343b1 \\\n    --hash=sha256:3f68fe400ceec235f3e4a4b02a28c2fd2d283584a193223c921dd4c48f1d0754 \\\n    --hash=sha256:3fb1d30d4b6d6e2c4a08e5ac6fffdb2b572d2cfcca15a5509cf4e7a1350f955c \\\n    --hash=sha256:41e579025a6e13a99e6d71e39c9cfc621a0dcdbbf19106325e145fa858f2d794 \\\n'
  '    --hash=sha256:421b94f3ba7067958d02e38960d987756347aa150df06df11aa68ae1af78c619 \\\n    --hash=sha256:427b62d62d4f967fbb10b82a3813e4875c2a6e7e7634739f17265b650c7f65a6 \\\n    --hash=sha256:436e11b4dd966afe5f7f665e4cc4c5485ffe3ceb42f25a22e1701d236abf1853 \\\n    --hash=sha256:43bcf2a871f28f16135545415cab3ec43904d4c80425a64598a9e6cebfb2b5ba \\\n    --hash=sha256:43e5f9169e73d0f0db33b5f6b8554bcce69ac278c966daf83d5eb4eb2f13829f \\\n    --hash=sha256:440c401e146ce64bdb3beb8ff0c84677b6f21307c28a34779071cecee5d4d70c \\\n    --hash=sha256:44ab12e8cd17d4f001769f00ad465208b4bcb897ed29e65f058f74466b57a98f \\\n    --hash=sha256:4528cf80ebbbf57d40edfb31521ae265daa6dd636d615b1cf0ac86209579e59d \\\n    --hash=sha256:45e88111ebe331de478ef8d4293efbe88f3cf8b863386c9a2357136b838e1af0 \\\n    --hash=sha256:4741d42d59e4e5fa1a86c17ab9c27dc8ea459c700d91b6742fdb9138d9a516cb \\\n    --hash=sha256:4751f1d7eecae6b2d2a773630f1a7248f125c9a92a456694d03c15bceffc9d68 \\\n'
  '    --hash=sha256:488ca5c5e28ef56ec4bbb12f835b3f1cbecc5f3510062e70117bc6594851932a \\\n    --hash=sha256:4972332c079d6aad69c4620a68d015a4ecb33141583f70d642cf9edf6a713763 \\\n    --hash=sha256:4a252fb862b0ae2590587e625f47a0e03da05cf0205e8830b67b6596c06038b1 \\\n    --hash=sha256:4a76345f5aceb4ec404918edf9c7f2b5507db864dc0d7455982009ac0890b57b \\\n    --hash=sha256:4af350bc3f329970c0e3a59af84a8a30998bf8a9167eb50cd48e59baaa1d7bec \\\n    --hash=sha256:4bbf3ff651e0f1a19beb5d0f48e0874a9bad2482a588c9d214c96ef1fff1cd9c \\\n    --hash=sha256:4e5141543c7f7fe3087500bbb4ac2845cb528a980aa91f8f1e661e2292ff4a5d \\\n    --hash=sha256:4f5e5c6df4b703afcbe9352d238a51efd97c3b91fdc3a2052e40fdacb1e7505f \\\n    --hash=sha256:515a822c73abbf6a0b7c70976d9662be342835c9d78b8dc7c023411f39c35dbc \\\n    --hash=sha256:554f87034635bcec47c5d72447bf3db7e02da1bf493a0ada010db28a76f891c6 \\\n    --hash=sha256:567cbc630302a46a8ecfd943b309ccf5372bb3718f1f3762d452df30f033bcf0 \\\n'
  '    --hash=sha256:57d7fa8f23908d173001c21a9e82bfc6ad997d1b6c270fb121812b7ed158891c \\\n    --hash=sha256:5adf927dca8c47fde7e683fe69efdd81bc865c4db1fb6bb00b391e2b6185207b \\\n    --hash=sha256:5b7875ac1a2edcb691f27642b8b94b904baa6bcecb7d79c72df2228ba8cb5c51 \\\n    --hash=sha256:5b7979f71d06ae45a769de0699900a246d8cb632db1e8bfdc79ec019063a503c \\\n    --hash=sha256:5c2d525a3afabcd8e3549d85fc7e111fde6bc302d06a1893fe73adb79823415e \\\n    --hash=sha256:5dc434c946012e6d8a72b10f970ea30755b718251dd7591dbfdabafd3bcb21bc \\\n    --hash=sha256:5f1ea31d61bcd2cd2f3ec4ca80a64187bbd7948f490b63cf0dcbc6e717b4c1e9 \\\n    --hash=sha256:62198213fc3e0c56e567894b318ba45834e007d065f84ba6dc9165d21546fc56 \\\n    --hash=sha256:63aa52659bc32bb9bd7cb5caf523b4d14429a477762cfac886132d687c1f80fc \\\n    --hash=sha256:649f2682c090cca1ac4037866381f3652eaacbd56e5178030f4ce1325b8f945b \\\n    --hash=sha256:67e57b834e07ed973cee7b6da1548ff28a56458d77696fd2a5f397f340694848 \\\n'
  '    --hash=sha256:684160b3c0a9b62c6f0de90f44e11dc5d8643dcfa18a5856b45fb1c47478bb71 \\\n    --hash=sha256:6a8c5ce76b94ba49f3be8a8f2611abc6564210702c72ac9e237ca2bebfd17794 \\\n    --hash=sha256:6a9f98af872355e0c02439e48583958eee00e60b928bb20476460d9d40cb7b4e \\\n    --hash=sha256:6c45258a37fc22721395c09927cb982d3e7a83607cab15be7e2416501bd3a330 \\\n    --hash=sha256:6cbf4e21ef0890804b5bb9ad25c48f9c127758d7f6c66bef374efcacc63c738a \\\n    --hash=sha256:6cf633df84d80a1668fcf61e330791dae46825e395549e7d34f376411e75088a \\\n    --hash=sha256:6efb8f21cc136c79b3e5bb747c8682d37916fb202cdbbc32182de5c4e47f821f \\\n    --hash=sha256:70129ebb8f20e1ac1da58b78ed381624bd689a43a9a7366560bd8fabea145105 \\\n    --hash=sha256:704381264b36a18b9c62ecbabe2e71d0fc58c77c129c15355c989b10bf05b6b0 \\\n    --hash=sha256:7236be540d6be9ce448d98b940dd26ddf70ca41012e8a14a53fd9354cefe4e8d \\\n    --hash=sha256:72f34834518157a75e7090f328ee7a16c70c804cfc7c694fa069cc888e9fc03e \\\n'
  '    --hash=sha256:74379a577a9f3b6afbdedf1b90e5c7764467051977f18a326d7d607336d743bd \\\n    --hash=sha256:74a164e8b63f1e9cf35c9a7809d082b033d1a00e7375d5d814415436e7867e57 \\\n    --hash=sha256:760de77279e9cf9c81d012ce0705cba13afccee9b09c480f17d778c8c5cefae8 \\\n    --hash=sha256:764b32d52d15b8b95ac8160e540772fa1adeb611fe40bffaeb42e7bf98279e44 \\\n    --hash=sha256:79a3203aadf39637869dfea1185227d8452844d78b837e54fb1117b4d34ba5c3 \\\n    --hash=sha256:7c343ee174d417a44d0c3355602c0cbbfa52a04d1bbbf1723378c7d2c8f60626 \\\n    --hash=sha256:7e27dbed5c4ba033919e4b4ed8dc14e029e91d14a93cd9f920d25277c7df6781 \\\n    --hash=sha256:81507a68ba84c55241fb61cce1469f473a5da4205fc8ef6f698e5948eea8dd88 \\\n    --hash=sha256:81664268dba92e037b740ecf37fa02f1cab4a391f93f28e35792b3341c60648f \\\n    --hash=sha256:839f58c5bd9989875be0fd28446dbf32cace2c2cd8bf2f6762acdc38a95cd1aa \\\n    --hash=sha256:83b8c2013edb5dc1f9e7268b6496130705bc48d79c86bb8817b3d210b81a5513 \\\n'
  '    --hash=sha256:84df5f8da574caadbc0cb1b8866ecc2368cc941f0cd05f677756c802f370dafa \\\n    --hash=sha256:8580aab306888224074c7edeec734de0c3c5ccde65b2da4e6c9a5e28f7c0a1bd \\\n    --hash=sha256:85bdd40cb505a11e0ca04191711266c5fd696ed786ae83849955e457774edc96 \\\n    --hash=sha256:85e402dab0f9acd3604539747c6fcc57dc188a18af6ab07eb8189351cd32466c \\\n    --hash=sha256:863f3d3b44110f7243e86cf994aa5c5d88f2348b6e84ab4402fadadfbf9f7da7 \\\n    --hash=sha256:86b2b12bec60c678ed8f5cca0258ad93a8928ebddb6ca7732f0875afe1451d1a \\\n    --hash=sha256:87aa309a93bd5ec13f14309a305ff4e9bf74c5363fc46c264c0a22edfd5b0670 \\\n    --hash=sha256:87cbdec1a7dd930079671a60b249f3ca4e773e6fbd0676e21e36fdc9dd0f3b00 \\\n    --hash=sha256:87da13df72c5612771cd905a8b121e0bfea62d7659b1c92198736eb722220e83 \\\n    --hash=sha256:88d87719fe6bddf117238b341c5db851f8e96ba68ad9832b450e4a43dc60b37f \\\n    --hash=sha256:8b4477edc03091f51f5309406d230851c23cf4822029e3bf40b8df53093fff1c \\\n'
  '    --hash=sha256:8b99ebaf9e816ac5069423b1367ee7e8078fbcebcf62545506bb0608d2f4f468 \\\n    --hash=sha256:8ba782ca3bf1e81492611152b9a0d5264971339e95e34d69de0ac2c926be496d \\\n    --hash=sha256:8bcba9456242ebf180a04d9443812fd85ffe6bd12bda464dd116fcece8886ff3 \\\n    --hash=sha256:8c9fe122444e129881afd1d4d1c7ac0d3ce2d91b68c2b40173b6025ff1c31f9a \\\n    --hash=sha256:8ec4777d92fd61a5c8fdeddab894fd65bea301a8092fb5419ec6472aa4d458d7 \\\n    --hash=sha256:90cb2a1c9cc503a054a19612b48ff6e8e47805f618bdb3224a07568aad03a37e \\\n    --hash=sha256:9283d9dd6b44acad35118e2976fc763a065509e4118debdb61916ec322ed17b9 \\\n    --hash=sha256:94ac8a6b8c47951173f0b67bf862bcb971bf24e493b9fbbdb0e010cbbc7d9f54 \\\n    --hash=sha256:96d8de55029d42251945531f6aa7590c32b48163c66a43bf29d8657d7446a377 \\\n    --hash=sha256:96dedccfb09a73a25751053a183159b88f4ee75f388df8166040c152ac0531c6 \\\n    --hash=sha256:9761ff4a0ffa583fe850731ad24fe82c88cccb7a2294727db0955f3279a4cb3f \\\n'
  '    --hash=sha256:97b455de3e8b1b0b1e4594cb61a468992563f03ca264062fbb0a66b393c01d90 \\\n    --hash=sha256:97b94fb29abf21f5f0bde15f7dbdd3a4aa2dc59f37026adc7b4bee8563b84375 \\\n    --hash=sha256:99054b838b74d8d3995ea0d410976ae967c46207ae22d6ddfc535e809197dab9 \\\n    --hash=sha256:99166cc98637e8bf550cda2aab07f4f1d5f899c45fbd721801aeabcc9d404824 \\\n    --hash=sha256:9a51b061d54cda8b83e62c44458bfbf0dabbef9b975dd9649952ba5076b9f349 \\\n    --hash=sha256:9b1dddc257279417d93c9e59420d49ef90aece90d7a01996db3aade74b0281b1 \\\n    --hash=sha256:9c3c4b9aa9a27196b921197f7daf9e6c1412739df06a99cfa6e923879362eff6 \\\n    --hash=sha256:a14578102a6081465aec9cf73c76c3cd3f79f0709bdb3b8ae7ab0b54c9d8b089 \\\n    --hash=sha256:a16a3fa6936e36bb1414d16a6bd012c9033e5161b68b426805b61d895392437d \\\n    --hash=sha256:a33de7633c948ab2dc144af370a66e7e7af29b425dcd0f7e4f59689fb9391b53 \\\n    --hash=sha256:a43418e1a90b4809a9caf64aeb8b0696e3e1f300a323acc1e6ee2f93ae319fcf \\\n'
  '    --hash=sha256:a4553d36cc0b7fce1f35ba8a94dfd775aa3ed12f5eab2dc3b46ac75a0706b0bb \\\n    --hash=sha256:a5b21b42a01a343096a1c018d35e9b7aec9c7065dda53ae8da071e37478b2cea \\\n    --hash=sha256:a65785e653573fcd1e33062760ab4c3c3440e8e910765018e4b6ed4ad07b54a0 \\\n    --hash=sha256:a6671a8f6ea4f2101ce11fab5023a2e59391cff249fc3928cecb69d971525fd5 \\\n    --hash=sha256:a69e8946e4902ea11fc1c557740cdbfe7d75c78fcc5e4324ff89a696a634357d \\\n    --hash=sha256:a6e3653df1a70b8ac4191216324242e4be2bca18c9a7c10934e1bd56dc7ca15e \\\n    --hash=sha256:a865d2d470220e659220fdb59d5b6c4422802d8d6098e1324bc4d12444798914 \\\n    --hash=sha256:a949b072ea59c6eca0811ccd9e95133cc50d2afda8d464b5b077c78f78efa269 \\\n    --hash=sha256:aa6ccc7f31018484d652cf52db020003433f3c9fa83189c028bd807d2adde503 \\\n    --hash=sha256:ac0f291ab6485bd71f33941f9b92771318332a05d505460b41e893a549caadc0 \\\n    --hash=sha256:acb31ecdd1a97fab5cd39a84ee9f515e727d319f796fec48703b8339b9998360 \\\n'
  '    --hash=sha256:acf52474b2494ef66dc7e0fb6d5e2b50c18313039ad4d275fbf9f9907c804bc5 \\\n    --hash=sha256:ad889d58361a26ba75f5d6a1a0da08ed4950ec4ac8a6da86e1c5ce1b95ccb43f \\\n    --hash=sha256:adbd48b30e3f82c89fb2b3e6a87cdd28d113b190a5ed0ee2dee286323ee9a621 \\\n    --hash=sha256:af05a3f650220a6c59fa0ad2410249f2d2470a05225807c378fb67458693f8df \\\n    --hash=sha256:b3662719007e059abde7eddacf8517142ba076ddc7b30c807260e57d28c3c191 \\\n    --hash=sha256:b3bece52127ac20044311ee73567f9f0893b5de64f9028aecc90cc740cfd525a \\\n    --hash=sha256:b4c8842fb19d78b5e8c2a52baf4c8357658cc56c62bc822b86ce0f942f28e286 \\\n    --hash=sha256:b659fad79c99b0238c7ad7e9d7dbf4eebfea9097c2dba65fa0a4d18a25b29a2f \\\n    --hash=sha256:b6c1f9c59bbe593f88a0aad30be4150f15bd57bd64efb95feeabcb8e563f1ecd \\\n    --hash=sha256:bdd16718b63aa3ebd68aabb79021a40e47c81374852d41a306b9453141bbcbee \\\n    --hash=sha256:bf430c587f447a554c53768ad76b9846fe7c5632180ef6f69c4fce8b0552fbd0 \\\n'
  '    --hash=sha256:bfed61996d618eb90d6eaae0178002e3466a28b06bfc557a7a3a7266378d8c5a \\\n    --hash=sha256:c09ada495567c9c9a8156c5ebcfb93be7fece0755062d738c972dcbecd0d84b5 \\\n    --hash=sha256:c0e6ccc2b19ec8a726b2e26062ac71ea63e15500d6bf85910e42481844fdffc1 \\\n    --hash=sha256:c101180495cb4ba3617b279a944345c53a5e73b0c150053d1fa8d8af32de9579 \\\n    --hash=sha256:c10b9206753b64aa791b35b201485477525b26fdec5bf86e8364c388a03e2592 \\\n    --hash=sha256:c3074db513c81f764053e3da079312ecf85a50d8350c71f4cc0105d9662a9e6c \\\n    --hash=sha256:c30dd1af66a820820398b26e0d74e7a9aa43cae705924f23ed828cd8e5c26c3d \\\n    --hash=sha256:c57963970d359a72262f7fe6be88f945e2334d4bc41462b7f08c37b0abf35ca6 \\\n    --hash=sha256:c6301d92545c591ad31c3e050aa40a5f8a4c16413f1f9e6f9322c6f0f9d2b736 \\\n    --hash=sha256:c6370189e8e66b7e608f533b939a9de092ddca6cce084ca0d3d414d2ed5b5d59 \\\n    --hash=sha256:c6fc415b5568bd9accc7187f1729a99707330c0a67a8b9f93c1149ed573ed75d \\\n'
  '    --hash=sha256:c7484fea54964edd417cc3a104d5180562514aa7c4e2a2bc26d776ef0c4cb4a1 \\\n    --hash=sha256:cba763d84b06bda2c38d5185dee76f1b9dfdc0789e96e476d9e10005526d0788 \\\n    --hash=sha256:cd878d32f5c6cbce9783f8d6897561fb772211edba9dde49d85672b88ed45276 \\\n    --hash=sha256:ce6d5cc94a50291d080259a126cbf1e9ba4ac861e6429d2f3cdbb1474f51945d \\\n    --hash=sha256:d0d24a4f3fb63852cd09af46ae4b7a4d00cc8b8615a046dca543786e728d1056 \\\n    --hash=sha256:d1e0d1ea6e44f51808a9e8469c8afdebcdf6fa23d1ea524a0303d57d23919712 \\\n    --hash=sha256:d54b8ae068af532c8cdf56abb9e09a60fbe7b10792444c9c27987bb6d3b450fa \\\n    --hash=sha256:d55bf4ef10eb09b8b6866790e083d26d087d84caa3cc0946ba87c3ca7ecaf7b7 \\\n    --hash=sha256:d9f3848ffaf010bdbabdbf4c25641fa258b6227ff27bc74a4d06edef521a4873 \\\n    --hash=sha256:da0264844a09b538c894e5eff25313d941deb4dedec2131b98418a71a3c9944e \\\n    --hash=sha256:da544672efd9ad76077928a3e6c5d894e52ce82d3bf14002db4a1bf17d1a36a2 \\\n'
  '    --hash=sha256:daade8936c4deaaf7b01561324ce438ba4f885d717e9adc62b4d67212ad7d7bd \\\n    --hash=sha256:dd649663ddeafbfd4734eb8abae921dd5baa1242f20bda54e8bc927369ccded4 \\\n    --hash=sha256:deca2a30d983d240b8375ec2ee0a4288e72042827fc61df2f7671f8467e4cb2f \\\n    --hash=sha256:e259bb7e1e2d8de6b35f430f5c7220b1c0ebf3962d1ba7ec7545980d5931edb8 \\\n    --hash=sha256:e3996ff9b6f99180357024336bf5749a8ad6476a9a2523e535c5212b995b12a2 \\\n    --hash=sha256:e3eba72f9bb84fe696516f4cbca68d3d74a376157e68bacddbb7f2516af61523 \\\n    --hash=sha256:e4296fcc790876a8b0f297edc83d3b088457b774d8f67b4636807f8a2ec69a79 \\\n    --hash=sha256:e53926e76131a74e79cc0b39fa712c227875f180afc68646bd1e1d8a17e60313 \\\n    --hash=sha256:e681a6fc7e4f715252b9b5acfb30536ec7dd1f75033a32dc617e6fa95af1a3fd \\\n    --hash=sha256:e71b34978e77868cbf2d18c5206a4603f9c644dd7181bec5643bd40141d3b8c5 \\\n    --hash=sha256:e8cda075b10bb3917b002c74a04f9e02b7d13b5bf732571404d51c52b11c7329 \\\n'
  '    --hash=sha256:e90b4bcf1d9eb1010fdaee7c9209fb667e74c0684f3ba17f9032bd7319da90c9 \\\n    --hash=sha256:e961093277ff9d42addb9dad5614dfb7800ccba07c245c39c8e9b4daa35d160c \\\n    --hash=sha256:e9701c073bd062fb6bf6be51b47186ad15f1e87feedf4ea07198e0333ec068dc \\\n    --hash=sha256:e998cb3685b92101ec5de0fb4d9485cf01e50bc418211955c55d98064664cf4c \\\n    --hash=sha256:ea5ecf800b45bdb34afe05a1d0dae1f8ea02a290e50636dccd399063f6b180f8 \\\n    --hash=sha256:ec1a470c6db94ac4589c203921e89ac1bc13e796a8b1784d8135e1893559cd3b \\\n    --hash=sha256:edccc2ec58435a580f96a48a3ccae8cd0a480824119165dd90108718ad81ae6e \\\n    --hash=sha256:f00330ac7e24769e2032203f2b01794d670916b0c1799fd261340f1af9499875 \\\n    --hash=sha256:f09ee747e2a5f876cc5ad56947734811828335e13b403dd8ea1e06d77a9dd48d \\\n    --hash=sha256:f18732adcc271741bd651c3e56fa519d8a237d2cccda01fe3afb226bf87f783b \\\n    --hash=sha256:f1b603d0686c99fa0879f104a74e7db58367634c6e50ba827bee9aa095e23205 \\\n'
  '    --hash=sha256:f33cf0baa91eccd2cb7b62bf00f10c2264ef578b71dd33a12962e71a36eb4d32 \\\n    --hash=sha256:f3e1a44af01b6692de0ec6caba5f0bf93ceb36896e02b7fc00952c6ea7ef39e1 \\\n    --hash=sha256:f484ed57bb3e4142f9d6439568658c38be5f94b702ba00a1ff32c69783b6c66d \\\n    --hash=sha256:f5d031f35962e5483a613214e61f09fe24ab523062c3646d592dc16c4a217451 \\\n    --hash=sha256:f6247f5e23ee94f2557ac9dab738a336f607c6ff476fcf66ca70c3aef5eee15a \\\n    --hash=sha256:f7db035447a0ac8959aa230c5d36545ecf9f547413eb1711c0ca6f0ba1418925 \\\n    --hash=sha256:f83295394d34e1287e5b30fcc496c13b92cf886a131f3dae5444e38da8757efb \\\n    --hash=sha256:fac4832b638000106207bc44e44b9616a6a416aaee56c62b01d61f3705e49f58 \\\n    --hash=sha256:fb59a0dd61fb2ad481c03fda399d78ce57dab6bb62c2c8fdb446a7ba4754b89a \\\n    --hash=sha256:fc737c05ca2d48e5dcdbbb249314df3fc6c2a0be6da8b0aa28e13d72afaad7cd \\\n    --hash=sha256:ff48915bf1871a1f19f74c11834c6329443d306cedc0c05fe7fe617810422a80 \\\n'
  '    --hash=sha256:ffa44b4c7c5d0ffa31356b4428659516c0e47647825c74079a296b3857b6d99d\n    # via datasets\nyarl==1.25.1 \\\n    --hash=sha256:0136d640dfa9b0523853e411430a99f8a91eca85774c6420285a33b755bc6de3 \\\n    --hash=sha256:03dd38de09bc213e9a8b29761eec33ee1d5318dac0e49d8af36e4d27830e23a7 \\\n    --hash=sha256:0a191bfdb30a79b98e5d175d75285f9fcb78bf0e46ba5efda042e1c72071a0de \\\n    --hash=sha256:0a66db89ea473abeac4b70523cafd94db3772380e565f9d28af7a179b7af71fa \\\n    --hash=sha256:0ae12ff2b805fa02c4dab838005caef735e39986322698c48588d3beacb65c62 \\\n    --hash=sha256:0f12afda4eea8c8994a76d4df1875c765194f5fbe8a9d197929ea303caee29ec \\\n    --hash=sha256:10b2fd95332f0d716d5eee3c9fb2ce8eada19082de7fee83d32e37992fd75c26 \\\n    --hash=sha256:126a2533570c554719ca40a1288fdee1700b6bc82e7131aa69fa85252d92e651 \\\n    --hash=sha256:12b6bc4906e11f5e1a1cdcb12296e7afbd366c783cc8073403cd2fb74334e453 \\\n    --hash=sha256:142c06c4d6a35ee3ec5da08499805e879cb3ca7c1fbfbecb0140fe72403818d6 \\\n'
  '    --hash=sha256:14b79a30a93a3ce2e8832603fd0ab780ada281b0ba5110b519a634f2d7d7d1fc \\\n    --hash=sha256:17c9877a89fb6e2bca6f9087eb24cd7fb434653946ef5075e470d23d49b52287 \\\n    --hash=sha256:192a866877a49993949ef1975864ad8728bea28ee810f6abe1a0729c2b500426 \\\n    --hash=sha256:1ab7618921a93767387a4b83776f751588f5b5ae9bb5bc96620e2e2e00bca868 \\\n    --hash=sha256:1e80dcf1446e1b080b1932b0d103c464a04112f5bc31f0f983ad418172063cde \\\n    --hash=sha256:1f51020b2eb8a003c84925638ec63c21a750a4bddd3a22ec8eac6a742dadf1b9 \\\n    --hash=sha256:1fb2a01ba8cd9c5d2c5dc1ec35e0fc951d04b4f037541d4ac090c993ce58b3d7 \\\n    --hash=sha256:2239a02249d9326655419e0168a28ca9008938eaab31dc29fc875c217927a6c0 \\\n    --hash=sha256:23bf5b403c879a54964e0feac7285688e04bb220074878d737d331522da0a5bf \\\n    --hash=sha256:24ce942011a61953e7d313438038f4d32ff21387b775f58a957f7a07dd55ef95 \\\n    --hash=sha256:25868beca8b6765f8f7d0e11fe6dd7c66dd4b0793b9500286d20cc92352126a5 \\\n'
  '    --hash=sha256:287e99ff5aa4dc1c7630bfc683ded6f106d756c99dec432a2d7f197a784f51c6 \\\n    --hash=sha256:29273edf1530e397bd07cb784db1fbe0d2590b77569f2e24679a9c0a2d763b94 \\\n    --hash=sha256:2b49375d22299b0a834c2bca72f39aaecc270d96fb24c30424899676f487b22a \\\n    --hash=sha256:30eec96e8a91bd588ce897c9543f6d5d8d34b28fbcba28a4dedf20ebeae9fe57 \\\n    --hash=sha256:319e070a01db9920fb63761843f96a104c8e2b9427266731810dc1e22595b17c \\\n    --hash=sha256:35dcbea443fafb3eece757ad4e514560ddeb6c34cfae1582c620d7b293d7feee \\\n    --hash=sha256:3f4d48a6112712973e676bd792121fee470e432d749177162d9949d5c9460a1b \\\n    --hash=sha256:3feb99222553a8cbedfa52c2f59dd84c3f50d5b582c728d522caf8d72769a54b \\\n    --hash=sha256:419f392a1da624877975709e3864dfe833af6cc7671b39318086d456e288380c \\\n    --hash=sha256:42a66563d8cc056ee32e6191e05097a7b2b3bc302e0bc3133daf8710eb18bd26 \\\n    --hash=sha256:48796ea00a303961507dc6c8437c4b325a6fc3f95f7c36c71b91ea9a8150963c \\\n'
  '    --hash=sha256:4bd6340d20ae2c7ca719b87b426e808e90743b676d05d4c26c4fb5ca71f41184 \\\n    --hash=sha256:4ca89e4e21854ed27ec753297dde84b16c9f8e53b14a4866fb44457d643c19f8 \\\n    --hash=sha256:4d781294bb815ecb5ea57ff6bbf8038e0a31a95fdf3e1788f66e0dc100d64b58 \\\n    --hash=sha256:4f1c91f5a5980a937ff8e238e98e6897e1ad74a4b1e2c0d68c73b5ffbb3f5c0b \\\n    --hash=sha256:564fdc7085d2245ab84f88882fdb1d6ac0723124bff6ded35bfb1c00f812630d \\\n    --hash=sha256:59ba3a6e1aa8cfe5adf4bd270fd965db21955401b7ca6f1696010c55ed4daec2 \\\n    --hash=sha256:5df89f769cc8ff94c3d7e7603386fba309d25ce5240132d26c15baa8d0e96c4c \\\n    --hash=sha256:632da579b2d879f6bad20f2cfa35ded1efe2f4f77f8abb26a6234a5b236acd2f \\\n    --hash=sha256:65b5b2066651b7432d389e9799d979c703bcc6ef44266bb8153ef54e91e4aab3 \\\n    --hash=sha256:664ec6a520b74a1df2810666eb67695fcb77fa663e6ea0a25aaf2e529cb24dfa \\\n    --hash=sha256:681c758b0490f9e96b78e5fa8e8dc6e648e9185bb6eaebe73183c33ea0c445f3 \\\n'
  '    --hash=sha256:683e362b8ba453080f7489c66f4ea794e751c35b72e7eab3575ef784c2fbc7fb \\\n    --hash=sha256:68782fdb4027b8d1eee25ec35e9a6db05e863b899eb0310b3a33b6c3fef55707 \\\n    --hash=sha256:6efaf45df6a849cef613a03a94c845647456662f85438c886bb67a9c027c8c2c \\\n    --hash=sha256:71f42c5b9a948c113bbdebfa544598321431d064ff959d32e99b1feb61d68345 \\\n    --hash=sha256:72849d892954be4d09e569b8b831ac39ce58417fedc767d4308a0fe542018a40 \\\n    --hash=sha256:72c34ac7ad4314c19362d5ce27626dcc8429bd30bbf8c179f4234078851f9492 \\\n    --hash=sha256:734f6e5400352ac4254456003d462866c684703570929cff7a7bde015d0cb371 \\\n    --hash=sha256:75baa6cf9b6d1c52f3e111a130e202fd8cf0a5b3a066c3f73d615e885092e4ec \\\n    --hash=sha256:77716e245c90f058466a05e6a465bb8600f767a8f4b18b4d40f3aff958e5f73c \\\n    --hash=sha256:77e5099b99b37f3cf79c246998ca9f7313a78054cd1809ec46bc1afad47e1c4c \\\n    --hash=sha256:783dd1467083f4d3f7722ad6a313f24c173e7571372738fcb7a6e6d1ba48df25 \\\n'
  '    --hash=sha256:7a5c3115595995779ee21f2567035793911c3802a43c74f3fbb0314929ec67ac \\\n    --hash=sha256:7c88edaec8c349ad4c5ad4c486a3defcc4b80ceb2f074436ffa0a87caf5e76a6 \\\n    --hash=sha256:7cb414a73e21a7ab58254926073f2930cb22f5b4314ea4260a687e2b3fd4dce3 \\\n    --hash=sha256:7d42e7e3ca399555578b4d617e3a6ecf13371b3743a115995fa010c7bf341459 \\\n    --hash=sha256:7d575b54cb3863ef9bc290ea4b009999d55dc237326131e4853cf33e888fee03 \\\n    --hash=sha256:7e4de3ac4adbad3d0bc7c6f4360a7dbff5de2f15e3b723be3198074e17fd9c40 \\\n    --hash=sha256:7efc9f082dfed77c316edffa9deb52888e1bc6789171887cc1f68e06d65465c8 \\\n    --hash=sha256:80a063f8297fc796296f00f100be520f209b23dc98f93ce8eba6ee7122598209 \\\n    --hash=sha256:80e47012e730da131c9f059c80936783f9659aae22dc31c03c0595590d11ed54 \\\n    --hash=sha256:83d4a37e4b95da4d8bda930d6d35b75b4cdadbacbb4980cae290ea3100b5d51d \\\n    --hash=sha256:83e9f4a25085bd4b7214701a0794ff1f50fc633ffb8bdfebf07abdd81c2db126 \\\n'
  '    --hash=sha256:85a18376073f8a39aa07be34f9fc77e2869aa72c55c441efdd2cf79a0407504d \\\n    --hash=sha256:87796fedc3ba97ec14fab55acb48584276e6c1e4c1e89c422bda62c838e754a9 \\\n    --hash=sha256:882569ff613758cac762a457a5d72d6e211b28d4bcfea89d1d71ea942b02eac0 \\\n    --hash=sha256:8ce4d6ccafb33d39bd78444612d14938ead674c25702ded2ee9c54a47735d225 \\\n    --hash=sha256:8e7d98cdbb6d71e726f7d525952867096053d1f290dd4e3c50d7d313a136f414 \\\n    --hash=sha256:8ee202350cf57abf0e9502a41601841019c25d3db7ff52d980aaf31446254059 \\\n    --hash=sha256:8fb0eb4955adf0579001581f2f71a126e8781ba61bcd120f127b0401163c6c2d \\\n    --hash=sha256:90c30ed53546da833c700115c0064c22120d1b1560f474699fd31f22dd668233 \\\n    --hash=sha256:9489e6abf47ba37f332075a91444c7cfedb03e6ce99fbb2f116bfe1ce810da3b \\\n    --hash=sha256:94d7aa6debf92a1dd14cb5280b083a764169a13cfb23a452111160274ed989f4 \\\n    --hash=sha256:98d370568f393215d605304cdb77b3d5539bd192c75b623c7304c42c8d6d8273 \\\n'
  '    --hash=sha256:9b1bdaae98bc016825dd3c9d8ee1832f829b3341f9cc6ebd1a1b0a7fef7367cc \\\n    --hash=sha256:9d693bf4bf534e9ba3ae2780cfd577f5135629f7b5ac653490859d0b77864865 \\\n    --hash=sha256:9d6ed3d17bccce4c05343e1ca8da13bc5c02c812a4e7282ddd05e8769322d3fc \\\n    --hash=sha256:9e23c82b63cd7652fc24d33ed6cc17099d607aa3b4fc4ddc75e95062f3d82df4 \\\n    --hash=sha256:a1daf47cd95a7c3a63456336bc5aaa8c86dd3a47d07ed3d0e76132ae4666a5a1 \\\n    --hash=sha256:a1e32763e641a1566507d90a8d3b19bfc3cc04a9d4e5ae3e32189874ed4b58a3 \\\n    --hash=sha256:a2059a2d891bd156bc5184e7ab7a56e78a84dfcfdeac8c501b552533ad1c36ee \\\n    --hash=sha256:a2ed0ba415ccdf08f14bf544cb78346d0f76086707ffee24921a2c84dbf1305a \\\n    --hash=sha256:a3faadac7d812ddac258feb57b9846b60c1b437c4f4b9ad42595c6f6fe4390df \\\n    --hash=sha256:a5877f2255aab518ebe528289037699201d5dc5f045f2396cb30aa02db22f57f \\\n    --hash=sha256:a78b50b4f7918a3de71105d5c0b93bbc57bb8339a4d03a9dfd449f9068e76f3d \\\n'
  '    --hash=sha256:a8c2b841478068440d8b733005d13a5ef535b9928cbc05f17182d410f32ba449 \\\n    --hash=sha256:a9ca696eb02e5c02a8afd872ada510eba9b7fe6e68b9572c2e9a9b1941e31e2e \\\n    --hash=sha256:aa4ed3dd308548f9e707d9caaf005d2d7f8c1e7868f858dfeb47fe76e16b391d \\\n    --hash=sha256:ab2054c5531af2a9ba7b69b8ec91e4f884420e83a8c5e579b013084cb57e5e5d \\\n    --hash=sha256:abb1384477f5901d436b5d2e5465954de46ea6098f59163d243660b5c4461d35 \\\n    --hash=sha256:acae6b45d1ace09b6ba3876da43b88366ef368f73b988c7f57e14231753d4420 \\\n    --hash=sha256:acfa7e22aa6c6e7a5996a41d275bfa01efa7ea56ab890590280e9063e2cf5c1b \\\n    --hash=sha256:af4ea5b37403ef4e30f3927eaed540db942bde01d8d3ff083527c0704d1c9c68 \\\n    --hash=sha256:b10dd0557ba422715b5206b3743192135a6022acca8baec51aa127d0a75db8fe \\\n    --hash=sha256:b13b88747769537f3d32e89e3a735da10c0a9e35d7322928c701b5f93d3afffd \\\n    --hash=sha256:b51c159a9794633f5e0db7ecec7b2b6e3734eca1f5d17dc989ff3552a43ff78b \\\n'
  '    --hash=sha256:b5402a340723fa7da00b5cff987ddab61276be6d11251ea71ae02bcac54890d8 \\\n    --hash=sha256:b7abffdf37af1cec6a2ad69b827aa84320db5894791bc8ed932dc93fb274b7e9 \\\n    --hash=sha256:b8075fe90bc08e40b8b8a1874fab42ee4c7b56af05c5886e9cc841397f916908 \\\n    --hash=sha256:bc3ac7bf569f6b64dad04dd7808c7872dae8a97df657856eac05e9b7e3614a85 \\\n    --hash=sha256:bd0912757081f89b107d6c00b2ff8a194401b0b87eadcf4481de2b865a8fd44f \\\n    --hash=sha256:bdc8d8b8c22e9e43ac68316b5e6cf083dec537f4ec213cb4aa967b583bc3fa64 \\\n    --hash=sha256:be80550d9bfe83d9b62398a37081a90434e6df2d978ec345c3d2820de6beddab \\\n    --hash=sha256:c6f117789d22dce188e5754e8bc65b7e6ebf8cb73963b9fa761f672a5883769d \\\n    --hash=sha256:ca32926d7d77bcc8838425c4c95e040a3ace1cb7dfdae599013458dcda2607ca \\\n    --hash=sha256:cce0727fd5ac04d372fa9bbfde9febc2bcf209aadfcf0468e45dec72719895d1 \\\n    --hash=sha256:d0f1489233a254bb3643d2f05de7d59019254d81daeca6b9162fe9edef57e0c7 \\\n'
  '    --hash=sha256:d1c557dfd5e3db046053a0bdc72261ade790ebe8e2c7a41b36b0ca1f14cb95f3 \\\n    --hash=sha256:d21f0fa80a02d05299207eeaafef345d812ace96d5306e4ef265e1d419a615fa \\\n    --hash=sha256:d45673badd08456d0340e9364eddafe1c53a9d2896424294de4d7dd71ad3ee57 \\\n    --hash=sha256:d5add7b4ca7afeea91d52e4d4e4db3b1fe9885b71f07054560d8c4296b7441a2 \\\n    --hash=sha256:d5f90e44653c4e0f78501ed9bb7d3fce835a8d62b7c6ed0cb16557534087e743 \\\n    --hash=sha256:d7306dee25b8a0e737363f347362b875094b4dc4e367311470656ae420fdbf8e \\\n    --hash=sha256:dbcef5a9119ef653653132cccaf999b30a0af6f33bb0a4ba80bec30056868487 \\\n    --hash=sha256:def538065f9e4d4cf1ae164bd59aba00dfa84f03923e0de4c3788f252d6bcd17 \\\n    --hash=sha256:df23df54b5114a17c2d0ef192433e2e5a9f0c5178c32375e90b7cfc965f349d0 \\\n    --hash=sha256:dfbf531053a0935f2e871bcd4753f90313688772ff8c017f5ea402e315a78c1f \\\n    --hash=sha256:e029648f9c951db30e98a7d7ec90835db88ec4b32820efe2a9bdc2287e032eb6 \\\n'
  '    --hash=sha256:e07595c7d6f4db270ceede356a1bd1c07a34f1c26f958d1ed0cd7b48e0d2bba3 \\\n    --hash=sha256:e12c538e00e7c1b286a07061046b90e8124e6a9793efae2c70db6a4aad07faad \\\n    --hash=sha256:e546fe1d4a93ebc2910f0d768baff19faa09843ab3f2036a67ed6e69fae4419d \\\n    --hash=sha256:e5637ca8d0bd7fb72648a6c7934af4baaccb697657f7438c9d264fc2abb8b0b1 \\\n    --hash=sha256:e636b64d24fd9c38053c5e389a1174c66361fa49dcfd220f4dd35b4abde7cb89 \\\n    --hash=sha256:e7011b8fb8c4054bf0c12e5edc6cd83778b0028e99ce59b18586ed036f92cfdc \\\n    --hash=sha256:e80f557716fd765439577131e526b8942ffc2c07bdbc5e39fa62f660ba1e963f \\\n    --hash=sha256:e92b6bcc741b86d67606c40d3cb9c7cc8e6c737f81e31f4a94efc204456c92e3 \\\n    --hash=sha256:eb96ed1ae6c7d072d60840c0434aef07a2df611812810807fbc54263a6053e9a \\\n    --hash=sha256:eda19ea5ee88742f47a2340816e6f2d40b53bed3ab5b69794769f36af9f35bb4 \\\n    --hash=sha256:ef74070ac553c59eb4f04258722066d6c6135b7baa03b2e9f2da65c096e96d98 \\\n'
  '    --hash=sha256:efb01a106f971cb3752856bca2318bbdf7f01bd8823779c461586cbe5ffd5258 \\\n    --hash=sha256:f074e8d4aa0a5798920ddb6de3d08b228c614ff3724c3e8bd7577f4bafea867b \\\n    --hash=sha256:f38a70074041d3b7e138e452799f5174198bae5bd5ab2000917badf403908c5f \\\n    --hash=sha256:f41753a76f4f63927d03a0d8ba8f5ce0f2083bec29a8cfaccc55371b1564b96b \\\n    --hash=sha256:f53dcd26694f148f738edc052b5a69234833e739f10f4c3287bdfd8ec0f7b326 \\\n    --hash=sha256:f61964f235a43738bfac50da46fc4254943a7eea3051aeb0b6fc7c992c29fadc \\\n    --hash=sha256:fe01645169a2112aa1d4ebc3e4c5f029c5c8f97adfc32e5d37c993b39a994d75\n    # via aiohttp\n'
 ),
}
CARRIED_HASHES = {
 'table_intelligence_workshop.py': 'de50ace70683906627575a7b6d577ba6441321064f9c7f20dafaec3cded89fec',
 'requirements.lock.txt': '4041c3faaa0865421fd2a80992e569833b1ade61952eb11a1492019a9af1c51d',
}
WORK_ROOT = Path(WORK_DIR)
RUNNER_ROOT = WORK_ROOT / 'runner'
for name, text in CARRIED_FILES.items():
    path = RUNNER_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8', newline='\n')
    if __import__('hashlib').sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name)
print('Carried stage file and hash lock verified:', sorted(CARRIED_FILES))

Carried stage file and hash lock verified: ['requirements.lock.txt', 'table_intelligence_workshop.py']


In [ ]:
import ast
import hashlib
import io
import json
import os
import platform
import subprocess
import time
import urllib.error
import urllib.request
import zipfile

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This notebook builds a Linux x86_64 environment: use Google Colab, Kaggle "
                       "or a Linux x86_64 Jupyter server (Windows and macOS kernels are not supported).")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "none detected")
except FileNotFoundError:
    print("GPU: none detected; the canonical runtime is a T4 (CPU works but is much slower).")

ENV_ROOT = WORK_ROOT / "env"
PYTHON = ENV_ROOT / "bin" / "python"
LOCK_PATH = RUNNER_ROOT / "requirements.lock.txt"
STAGE_SCRIPT = RUNNER_ROOT / "table_intelligence_workshop.py"
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1", DO_NOT_TRACK="1")
# The kernel's own settings must not leak into the isolated environment: hosted kernels export an
# inline plotting backend and a PYTHONPATH that exist only in the kernel.
for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
    ENV.pop(name, None)
ENV["MPLBACKEND"] = "Agg"

built_from = ENV_ROOT / ".dimer-lock-sha256"
if PYTHON.is_file() and built_from.is_file() and built_from.read_text().strip() == CARRIED_HASHES["requirements.lock.txt"]:
    print("Reusing the isolated environment built from this lock:", ENV_ROOT)
else:
    UV_URL = "https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
    UV_SHA256 = "aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(20081405)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    if len(wheel) != 20081404 or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("uv wheel size/hash mismatch")
    UV = WORK_ROOT / "uv"
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(n for n in archive.namelist() if n.endswith(".data/scripts/uv"))
        UV.write_bytes(archive.read(member))
    UV.chmod(0o700)
    print("Creating the isolated Python 3.12.12 environment...", flush=True)
    subprocess.run([str(UV), "venv", "--clear", "--managed-python", "--python", "3.12.12", str(ENV_ROOT)], env=ENV, check=True)
    print("Installing the hashed dependency lock (several minutes; CUDA wheels are large)...", flush=True)
    subprocess.run([str(UV), "pip", "install", "--python", str(PYTHON), "--require-hashes", "--only-binary", ":all:",
                    "--index-url", "https://pypi.org/simple", "-r", str(LOCK_PATH)], env=ENV, check=True)
    built_from.write_text(CARRIED_HASHES["requirements.lock.txt"])


def run_stage(stage, *options):
    """Run one stage of the carried file with the isolated interpreter; stream and keep its log."""
    config = {
        "use_byod": USE_BYOD, "byod_path": BYOD_PATH,
        "detection_threshold": DETECTION_THRESHOLD, "structure_threshold": STRUCTURE_THRESHOLD,
        "crop_padding": CROP_PADDING, "grid_nms_iou": GRID_NMS_IOU,
        "end_to_end_tables": END_TO_END_TABLES, "questions_per_table": QUESTIONS_PER_TABLE,
        "run_complex_structure_probe": RUN_COMPLEX_STRUCTURE_PROBE,
        "complex_structure_tables": COMPLEX_STRUCTURE_TABLES,
        "output_dir": OUTPUT_DIR, "work_dir": str(WORK_ROOT),
    }
    config_path = WORK_ROOT / "stage_config.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    log = WORK_ROOT / "logs" / (stage + ".log")
    log.parent.mkdir(parents=True, exist_ok=True)
    command = [str(PYTHON), "-u", str(STAGE_SCRIPT), "--config", str(config_path), "--stage", stage, *options]
    with log.open("w", encoding="utf-8") as output:
        process = subprocess.Popen(command, env=ENV, cwd=os.getcwd(), stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            print(line.rstrip("\n"), flush=True)
        process.wait()
    if process.returncode:
        # The stage's own output, including its traceback, is printed above; its last line names the problem.
        last = [line for line in log.read_text(encoding="utf-8").splitlines() if line.strip()][-1:] or ["no output"]
        raise RuntimeError(f"{stage} failed with exit {process.returncode}: {last[0]} (full log: {log})")


from IPython.display import Markdown, display


def show_source(*names):
    """Display functions of the carried stage file (they run in the isolated environment, not here)."""
    text = STAGE_SCRIPT.read_text(encoding="utf-8")
    found = {node.name: ast.get_source_segment(text, node) for node in ast.parse(text).body
             if isinstance(node, ast.FunctionDef)}
    display(Markdown("```python\n" + "\n\n".join(found[name] for name in names) + "\n```"))


run_stage("environment")


GPU: Tesla T4


Creating the isolated Python 3.12.12 environment...


Installing the hashed dependency lock (several minutes; CUDA wheels are large)...


{'python': '3.12.12', 'executable': '/content/work/env/bin/python', 'torch': '2.14.0+cu130', 'torchvision': '0.29.0+cu130', 'transformers': '4.57.6', 'timm': '1.0.30', 'datasets': '4.1.1', 'numpy': '2.5.3', 'numpy_source': 'hash lock (isolated environment)', 'pandas': '2.2.3', 'pillow': '11.3.0', 'lock_pins_checked': 66, 'device': 'cuda:0', 'gpu_name': 'Tesla T4'}


## 5. Immutable model provenance

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Every model is staged only from its exact immutable revision and verified against the DIMER manifest before loading.

No upstream `.bin` file is used.

In [ ]:
# Stage each model from its immutable revision and verify every file's size and SHA-256 against the DIMER manifest.
run_stage("models")

{'model_id': 'microsoft/table-transformer-detection', 'revision': '2357cbe2b5a5d1c03e54f32764f06058933b65ab', 'bytes': 115320191}


{'model_id': 'microsoft/table-transformer-structure-recognition-v1.1-all', 'revision': '7587a7ef111d9dcbf8ac695f1376ab7014340a0c', 'bytes': 115515347}


{'model_id': 'google/tapas-large-finetuned-wtq', 'revision': 'f58317ab2577d17647d9acafa790c744a0388b30', 'bytes': 1347256837}


## 6. SciTSR-PD provenance

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Pinned dataset:

`bevaya/SciTSR-pd @ dae336efa7af07d69194a510ff5568980e8ef253`

The dataset contains 108 public-domain scientific tables (89 train / 19 test) with:

- table image;
- logical cells;
- text chunks;
- paper identity;
- source license metadata.

The two source parquet files are verified by byte size and SHA-256 before they are read.

In [ ]:
# Download the two pinned SciTSR-PD parquet shards and verify their size and SHA-256.
run_stage("dataset")

train weights/scitsr-pd/data/train-00000-of-00001.parquet d295b918777425a26fa13a5c3ba076517429682ede8c71b671a8b6115968e465


test weights/scitsr-pd/data/test-00000-of-00001.parquet 73af6317219349a08acdf1c8fee7cff21b463cdded64111b643123f7df4f8bed


## 7. Build the table-geometry/text companion from SciTSR-PD

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

The source dataset stores logical cells (with their text) and PDF-coordinate text chunks, but no pixel boxes. The notebook combines two things:

- **Pinned carrier geometry.** The DIMER structure carrier derived pixel-space `table` / `table row` / `table column` / `table spanning cell` boxes for 94 of the 108 SciTSR-PD tables, once, and recorded the whole-pixel offset that places each table's text chunks on its image. The carried stage file (§4) embeds that geometry and its SHA-256; every stage checks the digest when it starts, and the next cell prints it.
- **Annotation text.** Each logical cell's text is matched to its PDF text chunks, and the chunk boxes are placed on the image with the carrier's offset. This gives every cell a pixel box, its row/column span and its text.

This is a deterministic data transformation, not model inference and not OCR. The notebook stops if any carrier table fails to rebuild, if its pixels differ from the carrier's, or if its cell spans disagree with the carrier's row and column counts.

In [ ]:
# The pinned carrier geometry (94 tables) and its SHA-256 are embedded in the carried stage file; every stage
# checks the digest on start. This prints the table count and the digest.
run_stage("geometry")

{'carrier_tables': 94, 'geometry_sha256': 'd6d524c367a151ed8a41bda7eadf7679977256953888862c9d85186eeda2c66a'}


In [ ]:
# Rebuild the table-geometry/text companion from the verified shards (stage_companion in the carried file).
run_stage("companion")

Generating train split: 0 examples [00:00, ? examples/s]


Generating train split: 89 examples [00:00, 2631.98 examples/s]


Generating test split: 0 examples [00:00, ? examples/s]


Generating test split: 19 examples [00:00, 1845.70 examples/s]


{'source_tables': 108, 'carrier_tables': 94, 'derived_tables': 94}


## 8. Paper-disjoint split

> **Evaluation practice**

The corpus is split by paper, not by individual table: tables from one paper never appear in two splits.

The notebook reproduces the carrier's split exactly (seed 42, papers drawn in the order test → validation → train):

| split | papers | tables |
|---|---:|---:|
| test | 10 | 21 |
| validation | 7 | 23 |
| train | 29 | 50 |

The notebook performs no training, so the 50 training tables are never used. The ten canonical QA tables (§9) and the spanning-cell probe (§19) come from the 21 held-out **test** tables. The threshold activity after §18 uses the 23 **validation** tables, so experimenting with it never touches the test results. The next cell stops if the counts differ.

In [ ]:
show_source("stage_split")
run_stage("split")

```python
def stage_split(args):
    global splits
    rng=random.Random(42)
    by_paper=defaultdict(list)
    for item in derived:
        by_paper[item["paper_id"]].append(item)

    papers=sorted(by_paper)
    rng.shuffle(papers)
    if len(papers)!=46:
        raise RuntimeError(f"Expected the carrier's 46 papers, derived {len(papers)}")

    # Iterate the shuffled paper list, never a set: set order depends on per-process string hashing.
    test_papers=papers[:10]
    val_papers=papers[10:17]
    train_papers=papers[17:]

    splits={
        "test":[x for p in test_papers for x in by_paper[p]],
        "validation":[x for p in val_papers for x in by_paper[p]],
        "train":[x for p in train_papers for x in by_paper[p]],
    }
    expected={"test":21,"validation":23,"train":50}
    if {k:len(v) for k,v in splits.items()}!=expected:
        raise RuntimeError(f"Paper-disjoint split does not match the carrier: {({k:len(v) for k,v in splits.items()})}")
    print({k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()})
```

{'test': {'tables': 21, 'papers': 10}, 'validation': {'tables': 23, 'papers': 7}, 'train': {'tables': 50, 'papers': 29}}


## 9. Select ten QA-eligible rectangular tables

> **Evaluation practice**

The full end-to-end QA workflow intentionally excludes spanning-cell tables.

Eligibility:

- no spanning cells;
- 3–20 rows;
- 2–12 columns;
- complete first row with unique non-empty headers;
- complete first body column with unique row identifiers;
- at least one numeric body column;
- cells ≤200 characters.

Selection is deterministic and independent of model output.

In [ ]:
show_source("parse_number", "eligible", "stage_select")
run_stage("select")

```python
def parse_number(text):
    s=str(text).strip().replace(",","")
    if s.endswith("%"): s=s[:-1]
    if not NUMBER_RE.match(str(text).strip()):
        try:
            # permit comma-stripped representation
            float(s)
        except Exception:
            return None
    try:return float(s)
    except Exception:return None

def eligible(item):
    if item["has_spans"]: return False
    if not (3<=item["n_rows"]<=20 and 2<=item["n_cols"]<=12): return False
    grid=item["grid"]
    if any(not str(x).strip() or len(str(x))>200 for row in grid for x in row): return False
    headers=[norm_text(x) for x in grid[0]]
    if any(not x for x in headers) or len(set(headers))!=len(headers): return False
    row_ids=[norm_text(row[0]) for row in grid[1:]]
    if any(not x for x in row_ids) or len(set(row_ids))!=len(row_ids): return False
    numeric_cols=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in grid[1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            numeric_cols.append(c)
    return bool(numeric_cols)

def stage_select(args):
    global canonical_tables
    eligible_tables=[x for x in splits["test"] if eligible(x)]
    eligible_tables.sort(key=lambda x:hashlib.sha256(f"42:{x['source_id']}".encode()).hexdigest())
    canonical_tables=eligible_tables[:END_TO_END_TABLES]
    if len(canonical_tables)!=END_TO_END_TABLES:
        # Spec §18: canonical tables come from the paper-disjoint test split only; never borrow train/validation tables.
        raise RuntimeError(f"Only {len(canonical_tables)} QA-eligible test tables were derived")

    print([x["source_id"] for x in canonical_tables])
```

['1702.02925v1.7', '1605.06770v1.1', '1606.09371v1.5', '1407.3745v1.1', '1702.02925v1.3', '1108.4723v1.4', '1611.09238v1.1', '1108.4723v1.3', '1702.02925v1.6', '1606.09371v1.2']


## 10. Create deterministic document pages

> **Evaluation practice**

The real SciTSR table pixels are placed unchanged on a simple white page.

This creates measurable page-level table-detection ground truth without introducing a second table.

In [ ]:
show_source("make_page", "stage_pages")
run_stage("pages")

```python
def make_page(item):
    table=item["image"]
    left,top,right,bottom=120,160,120,120
    page=Image.new("RGB",(table.width+left+right,table.height+top+bottom),"white")
    draw=ImageDraw.Draw(page)
    draw.text((left,48),f"Table {item['source_id']}",font=FONT,fill="black")
    page.paste(table,(left,top))
    draw.text((left,top+table.height+40),"Scientific table excerpt",font=FONT,fill="gray")
    gt_box=[left,top,left+table.width,top+table.height]
    return page,gt_box

def stage_pages(args):
    for item in canonical_tables:
        item["page"],item["page_gt_box"]=make_page(item)
        item["page_sha256"]=image_digest(item["page"])

    print({"pages":len(canonical_tables),"page_sizes":[x["page"].size for x in canonical_tables[:3]]})
```

{'pages': 10, 'page_sizes': [(765, 537), (756, 492), (626, 416)]}


## 11. Common detection/structure metrics

> **Evaluation practice**

Bounding-box comparisons use pixel-space IoU.

AP is Pascal-VOC-style all-point AP over score-ranked predictions with one-to-one matching.

In [ ]:
# These metric functions run inside the isolated environment; they are shown here and used from §13 on.
show_source("box_iou", "ap_for_label")

```python
def box_iou(a,b):
    x0=max(float(a[0]),float(b[0])); y0=max(float(a[1]),float(b[1]))
    x1=min(float(a[2]),float(b[2])); y1=min(float(a[3]),float(b[3]))
    inter=max(0,x1-x0)*max(0,y1-y0)
    aa=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))
    bb=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))
    u=aa+bb-inter
    return inter/u if u>0 else 0.0

def ap_for_label(preds,refs,label,thr):
    det=[]
    n_gt=0
    for i,(p,r) in enumerate(zip(preds,refs,strict=True)):
        gt=[x["box"] for x in r if x["label"]==label]
        n_gt+=len(gt)
        det += [(float(x["score"]),i,x["box"]) for x in p if x["label"]==label]
    det.sort(key=lambda x:-x[0])
    matched=defaultdict(set); hits=[]
    for score,i,box in det:
        gt=[x["box"] for x in refs[i] if x["label"]==label]
        best,bj=0,None
        for j,g in enumerate(gt):
            if j in matched[i]: continue
            v=box_iou(box,g)
            if v>best: best,bj=v,j
        if bj is not None and best>=thr:
            matched[i].add(bj); hits.append(1)
        else:hits.append(0)
    if not hits or n_gt==0:return 0.0
    tp=np.cumsum(hits); recall=tp/n_gt; precision=tp/np.arange(1,len(hits)+1)
    mr=np.concatenate([[0],recall,[1]]); mp=np.concatenate([[0],precision,[0]])
    for i in range(len(mp)-2,-1,-1):mp[i]=max(mp[i],mp[i+1])
    idx=np.where(mr[1:]!=mr[:-1])[0]
    return float(np.sum((mr[idx+1]-mr[idx])*mp[idx+1]))
```

## 12. Stage 1 — Table Transformer Detection

> **Core concept**

The detector runs on the synthetic page. The highest-scoring surviving `table` / `table rotated` detection becomes the target crop.

A miss is a real pipeline failure and propagates downstream.

**Question tested:** does a detector trained on document pages find a real scientific table placed on a plain synthetic page?

**Predict:** how many of the ten pages will yield a `table` detection at threshold 0.90? Will the best-box IoU be close to 1.0?

In [ ]:
show_source("load_detection_model", "detect_table", "stage_detection")
run_stage("detection")

```python
def load_detection_model():
    """Load the verified detection snapshot (used here and by BYOD in §28)."""
    processor=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)
    model=TableTransformerForObjectDetection.from_pretrained(
        str(DETECTION_DIR),
        local_files_only=True,
        trust_remote_code=False,
        use_pretrained_backbone=False,
    ).to(DEVICE).eval()
    for p in model.parameters():p.requires_grad_(False)
    return processor,model

def detect_table(image,threshold=None):
    threshold=DETECTION_THRESHOLD if threshold is None else threshold
    inputs=det_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=det_model(**inputs)
    result=det_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=det_model.config.id2label[int(label)]
        if name in ("table","table rotated"):
            rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

def stage_detection(args):
    global det_processor,det_model,det_load_seconds,det_params,det_times,det_peak
    load_transformers()
    t0=time.perf_counter()
    det_processor,det_model=load_detection_model()
    det_load_seconds=time.perf_counter()-t0
    det_params=sum(p.numel() for p in det_model.parameters())

    _ = detect_table(canonical_tables[0]["page"])
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    det_times=[]
    for i,item in enumerate(canonical_tables):
        if torch.cuda.is_available():torch.cuda.synchronize()
        t=time.perf_counter(); preds=detect_table(item["page"])
        if torch.cuda.is_available():torch.cuda.synchronize()
        det_times.append(time.perf_counter()-t)
        item["detections"]=preds
        item["selected_detection"]=preds[0] if preds else None
    det_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

    print({"mean_latency":float(np.mean(det_times)),"detections":[len(x["detections"]) for x in canonical_tables]})

    # The detector is no longer needed: unload it before the structure model (sequential loading, spec §67).
    del det_model,det_processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
```

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Some weights of the model checkpoint at weights/table-transformer-detection were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']


- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).


- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'mean_latency': 0.05740664119999792, 'detections': [1, 1, 1, 1, 1, 0, 1, 1, 1, 1]}


## 13. Detection metrics

> **Evaluation practice**

Each page has exactly one ground-truth table box.

The tutorial reports AP50/AP75, hit rate and best-box IoU across the ten pages.

In [ ]:
show_source("stage_detection_metrics")
run_stage("detection-metrics")

```python
def stage_detection_metrics(args):
    global det_rows,det_metrics
    det_pred_lists=[]
    det_refs=[]
    det_rows=[]
    for item in canonical_tables:
        preds=[{"label":"table","score":x["score"],"box":x["box"]} for x in item["detections"]]
        det_pred_lists.append(preds)
        det_refs.append([{"label":"table","box":item["page_gt_box"]}])
        best=max([box_iou(x["box"],item["page_gt_box"]) for x in item["detections"]],default=0.0)
        det_rows.append({
            "table_id":item["source_id"],"predicted":bool(item["detections"]),
            "best_iou":best,"hit50":best>=0.5,"hit75":best>=0.75,
        })

    det_metrics={
        "ap50":ap_for_label(det_pred_lists,det_refs,"table",0.5),
        "ap75":ap_for_label(det_pred_lists,det_refs,"table",0.75),
        "hit50":float(np.mean([r["hit50"] for r in det_rows])),
        "hit75":float(np.mean([r["hit75"] for r in det_rows])),
        "mean_best_iou":float(np.mean([r["best_iou"] for r in det_rows])),
        "median_best_iou":float(np.median([r["best_iou"] for r in det_rows])),
    }
    print(det_metrics)
```

{'ap50': 0.9, 'ap75': 0.9, 'hit50': 0.9, 'hit75': 0.9, 'mean_best_iou': 0.7782565328815216, 'median_best_iou': 0.8561918726184901}


### What to notice — detection

- `hit50` is the fraction of pages whose best detection overlaps the placed table with IoU ≥ 0.50. A page with no detection above the threshold is a **miss**; every later stage records it as `detection_miss` instead of dropping it, so it still counts against end-to-end accuracy.
- Best-box IoU stays below 1.0 even for a good detection. The reference box is the whole pasted SciTSR image, including its white margin, while the detector tends to hug the ruled table. The red boxes in the §26 panels show the difference.
- AP50 and AP75 are close when every hit is tightly localized; a large gap between them would mean loose boxes.

## 14. Build gold and detected crops

> **Core concept**

The structure recognizer is evaluated twice:

- **gold crop** — exact known table location, with 10 px page padding;
- **detected crop** — predicted box, with the same padding.

Ground-truth structure boxes are transformed into each crop's coordinate frame.

### Three coordinate frames

Every stage works in its own frame, so a box has to be translated as it moves through the pipeline:

- **table frame**: pixels of the original SciTSR table image, where the carrier's reference boxes live;
- **page frame**: the synthetic page; §10 pastes the table at `(120, 160)`, so page = table + (120, 160);
- **crop frame**: the padded crop given to the structure model; crop = page − (crop left, crop top).

The next cell builds both crops and prints one reference row box in all of these frames.

**Checkpoint: trace a box.** A row box is `[3, 30, 560, 58]` in the table frame, and the detected crop starts at page pixel `(112, 150)`. Where is the box in the page frame and in the detected-crop frame? Work it out, then compare your method with the printed trace.

<details>
<summary>Sample answer</summary>

Page frame: add the paste offset, `[3+120, 30+160, 560+120, 58+160] = [123, 190, 680, 218]`. Detected-crop frame: subtract the crop origin, `[123−112, 190−150, 680−112, 218−150] = [11, 40, 568, 68]`. A box that ends up partly outside the crop is clipped to it, which is how a tight detection can cut a row or column out of the structure model's view.

</details>

In [ ]:
show_source("padded_crop", "translate_gold_objects", "to_crop_frame", "stage_crops")
run_stage("crops")

```python
def padded_crop(page,box,pad=None):
    pad=CROP_PADDING if pad is None else pad
    x0,y0,x1,y1=[float(v) for v in box]
    cx0=max(0,int(math.floor(x0-pad))); cy0=max(0,int(math.floor(y0-pad)))
    cx1=min(page.width,int(math.ceil(x1+pad))); cy1=min(page.height,int(math.ceil(y1+pad)))
    return page.crop((cx0,cy0,cx1,cy1)),[cx0,cy0,cx1,cy1]

def translate_gold_objects(item,crop_box):
    page_x0,page_y0=120,160
    cx0,cy0,cx1,cy1=crop_box
    out=[]
    for obj in item["objects"]:
        x0,y0,x1,y1=obj["box"]
        b=[x0+page_x0-cx0,y0+page_y0-cy0,x1+page_x0-cx0,y1+page_y0-cy0]
        b=[max(0,b[0]),max(0,b[1]),min(cx1-cx0,b[2]),min(cy1-cy0,b[3])]
        if b[2]>b[0] and b[3]>b[1]:
            out.append({"label":obj["label"],"box":b})
    return out

def to_crop_frame(box,crop_box):
    return [box[0]-crop_box[0],box[1]-crop_box[1],box[2]-crop_box[0],box[3]-crop_box[1]]

def stage_crops(args):
    for item in canonical_tables:
        gold_crop,gold_crop_box=padded_crop(item["page"],item["page_gt_box"])
        item["gold_crop"]=gold_crop; item["gold_crop_box"]=gold_crop_box
        item["gold_crop_objects"]=translate_gold_objects(item,gold_crop_box)

        if item["selected_detection"] is not None:
            det_crop,det_crop_box=padded_crop(item["page"],item["selected_detection"]["box"])
            item["det_crop"]=det_crop; item["det_crop_box"]=det_crop_box
            item["det_crop_objects"]=translate_gold_objects(item,det_crop_box)
        else:
            item["det_crop"]=None; item["det_crop_box"]=None; item["det_crop_objects"]=[]

    # Trace one reference row box through the frames: table -> page -> gold crop / detected crop.
    trace_item=next((x for x in canonical_tables if x["selected_detection"] is not None),canonical_tables[0])
    trace_box=next(o["box"] for o in trace_item["objects"] if o["label"]=="table row")
    trace_page=[trace_box[0]+120,trace_box[1]+160,trace_box[2]+120,trace_box[3]+160]
    print({
        "table":trace_item["source_id"],
        "row_box_table_frame":[round(v,1) for v in trace_box],
        "page_frame":[round(v,1) for v in trace_page],
        "gold_crop_origin":trace_item["gold_crop_box"][:2],
        "gold_crop_frame":[round(v,1) for v in to_crop_frame(trace_page,trace_item["gold_crop_box"])],
        "detected_crop_origin":trace_item["det_crop_box"][:2] if trace_item["det_crop_box"] else None,
        "detected_crop_frame":[round(v,1) for v in to_crop_frame(trace_page,trace_item["det_crop_box"])] if trace_item["det_crop_box"] else None,
    })
```

{'table': '1702.02925v1.7', 'row_box_table_frame': [3.0, 3.0, 522.0, 29.1], 'page_frame': [123.0, 163.0, 642.0, 189.1], 'gold_crop_origin': [110, 150], 'gold_crop_frame': [13.0, 13.0, 532.0, 39.1], 'detected_crop_origin': [124, 154], 'detected_crop_frame': [-1.0, 9.0, 518.0, 35.1]}


## 15. Stage 2 — Table Transformer Structure Recognition

> **Core concept**

Raw model output is preserved.

Rows and columns are later subjected to deterministic same-label NMS for grid reconstruction. NMS is a composition rule of this notebook, not part of the model.

**Question tested:** does structure recognition depend on the crop it is given?

**Predict:** will the gold-crop path or the detected-crop path produce more exact grids (row count and column count both right)?

In [ ]:
show_source("load_structure_model", "recognize_structure", "timed_structure", "stage_structure")
run_stage("structure")

```python
def load_structure_model():
    """Load the verified structure snapshot; §15, the activity after §18 and BYOD (§28) all use this loader."""
    processor=AutoImageProcessor.from_pretrained(
        str(STRUCTURE_DIR),
        local_files_only=True,
        trust_remote_code=False,
        size={"shortest_edge":800,"longest_edge":800},
    )
    model=TableTransformerForObjectDetection.from_pretrained(
        str(STRUCTURE_DIR),
        local_files_only=True,
        trust_remote_code=False,
        use_pretrained_backbone=False,
    ).to(DEVICE).eval()
    for p in model.parameters():p.requires_grad_(False)
    return processor,model

def recognize_structure(image,threshold=None):
    threshold=STRUCTURE_THRESHOLD if threshold is None else threshold
    inputs=struct_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=struct_model(**inputs)
    result=struct_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=struct_model.config.id2label[int(label)]
        rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

def timed_structure(crop):
    if torch.cuda.is_available():torch.cuda.synchronize()
    t=time.perf_counter(); raw=recognize_structure(crop)
    if torch.cuda.is_available():torch.cuda.synchronize()
    struct_times.append(time.perf_counter()-t)
    return raw

def stage_structure(args):
    global struct_processor,struct_model,struct_load_seconds,struct_params,struct_times,struct_peak
    load_transformers()
    t0=time.perf_counter()
    struct_processor,struct_model=load_structure_model()
    struct_load_seconds=time.perf_counter()-t0
    struct_params=sum(p.numel() for p in struct_model.parameters())

    _ = recognize_structure(canonical_tables[0]["gold_crop"])
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    struct_times=[]  # one entry per crop (gold or detected), synchronised like the detection timings

    for item in canonical_tables:
        item["gold_structure_raw"]=timed_structure(item["gold_crop"])
        item["det_structure_raw"]=timed_structure(item["det_crop"]) if item["det_crop"] is not None else []
    struct_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    print({"mean_seconds_per_crop":float(np.mean(struct_times)),"crops":len(struct_times)})

    del struct_model,struct_processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
```

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{'mean_seconds_per_crop': 0.03611019899999759, 'crops': 19}


## 16. Grid reconstruction

> **Core concept**

Rows are sorted top-to-bottom; columns left-to-right.

Same-label duplicate boxes are removed with NMS at IoU 0.50 before grid construction.

In [ ]:
show_source("nms_items", "grid_from_structure", "stage_grid")
run_stage("grid")

```python
def nms_items(items,iou_threshold=None):
    iou_threshold=GRID_NMS_IOU if iou_threshold is None else iou_threshold
    kept=[]
    for item in sorted(items,key=lambda x:-x["score"]):
        if all(box_iou(item["box"],k["box"])<iou_threshold for k in kept):
            kept.append(item)
    return kept

def grid_from_structure(raw):
    rows=nms_items([x for x in raw if x["label"]=="table row"])
    cols=nms_items([x for x in raw if x["label"]=="table column"])
    rows.sort(key=lambda x:(x["box"][1]+x["box"][3])/2)
    cols.sort(key=lambda x:(x["box"][0]+x["box"][2])/2)
    cells=[]
    valid=bool(rows and cols)
    for ri,r in enumerate(rows):
        row=[]
        for ci,c in enumerate(cols):
            b=[
                max(r["box"][0],c["box"][0]),max(r["box"][1],c["box"][1]),
                min(r["box"][2],c["box"][2]),min(r["box"][3],c["box"][3]),
            ]
            if b[2]<=b[0] or b[3]<=b[1]:valid=False
            row.append(b)
        cells.append(row)
    return {"rows":rows,"columns":cols,"cells":cells,"valid_geometry":valid}

def stage_grid(args):
    for item in canonical_tables:
        item["gold_grid_pred"]=grid_from_structure(item["gold_structure_raw"])
        item["det_grid_pred"]=grid_from_structure(item["det_structure_raw"]) if item["det_crop"] is not None else {"rows":[],"columns":[],"cells":[],"valid_geometry":False}
```

## 17. Assign SciTSR source cell text to predicted cells

> **Core concept**

For each gold logical cell, its pixel center is transformed into the current crop and located in the predicted row and column.

No gold row/column index is supplied to the reconstruction algorithm.

In [ ]:
show_source("source_cells_in_crop", "contains", "reconstruct_table", "stage_reconstruct")
run_stage("reconstruct")

```python
def source_cells_in_crop(item,crop_box):
    cx0,cy0,_,_=crop_box
    page_left,page_top=120,160
    out=[]
    for c in item["cells"]:
        b=c["box"]
        out.append({
            **c,
            "crop_box":[b[0]+page_left-cx0,b[1]+page_top-cy0,b[2]+page_left-cx0,b[3]+page_top-cy0],
        })
    return out

def contains(box,x,y):
    return box[0]<=x<=box[2] and box[1]<=y<=box[3]

def reconstruct_table(item,grid,crop_box):
    if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
        return {"valid":False,"reason":"grid_invalid","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}
    strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
    mapped=unmapped=ambiguous=correct=0
    simple=[c for c in source_cells_in_crop(item,crop_box) if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]]
    for c in simple:
        b=c["crop_box"]; x=(b[0]+b[2])/2; y=(b[1]+b[3])/2
        ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
        cis=[i for i,col in enumerate(grid["columns"]) if contains(col["box"],x,y)]
        if len(ris)==1 and len(cis)==1:
            ri,ci=ris[0],cis[0]
            strings[ri][ci].append(c["text"])
            mapped+=1
            if ri==c["row_start"] and ci==c["col_start"]:correct+=1
        elif len(ris)==0 or len(cis)==0:
            unmapped+=1
        else:
            ambiguous+=1
    table=[[" ".join(cell).strip() for cell in row] for row in strings]
    headers=table[0] if table else []
    norm_headers=[norm_text(x) for x in headers]
    valid_headers=bool(headers) and all(norm_headers) and len(set(norm_headers))==len(norm_headers)
    valid=grid["valid_geometry"] and valid_headers and len(table)<=64 and len(headers)<=32
    return {
        "valid":valid,
        "reason":"ok" if valid else "header_or_limit_invalid",
        "table":table,
        "mapped":mapped,"unmapped":unmapped,"ambiguous":ambiguous,
        "assignment_accuracy":correct/max(1,len(simple)),
        "headers_valid":valid_headers,
    }

def stage_reconstruct(args):
    for item in canonical_tables:
        item["gold_recon"]=reconstruct_table(item,item["gold_grid_pred"],item["gold_crop_box"])
        if item["det_crop"] is not None:
            item["det_recon"]=reconstruct_table(item,item["det_grid_pred"],item["det_crop_box"])
        else:
            item["det_recon"]={"valid":False,"reason":"detection_miss","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}
```

## 18. Structure and reconstruction metrics

> **Evaluation practice**

Rows/columns are scored independently on gold crops and detected crops.

Header predictions are not penalized because SciTSR-PD does not annotate the two PubTables header classes used by the structure checkpoint.

In [ ]:
show_source("refs_for", "preds_for", "stage_structure_metrics")
run_stage("structure-metrics")

```python
def refs_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

def preds_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

def stage_structure_metrics(args):
    global structure_rows
    structure_rows=[]
    for path_name,pred_key,ref_key in [
        ("gold_crop","gold_structure_raw","gold_crop_objects"),
        ("detected_crop","det_structure_raw","det_crop_objects"),
    ]:
        for item in canonical_tables:
            gp=item["gold_grid_pred"] if path_name=="gold_crop" else item["det_grid_pred"]
            # A page without a detection has no crop to recognize: a failed grid on this path, never an exact one.
            missed=path_name=="detected_crop" and item["det_crop"] is None
            gold_rows=refs_for(item,ref_key,"table row")
            gold_cols=refs_for(item,ref_key,"table column")
            pred_rows=preds_for(item,pred_key,"table row")
            pred_cols=preds_for(item,pred_key,"table column")
            best_r=[max([box_iou(g["box"],p["box"]) for p in pred_rows],default=0) for g in gold_rows]
            best_c=[max([box_iou(g["box"],p["box"]) for p in pred_cols],default=0) for g in gold_cols]
            # Counts are compared with the table's own grid, not with the reference boxes left inside a (possibly clipped) crop.
            rows_exact=not missed and len(gp["rows"])==item["n_rows"]
            cols_exact=not missed and len(gp["columns"])==item["n_cols"]
            structure_rows.append({
                "table_id":item["source_id"],"path":path_name,
                "status":"detection_miss" if missed else "ok",
                "gold_rows":item["n_rows"],"predicted_rows":len(gp["rows"]),
                "gold_columns":item["n_cols"],"predicted_columns":len(gp["columns"]),
                "row_count_exact":rows_exact,
                "column_count_exact":cols_exact,
                "grid_shape_exact":rows_exact and cols_exact,
                "mean_row_iou":float(np.mean(best_r)) if best_r else 0.0,
                "mean_column_iou":float(np.mean(best_c)) if best_c else 0.0,
            })

    for path_name in ("gold_crop","detected_crop"):
        subset=[r for r in structure_rows if r["path"]==path_name]
        print(path_name,{
            "grid_exact":float(np.mean([r["grid_shape_exact"] for r in subset])),
            "mean_row_iou":float(np.mean([r["mean_row_iou"] for r in subset])),
            "mean_column_iou":float(np.mean([r["mean_column_iou"] for r in subset])),
        })
```

gold_crop {'grid_exact': 1.0, 'mean_row_iou': 0.7618614681496387, 'mean_column_iou': 0.8883845937466599}


detected_crop {'grid_exact': 0.8, 'mean_row_iou': 0.6934404567206169, 'mean_column_iou': 0.8087576124892546}


### What to notice — structure

- `grid_exact` is the gate for reconstruction: when the row or column count is wrong, text is assigned to the wrong cells no matter how good the boxes look.
- Mean row/column IoU compares each reference row or column with its best-matching prediction. Exact counts with IoU well below 1.0 are normal here: the carrier's reference rows and columns tile the whole table from mid-gap to mid-gap (the PubTables-1M convention), while predicted boxes follow the model's own boundaries. Boundary offsets matter only when a cell's center falls on the wrong side of a boundary.
- The detected-crop path can lose a table entirely or gain or lose a row or column when the crop clips or adds margin. A page with no detection is scored as a failed detected-crop grid (`status` = `detection_miss` in `structure_metrics.csv`), so the detected-crop exact-grid count can never exceed the number of detection hits.
- Column-header and projected-row-header predictions are not scored, because SciTSR-PD does not annotate them.

### Checkpoint — structure

1. A predicted grid has the right number of columns but one row too few. What happens to the text of the cells in the missing row?
2. Why can a detected crop change the structure result even when its detection IoU is above 0.8?

<details>
<summary>Sample answer</summary>

1. The centers of the missing row's cells fall into a neighbouring predicted row, and their text is merged into that row's cells, or into no row at all, and they become unmapped. Either way the reconstructed table no longer says what the source table says, so questions about that row can fail although TAPAS itself is unchanged.
2. The structure model sees a different image. The crop's margins, and therefore its scale after resizing to 800 px, change, and a clipped rule or header line removes evidence. IoU measures overlap with the reference box, not whether every rule and header line is inside the crop.

</details>

### Activity — Predict → Change one thing → Run → Observe → Explain

This activity uses the 23 **validation** tables only, so the canonical test-table results above and the exported files are unchanged.

1. **Predict.** If the structure threshold rises from 0.50 to `ACTIVITY_STRUCTURE_THRESHOLD`, will the model report more rows and columns, fewer, or the same? Which kinds of tables will change?
2. **Change one thing.** Set `ACTIVITY_STRUCTURE_THRESHOLD` in the next cell (default 0.95; then try 0.98 and 0.30).
3. **Run** the cell. It recognizes structure on each validation table's gold crop at both thresholds. You can rerun it as often as you like after `Run all`: each run is a separate process that loads the structure model from the verified snapshot and unloads it when it finishes.
4. **Observe** the exact-grid counts and the tables whose row/column counts changed.
5. **Explain** the tradeoff: what does a higher threshold remove, and what does that cost?

<details>
<summary>Sample answer</summary>

A higher threshold keeps only the more confident boxes, so it can only remove predictions. On these clean, ruled scientific tables the model is confident about nearly every real row and column, so lowering the threshold changes little, while raising it toward 1.0 starts removing real rows or columns — typically short or sparse ones — and the grid comes out a row or column short. On noisier documents a lower threshold can also admit spurious or duplicated boxes that NMS must then suppress. Because reconstruction needs exact counts, the useful threshold is the one that gets counts right most often on held-out validation tables. That is why the experiment runs on validation tables, not on the test tables the notebook reports.

</details>

In [ ]:
# Change one thing: the structure threshold, on validation tables only (canonical outputs are unaffected).
ACTIVITY_STRUCTURE_THRESHOLD = 0.95  # @param {type:"number"}
if not 0 <= ACTIVITY_STRUCTURE_THRESHOLD <= 1:
    raise ValueError("ACTIVITY_STRUCTURE_THRESHOLD must be in [0,1]")

# Each run is its own process: it loads the verified structure snapshot, runs both thresholds and unloads it.
show_source("grid_counts", "stage_activity")
run_stage("activity", "--threshold", str(ACTIVITY_STRUCTURE_THRESHOLD))

```python
def grid_counts(crop,threshold):
    grid=grid_from_structure(recognize_structure(crop,threshold=threshold))
    return len(grid["rows"]),len(grid["columns"])

def stage_activity(args):
    global struct_processor,struct_model,activity_rows
    load_transformers()
    # Change one thing: the structure threshold, on validation tables only (canonical outputs are unaffected).
    ACTIVITY_STRUCTURE_THRESHOLD = float(args.threshold)
    if not 0 <= ACTIVITY_STRUCTURE_THRESHOLD <= 1:
        raise ValueError("ACTIVITY_STRUCTURE_THRESHOLD must be in [0,1]")

    # Each stage process loads the structure model only if it is absent, and unloads what it loaded.
    activity_loaded_model="struct_model" not in globals()
    if activity_loaded_model:
        struct_processor,struct_model=load_structure_model()

    activity_rows=[]
    for item in splits["validation"]:
        page,page_box=make_page(item)
        crop,_=padded_crop(page,page_box)
        activity_rows.append({
            "table":item["source_id"],
            "gold":(item["n_rows"],item["n_cols"]),
            "at_reference":grid_counts(crop,STRUCTURE_THRESHOLD),
            "at_changed":grid_counts(crop,ACTIVITY_STRUCTURE_THRESHOLD),
        })

    for name,threshold in (("at_reference",STRUCTURE_THRESHOLD),("at_changed",ACTIVITY_STRUCTURE_THRESHOLD)):
        exact=sum(r[name]==r["gold"] for r in activity_rows)
        print(f"threshold {threshold:.2f}: exact grids {exact} / {len(activity_rows)} validation tables")
    changed=[r for r in activity_rows if r["at_reference"]!=r["at_changed"]]
    print(f"Tables whose (rows, columns) changed: {len(changed)}")
    for r in changed:
        print(" ",r)

    if activity_loaded_model:
        del struct_model,struct_processor
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
```

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


threshold 0.50: exact grids 23 / 23 validation tables


threshold 0.95: exact grids 20 / 23 validation tables


Tables whose (rows, columns) changed: 3


  {'table': '1711.04434v1.7', 'gold': (7, 3), 'at_reference': (7, 3), 'at_changed': (5, 3)}


  {'table': '1706.08653v2.6', 'gold': (14, 13), 'at_reference': (14, 13), 'at_changed': (14, 12)}


  {'table': '1706.08653v2.12', 'gold': (8, 7), 'at_reference': (8, 7), 'at_changed': (8, 6)}


## 19. Complex spanning-cell probe

> **Core concept**

The first three held-out spanning-cell tables are run through structure recognition on their gold crops. Each panel in `complex_probe/` shows the gold spanning cells beside the predicted rows, columns and spanning cells.

They are intentionally excluded from rectangular TAPAS reconstruction because simple row × column intersections cannot faithfully represent their semantics.

In [ ]:
show_source("outline", "stage_complex_probe")
run_stage("complex-probe")

```python
def outline(image,items,colors):
    out=image.convert("RGB").copy()
    d=ImageDraw.Draw(out)
    for x in items:
        d.rectangle(x["box"],outline=colors.get(x["label"],"orange"),width=2)
    return out

def stage_complex_probe(args):
    global struct_processor,struct_model,complex_tables
    load_transformers()
    complex_tables=[x for x in splits["test"] if x["has_spans"]][:COMPLEX_STRUCTURE_TABLES] if RUN_COMPLEX_STRUCTURE_PROBE else []
    complex_dir=Path(OUTPUT_DIR)/"complex_probe"
    complex_dir.mkdir(parents=True,exist_ok=True)
    for old_panel in complex_dir.glob("*.png"):
        old_panel.unlink()  # keep only this run's panels

    if complex_tables and "struct_model" not in globals():
        struct_processor,struct_model=load_structure_model()
    for item in complex_tables:
        page,page_box=make_page(item)
        crop,crop_box=padded_crop(page,page_box)
        gold_spans=[x for x in translate_gold_objects(item,crop_box) if x["label"]=="table spanning cell"]
        raw=recognize_structure(crop)
        grid=grid_from_structure(raw)
        pred_spans=[x for x in raw if x["label"]=="table spanning cell"]
        left=outline(crop,gold_spans,{"table spanning cell":"orange"})
        right=outline(crop,grid["rows"]+grid["columns"]+pred_spans,PROBE_COLORS)
        panel=Image.new("RGB",(left.width+right.width+20,left.height+60),"white")
        panel.paste(left,(0,40)); panel.paste(right,(left.width+20,40))
        d=ImageDraw.Draw(panel)
        d.text((5,8),"gold spanning cells (orange)",fill="black")
        d.text((left.width+25,8),"predicted rows / columns / spanning cells",fill="black")
        panel.save(complex_dir/f"{item['source_id'].replace('/','_')}.png")
        print({
            "table":item["source_id"],
            "gold_grid":[item["n_rows"],item["n_cols"]],"predicted_grid":[len(grid["rows"]),len(grid["columns"])],
            "gold_spanning_cells":len(gold_spans),"predicted_spanning_cells":len(pred_spans),
        })
    if complex_tables:
        print("These tables are intentionally excluded from the simple rectangular QA reconstruction contract.")

    # The structure recognizer is no longer needed: unload it before TAPAS (sequential loading, spec §67).
    if "struct_model" in globals():
        del struct_model,struct_processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
```

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{'table': '1604.06285v1.2', 'gold_grid': [14, 2], 'predicted_grid': [13, 2], 'gold_spanning_cells': 3, 'predicted_spanning_cells': 0}


{'table': '1604.06285v1.4', 'gold_grid': [5, 5], 'predicted_grid': [5, 5], 'gold_spanning_cells': 2, 'predicted_spanning_cells': 2}


{'table': '1605.06770v1.2', 'gold_grid': [5, 4], 'predicted_grid': [3, 4], 'gold_spanning_cells': 2, 'predicted_spanning_cells': 1}


These tables are intentionally excluded from the simple rectangular QA reconstruction contract.


## 20. Generate 50 table-QA questions from the gold logical tables

> **Evaluation practice**

Each table contributes:

- 2 lookup (`NONE`) questions;
- 1 `SUM`;
- 1 `AVERAGE`;
- 1 `COUNT`.

Generation uses only the gold logical table and happens independently of model predictions.

In [ ]:
show_source("gold_table_dict", "numeric_columns", "stage_questions")
run_stage("questions")

```python
def gold_table_dict(item):
    headers=item["grid"][0]
    body=item["grid"][1:]
    return {headers[c]:[row[c] for row in body] for c in range(len(headers))}

def numeric_columns(item):
    out=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in item["grid"][1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            out.append((c,vals))
    return out

def stage_questions(args):
    global qa_records
    qa_records=[]
    for item in canonical_tables:
        grid=item["grid"]; headers=grid[0]; body=grid[1:]
        row_ids=[row[0] for row in body]
        candidates=[]
        for r in range(len(body)):
            for c in range(1,len(headers)):
                candidates.append((r,c))
        candidates.sort(key=lambda rc:hashlib.sha256(f"{item['source_id']}:{rc[0]}:{rc[1]}".encode()).hexdigest())
        for j,(r,c) in enumerate(candidates[:2]):
            qa_records.append({
                "id":f"{item['source_id']}-lookup-{j}",
                "table_id":item["source_id"],
                "question":f"What is the {headers[c]} for {row_ids[r]}?",
                "category":"NONE",
                "gold_aggregation":"NONE",
                "gold_coordinates":[[r,c]],
                "gold_denotation":[body[r][c]],
            })
        c,vals=numeric_columns(item)[0]
        coords=[[r,c] for r in range(len(body))]
        qa_records += [
            {
                "id":f"{item['source_id']}-sum","table_id":item["source_id"],
                "question":f"What is the total {headers[c]}?","category":"SUM",
                "gold_aggregation":"SUM","gold_coordinates":coords,
                "gold_denotation":float(sum(vals)),
            },
            {
                "id":f"{item['source_id']}-avg","table_id":item["source_id"],
                "question":f"What is the average {headers[c]}?","category":"AVERAGE",
                "gold_aggregation":"AVERAGE","gold_coordinates":coords,
                "gold_denotation":float(sum(vals)/len(vals)),
            },
            {
                "id":f"{item['source_id']}-count","table_id":item["source_id"],
                "question":f"How many {headers[c]} entries are listed?","category":"COUNT",
                "gold_aggregation":"COUNT","gold_coordinates":coords,
                "gold_denotation":float(len(vals)),
            },
        ]

    if len(qa_records)!=50:
        raise RuntimeError(f"Expected 50 questions, got {len(qa_records)}")
    counts=defaultdict(int)
    for r in qa_records:counts[r["category"]]+=1
    if dict(counts)!={"NONE":20,"SUM":10,"AVERAGE":10,"COUNT":10}:
        raise RuntimeError(f"Question mix mismatch: {dict(counts)}")
    print(dict(counts))
```

{'NONE': 20, 'SUM': 10, 'AVERAGE': 10, 'COUNT': 10}


## 21. Stage 3 — TAPAS

> **Core concept**

TAPAS receives a string table and a question. It selects cells at threshold 0.5 and predicts one aggregation operator.

For `SUM`, `AVERAGE` and `COUNT`, the notebook computes the numeric answer from selected cells exactly as the DIMER reference carrier does.

**Question tested:** how much QA accuracy is lost when TAPAS reads a reconstructed table instead of the gold one?

**Predict:** rank the three paths (gold table, structure-only, end-to-end) by denotation accuracy, and estimate the gap between the first and the last.

In [ ]:
# TAPAS is loaded by the QA stage in the next cell, in the same process that answers the questions.
show_source("load_tapas", "compute_numeric", "table_to_df", "tapas_answer")

```python
def load_tapas():
    global tapas_tokenizer,tapas_model,tapas_load_seconds,tapas_params
    tapas_tokenizer=TapasTokenizer.from_pretrained(str(TAPAS_DIR),local_files_only=True)
    t0=time.perf_counter()
    tapas_model=TapasForQuestionAnswering.from_pretrained(
        str(TAPAS_DIR),local_files_only=True,dtype=torch.float32
    ).to(DEVICE).eval()
    tapas_load_seconds=time.perf_counter()-t0
    for p in tapas_model.parameters():p.requires_grad_(False)
    tapas_params=sum(p.numel() for p in tapas_model.parameters())

def compute_numeric(cells,aggregation):
    vals=[parse_number(x) for x in cells]
    if aggregation=="COUNT":return float(len(cells))
    if any(v is None for v in vals):return None
    if aggregation=="SUM":return float(sum(vals))
    if aggregation=="AVERAGE":return float(sum(vals)/len(vals)) if vals else None
    return None

def table_to_df(table):
    headers=table[0]
    body=table[1:]
    return pd.DataFrame(body,columns=headers,dtype=object)

def tapas_answer(table,question):
    df=table_to_df(table)
    inputs=tapas_tokenizer(
        table=df,
        queries=[question],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    model_inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.inference_mode():
        out=tapas_model(**model_inputs)
    coords,agg=tapas_tokenizer.convert_logits_to_predictions(
        inputs,
        out.logits.detach().cpu(),
        out.logits_aggregation.detach().cpu(),
    )
    selected=[list(x) for x in coords[0]]
    aggregation=AGGREGATIONS[int(agg[0])]
    cells=[str(df.iat[r,c]) for r,c in selected]
    numeric=compute_numeric(cells,aggregation)
    answer=(f"{aggregation} > " if aggregation!="NONE" else "")+", ".join(cells)
    return {
        "coordinates":selected,"cells":cells,"aggregation":aggregation,
        "numeric_answer":numeric,"answer":answer,
    }
```

## 22. QA scoring

> **Evaluation practice**

Denotation correctness is the headline downstream metric:

- lookup: same multiset of normalized selected cell strings;
- numeric aggregation: numeric answer matches within `1e-6`.

Gold-table evaluation also reports aggregation and exact selected-coordinate accuracy.

In [ ]:
show_source("norm_cell", "denotation_correct", "score_path", "gold_provider", "struct_provider", "full_provider", "qa_summary", "stage_qa")
run_stage("qa")

```python
def norm_cell(s):
    return " ".join(str(s).lower().split())

def denotation_correct(result,record):
    gold=record["gold_denotation"]
    if isinstance(gold,list):
        return sorted(norm_cell(x) for x in result["cells"])==sorted(norm_cell(x) for x in gold)
    value=result["numeric_answer"]
    return value is not None and abs(float(value)-float(gold))<=1e-6

def score_path(path_name,table_provider):
    rows=[]
    times=[]
    for record in qa_records:
        item=next(x for x in canonical_tables if x["source_id"]==record["table_id"])
        table,status=table_provider(item)
        if table is None:
            rows.append({
                "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
                "question":record["question"],"gold_aggregation":record["gold_aggregation"],
                "predicted_aggregation":None,"gold_denotation":record["gold_denotation"],
                "predicted_denotation":None,"denotation_correct":False,
                "aggregation_correct":False,"selected_cells":[],"coordinates":[],
                "pipeline_status":status,"latency_seconds":None,
            })
            continue
        t=time.perf_counter()
        result=tapas_answer(table,record["question"])
        times.append(time.perf_counter()-t)
        rows.append({
            "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
            "question":record["question"],"gold_aggregation":record["gold_aggregation"],
            "predicted_aggregation":result["aggregation"],"gold_denotation":record["gold_denotation"],
            "predicted_denotation":result["numeric_answer"] if record["gold_aggregation"]!="NONE" else result["cells"],
            "denotation_correct":denotation_correct(result,record),
            "aggregation_correct":result["aggregation"]==record["gold_aggregation"],
            "selected_cells":result["cells"],"coordinates":result["coordinates"],
            "pipeline_status":"ok","latency_seconds":times[-1],
        })
    return rows,times

def gold_provider(item):
    return item["grid"],"ok"

def struct_provider(item):
    r=item["gold_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

def full_provider(item):
    if item["selected_detection"] is None:return None,"detection_miss"
    r=item["det_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

def qa_summary(rows):
    return {
        "n":len(rows),
        "correct":int(sum(r["denotation_correct"] for r in rows)),
        "denotation_accuracy":float(np.mean([r["denotation_correct"] for r in rows])),
        "aggregation_accuracy":float(np.mean([r["aggregation_correct"] for r in rows])),
        "valid_questions":sum(r["pipeline_status"]=="ok" for r in rows),
    }

def stage_qa(args):
    global gold_qa,gold_tapas_times,structure_qa,structure_tapas_times,end_to_end_qa,end_tapas_times,tapas_peak
    load_transformers()
    load_pandas()
    load_tapas()
    print({"parameters":tapas_params,"load_seconds":tapas_load_seconds})
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()  # TAPAS peak only, not the earlier stages
    gold_qa,gold_tapas_times=score_path("gold_table",gold_provider)
    structure_qa,structure_tapas_times=score_path("structure_only",struct_provider)
    end_to_end_qa,end_tapas_times=score_path("end_to_end",full_provider)
    tapas_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

    print("Gold:",qa_summary(gold_qa))
    print("Structure-only:",qa_summary(structure_qa))
    print("End-to-end:",qa_summary(end_to_end_qa))
```

{'parameters': 336734214, 'load_seconds': 0.8247394160000567}


/content/work/env/lib/python3.12/site-packages/transformers/models/tapas/tokenization_tapas.py:2700: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


  text = normalize_for_match(row[col_index].text)


/content/work/env/lib/python3.12/site-packages/transformers/models/tapas/tokenization_tapas.py:1494: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


...

/content/work/env/lib/python3.12/site-packages/transformers/models/tapas/tokenization_tapas.py:1494: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


  cell = row[col_index]


Gold: {'n': 50, 'correct': 40, 'denotation_accuracy': 0.8, 'aggregation_accuracy': 0.62, 'valid_questions': 50}


Structure-only: {'n': 50, 'correct': 40, 'denotation_accuracy': 0.8, 'aggregation_accuracy': 0.62, 'valid_questions': 50}


End-to-end: {'n': 50, 'correct': 30, 'denotation_accuracy': 0.6, 'aggregation_accuracy': 0.5, 'valid_questions': 40}


### What to notice — QA

- The gold-table path is TAPAS's own ceiling on these questions. Its errors are QA-model errors (wrong cells or wrong aggregation), not pipeline errors.
- The structure-only path matches the gold path only when every gold-crop reconstruction is valid and puts each cell's text in the right place; any drop below the gold path was introduced by structure recognition or reconstruction.
- The end-to-end path additionally pays for detection misses and detected-crop structure errors. Questions whose table could not be reconstructed count as wrong (`pipeline_status` records why), so all three paths share the same 50-question denominator.
- A lookup answer counts as correct when the selected cells are right, even if TAPAS also attached an aggregation operator, so aggregation accuracy can sit below denotation accuracy.

## 23. Pipeline waterfall

> **Evaluation practice**

The waterfall counts failures rather than dropping them from downstream denominators.

In [ ]:
show_source("stage_waterfall")
run_stage("waterfall")

```python
def stage_waterfall(args):
    global waterfall
    waterfall={
        "tables":len(canonical_tables),
        "unit":"detection, structure and reconstruction values count tables out of `tables`; qa gives accuracy and correct-answer counts",
        "detection":{
            "hit50":sum(r["hit50"] for r in det_rows),
            "hit75":sum(r["hit75"] for r in det_rows),
        },
        "structure_gold_crop":{
            "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="gold_crop"),
        },
        "structure_detected_crop":{
            "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="detected_crop"),
        },
        "reconstruction":{
            "valid_gold_crop":sum(x["gold_recon"]["valid"] for x in canonical_tables),
            "valid_detected_crop":sum(x["det_recon"]["valid"] for x in canonical_tables),
        },
        "qa":{
            "questions":len(qa_records),
            "gold_table_correct":qa_summary(gold_qa)["correct"],
            "structure_only_correct":qa_summary(structure_qa)["correct"],
            "end_to_end_correct":qa_summary(end_to_end_qa)["correct"],
            "gold_table_denotation_accuracy":qa_summary(gold_qa)["denotation_accuracy"],
            "structure_only_denotation_accuracy":qa_summary(structure_qa)["denotation_accuracy"],
            "end_to_end_denotation_accuracy":qa_summary(end_to_end_qa)["denotation_accuracy"],
        },
    }
    print(json.dumps(waterfall,indent=2))
```

{


  "tables": 10,


  "unit": "detection, structure and reconstruction values count tables out of `tables`; qa gives accuracy and correct-answer counts",


  "detection": {


...

    "gold_table_denotation_accuracy": 0.8,


    "structure_only_denotation_accuracy": 0.8,


    "end_to_end_denotation_accuracy": 0.6


  }


}


### What to notice — waterfall

Read the waterfall top to bottom. Detection, structure and reconstruction are counts of tables (out of 10); the QA block gives each path's accuracy and its number of correct answers (out of 50, five per table). Along the end-to-end path, the counts generally shrink from detection hits, to exact detected-crop grids, to valid detected-crop tables, to correct answers. The step with the largest drop is where most of the end-to-end loss entered. Compare each gold-crop count with its detected-crop count to separate structure errors from errors caused by the crop.

## 24. Checkpoint — trace the losses

> **Evaluation practice**

Return to the predictions you wrote down in §2. The next cell prints, for each canonical table, the detection IoU, whether each path produced a valid table, the predicted grid shapes, and the cell-assignment accuracy on each crop. Compare them with the waterfall above.

1. Which stage reduced end-to-end accuracy the most in this run?
2. Can perfect table detection guarantee correct structure?
3. Can perfect structure guarantee correct QA?
4. Why does TAPAS need strings rather than row/column boxes?
5. Where would OCR enter a production version of this workflow?
6. What happens if one predicted row is missing?
7. What happens if the first reconstructed row is mistaken?

<details>
<summary>Sample answer</summary>

1. Look for the largest step in the waterfall. When the gold-crop path is nearly perfect, the loss enters with detection (misses) and with detected-crop structure errors (a row or column gained or lost through the crop).
2. No. Structure recognition sees only the crop; margins, clipped rules and faint rows can still change the predicted grid.
3. No. The gold-table path is the ceiling: TAPAS still selects wrong cells or the wrong aggregation on some questions with a perfect table.
4. TAPAS is a language model over a table of strings: it embeds cell text together with row/column position ids. Boxes carry no words, so something must place text into the grid first.
5. Between structure recognition and reconstruction: OCR or PDF text extraction supplies the words and their boxes, which are then assigned to predicted cells. This notebook uses SciTSR annotation text in that position.
6. The texts of that row merge into a neighbouring row or become unmapped; the table changes meaning, and questions about the row fail.
7. The first row becomes the header. With a wrong or duplicate header the table is invalid for TAPAS here; with a plausible but wrong header, questions that name a column select the wrong one.

</details>

In [ ]:
show_source("stage_checkpoint")
run_stage("checkpoint")

```python
def stage_checkpoint(args):
    for item in canonical_tables:
        print({
            "table":item["source_id"],
            "detection_iou":next(r["best_iou"] for r in det_rows if r["table_id"]==item["source_id"]),
            "gold_crop_valid":item["gold_recon"]["valid"],
            "detected_crop_valid":item["det_recon"]["valid"],
            "gold_shape":[item["n_rows"],item["n_cols"]],
            "gold_crop_pred_shape":[len(item["gold_grid_pred"]["rows"]),len(item["gold_grid_pred"]["columns"])],
            "det_crop_pred_shape":[len(item["det_grid_pred"]["rows"]),len(item["det_grid_pred"]["columns"])],
            "cell_accuracy_gold_crop":item["gold_recon"].get("assignment_accuracy"),
            "cell_accuracy_detected_crop":item["det_recon"].get("assignment_accuracy"),
        })
```

{'table': '1702.02925v1.7', 'detection_iou': 0.8917366626625095, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [10, 7], 'gold_crop_pred_shape': [10, 7], 'det_crop_pred_shape': [10, 7], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


{'table': '1605.06770v1.1', 'detection_iou': 0.8817291072271487, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [8, 2], 'gold_crop_pred_shape': [8, 2], 'det_crop_pred_shape': [8, 2], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


{'table': '1606.09371v1.5', 'detection_iou': 0.8330993914356506, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [5, 4], 'gold_crop_pred_shape': [5, 4], 'det_crop_pred_shape': [5, 4], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


{'table': '1407.3745v1.1', 'detection_iou': 0.8346099613787515, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [4, 4], 'gold_crop_pred_shape': [4, 4], 'det_crop_pred_shape': [4, 4], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


...

{'table': '1108.4723v1.4', 'detection_iou': 0.0, 'gold_crop_valid': True, 'detected_crop_valid': False, 'gold_shape': [3, 4], 'gold_crop_pred_shape': [3, 4], 'det_crop_pred_shape': [0, 0], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': None}


{'table': '1611.09238v1.1', 'detection_iou': 0.8335369509231655, 'gold_crop_valid': True, 'detected_crop_valid': False, 'gold_shape': [4, 5], 'gold_crop_pred_shape': [4, 5], 'det_crop_pred_shape': [4, 6], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 0.8}


{'table': '1108.4723v1.3', 'detection_iou': 0.8243800958499397, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [4, 4], 'gold_crop_pred_shape': [4, 4], 'det_crop_pred_shape': [4, 4], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


{'table': '1702.02925v1.6', 'detection_iou': 0.8904508512756362, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [10, 7], 'gold_crop_pred_shape': [10, 7], 'det_crop_pred_shape': [10, 7], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


{'table': '1606.09371v1.2', 'detection_iou': 0.8777737838582285, 'gold_crop_valid': True, 'detected_crop_valid': True, 'gold_shape': [7, 4], 'gold_crop_pred_shape': [7, 4], 'det_crop_pred_shape': [7, 4], 'cell_accuracy_gold_crop': 1.0, 'cell_accuracy_detected_crop': 1.0}


## 25. Resource comparison

> **Evaluation practice**

Models are loaded sequentially. The table records model size, load time, stage latency and peak accelerator memory where available.

Latency units differ by stage, so compare them with care: detection is timed per page, structure per crop and TAPAS per question (`latency_unit`). Peak memory is reset before each stage is measured. These are measurements from this runtime only (the printed `device` and `gpu_name`).

In [ ]:
show_source("median", "stage_resources")
run_stage("resources")

```python
def median(x):return float(np.median(x)) if x else None

def stage_resources(args):
    global resource_rows
    resource_rows=[
        {
            "stage":"detection","model":"Table Transformer Detection",
            "model_id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"],
            "parameter_count":det_params,
            "weight_bytes":next(x["bytes"] for x in DETECTION_MANIFEST["files"] if x["path"]=="model.safetensors"),
            "load_seconds":det_load_seconds,"latency_unit":"per page","mean_latency_seconds":float(np.mean(det_times)),
            "median_latency_seconds":median(det_times),"peak_gpu_memory_bytes":det_peak,
        },
        {
            "stage":"structure","model":"Table Transformer Structure",
            "model_id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"],
            "parameter_count":struct_params,
            "weight_bytes":next(x["bytes"] for x in STRUCTURE_MANIFEST["files"] if x["path"]=="model.safetensors"),
            "load_seconds":struct_load_seconds,"latency_unit":"per crop","mean_latency_seconds":float(np.mean(struct_times)),
            "median_latency_seconds":median(struct_times),"peak_gpu_memory_bytes":struct_peak,
        },
        {
            "stage":"qa","model":"TAPAS Large WTQ",
            "model_id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"],
            "parameter_count":tapas_params,
            "weight_bytes":next(x["bytes"] for x in TAPAS_MANIFEST["files"] if x["path"]=="model.safetensors"),
            "load_seconds":tapas_load_seconds,"latency_unit":"per question",
            "mean_latency_seconds":float(np.mean(gold_tapas_times+structure_tapas_times+end_tapas_times)),
            "median_latency_seconds":median(gold_tapas_times+structure_tapas_times+end_tapas_times),
            "peak_gpu_memory_bytes":tapas_peak,
        },
    ]
    for row in resource_rows:print(row)
```

{'stage': 'detection', 'model': 'Table Transformer Detection', 'model_id': 'microsoft/table-transformer-detection', 'revision': '2357cbe2b5a5d1c03e54f32764f06058933b65ab', 'parameter_count': 28799431, 'weight_bytes': 115317516, 'load_seconds': 0.43605294000008143, 'latency_unit': 'per page', 'mean_latency_seconds': 0.05740664119999792, 'median_latency_seconds': 0.055463757500035626, 'peak_gpu_memory_bytes': 235753472}


{'stage': 'structure', 'model': 'Table Transformer Structure', 'model_id': 'microsoft/table-transformer-structure-recognition-v1.1-all', 'revision': '7587a7ef111d9dcbf8ac695f1376ab7014340a0c', 'parameter_count': 28828619, 'weight_bytes': 115437156, 'load_seconds': 0.4077689239999245, 'latency_unit': 'per crop', 'mean_latency_seconds': 0.03611019899999759, 'median_latency_seconds': 0.03304995799999233, 'peak_gpu_memory_bytes': 216211968}


{'stage': 'qa', 'model': 'TAPAS Large WTQ', 'model_id': 'google/tapas-large-finetuned-wtq', 'revision': 'f58317ab2577d17647d9acafa790c744a0388b30', 'parameter_count': 336734214, 'weight_bytes': 1346985282, 'load_seconds': 0.8247394160000567, 'latency_unit': 'per question', 'mean_latency_seconds': 0.12951616581428554, 'median_latency_seconds': 0.12107391349991303, 'peak_gpu_memory_bytes': 1404542464}


## 26. Qualitative four-stage panels

> **Evaluation practice**

Each canonical table gets a panel in `examples/` showing:

1. synthetic document page + predicted table box;
2. detected crop + row/column structure;
3. reconstructed table text;
4. one representative TAPAS result.

Visualizations supplement machine-readable outputs.

In [ ]:
show_source("draw_boxes", "stage_panels")
run_stage("panels")

```python
def draw_boxes(image,items):
    out=image.convert("RGB").copy()
    d=ImageDraw.Draw(out)
    for item in items:
        # PIL draws no outline unless a colour is given.
        d.rectangle(item["box"],outline=BOX_COLORS.get(item["label"],"orange"),width=2)
        if item["label"]=="table":
            d.text((item["box"][0]+4,item["box"][1]-14),f"table {item.get('score',0):.2f}",fill="red")
    return out

def stage_panels(args):
    examples_dir=Path(OUTPUT_DIR)/"examples"
    examples_dir.mkdir(parents=True,exist_ok=True)

    for item in canonical_tables:
        page=draw_boxes(item["page"],item["detections"][:3])
        crop=item["det_crop"] if item["det_crop"] is not None else item["gold_crop"]
        struct=item["det_structure_raw"] if item["det_crop"] is not None else item["gold_structure_raw"]
        crop_vis=draw_boxes(crop,[x for x in struct if x["label"] in ("table row","table column","table spanning cell")])
        page.thumbnail((700,700)); crop_vis.thumbnail((700,700))
        recon=item["det_recon"] if item["det_crop"] is not None else item["gold_recon"]
        recon_lines=[" | ".join(row) for row in (recon["table"] or [])[:3]] or [f"(no table: {recon['reason']})"]
        canvas=Image.new("RGB",(max(page.width,crop_vis.width),page.height+crop_vis.height+120+30*len(recon_lines)),"white")
        canvas.paste(page,(0,0)); canvas.paste(crop_vis,(0,page.height))
        q=next(r for r in end_to_end_qa if r["table_id"]==item["source_id"])
        d=ImageDraw.Draw(canvas)
        y=page.height+crop_vis.height+8
        for line in recon_lines:
            d.text((5,y),f"Reconstructed: {line[:110]}",fill="black"); y+=30
        d.text((5,y),f"Q: {q['question']}",fill="black")
        d.text((5,y+30),f"Gold: {q['gold_denotation']}",fill="black")
        d.text((5,y+60),f"Path status: {q['pipeline_status']}  TAPAS: {q['predicted_denotation']}  aggregation: {q['predicted_aggregation']}  cells: {q['selected_cells'][:4]}",fill="black")
        path=examples_dir/f"{item['source_id'].replace('/','_')}.png"
        canvas.save(path)
        print(path)
```

outputs/table_intelligence/examples/1702.02925v1.7.png


outputs/table_intelligence/examples/1605.06770v1.1.png


outputs/table_intelligence/examples/1606.09371v1.5.png


outputs/table_intelligence/examples/1407.3745v1.1.png


...

outputs/table_intelligence/examples/1108.4723v1.4.png


outputs/table_intelligence/examples/1611.09238v1.1.png


outputs/table_intelligence/examples/1108.4723v1.3.png


outputs/table_intelligence/examples/1702.02925v1.6.png


outputs/table_intelligence/examples/1606.09371v1.2.png


## 27. Machine-readable exports

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Outputs include stage metrics, reconstructed tables, QA records, waterfall, resource metrics and provenance.

In [ ]:
# Write the CSV/JSON exports, per-table JSON and provenance under OUTPUT_DIR (stage_exports in the carried file).
run_stage("exports")

Outputs:


  outputs/table_intelligence/complex_probe


  outputs/table_intelligence/examples


  outputs/table_intelligence/input_manifest.json


  outputs/table_intelligence/provenance.json


...

  outputs/table_intelligence/resource_metrics.csv


  outputs/table_intelligence/structure_metrics.csv


  outputs/table_intelligence/table_detection.csv


  outputs/table_intelligence/tables


  outputs/table_intelligence/waterfall.json


## 28. BYOD (optional)

> **Optional** — skipped on the canonical `Run all` path (`USE_BYOD = False`). To use it, set `USE_BYOD = True` and `BYOD_PATH` in §3, then choose `Run all` again; the sample path runs first.

A production table-intelligence workflow needs text to populate predicted cells. This notebook does not run OCR itself, so BYOD takes page images plus OCR/text boxes you produced elsewhere, and optional questions.

Folder layout:

```text
dataset/
  pages/
    page001.png        # .png .jpg .jpeg .webp .tif .tiff .bmp; each side 16..4096 px
  ocr.csv
  questions.csv        # optional
```

`ocr.csv`, one row per word or text box:

```text
page_id,text,x0,y0,x1,y1,order
page001,Method,130.0,172.5,196.0,190.0,1
```

- `page_id` is the image file name without its extension (`page001` for `pages/page001.png`).
- `x0,y0,x1,y1` are **page-image pixels with the origin at the top-left corner**, as the image is stored (no rescaling). Convert PDF points or a bottom-left origin first: a box outside the image is rejected with the page and lines named.
- `order` is an integer reading order; words that fall in the same cell are joined in this order.

`questions.csv` (optional) has `id,page_id,question` and an optional `accepted_answer`. A numeric accepted answer is compared numerically (within `1e-6`); anything else is compared as normalized text.

What happens: each page goes through the same detection, crop, structure, grid-reconstruction and TAPAS steps as the sample path, and OCR words are assigned to predicted cells by word-centre containment. Every row of both CSV files is validated before a model runs, and an error names the file, line and field. A page without OCR rows, or with an unsupported file type, is listed as skipped with its reason. The cell prints each reconstructed table and answer, and writes `outputs/table_intelligence/byod/byod_results.json` plus one CSV per reconstructed table under `byod/tables/`; earlier BYOD outputs are removed first.

### Privacy

Your files stay inside this notebook runtime: this branch sends nothing to an external service and writes only under `outputs/table_intelligence/byod/`. Tables may contain financial, personal, employee, medical or proprietary information. Do not upload restricted or regulated documents to a hosted runtime unless you are authorized to process them there.

In [ ]:
# Optional BYOD: runs only when USE_BYOD is True (§3). Every CSV row is validated before any model loads.
show_source("accepted_answer_matches", "read_byod_csv", "stage_byod")
run_stage("byod")

```python
def accepted_answer_matches(result,accepted):
    """Numbers compare within 1e-6 when both sides parse as numbers; anything else compares as normalized text."""
    predicted=result["numeric_answer"] if result["aggregation"]!="NONE" else ", ".join(result["cells"])
    if predicted is None:
        return False
    p,a=parse_number(predicted),parse_number(accepted)
    if p is not None and a is not None:
        return abs(p-a)<=1e-6
    return norm_cell(predicted)==norm_cell(accepted)

def read_byod_csv(path,required):
    """Rows with their file line numbers (line 1 is the header); names any missing column."""
    with open(path,encoding="utf-8-sig",newline="") as f:
        reader=csv.DictReader(f)
        lacking=sorted(required-set(reader.fieldnames or []))
        if lacking:
            raise ValueError(f"{path.name}: missing column(s) {lacking}; required columns are {sorted(required)}")
        return list(enumerate(reader,start=2))

def stage_byod(args):
    global det_processor,det_model,struct_processor,struct_model,byod_result
    load_transformers()
    load_pandas()
    byod_result=None
    if USE_BYOD:
        if not str(BYOD_PATH).strip():
            raise ValueError("USE_BYOD is True but BYOD_PATH is empty: set BYOD_PATH in §3 to the folder that holds pages/ and ocr.csv")
        root=Path(BYOD_PATH)
        pages_dir=root/"pages"
        ocr_path=root/"ocr.csv"
        questions_path=root/"questions.csv"
        absent=[name for name,ok in (("pages/",pages_dir.is_dir()),("ocr.csv",ocr_path.is_file())) if not ok]
        if absent:
            raise FileNotFoundError(f"BYOD_PATH {str(root)!r} is missing {absent}; use the layout shown in §28")

        # Validate every row before any model runs.
        ocr_by_page=defaultdict(list)
        for line,row in read_byod_csv(ocr_path,{"page_id","text","x0","y0","x1","y1","order"}):
            page_id=str(row["page_id"] or "").strip()
            if not page_id:
                raise ValueError(f"ocr.csv line {line}: page_id is empty")
            try:
                box=[float(row[k]) for k in ("x0","y0","x1","y1")]
            except (TypeError,ValueError):
                raise ValueError(
                    f"ocr.csv line {line}: x0,y0,x1,y1 must be numbers in page-image pixels, got "
                    f"{[row[k] for k in ('x0','y0','x1','y1')]}"
                ) from None
            if not all(math.isfinite(v) for v in box) or box[0]>box[2] or box[1]>box[3]:
                raise ValueError(f"ocr.csv line {line}: box {box} needs finite values with x0<=x1 and y0<=y1 (top-left origin)")
            try:
                order=int(str(row["order"]).strip())
            except ValueError:
                raise ValueError(f"ocr.csv line {line}: order must be a whole number, got {row['order']!r}") from None
            ocr_by_page[page_id].append({"text":str(row["text"] or ""),"box":box,"order":order,"line":line})
        if not ocr_by_page:
            raise ValueError("ocr.csv has a header but no rows")
        for page_id in ocr_by_page:
            ocr_by_page[page_id].sort(key=lambda x:x["order"])

        q_by_page=defaultdict(list)
        if questions_path.is_file():
            for line,row in read_byod_csv(questions_path,{"id","page_id","question"}):
                if not str(row["question"] or "").strip() or not str(row["page_id"] or "").strip():
                    raise ValueError(f"questions.csv line {line}: page_id and question must be non-empty")
                q_by_page[str(row["page_id"]).strip()].append(row)

        byod_skipped=[]
        byod_inputs=[]
        for image_path in sorted(pages_dir.iterdir()):
            if not image_path.is_file():
                continue
            if image_path.suffix.lower() not in BYOD_IMAGE_SUFFIXES:
                byod_skipped.append({"file":image_path.name,"reason":"unsupported_extension"})
            elif image_path.stem not in ocr_by_page:
                byod_skipped.append({"file":image_path.name,"reason":"no_ocr_rows"})
            else:
                byod_inputs.append(image_path)
        ocr_without_image=sorted(set(ocr_by_page)-{p.stem for p in byod_inputs})
        if not byod_inputs:
            raise ValueError(
                f"No page in {pages_dir} has OCR rows. Skipped: {byod_skipped}. "
                f"ocr.csv page_ids: {sorted(ocr_by_page)[:10]}. A page_id must equal its image file name without the extension."
            )

        byod_images={}
        for image_path in byod_inputs:
            image=Image.open(image_path); image.load(); image=image.convert("RGB")
            if min(image.size)<16 or max(image.size)>4096:
                raise ValueError(f"{image_path.name}: page is {image.width}x{image.height}; each side must be 16..4096 px")
            outside=[w["line"] for w in ocr_by_page[image_path.stem]
                     if w["box"][0]<-1 or w["box"][1]<-1 or w["box"][2]>image.width+1 or w["box"][3]>image.height+1]
            if outside:
                raise ValueError(
                    f"ocr.csv lines {outside[:5]}: boxes for {image_path.stem} extend beyond its {image.width}x{image.height} image. "
                    "Boxes must be page-image pixels with a top-left origin; convert PDF points or a bottom-left origin first"
                )
            byod_images[image_path.stem]=image

        # Detection, then structure: loaded one at a time with the same loaders as the sample path.
        det_processor,det_model=load_detection_model()
        byod_pages=[]
        for page_id,image in byod_images.items():
            detections=detect_table(image)
            byod_pages.append({"page_id":page_id,"image":image,"detection":detections[0] if detections else None})
        del det_model,det_processor
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()

        struct_processor,struct_model=load_structure_model()
        if any(q_by_page.values()) and "tapas_model" not in globals():
            load_tapas()
        byod_outputs=[]
        for page in byod_pages:
            record={"page_id":page["page_id"],"detection":page["detection"],"table":None,"answers":[],
                    "words":len(ocr_by_page[page["page_id"]]),"words_assigned":0}
            if page["detection"] is None:
                byod_outputs.append({**record,"status":"detection_miss"})
                continue
            crop,crop_box=padded_crop(page["image"],page["detection"]["box"])
            grid=grid_from_structure(recognize_structure(crop))
            if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
                byod_outputs.append({**record,"status":"grid_invalid"})
                continue

            # Assign OCR word centers into predicted row/column cells (crop frame = page frame - crop origin).
            strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
            cx0,cy0,_,_=crop_box
            for word in ocr_by_page[page["page_id"]]:
                x=(word["box"][0]+word["box"][2])/2-cx0
                y=(word["box"][1]+word["box"][3])/2-cy0
                ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
                cis=[i for i,c in enumerate(grid["columns"]) if contains(c["box"],x,y)]
                if len(ris)==1 and len(cis)==1:
                    strings[ris[0]][cis[0]].append((word["order"],word["text"]))
                    record["words_assigned"]+=1
            table=[[" ".join(t for _o,t in sorted(cell)).strip() for cell in row] for row in strings]
            record["table"]=table
            headers=[norm_text(x) for x in table[0]] if table else []
            valid=bool(table and all(headers) and len(set(headers))==len(headers) and len(table)<=64 and len(table[0])<=32)
            if not valid:
                byod_outputs.append({**record,"status":"header_or_limit_invalid"})
                continue
            for q in q_by_page.get(page["page_id"],[]):
                result=tapas_answer(table,q["question"])
                row={"id":q["id"],"question":q["question"],"answer":result["answer"],"aggregation":result["aggregation"],
                     "cells":result["cells"],"numeric_answer":result["numeric_answer"]}
                accepted=(q.get("accepted_answer") or "").strip()
                if accepted:
                    row["accepted_answer"]=accepted
                    row["accepted_answer_match"]=accepted_answer_matches(result,accepted)
                record["answers"].append(row)
            byod_outputs.append({**record,"status":"ok"})
        del struct_model,struct_processor
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()

        # Show and export this run's BYOD results; earlier BYOD outputs are removed first.
        byod_dir=Path(OUTPUT_DIR)/"byod"
        if byod_dir.exists():
            shutil.rmtree(byod_dir)
        (byod_dir/"tables").mkdir(parents=True)
        for out in byod_outputs:
            print(f"\n[{out['page_id']}] status={out['status']}  OCR words assigned to cells: {out['words_assigned']}/{out['words']}")
            if out["table"] is not None:
                with open(byod_dir/"tables"/f"{out['page_id']}.csv","w",encoding="utf-8",newline="") as f:
                    csv.writer(f).writerows(out["table"])
                header=out["table"][0]
                columns=header if len(set(header))==len(header) else [f"{i}:{h}" for i,h in enumerate(header)]
                print(pd.DataFrame(out["table"][1:],columns=columns).to_string(index=False))
            for a in out["answers"]:
                verdict=f"  (accepted {a['accepted_answer']!r}: {'match' if a['accepted_answer_match'] else 'no match'})" if "accepted_answer" in a else ""
                print(f"  Q {a['id']}: {a['question']}\n    -> {a['answer']}{verdict}")
        byod_result={
            "byod_path":str(root),"pages":len(byod_outputs),"skipped":byod_skipped,
            "ocr_page_ids_without_image":ocr_without_image,
            "thresholds":{"detection":DETECTION_THRESHOLD,"structure":STRUCTURE_THRESHOLD,"grid_nms_iou":GRID_NMS_IOU},
            "outputs":byod_outputs,
        }
        (byod_dir/"byod_results.json").write_text(json.dumps(byod_result,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
        print({"byod_pages":len(byod_outputs),"status_counts":dict(Counter(x["status"] for x in byod_outputs)),
               "skipped":byod_skipped,"outputs":str(byod_dir)})
    else:
        print("BYOD disabled on canonical Run all path.")
```

BYOD disabled on canonical Run all path.


## 29. Interpretation and limitations

### Table Intelligence is a pipeline

Detection, structure, text extraction, reconstruction and QA are separable failure points.

### Geometry is not text

Table Transformer predicts boxes. A real deployment still needs PDF text extraction, OCR or another document-extraction model.

### Gold-table QA is an upper-stage diagnostic

Strong TAPAS performance on the gold logical table does not imply strong end-to-end performance.

### Structure errors alter semantics

Missing or duplicated rows/columns can turn a correct source table into a different structured table before TAPAS sees it.

### Complex tables need richer reconstruction

Spanning cells, projected headers and nested structure require more than simple row × column intersection.

### Tutorial evidence is not a benchmark

This notebook uses a small, deterministic public-domain scientific-table sample and one seeded composition policy. Results are measurements from this notebook, not production-quality claims.

## Your conclusion

Complete this template from the outputs above, citing the printed number or exported file you used for each claim. Do not generalise beyond ten scientific tables, 50 generated questions and one run.

- **Task.** This notebook composed ___ → ___ → ___ to answer questions about tables on document pages.
- **Principal result.** End-to-end denotation accuracy was ___ on 50 questions, against ___ on the gold table (the reference) and ___ on the structure-only path.
- **Where the loss entered.** The largest drop occurred between ___ and ___ (evidence: ___ in `waterfall.json` or `reconstruction_metrics.csv`), mainly because ___.
- **Failure mode or uncertainty.** One important failure was ___. With ten tables, a single table moves a per-table rate by 10 percentage points and five questions.
- **Limitations.** Cell text came from annotations rather than OCR; each synthetic page held one table; only rectangular tables were used for QA; one paper-disjoint sample and one composition policy were measured.

<details>
<summary>Sample conclusion (illustrative; your numbers may differ)</summary>

On ten held-out SciTSR-PD tables, TAPAS answered 80% of 50 questions correctly from the gold tables and from gold-crop reconstructions, but 60% end to end. Every gold crop produced an exact grid, so structure recognition on a well-framed crop was not the bottleneck here. The loss entered at detection (one page had no detection above 0.90, which made its five questions unanswerable) and on one detected crop whose grid gained a column. These are measurements on ten tables with annotation text; they do not predict accuracy on scanned documents, OCR text, or tables with spanning cells.

</details>

## Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| Run is very slow; `device` prints `cpu` | no GPU runtime selected | select a T4 GPU and `Run all` again (CPU works, but TAPAS takes about a second per question) |
| `This notebook builds a Linux x86_64 environment` | a Windows or macOS kernel | use Google Colab, Kaggle or a Linux x86_64 Jupyter server |
| `uv wheel size/hash mismatch`, or a `uv pip install` error | package index unreachable or a truncated download | rerun the Runtime cell (§4); no restart is needed |
| `Installed versions differ from the hash lock` | packages in `work/env` were changed by hand | delete the `work/env` folder and rerun the Runtime cell (§4) |
| `<stage> failed with exit …` | an error inside the isolated environment | read the log tail printed above it (`work/logs/<stage>.log`) |
| `… is missing: run the earlier notebook cells first` or a `NameError` in a stage | a cell was run before the cells it depends on | run the notebook from the top (`Run all`) |
| model size or SHA-256 mismatch | incomplete or changed download | delete that model's folder under `weights/` and rerun; never bypass the check |
| SciTSR-PD shard size or SHA-256 mismatch | incomplete download | delete `weights/scitsr-pd` and rerun |
| `Embedded carrier geometry does not match its SHA-256`, or `Carried file integrity failure` | the carried stage file was edited | restore the notebook from the published copy; never bypass the check |
| `carrier tables failed to derive` or `source pixels differ` | a different dataset revision was read | keep the pinned `SCITSR_REVISION`; do not relax the check |
| split or eligibility `RuntimeError` | seed, derivation or eligibility rules were edited | restore the defaults; never borrow train/validation tables for the QA set |
| CUDA out of memory | another notebook shares the GPU | close the other notebook and rerun the failed cell; each stage process loads one model at a time and releases it when it exits |
| every reconstruction invalid | Configuration thresholds were changed | restore the defaults and experiment in the activity cell instead |
| BYOD: `BYOD_PATH is empty` or `is missing ['pages/', …]` | path not set, or wrong folder layout | set `BYOD_PATH` in §3 to the folder that holds `pages/` and `ocr.csv` (layout in §28) |
| BYOD: `ocr.csv line N: …` or `questions.csv line N: …` | that row breaks the file's contract | fix the named line and field |
| BYOD: `boxes … extend beyond its W×H image` | OCR boxes in PDF points, or with a bottom-left origin | convert them to page-image pixels with a top-left origin |
| BYOD: a page listed under `skipped` | no OCR rows for that `page_id`, or an unsupported file type | make the `page_id` equal the image file name without its extension; use a listed image type |

## Glossary

| Term | Meaning |
|---|---|
| **Table detection** | Finding the bounding box of each table on a page |
| **Structure recognition** | Finding the rows, columns, headers and spanning cells inside a table crop |
| **Spanning cell** | A cell that covers more than one row or column |
| **Crop / coordinate frame** | A sub-image; boxes must be shifted between page, crop and table coordinates |
| **IoU** | Intersection over union of two boxes: 1.0 is identical, 0 is disjoint |
| **AP50 / AP75** | Average precision counting a detection as correct at IoU ≥ 0.50 / ≥ 0.75 |
| **NMS** | Non-maximum suppression: drop a box that overlaps a higher-scoring box of the same label |
| **Grid reconstruction** | Intersecting predicted rows and columns to form cells |
| **Cell-text assignment** | Placing each text item into the predicted cell that contains its center |
| **TAPAS** | A BERT-style model that answers questions over a table of strings by selecting cells and an aggregation |
| **WTQ** | WikiTableQuestions, the QA dataset TAPAS Large WTQ was fine-tuned on |
| **Aggregation operator** | `NONE`, `SUM`, `AVERAGE` or `COUNT` applied to the selected cells |
| **Denotation** | The final answer value (cell strings or a number), compared with the gold answer |
| **Paper-disjoint split** | Tables from one paper never appear in two splits |
| **Carrier** | The DIMER repository whose pinned model and data this notebook reproduces |
| **Error waterfall** | Stage-by-stage counts showing where end-to-end failures entered |

## 30. Terminal summary

The final cell verifies required outputs and prints the stage waterfall without declaring a winner.

In [ ]:
run_stage("summary")

DIMER Table Intelligence Workshop


----------------------------------


Canonical tables: 10


Questions: 50


Stage 1 — Detection


...

  gold-table denotation: 0.800


  structure-only denotation: 0.800


  end-to-end denotation: 0.600


  correct answers of 50: gold 40, structure-only 40, end-to-end 30


Outputs: outputs/table_intelligence/
